# Kiểm tra cấu hình Kaggle offline cho notebook 1 và 3

Mở **phiên Kaggle mới**, chọn đúng GPU dự định dùng và **Internet Off**.
Gắn các Dataset **models, packages và PDF** hiện tại rồi **Run All**; không cần chạy
setup notebook 1/3 trước. Nếu chưa gắn Input, vẫn lấy được báo cáo kernel.

**Mặc định kiểm tra môi trường đầy đủ cho OCR và restoration**, gồm inventory mọi
distribution, dependency, ABI, CUDA/native ops nhỏ, API và runtime đã tồn tại.
Đọc metadata wheel và dependency closure của các runtime plan với kernel thật.
Ngân sách tối đa 900 giây và timeout riêng; nếu hết thời gian, log ghi `partial`.
Không cài package, build, đọc toàn bộ weights hoặc inference model.
DocEnTR/DE-GAN có inventory size/source/license và Keras backend Torch/HDF5 API.
Thiếu package trong kernel có thể đã được cung cấp bởi overlay offline.

Giữ `RESTORATION_GPU_CHECK='off'` để chỉ gửi log môi trường. Các lựa chọn
`'build_and_probe'` và `'unihir_smoke'` vẫn có nếu bạn chủ động bật phép kiểm dài
trên Kaggle; local luôn bỏ qua. Log/JSON giữ riêng metadata và inference,
kể cả khi thiếu Input hoặc bị ngắt. Không OCR hoặc chạy cả corpus.

Metadata, file trong Input, import, tensor GPU và inference model là các mức kiểm
chứng khác nhau. Thiếu package trong kernel có thể được giải quyết bằng runtime
riêng; báo cáo này không tự kết luận cả pipeline đã chạy được.


In [ ]:
from pathlib import Path
if not Path('/kaggle/input').is_dir() or not Path('/kaggle/working').is_dir():
    raise RuntimeError('Notebook này chỉ chạy trên Kaggle với Internet Off; hãy gắn Dataset packages/models/PDF.')

INPUT_ROOT = Path('/kaggle/input')
# Để trống để tự tìm qua manifest; nếu có nhiều Dataset cùng loại, điền đường dẫn.
PACKAGES_DIR = ''
MODELS_DIR = ''
DATA_DIR = ''
PREPARED_IMAGES = ''  # tùy chọn, thư mục có dataset.json; chỉ đọc descriptor

# Chỉ điền nếu runtime riêng ĐÃ tồn tại; GPU check tạo overlay riêng khi cần.
RUNTIME_PYTHONS = {
    # 'unihir_restoration': '/path/to/verified/environment/bin/python',
    # 'tadisr_x4': '/path/to/overlay/bin/python',
}
RUN_PROBES = True
CHECK_OPTIONS = {
    'verify_asset_hashes': False,  # mặc định chỉ stat kích thước; không đọc hàng chục GB weights
    'check_existing_runtimes': True,
    'quick_restoration_checks': False,  # True nếu chỉ cần nhóm probe restoration ngắn
    'max_runtime_checks': 8,
    'max_scan_dirs': 500,
    'max_asset_records': 12000,
    'probe_timeout_cap': 180,
    'time_budget_seconds': 900,
}
OUTPUT_DIR = Path('/kaggle/working')

# Kaggle-only; không chạy build/model trên local. Điền DATA_DIR để thử inference PDF.
RESTORATION_GPU_CHECK = 'off'  # off | build_and_probe | unihir_smoke; chủ động bật nếu cần
RESTORATION_CHECK_OPTIONS = {
    'books': [], 'pages': '1',  # [] chọn PDF đầu; giới hạn cứng 1 PDF / 1 trang / crop page
    'steps': 10, 'ref_count': 1,
    'gpu_ids': [0,1,2,3], 'placement':'sharded',
    'max_memory':'18GiB', 'activation_reserve_gib':4, 'entry_reserve_gib':2,
    'max_source_pixels': 16_000_000,
    'load_timeout_seconds': 900, 'inference_timeout_seconds': 900,
    'total_timeout_seconds': 5400,
    'runtime_dir': '/tmp/vtm_image_runtime',
}


In [ ]:
"""Read-only environment inventory; no installs, downloads, weights or OCR."""
from __future__ import annotations

from datetime import datetime, timezone
from email.parser import Parser
import importlib.metadata as metadata
import importlib.util
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import sysconfig
import hashlib
import time
import zipfile
from pathlib import PurePosixPath


PACKAGES = (
    "torch", "torchvision", "transformers", "tokenizers", "accelerate", "safetensors",
    "huggingface-hub", "sentencepiece", "protobuf", "numpy", "scipy", "pillow", "opencv-python", "opencv-python-headless",
    "paddlepaddle", "paddlepaddle-gpu", "paddleocr", "paddlex", "mmcv", "mmcv-full",
    "mmcv-lite", "mmengine", "mmdet", "mmrotate", "timm", "einops", "flash-attn",
    "triton", "pyyaml", "pypdfium2", "pandas", "tqdm", "pip", "setuptools", "wheel",
    "packaging", "ninja", "cmake", "uv", "diffusers", "peft", "omegaconf",
    "pytorch-lightning", "lightning-utilities", "torchmetrics", "zhconv", "fonttools",
    "decord", "bitsandbytes", "sentence-transformers", "opencv-contrib-python",
    "vit-pytorch", "keras", "h5py", "ml-dtypes", "optree", "namex",
)

PROBES = {
    "torch_cuda": ("torch", 90, """import torch, json
print('stage: torch imported', flush=True)
devices = []
for i in range(torch.cuda.device_count()):
    x = torch.ones((2,2), device=f'cuda:{i}')
    value = float((x @ x).sum().item())
    torch.cuda.synchronize(i)
    devices.append({'index':i,'name':torch.cuda.get_device_name(i),
        'VRAM_GiB':round(torch.cuda.get_device_properties(i).total_memory/1024**3,2),
        'free_VRAM_GiB':round(torch.cuda.mem_get_info(i)[0]/1024**3,2),
        'compute_capability':list(torch.cuda.get_device_capability(i)),'gpu_matmul_sum':value})
result = {'torch': torch.__version__, 'built_cuda': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'devices': devices,
          'compiled_arches': getattr(torch.cuda, 'get_arch_list', lambda: [])() if torch.cuda.is_available() else [],
          'module_file': torch.__file__, 'native_file': torch._C.__file__,
          'cxx11_abi': getattr(torch._C, '_GLIBCXX_USE_CXX11_ABI', None)}
result['gpu_count'] = len(devices)
result['peer_access'] = [[torch.cuda.can_device_access_peer(i,j) if i!=j else True for j in range(len(devices))] for i in range(len(devices))]
result['memory_is_per_device'] = True
print(json.dumps(result))
assert result['cuda_available'], 'CUDA unavailable; this is not a verified GPU runtime'
"""),
    "torchvision_ops": ("torchvision", 90, """import torch, torchvision, json
from torchvision.ops import nms
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
boxes = torch.tensor([[0.,0.,2.,2.],[0.,0.,2.,2.]],device=device)
keep = nms(boxes, torch.tensor([.9,.8],device=device), .5)
assert keep.numel() == 1
print(json.dumps({'torchvision': torchvision.__version__, 'nms_device': device}))
"""),
    "torch_numpy_bridge": ("torch", 45, """import torch, numpy as np, json
x = torch.from_numpy(np.zeros((2,2),dtype=np.float32))
assert x.shape == (2,2) and x.numpy().shape == (2,2)
print(json.dumps({'torch':torch.__version__,'numpy':np.__version__,'numpy_bridge':'OK'}))
"""),
    "numpy_cv2": ("cv2", 45, """import numpy as np, cv2, json
from PIL import Image
x = np.zeros((4,4,3),dtype=np.uint8)
assert cv2.cvtColor(x,cv2.COLOR_RGB2GRAY).shape == (4,4)
assert Image.fromarray(x).size == (4,4)
print(json.dumps({'numpy': np.__version__, 'opencv': cv2.__version__, 'array_image_bridge': 'OK'}))
"""),
    "transformers_hunyuan": ("transformers", 180, """import json
print('stage: transformers import', flush=True)
import transformers
print('stage: AutoProcessor', flush=True)
from transformers import AutoProcessor
print('stage: Hunyuan class', flush=True)
from transformers import HunYuanVLForConditionalGeneration
print(json.dumps({'transformers': transformers.__version__, 'hunyuan_api': 'OK'}))
"""),
    "paddle_cuda": ("paddle", 90, """import paddle, json
result = {'paddle': paddle.__version__, 'compiled_with_cuda': paddle.is_compiled_with_cuda()}
result['devices'] = paddle.device.cuda.device_count() if result['compiled_with_cuda'] else 0
if result['devices']:
    paddle.set_device('gpu:0')
    x = paddle.ones((2,2))
    result['gpu_matmul_sum'] = float(paddle.matmul(x,x).sum().item())
print(json.dumps(result))
"""),
    "paddleocr_apis": ("paddleocr", 120, """import paddleocr, json
from paddleocr import PaddleOCR, PaddleOCRVL
print(json.dumps({'paddleocr': paddleocr.__version__, 'PaddleOCR': 'OK', 'PaddleOCRVL': 'OK'}))
"""),
    "mmcv_compiled_ops": ("mmcv", 120, """import torch, mmcv, mmcv.ops, mmdet, mmrotate, json
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
boxes = torch.tensor([[5.,5.,2.,2.,0.]],device=device)
scores = torch.tensor([.9],device=device)
_, keep = mmcv.ops.nms_rotated(boxes,scores,.5)
assert keep.numel() == 1
print(json.dumps({'mmcv': mmcv.__version__, 'mmdet': mmdet.__version__, 'mmrotate': mmrotate.__version__, 'rotated_nms_device': device}))
"""),
    "pdf_renderer": ("pypdfium2", 45, """import pypdfium2 as pdfium, json
doc = pdfium.PdfDocument.new()
page = doc.new_page(32,32)
bitmap = page.render(scale=1)
image = bitmap.to_pil()
assert image.size == (32,32)
image.close(); bitmap.close(); page.close(); doc.close()
print(json.dumps({'blank_page_32x32': 'OK'}))
"""),
    "torch_attention": ("torch", 90, """import torch, json
from torch.nn.functional import scaled_dot_product_attention
assert torch.cuda.is_available(), 'CUDA required for attention probe'
tested = []
for dtype in [torch.float16] + ([torch.bfloat16] if torch.cuda.is_bf16_supported() else []):
    q = torch.randn(1, 2, 16, 64, device='cuda', dtype=dtype)
    y = scaled_dot_product_attention(q, q, q)
    torch.cuda.synchronize()
    assert y.shape == q.shape and bool(torch.isfinite(y).all())
    tested.append(str(dtype))
print(json.dumps({'tiny_sdpa_cuda': 'OK', 'tested_dtypes': tested, 'model_inference': False}))
"""),
    "diffusers_cogview4": ("diffusers", 180, """import json, diffusers
print('stage: CogView4 classes', flush=True)
from diffusers import CogView4Transformer2DModel, CogView4Pipeline
print(json.dumps({'diffusers':diffusers.__version__, 'module_file':diffusers.__file__,
 'CogView4Transformer2DModel':'OK', 'CogView4Pipeline':'OK', 'weights_loaded':False}))
"""),
    "transformers_glm": ("transformers", 180, """import json, transformers
from transformers import GlmModel
print(json.dumps({'transformers':transformers.__version__, 'GlmModel':'OK', 'weights_loaded':False}))
"""),
    "peft_api": ("peft", 120, """import peft, json
from peft import PeftModel
print(json.dumps({'peft':peft.__version__, 'module_file':peft.__file__, 'PeftModel':'OK'}))
"""),
    "restoration_dependencies": ("accelerate", 90, """import importlib, json
names = ['accelerate','safetensors','sentencepiece','huggingface_hub','PIL','numpy']
result = {}
for name in names:
    module = importlib.import_module(name)
    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}
print(json.dumps(result))
"""),
    "difftsr_dependencies": ("omegaconf", 120, """import importlib, json
result = {}
for name in ['torch','torchvision','pytorch_lightning','omegaconf','zhconv','einops','torchmetrics']:
    module = importlib.import_module(name)
    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}
print(json.dumps({'third_party_imports':result,'difftsr_model_code_tested':False}))
"""),
    "unihir_dependencies": ("accelerate", 120, """import importlib, json
result = {}
for name in ['torch','transformers','accelerate','fontTools','safetensors','einops']:
    module = importlib.import_module(name)
    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}
print(json.dumps({'third_party_imports':result,'unihir_model_code_tested':False}))
"""),
    "flash_attention": ("flash_attn", 90, """import torch, flash_attn, json
from flash_attn import flash_attn_func
assert torch.cuda.is_available(), 'CUDA required for FlashAttention probe'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
q = torch.randn(1,16,2,64,device='cuda',dtype=dtype)
y = flash_attn_func(q,q,q)
torch.cuda.synchronize()
assert y.shape == q.shape and bool(torch.isfinite(y).all())
print(json.dumps({'flash_attn':flash_attn.__version__, 'dtype':str(dtype),
 'module_file':flash_attn.__file__, 'tiny_cuda_forward':'OK', 'model_inference':False}))
"""),
    "torch_build": ("torch", 90, """import torch, json, sysconfig, pathlib
from torch.utils.cpp_extension import CUDA_HOME
print(json.dumps({'CUDA_HOME':str(CUDA_HOME) if CUDA_HOME else None,
 'torch_build_config':torch.__config__.show(), 'Python_h_exists':pathlib.Path(sysconfig.get_path('include'),'Python.h').is_file(),
 'compiled_arches':torch.cuda.get_arch_list() if torch.cuda.is_available() else [], 'extension_compilation_performed':False}))
"""),
    "torch_restoration_ops": ("torch", 90, """import torch, json
from torch.nn import functional as F
assert torch.cuda.is_available(), 'CUDA required for restoration tensor ops'
x = torch.ones(1,3,16,16,device='cuda',dtype=torch.float32)
y = F.conv2d(x,torch.ones(12,3,3,3,device='cuda'),padding=1)
y = F.pixel_shuffle(y,2)
y = F.interpolate(y,scale_factor=2,mode='bicubic',align_corners=False)
torch.cuda.synchronize()
assert tuple(y.shape)==(1,3,64,64) and bool(torch.isfinite(y).all())
print(json.dumps({'conv2d_pixel_shuffle_bicubic_cuda':'OK','dtype':'float32','model_inference':False}))
"""),
    "vit_pytorch_api": ("vit_pytorch", 90, """import vit_pytorch, json, importlib.metadata as md
from vit_pytorch import ViT
print(json.dumps({'version':md.version('vit-pytorch'),'module_file':vit_pytorch.__file__,
 'ViT_api':'OK','model_constructed':False,'model_inference':False}))
"""),
    "keras_torch_hdf5": ("keras", 120, """import os, io, json
os.environ['KERAS_BACKEND']='torch'
import keras, torch, h5py, numpy as np
from keras.layers import Input, Conv2D, MaxPooling2D, Dropout, UpSampling2D, concatenate
assert keras.backend.backend()=='torch', 'Keras Torch backend unavailable'
stream=io.BytesIO()
with h5py.File(stream,'w') as handle: handle['tiny']=np.arange(4,dtype=np.float32)
stream.seek(0)
with h5py.File(stream,'r') as handle: assert np.array_equal(handle['tiny'][:],np.arange(4))
print(json.dumps({'keras':keras.__version__,'backend':keras.backend.backend(),
 'torch':torch.__version__,'h5py':h5py.__version__,'hdf5':h5py.version.hdf5_version,
 'keras_file':keras.__file__,'h5py_file':h5py.__file__,'HDF5_numpy_bridge':'OK',
 'generator_constructed':False,'weights_loaded':False,'model_inference':False}))
"""),
}


PROBE_GUARD = '''import sys
sys.dont_write_bytecode = True
def _vtm_diagnostic_audit(event, args):
    if event in ('socket.connect', 'socket.getaddrinfo'):
        raise RuntimeError('Network disabled in environment diagnostics')
sys.addaudithook(_vtm_diagnostic_audit)
'''

DIAGNOSTIC_DEFAULTS = {'max_scan_dirs': 500, 'max_asset_records': 12000,
    'verify_asset_hashes': False, 'check_existing_runtimes': True,
    'quick_restoration_checks': False,
    'max_runtime_checks': 4, 'probe_timeout_cap': 180, 'time_budget_seconds': 600}

RUNTIME_INVENTORY = '''import sys, sysconfig, platform, json, importlib.metadata as md, importlib.util
records, seen = [], set()
for dist in md.distributions():
    name=dist.metadata.get('Name',''); key=name.lower().replace('_','-').replace('.','-')
    records.append({'name':name,'version':dist.version,'active':key not in seen,
        'location':str(dist.locate_file('')),'requires_python':dist.metadata.get('Requires-Python'),
        'requires':dist.requires or []})
    seen.add(key)
modules={}
for name in ['torch','torchvision','transformers','numpy','cv2','paddle','paddleocr','mmcv',
             'diffusers','peft','accelerate','omegaconf','flash_attn','pypdfium2','fontTools',
             'vit_pytorch','keras','h5py']:
    try:
        spec=importlib.util.find_spec(name)
        modules[name]={'available':spec is not None,'origin':getattr(spec,'origin',None)}
    except Exception as exc: modules[name]={'available':False,'error':str(exc)}
try:
    from packaging.tags import sys_tags
    tags=[str(tag) for tag in sys_tags()]
except Exception: tags=[]
print(json.dumps({'python':sys.version,'executable':sys.executable,'prefix':sys.prefix,
    'ABI':sysconfig.get_config_var('SOABI'),'machine':platform.machine(),
    'include_dir':sysconfig.get_path('include'),'Python_h_exists':__import__('pathlib').Path(sysconfig.get_path('include'),'Python.h').is_file(),
    'all_distributions':records,'module_specs':modules,'wheel_tags':tags}))
'''


def run_command(command, timeout=30):
    env = {**os.environ, 'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'HF_HUB_DISABLE_TELEMETRY': '1', 'PIP_NO_INDEX': '1', 'USE_TF': '0', 'USE_FLAX': '0', 'PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK': 'True', 'DIFFUSERS_OFFLINE': '1', 'HF_DATASETS_OFFLINE': '1', 'PYTHONDONTWRITEBYTECODE': '1', 'PYTHONUTF8': '1', 'PIP_DISABLE_PIP_VERSION_CHECK': '1', 'MPLBACKEND': 'Agg', 'TOKENIZERS_PARALLELISM': 'false'}
    for key in ('PYTHONHOME', 'PYTHONPATH', 'PYTHONUSERBASE'):
        env.pop(key, None)
    try:
        result = subprocess.run(command, capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=timeout, env=env)
        return {'status': 'ok' if result.returncode == 0 else 'failed', 'returncode': result.returncode, 'stdout': result.stdout, 'stderr': result.stderr}
    except subprocess.TimeoutExpired as exc:

        def decode(value):
            return value.decode('utf-8', errors='replace') if isinstance(value, bytes) else value
        return {'status': 'timeout', 'timeout_seconds': timeout, 'stdout': decode(exc.stdout), 'stderr': decode(exc.stderr)}
    except OSError as exc:
        return {'status': 'unavailable', 'detail': str(exc)}


def read_text(path):
    try:
        return Path(path).read_text(encoding="utf-8").strip()
    except OSError:
        return None


def hardware():
    meminfo = {}
    for line in read_text('/proc/meminfo').splitlines():
        key, value = line.split(':', 1)
        meminfo[key] = int(value.strip().split()[0]) * 1024
    limit = read_text('/sys/fs/cgroup/memory.max') or read_text('/sys/fs/cgroup/memory/memory.limit_in_bytes')
    current = read_text('/sys/fs/cgroup/memory.current') or read_text('/sys/fs/cgroup/memory/memory.usage_in_bytes')
    limit = int(limit) if limit and limit.isdigit() else None
    current = int(current) if current and current.isdigit() else None
    totals = [value for value in (meminfo.get('MemTotal'), limit) if value is not None]
    total = min(totals) if totals else None
    available = meminfo.get('MemAvailable')
    if limit is not None and current is not None:
        available = min(available if available is not None else limit, max(0, limit - current))
    quota = read_text('/sys/fs/cgroup/cpu.max')
    try:
        affinity = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else None
    except OSError:
        affinity = None
    cpu_model = next((line.split(':', 1)[1].strip() for line in read_text('/proc/cpuinfo').splitlines() if line.startswith('model name')), platform.processor())
    disks = {}
    for name in ('/', '/kaggle/input', '/kaggle/working', '/tmp'):
        if Path(name).is_dir():
            disk = shutil.disk_usage(name)
            disks[name] = {'total_GiB': round(disk.total / 1024 ** 3, 2), 'free_GiB': round(disk.free / 1024 ** 3, 2)}
    return {'os': platform.platform(), 'machine': platform.machine(), 'libc': platform.libc_ver(), 'python': sys.version, 'python_executable': sys.executable, 'python_ABI': sysconfig.get_config_var('SOABI'), 'cpu_model': cpu_model, 'cpu_logical': os.cpu_count(), 'cpu_affinity': affinity, 'cpu_quota_raw': quota, 'RAM_effective_GiB': round(total / 1024 ** 3, 2) if total is not None else None, 'RAM_available_GiB': round(available / 1024 ** 3, 2) if available is not None else None, 'disks': disks}


def distributions():
    records = []
    seen = set()
    for dist in metadata.distributions():
        name = dist.metadata.get('Name', '')
        key = name.lower().replace('_', '-').replace('.', '-')
        records.append({"name": name, "version": dist.version, 'active': key not in seen,
                        'location': str(dist.locate_file('')),
                        "requires_python": dist.metadata.get("Requires-Python"), "requires": dist.requires or []})
        seen.add(key)
    return sorted(records, key=lambda item: item["name"].lower())


def offline_inputs(root=Path("/kaggle/input"), max_dirs=150, max_files=80):
    found, visited = [], 0
    if not root.is_dir():
        return {"exists": False, "files": [], "truncated": False}
    for parent, dirs, files in os.walk(root):
        visited += 1
        if len(Path(parent).relative_to(root).parts) >= 6:
            dirs[:] = []
        for name in sorted(files):
            if name.endswith((".whl", ".tar.gz", ".tar.xz", ".zip", ".safetensors", ".pth")) or name in {"config.json", "inference.yml", "bundle_manifest.json"}:
                path = Path(parent) / name
                try:
                    size = path.stat().st_size
                except OSError:
                    continue
                found.append({"path": str(path), "bytes": size})
                if len(found) >= max_files:
                    return {"exists": True, "files": found, "truncated": True, "visited_dirs": visited}
        if visited >= max_dirs:
            return {"exists": True, "files": found, "truncated": True, "visited_dirs": visited}
    return {"exists": True, "files": found, "truncated": False, "visited_dirs": visited}


def diagnostic_options(options=None):
    if set(options or {}) - DIAGNOSTIC_DEFAULTS.keys():
        raise ValueError('Unknown diagnostic options')
    result = {**DIAGNOSTIC_DEFAULTS, **(options or {})}
    for key, default in DIAGNOSTIC_DEFAULTS.items():
        if isinstance(default, bool):
            if type(result[key]) is not bool:
                raise ValueError(f'{key} must be bool')
        elif type(result[key]) is not int or result[key] < 1:
            raise ValueError(f'{key} must be a positive integer')
    return result


def diagnostic_json(path):
    path = Path(path)
    if path.stat().st_size > 16*1024*1024:
        raise ValueError('Diagnostic metadata exceeds 16 MiB; not read')
    return json.loads(path.read_text(encoding='utf8'))


def diagnostic_payload(output):
    for line in reversed(output.splitlines()):
        try:
            value = json.loads(line)
            if isinstance(value, dict):
                return value
        except ValueError:
            continue
    return None


def discover_asset_roots(input_root, configured, *, max_dirs=500):
    """Bounded discovery prunes weights/objects/source trees; explicit paths take priority."""
    markers = {'models': 'models_manifest.json', 'packages': 'bundle_manifest.json',
               'prepared_images': 'dataset.json'}
    candidates = {key: [] for key in markers}
    root, visited, truncated = Path(input_root), 0, False
    if root.is_dir():
        for parent, dirs, files in os.walk(root):
            visited += 1
            for kind, marker in markers.items():
                if marker in files:
                    candidates[kind].append(str(Path(parent).resolve()))
            dirs[:] = sorted(d for d in dirs if d not in {'sources', 'wheels', 'objects', 'snapshots', '__pycache__', '.cache'})
            if len(Path(parent).relative_to(root).parts) >= 6:
                dirs[:] = []
            if visited >= max_dirs:
                truncated = True
                break
    found = {}
    for kind, marker in markers.items():
        explicit = configured.get(kind)
        if explicit:
            path = Path(explicit).resolve()
            if path.is_file():
                path = path.parent.parent if path.parent.name == 'snapshots' else path.parent
            for child in (path/kind, path/'kaggle_upload'/kind):
                if not (path/marker).is_file() and (child/marker).is_file():
                    path = child
                    break
            found[kind] = {'path': str(path), 'selection': 'explicit', 'exists': path.is_dir()}
        elif len(candidates[kind]) == 1:
            found[kind] = {'path': candidates[kind][0], 'selection': 'discovered', 'exists': True}
        else:
            found[kind] = {'path': None, 'selection': 'ambiguous' if candidates[kind] else 'not_found',
                           'candidates': candidates[kind], 'exists': False}
    found['discovery'] = {'visited_dirs': visited, 'truncated': truncated}
    found['pdfs'] = {'path': str(configured['pdfs']) if configured.get('pdfs') else None,
                     'selection': 'explicit' if configured.get('pdfs') else 'not_configured'}
    return found


def diagnostic_relative(root, relative):
    value = str(relative)
    parts = PurePosixPath(value)
    if ':' in value or '\\' in value or parts.is_absolute() or '..' in parts.parts:
        raise ValueError('Invalid portable asset path')
    root = Path(root).resolve()
    path = root.joinpath(*parts.parts)
    if not path.resolve().is_relative_to(root):
        raise ValueError('Asset reference escapes its root')
    return path


def inspect_asset_records(root, records, *, limit=12000, verify_hashes=False):
    """Default checks file presence and size only, without opening model weights."""
    expanded=[]
    for record in records:
        if record.get('type')=='directory':
            expanded.extend(dict(child,path=record['path'].rstrip('/')+'/'+child['path']) for child in record['files'])
        else:
            expanded.append(record)
    records=expanded
    missing, mismatches, checked, bytes_declared, hashes_verified = [], [], 0, 0, 0
    optional_bytecode = []
    for record in records[:limit]:
        relative = record.get('path', '')
        try:
            path = diagnostic_relative(root, relative)
            if '__pycache__' in PurePosixPath(relative).parts and path.suffix in ('.pyc', '.pyo'):
                optional_bytecode.append(relative)
                continue
            if not path.is_file():
                missing.append(relative)
                continue
            actual = path.stat().st_size
            if record.get('bytes') is not None and actual != record['bytes']:
                mismatches.append({'path': relative, 'kind': 'size', 'actual': actual, 'expected': record['bytes']})
            elif verify_hashes and record.get('sha256'):
                digest = hashlib.sha256()
                with path.open('rb') as stream:
                    for block in iter(lambda: stream.read(8*1024*1024), b''):
                        digest.update(block)
                if digest.hexdigest() != record['sha256']:
                    mismatches.append({'path': relative, 'kind': 'sha256'})
                else:
                    hashes_verified += 1
            checked += 1
            bytes_declared += record.get('bytes', actual)
        except (OSError, ValueError) as exc:
            mismatches.append({'path': relative, 'kind': 'unreadable_or_invalid', 'detail': str(exc)})
    return {'status': 'issues_found' if missing or mismatches else 'analysis_incomplete' if len(records) > limit
            else 'files_present_sizes_match', 'declared_files': len(records), 'checked_files': checked,
            'missing': missing, 'mismatches': mismatches, 'declared_bytes_checked': bytes_declared,
            'truncated': len(records) > limit, 'hashes_requested': verify_hashes,
            'hashes_verified_files': hashes_verified,
            'optional_bytecode_records': optional_bytecode,
            'model_inference_verified': False}


def inspect_runtime_kernel_requirements(plan):
    """Compare the measured kernel to a plan without importing ML packages."""
    mismatches = []
    for name, expected in plan.get('kernel_reuse', {}).items():
        try:
            actual = metadata.version(name)
        except metadata.PackageNotFoundError:
            actual = None
        if actual != expected:
            mismatches.append({'package': name, 'expected': expected, 'actual': actual})
    abi_matches = (str(sys.version_info.major)+str(sys.version_info.minor) == str(plan.get('python'))
                   and sys.platform == 'linux' and platform.machine() == 'x86_64')
    return {'state': 'kernel_metadata_match' if not mismatches and abi_matches else 'kernel_metadata_differs',
            'mismatches': mismatches, 'python': platform.python_version(),
            'python_platform_match': abi_matches, 'expected_python': plan.get('python'),
            'platform': sysconfig.get_platform(), 'model_inference_verified': False,
            'note': 'Metadata only; missing overlay packages in the kernel are allowed. No installation or model load.'}


def diagnostic_wheel_metadata(path):
    with zipfile.ZipFile(path) as archive:
        names = [n for n in archive.namelist() if n.count('/') == 1 and n.endswith('.dist-info/METADATA')]
        if len(names) != 1 or archive.getinfo(names[0]).file_size > 4*1024**2:
            raise ValueError('Invalid or oversized wheel METADATA')
        body = Parser().parsestr(archive.read(names[0]).decode('utf8'))
    return {'name': body['Name'], 'version': body['Version'],
            'requires_python': body.get('Requires-Python'), 'requires': body.get_all('Requires-Dist', [])}


def inspect_runtime_dependency_metadata(root, plan):
    """Resolve the planned overlay against this kernel without pip or imports."""
    try:
        from packaging.requirements import Requirement
        from packaging.utils import canonicalize_name
        from packaging.specifiers import SpecifierSet
        from packaging.markers import default_environment
        selected, problems, checked, extras = {}, [], [], []
        for record in plan.get('wheels', []):
            item = diagnostic_wheel_metadata(diagnostic_relative(root, record['path']))
            if (record.get('name') and canonicalize_name(record['name']) != canonicalize_name(item['name'])) or (
                    record.get('version') and record['version'] != item['version']):
                problems.append({'path':record['path'], 'state':'wheel_metadata_differs_from_plan'})
            selected[canonicalize_name(item['name'])] = dict(item, source='staged_overlay')
        pending = list(plan.get('requirements', []))
        roots_explicit = bool(pending)
        if not pending:
            pending = [name+'=='+version for name, version in plan.get('kernel_reuse', {}).items()]
            pending.extend(item['name']+'=='+item['version'] for item in selected.values())
        seen = set()
        marker_environment = {**default_environment(), 'extra': ''}
        while pending and len(seen) < 1500:
            raw = pending.pop()
            if raw in seen:
                continue
            seen.add(raw)
            requirement = Requirement(raw)
            if requirement.marker and not requirement.marker.evaluate(marker_environment):
                continue
            if requirement.extras:
                extras.append(raw)
            name = canonicalize_name(requirement.name)
            if name not in selected:
                try:
                    dist = metadata.distribution(name)
                    selected[name] = {'name':name, 'version':dist.version, 'source':'active_kernel',
                        'requires':dist.requires or [], 'requires_python':dist.metadata.get('Requires-Python')}
                except metadata.PackageNotFoundError:
                    problems.append({'requirement':raw, 'state':'missing_in_kernel_and_planned_wheels'})
                    continue
            item = selected[name]
            if not requirement.specifier.contains(item['version'], prereleases=True):
                problems.append({'requirement':raw, 'state':'version_conflict', 'actual':item['version'], 'source':item['source']})
            if item['requires_python'] and not SpecifierSet(item['requires_python']).contains(platform.python_version(), prereleases=True):
                problems.append({'package':name, 'state':'python_version_conflict', 'requires_python':item['requires_python']})
            checked.append({'requirement':raw, 'version':item['version'], 'source':item['source']})
            pending.extend(item['requires'])
        native = plan.get('native_sources', [])
        return {'state':'issues_found' if problems else 'partial_metadata' if pending or native or extras else 'metadata_closure_satisfied',
            'problems':problems, 'checked_requirements':checked, 'truncated':bool(pending),
            'optional_extras_not_evaluated':extras,
            'root_requirements_source':'explicit_plan' if roots_explicit else 'planned_kernel_reuse_and_wheels',
            'native_source_metadata_not_resolved':[r['path'] for r in native],
            'note':'Default dependency markers only; no optional extras, installation, native build or model execution verified.'}
    except (ImportError, OSError, ValueError, KeyError, zipfile.BadZipFile) as error:
        return {'state':'metadata_analysis_unavailable', 'detail':str(error)}


def inspect_offline_assets(roots, settings):
    reports = {}
    def inspect(name, root, manifest, required=()):
        result = {'manifest': str(manifest), 'required_missing': [], 'state': 'manifest_missing'}
        try:
            data = diagnostic_json(manifest)
            result.update(inspect_asset_records(root, data.get('files', []), limit=settings['max_asset_records'],
                                                verify_hashes=settings['verify_asset_hashes']))
            result['state'] = result['status']
            result['required_missing'] = [r for r in required if not diagnostic_relative(root, r).is_file()]
            indexed = {r.get('path') for r in data.get('files', [])}
            result['required_not_in_inventory'] = [r for r in required if r not in indexed]
            if result['required_missing'] or result['required_not_in_inventory']:
                result['state'] = 'missing_required_asset'
            if name == 'tadisr':
                shards = []
                for prefix, index in (('transformer', 'diffusion_pytorch_model.safetensors.index.json'),
                                      ('text_encoder', 'model.safetensors.index.json')):
                    index_path = Path(root)/'CogView4'/prefix/index
                    if index_path.is_file():
                        body = diagnostic_json(index_path)
                        for part in set(body['weight_map'].values()):
                            if (not diagnostic_relative(Path(root)/'CogView4'/prefix, part).is_file()
                                    or f'CogView4/{prefix}/{part}' not in indexed):
                                shards.append(prefix+'/'+part)
                result['missing_indexed_shards'] = shards
                if shards:
                    result['state'] = 'missing_required_asset'
                result['prompt_embeddings_present'] = (Path(root)/'CogView4/saved_prompt_tokens_nocfg.pt').is_file()
                result['prompt_note'] = 'Notebook 1 can derive the fixed prompt offline from verified base assets'
        except (OSError, ValueError, KeyError, TypeError) as exc:
            result['detail'] = str(exc)
            if Path(manifest).is_file():
                result['state'] = 'invalid_manifest'
        reports[name] = result
    models = roots.get('models', {}).get('path')
    if models:
        models = Path(models)
        inspect('ocr_models', models, models/'models_manifest.json', ['tonggu/layout/finetune_infer.pth'])
        requirements = {
            'docentr': ['model_8_2018_base.pt'],
            'degan': ['degan_weights.bundle'],
            'realesrgan': ['RealESRGAN_x4plus.pth'],
            'docres': ['docres.pkl'],
            'docdiff': ['init.pth','denoiser.pth'],
            'restormer': ['motion_deblurring.pth','single_image_defocus_deblurring.pth',
                          'real_denoising.pth','gaussian_color_denoising_blind.pth'],
            'unihir': ['model.safetensors', 'ae.safetensors', 'llm_config.json', 'vit_config.json',
                      'tokenizer.json', 'tokenizer_config.json', 'vocab.json', 'merges.txt'] +
                     ['font/'+n for n in ('KaiXinSongA.ttf','KaiXinSongB.ttf','TH-Tshyn-P0.ttf','TH-Tshyn-P1.ttf','TH-Tshyn-P16.ttf','TH-Tshyn-P2.ttf')],
            'tadisr': ['tadisr_cogview4_realce_17500.pkl'] + ['CogView4/'+n for n in (
                'model_index.json','vae/config.json','vae/diffusion_pytorch_model.safetensors',
                'transformer/config.json','transformer/diffusion_pytorch_model.safetensors.index.json',
                'text_encoder/config.json','text_encoder/model.safetensors.index.json',
                'tokenizer/tokenizer.json','tokenizer/tokenizer_config.json','tokenizer/special_tokens_map.json',
                'scheduler/scheduler_config.json')],
        }
        for name, required in requirements.items():
            root = models/'image_restoration'/name
            inspect(name, root, root/'assets.json', required)
        reports['difftsr']={'state':'excluded_by_configuration','required':False,
                            'assets_still_present':(models/'image_restoration/difftsr').exists()}
    packages = roots.get('packages', {}).get('path')
    if packages:
        packages = Path(packages)
        inspect('ocr_packages', packages, packages/'bundle_manifest.json',
                ['paddle_manifest.json','tonggu_manifest.json','system_assets_manifest.json'])
        restoration = packages/'image_restoration'
        try:
            plan = diagnostic_json(restoration/'manifests/tadisr_runtime.json')
            reports['tadisr_packages'] = inspect_asset_records(restoration, plan.get('wheels', []),
                limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes'])
            reports['tadisr_kernel_compatibility'] = inspect_runtime_kernel_requirements(plan)
            reports['tadisr_dependency_metadata'] = inspect_runtime_dependency_metadata(restoration, plan)
        except (OSError, ValueError, TypeError) as exc:
            reports['tadisr_packages'] = {'status': 'manifest_unavailable', 'detail': str(exc)}
        for name in ('document', 'unihir', 'learned_documents'):
            try:
                plan=diagnostic_json(restoration/f'manifests/{name}_runtime.json')
                reports[name+'_runtime_manifest'] = plan
                reports[name+'_kernel_compatibility'] = inspect_runtime_kernel_requirements(plan)
                reports[name+'_dependency_metadata'] = inspect_runtime_dependency_metadata(restoration, plan)
                records=plan.get('wheels',[])+plan.get('native_sources',[])
                if plan.get('bootstrap'):records.append(plan['bootstrap'])
                reports[name+'_runtime_assets']=inspect_asset_records(restoration,records,
                    limit=settings['max_asset_records'],verify_hashes=settings['verify_asset_hashes'])
            except (OSError, ValueError) as exc:
                reports[name+'_runtime_manifest'] = {'state': 'manifest_unavailable', 'detail': str(exc)}
        try:
            catalog = diagnostic_json(restoration/'manifests/asset_catalog.json')
            for name, group in catalog['groups'].items():
                records = [dict(r, path=r['path'].removeprefix('packages/'))
                           for r in group['files'] if r['kind'] == 'source']
                if not records:
                    continue
                reports[name+'_sources'] = inspect_asset_records(packages, records,
                    limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes'])
                reports[name+'_sources'].update(revision=group['revision'], license=group['license'])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            reports['restoration_catalog_sources'] = {'state':'manifest_unavailable', 'detail':str(exc)}
        try:
            records = diagnostic_json(restoration/'manifests/embedded_sources.json')
            records = [dict(r, path=r['path'].removeprefix('packages/')) for r in records]
            reports['embedded_upstream_sources'] = inspect_asset_records(packages, records,
                limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes'])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            reports['embedded_upstream_sources'] = {'state':'manifest_unavailable', 'detail':str(exc)}
    prepared = roots.get('prepared_images', {}).get('path')
    if prepared:
        try:
            data = diagnostic_json(Path(prepared)/'dataset.json')
            reports['prepared_images'] = {'format': data.get('format'), 'version': data.get('version'),
                'snapshot_id': data.get('snapshot_id'), 'pages': len(data.get('pages', [])),
                'restoration': data.get('restoration'), 'state': 'descriptor_read_only', 'pixels_verified': False}
        except (OSError, ValueError, TypeError) as exc:
            reports['prepared_images'] = {'state': 'descriptor_unavailable', 'detail': str(exc)}
    return reports


def runtime_candidates(configured, *, include_existing=True):
    result = {str(name): {'python': str(path), 'source': 'explicit'} for name, path in (configured or {}).items() if path}
    if include_existing:
        for path in (Path('/tmp/vtm_kaggle_runtime/runtime_config.json'), Path('/kaggle/working/runtime_config.json')):
            try:
                data = diagnostic_json(path)
                for name in ('paddle', 'tonggu'):
                    if data.get(name+'_python') and name not in result:
                        result[name] = {'python': data[name+'_python'], 'source': str(path)}
            except (OSError, ValueError, TypeError):
                pass
        path = Path('/kaggle/working/.image_runtime/preflight.json')
        try:
            for name, item in diagnostic_json(path).items():
                if isinstance(item, dict) and item.get('python') and name not in result:
                    result[name] = {'python': item['python'], 'source': str(path)}
        except (OSError, ValueError, TypeError):
            pass
    return result


def runtime_probe_names(name):
    common = ['torch_cuda', 'torch_numpy_bridge', 'numpy_cv2', 'pdf_renderer', 'torch_build']
    if name in {'paddle', 'ppocr'}:
        return ['numpy_cv2', 'pdf_renderer', 'paddle_cuda', 'paddleocr_apis']
    if name == 'tonggu':
        return common + ['torchvision_ops', 'mmcv_compiled_ops']
    if name == 'difftsr':
        return common + ['torchvision_ops', 'difftsr_dependencies']
    if name in {'unihir', 'unihir_restoration'}:
        return common + ['unihir_dependencies', 'flash_attention', 'torch_attention']
    if name in {'tadisr', 'tadisr_x4'}:
        return common + ['diffusers_cogview4', 'transformers_glm', 'peft_api', 'restoration_dependencies', 'torch_restoration_ops']
    if name in {'realesrgan', 'realesrgan_x4'}:
        return common + ['torch_restoration_ops']
    if name.startswith(('docentr', 'degan', 'learned_documents')):
        return common + ['vit_pytorch_api', 'keras_torch_hdf5']
    return common


def inspect_staged_wheels(roots, runtime_inventories):
    packages = roots.get('packages', {}).get('path')
    if not packages:
        return {'state': 'packages_not_configured', 'wheels': []}
    try:
        from packaging.utils import parse_wheel_filename
        result = []
        candidates = sorted(Path(packages).rglob('*.whl'))
        for path in candidates[:2000]:
            try:
                _, _, _, tags = parse_wheel_filename(path.name)
                wheel_tags = {str(tag) for tag in tags}
                result.append({'file': str(path), 'wheel_tags': sorted(wheel_tags),
                    'metadata':diagnostic_wheel_metadata(path),
                    'compatible_with_observed_interpreters': {name: bool(wheel_tags & set(inv['wheel_tags']))
                        if inv.get('wheel_tags') else None for name, inv in runtime_inventories.items()}})
            except (OSError, ValueError, zipfile.BadZipFile) as exc:
                result.append({'file': str(path), 'state': 'invalid_wheel_metadata_or_filename', 'detail': str(exc)})
        return {'state': 'tag_and_metadata_comparison', 'wheels': result,
                'total_wheels':len(candidates), 'truncated':len(candidates)>2000,
                'note': 'A cp310 wheel failing kernel cp313 tags can still belong to a separate cp310 runtime; tags do not prove CUDA support'}
    except ImportError:
        return {'state': 'packaging_module_unavailable', 'wheels': []}


def readiness_summary(report):
    """Per-test observations only: never promote these diagnostics to model readiness."""
    groups = {'notebook_1_classical': ['numpy_cv2', 'pdf_renderer'],
        'notebook_1_realesrgan': ['torch_cuda', 'torch_numpy_bridge', 'numpy_cv2', 'torch_restoration_ops'],
        'notebook_1_tadisr': ['torch_cuda', 'diffusers_cogview4', 'transformers_glm', 'peft_api', 'restoration_dependencies'],
        'notebook_1_unihir': ['torch_cuda', 'unihir_dependencies', 'flash_attention'],
        'notebook_1_docentr': ['torch_cuda', 'torch_numpy_bridge', 'vit_pytorch_api'],
        'notebook_1_degan': ['torch_cuda', 'keras_torch_hdf5'],
        'notebook_3_hunyuan': ['torch_cuda', 'torch_numpy_bridge', 'transformers_hunyuan'],
        'notebook_3_paddle_in_kernel': ['paddle_cuda', 'paddleocr_apis'],
        'notebook_3_tonggu_in_kernel': ['torch_cuda', 'mmcv_compiled_ops']}
    result = {}
    for name, probes in groups.items():
        states = {p: report['probes'].get(p, {}).get('status', 'not_run') for p in probes}
        result[name] = {'kernel_probe_states': states, 'model_inference_verified': False,
            'note': 'Kernel imports are separate from staged assets and private runtimes. Passing tiny ops does not verify full model inference.'}
    return result


def collect_environment(*, run_probes=True, input_root=Path('/kaggle/input'), configured_paths=None,
                        runtime_pythons=None, options=None, checkpoint_path=None):
    settings = diagnostic_options(options)
    started = time.monotonic()
    inventory = distributions()
    versions = {}
    for item in inventory:
        name = item['name'].lower().replace('_', '-')
        if name not in versions or item.get('active') is True:
            versions[name] = item['version']
    roots = discover_asset_roots(input_root, configured_paths or {}, max_dirs=settings['max_scan_dirs'])
    report = {"schema": 1, "diagnostics_version": 2, "collected_at_UTC": datetime.now(timezone.utc).isoformat(),
              "hardware": hardware(), "packages": {name: versions.get(name) for name in PACKAGES},
              "all_distributions": inventory, "probes": {}, "tools": {}, "offline_inputs": offline_inputs(Path(input_root)),
              "prior_vtm_runtime_exists": Path("/tmp/vtm_kaggle_runtime/runtime_config.json").is_file(),
              "asset_roots": roots, "options": settings, "assets": inspect_offline_assets(roots, settings),
              "runtimes": {}, "complete": False, "status": 'collecting', "execution_limits": [],
              "model_inference_performed": False,
              "configuration": {'input_root': str(input_root), 'configured_paths': {k:str(v) for k,v in (configured_paths or {}).items()},
                                'runtime_pythons': {k:str(v) for k,v in (runtime_pythons or {}).items()}},
              "build_inventory": {'Python_include': sysconfig.get_path('include'),
                  'Python_h_exists': Path(sysconfig.get_path('include'), 'Python.h').is_file(),
                  'sysconfig': {k:sysconfig.get_config_var(k) for k in ('CC','CXX','LDSHARED','LIBDIR','EXT_SUFFIX','SOABI')},
                  'selected_environment': {k:os.environ.get(k) for k in ('CUDA_HOME','CUDA_PATH','CC','CXX','CUDACXX','MAX_JOBS','CUDA_VISIBLE_DEVICES')}},
              "note": "Inventory phase only: metadata/imports/tiny ops, no installation or model loading in this phase. Optional build/inference results appear separately in restoration_gpu_check. Assets use size checks unless hashes explicitly enabled."}
    def checkpoint():
        report['elapsed_seconds'] = round(time.monotonic()-started, 3)
        if checkpoint_path:
            path = Path(checkpoint_path)
            if path.resolve().is_relative_to(Path('/kaggle/input').resolve()):
                raise ValueError('Diagnostic report must not be written to Kaggle Input')
            path.parent.mkdir(parents=True, exist_ok=True)
            temporary = path.with_name(path.name+'.tmp')
            temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2)+'\n', encoding='utf8')
            temporary.replace(path)
    def bounded(command, timeout=30):
        remaining = settings['time_budget_seconds']-(time.monotonic()-started)
        if remaining < 1:
            report['complete'] = False
            if 'time_budget_seconds' not in report['execution_limits']:
                report['execution_limits'].append('time_budget_seconds')
            return {'status': 'not_run_budget', 'detail': 'Diagnostic time budget exhausted; partial report retained'}
        return run_command(command, min(timeout, settings['probe_timeout_cap'], max(1, int(remaining))))
    checkpoint()
    if not run_probes:
        report['readiness'] = readiness_summary(report)
        report.update(complete=True, status='inventory_only')
        checkpoint()
        return report
    commands = {"gpu": ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,driver_version", "--format=csv,noheader"],
                "nvcc": ["nvcc", "--version"], "g++": ["g++", "--version"],
                "cmake": ["cmake", "--version"], "ninja": ["ninja", "--version"],
                "gcc": ["gcc", "--version"], "ldd": ["ldd", "--version"]}
    for name, command in commands.items():
        report["tools"][name] = bounded(command)
        checkpoint()
    report["pip_check"] = bounded([sys.executable, "-m", "pip", "check"], timeout=60)
    report['kernel_runtime_inventory'] = bounded([sys.executable, '-c', PROBE_GUARD+RUNTIME_INVENTORY], timeout=60)
    kernel_inventory = diagnostic_payload(report['kernel_runtime_inventory'].get('stdout', '')) or {}
    observed = {'kernel': kernel_inventory}
    report['kernel_runtime_inventory']['data'] = kernel_inventory
    checkpoint()
    for name, (module, timeout, code) in PROBES.items():
        if name=='difftsr_dependencies':continue
        if settings['quick_restoration_checks'] and name not in {
            'torch_cuda','torch_numpy_bridge','numpy_cv2','pdf_renderer','torch_attention',
            'flash_attention','torch_restoration_ops'}:continue
        print(f"Checking {name} (timeout {timeout}s)...", flush=True)
        try:
            present = importlib.util.find_spec(module) is not None
        except (ImportError, ValueError):
            present = False
        result = bounded([sys.executable, "-c", PROBE_GUARD+code], timeout) if present else {"status": "not_installed"}
        result['data'] = diagnostic_payload(result.get('stdout', ''))
        report["probes"][name] = result
        print(f"  {result['status']}", flush=True)
        checkpoint()
    candidates = runtime_candidates(runtime_pythons, include_existing=settings['check_existing_runtimes'])
    seen_python = {str(Path(sys.executable).resolve()): 'kernel'}
    checked = 0
    for name, candidate in candidates.items():
        python = str(Path(candidate['python']).expanduser().resolve())
        state = {**candidate, 'python': python, 'model_inference_verified': False, 'probes': {}}
        report['runtimes'][name] = state
        if python in seen_python:
            state.update(status='shared_interpreter', shares_with=seen_python[python])
            checkpoint()
            continue
        if checked >= settings['max_runtime_checks']:
            state['status'] = 'not_run_runtime_limit'
            report['complete'] = False
            if 'max_runtime_checks' not in report['execution_limits']:
                report['execution_limits'].append('max_runtime_checks')
            continue
        if not Path(python).is_file():
            state['status'] = 'interpreter_missing'
            continue
        checked += 1
        seen_python[python] = name
        print(f'Checking separate runtime {name}: {python}', flush=True)
        state['inventory'] = bounded([python, '-c', PROBE_GUARD+RUNTIME_INVENTORY], timeout=60)
        detail = diagnostic_payload(state['inventory'].get('stdout', ''))
        if state['inventory']['status'] != 'ok' or detail is None:
            state['status'] = 'inventory_unavailable'
            checkpoint()
            continue
        state['inventory']['data'] = detail
        observed[name] = detail
        state['pip_check'] = bounded([python, '-m', 'pip', 'check'], timeout=60)
        for probe_name in runtime_probe_names(name):
            module, timeout, code = PROBES[probe_name]
            present = detail.get('module_specs', {}).get(module, {}).get('available', False)
            print(f'  {name}/{probe_name}', flush=True)
            probe = bounded([python, '-c', PROBE_GUARD+code], timeout) if present else {'status':'not_installed'}
            probe['data'] = diagnostic_payload(probe.get('stdout', ''))
            state['probes'][probe_name] = probe
            checkpoint()
        state['status'] = 'probed; model_inference_unverified'
        checkpoint()
    report['staged_wheel_tags'] = inspect_staged_wheels(roots, observed)
    report['readiness'] = readiness_summary(report)
    report['limitations'] = ['Missing kernel packages may be supplied by a separate staged runtime.',
        'Wheel ABI tags and tiny GPU ops do not verify full model execution or restoration fidelity.',
        'This inventory phase does not execute model weights.',
        'Optional restoration GPU checks are reported separately from inventory; no network downloads.']
    report['complete'] = not report['execution_limits']
    report['status'] = 'completed_with_probe_results' if report['complete'] else 'partial_limits'
    checkpoint()
    return report


def save_report(report, output_dir):
    output = Path(output_dir)
    if output.resolve().is_relative_to(Path('/kaggle/input').resolve()):
        raise ValueError('Output must be outside Kaggle Input')
    output.mkdir(parents=True, exist_ok=True)
    json_path = output / 'kaggle_environment_report.json'
    text_path = output / 'kaggle_environment_report.txt'
    json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    lines = [json.dumps(report['hardware'], ensure_ascii=False, indent=2), '\nPackages:']
    lines.extend((f"{name:24s} {'not installed'}" for name, version in report['packages'].items()))
    for group in ('tools', 'probes'):
        for name, result in report[group].items():
            lines.append(f'\n[{name}]\n' + json.dumps(result, ensure_ascii=False, indent=2))
    lines.append('\n[pip_check]\n' + json.dumps(report.get('pip_check'), ensure_ascii=False, indent=2))
    for name in ('asset_roots', 'assets', 'runtimes', 'build_inventory', 'readiness', 'limitations', 'restoration_gpu_check'):
        if name in report:
            lines.append(f'\n[{name}]\n' + json.dumps(report[name], ensure_ascii=False, indent=2))
    text_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return (json_path, text_path)




In [ ]:
"""Kaggle-only restoration check using notebook 1's exact offline implementation.

Inventory remains read-only. The separately enabled GPU check may create a
private runtime, compile FlashAttention and run one UniHIR page. No downloads,
OCR, training, kernel package replacement, or corpus processing.
"""
from __future__ import annotations

import ast
import base64
from contextlib import contextmanager, ExitStack
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import shutil
import signal
import subprocess
import sys
import time
import traceback
from types import ModuleType
import uuid
import zipfile
import zlib


FLASH_GPU_PROBE = '''import json, torch, flash_attn
from flash_attn import flash_attn_func, flash_attn_varlen_func
assert torch.cuda.is_available(), 'CUDA unavailable'
torch.manual_seed(42)
devices=[]
for index in range(torch.cuda.device_count()):
    with torch.cuda.device(index):
        q,k,v=[torch.randn(1,16,2,64,device=f'cuda:{index}',dtype=torch.bfloat16) for _ in range(3)]
        y=flash_attn_func(q,k,v,dropout_p=0.,causal=False)
        reference=torch.nn.functional.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),dropout_p=0.).transpose(1,2)
        cu=torch.tensor([0,16],device=f'cuda:{index}',dtype=torch.int32)
        variable=flash_attn_varlen_func(q[0],k[0],v[0],cu,cu,16,16,dropout_p=0.,causal=False)
        torch.cuda.synchronize(index)
        assert y.shape==q.shape and variable.shape==q[0].shape
        assert bool(torch.isfinite(y).all()) and bool(torch.isfinite(variable).all())
        assert torch.allclose(y.float(),reference.float(),atol=.03,rtol=.03), 'FlashAttention differs from SDPA reference'
        assert torch.allclose(variable.float(),y[0].float(),atol=.03,rtol=.03), 'Variable-length kernel differs'
        devices.append(dict(index=index,name=torch.cuda.get_device_name(index),
            max_abs_difference=float((y.float()-reference.float()).abs().max().item())))
print(json.dumps(dict(status='passed',torch=torch.__version__,cuda=torch.version.cuda,
    devices=devices,capability=list(torch.cuda.get_device_capability(0)),
    flash_attention=flash_attn.__version__,module_file=flash_attn.__file__,
    fixed_and_variable_length_verified=True,finite=True,
    weights_loaded=False)))
'''


def check_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+'.tmp')
    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n',encoding='utf8')
    temporary.replace(path)


def check_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024**2),b''):h.update(block)
    return h.hexdigest()


def check_offline_env():
    env=dict(os.environ)
    for key in ('PYTHONPATH','PYTHONHOME','PYTHONUSERBASE'):env.pop(key,None)
    env.update(PYTHONNOUSERSITE='1',PYTHONUTF8='1',PIP_NO_INDEX='1',PIP_DISABLE_PIP_VERSION_CHECK='1',
        HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',HF_DATASETS_OFFLINE='1',DIFFUSERS_OFFLINE='1',
        HF_HUB_DISABLE_TELEMETRY='1',MPLBACKEND='Agg',TOKENIZERS_PARALLELISM='false')
    return env


def check_tail(path, maximum=2000):
    try:
        with Path(path).open('rb') as stream:
            stream.seek(0,2);stream.seek(max(0,stream.tell()-maximum))
            return stream.read().decode('utf8',errors='replace')
    except OSError:return ''


def check_stop(process):
    if process.poll() is not None:
        return
    if not getattr(process, '_vtm_owns_group', False):
        process.kill()
    else:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    process.wait()


def check_logged_process(command, log, *, timeout, env=None, report_path=None, new_session=True):
    """One process group; preserve bounded log tails and kill descendants on timeout."""
    log = Path(log)
    log.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    last_notice = started
    process = None
    with log.open('w', encoding='utf8') as output:
        try:
            process = subprocess.Popen(list(map(str, command)), stdout=output, stderr=subprocess.STDOUT, env=env or check_offline_env(), start_new_session=new_session, creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0))
            process._vtm_owns_group = new_session
            while process.poll() is None:
                elapsed = time.monotonic() - started
                if elapsed > timeout:
                    check_stop(process)
                    return dict(status='timeout', seconds=round(elapsed, 3), log=str(log), tail=check_tail(log))
                if time.monotonic() - last_notice >= 15:
                    phase = 'running'
                    if report_path and Path(report_path).is_file():
                        try:
                            phase = json.loads(Path(report_path).read_text(encoding='utf8')).get('phase', phase)
                        except (OSError, ValueError):
                            pass
                    tail = check_tail(log, 1000).splitlines()
                    native = next((line for line in reversed(tail) if line.startswith('FlashAttention build:')), '')
                    print(f'UniHIR check: {phase}, {elapsed:.0f}s; log: {log}' + (' | ' + native if native else ''), flush=True)
                    last_notice = time.monotonic()
                time.sleep(0.2)
            return dict(status='ok' if process.returncode == 0 else 'failed', returncode=process.returncode, seconds=round(time.monotonic() - started, 3), log=str(log), tail=check_tail(log))
        except BaseException:
            if process is not None:
                check_stop(process)
            raise


def check_payload(payload):
    raw=zlib.decompress(base64.b85decode(payload['data']))
    if hashlib.sha256(raw).hexdigest()!=payload['sha256']:raise ValueError('Notebook 1 definitions checksum mismatch')
    value=json.loads(raw)
    if value.get('format')!='vtm-notebook1-restoration-check-v1':raise ValueError('Unexpected definition bundle')
    return value


def check_native_receipt(python):
    folder=Path(python).parent.parent
    data=json.loads((folder/'install.json').read_text(encoding='utf8'))
    verified=[]
    for record in data.get('native',[]):
        wheel=folder/'native_build/wheels'/record['wheel']
        if not wheel.is_file() or check_hash(wheel)!=record['sha256']:
            raise ValueError('Compiled wheel missing or changed: '+str(wheel))
        verified.append(dict(record,path=str(wheel)))
    if not verified:raise ValueError('No compiled FlashAttention wheel receipt')
    return dict(status='built_or_reused_verified_wheel',native_wheels=verified,
                note='The separate CUDA probe executes the currently imported extension.')


def check_collect_logs(runtime, output):
    """Export logs/receipts only; never copy runtime binaries or model weights."""
    runtime,output=Path(runtime),Path(output)
    if not runtime.is_dir():return
    spec=output/'spec.json'
    cutoff=spec.stat().st_mtime if spec.is_file() else 0
    active_native=[]
    try:
        current=json.loads((output/'report.json').read_text(encoding='utf8'))
        active_native=[Path(r['path']).parent.parent for r in
            current.get('phases',{}).get('runtime_build',{}).get('receipt',{}).get('native_wheels',[])]
    except (OSError,ValueError):pass
    for pattern in ('build.log','build_config.json','build_progress.json','build.ninja','.ninja_log',
                    'install.json','preflight.json','worker.log','ready.json','response.json'):
        sources=sorted(runtime.rglob(pattern),key=lambda p:p.stat().st_mtime,reverse=True)[:32]
        for source in sources:
            if source.is_symlink() or source.stat().st_size>10*1024**2:continue
            if source.stat().st_mtime<cutoff and not (pattern in ('build.log','build_config.json','build_progress.json','build.ninja','.ninja_log','install.json') and
                    any(source.is_relative_to(p.parent) for p in active_native)):continue
            if pattern in ('build.ninja','.ninja_log'):
                short=hashlib.sha256(str(source.relative_to(runtime)).encode()).hexdigest()[:12]
                target=output/'runtime_logs/native_debug'/(short+'_'+pattern.lstrip('.')+'.txt')
            else:
                target=output/'runtime_logs'/source.relative_to(runtime)
            target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,target)


def check_inference_evidence(summary, validation):
    metrics=[m for m in summary.get('measurements',[]) if m.get('variant')=='unihir_restoration']
    if summary.get('failures') or not metrics or summary.get('variants_reused'):
        raise ValueError('Fresh, successful UniHIR inference was not established')
    for item in metrics:
        worker=item.get('worker',{})
        if worker.get('state')!='ok' or worker.get('peak_vram_bytes',0)<=0:
            raise ValueError('Missing successful GPU worker evidence')
    if not validation.get('restoration',{}).get('validated'):
        raise ValueError('Output manifest/pixels/geometry validation incomplete')
    return metrics


@contextmanager
def check_shared_definitions(sources):
    """Execute notebook definitions in a registered module, including dataclasses."""
    name = 'vtm_notebook1_check_definitions_' + uuid.uuid4().hex
    module = ModuleType(name)
    sys.modules[name] = module
    try:
        for index, source in enumerate(sources):
            exec(compile(source, f'notebook1-shared:{index}', 'exec'), module.__dict__)
        yield module.__dict__
    finally:
        if sys.modules.get(name) is module:
            del sys.modules[name]


def restoration_check_child(spec_path):
    """Executed only by the Kaggle job supervisor, never by notebook import."""
    spec=json.loads(Path(spec_path).read_text(encoding='utf8'))
    output=Path(spec['output']);runtime=Path(spec['runtime']);report_path=output/'report.json'
    report=dict(format='vtm-kaggle-restoration-gpu-check',version=1,status='running',phase='hardware',
        mode=spec['mode'],native_build_verified=False,flash_attention_cuda_verified=False,
        model_weights_loaded=False,model_inference_attempted=False,model_inference_verified=False,
        offline_requested=True,parameters=spec['options'],phases={},pid=os.getpid())
    def phase(phase_name,**details):
        report['phase']=phase_name;report['phases'][phase_name]=dict(status='running',**details)
        check_json(report_path,report);print('STAGE:',phase_name,flush=True)
    def passed(phase_name,**details):
        report['phases'][phase_name]={**details,'status':'passed'};check_json(report_path,report)
    def guard(event,args):
        if event=='socket.connect' and isinstance(args[1],str):return
        if event in ('socket.connect','socket.getaddrinfo'):raise RuntimeError('Offline restoration check blocks network access')
    sys.addaudithook(guard)
    shared_modules = ExitStack()
    try:
        if sys.platform!='linux' or not Path('/kaggle/working').is_dir():
            raise RuntimeError('This GPU/build job is Kaggle-only')
        import torch
        if not torch.cuda.is_available():raise RuntimeError('CUDA unavailable')
        devices=[dict(index=i,name=torch.cuda.get_device_name(i),vram_bytes=torch.cuda.get_device_properties(i).total_memory,
            capability=list(torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())]
        gpu=dict(name=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory,
            capability=list(torch.cuda.get_device_capability(0)),torch=torch.__version__,cuda=torch.version.cuda,
            python=sys.version,devices=devices,gpu_count=len(devices))
        report['hardware']=gpu
        if any(d['capability'][0]<8 for d in devices):
            raise RuntimeError('FlashAttention 2 CUDA/BF16 check needs a supported Ampere-or-newer GPU')
        sharded=spec['options'].get('placement','single')=='sharded'
        if spec['mode']=='unihir_smoke':
            if sharded:
                if len(devices)<2 or sum(d['vram_bytes'] for d in devices)<48*1024**3:
                    raise RuntimeError('Sharded UniHIR needs multiple GPUs with sufficient per-device headroom; no build/model load started')
            elif gpu['vram_bytes']<75*1024**3:
                raise RuntimeError('Single-GPU UniHIR needs >=75 GiB; select placement=sharded for 4 L4')
        if spec['options'].get('mode') not in (None,spec['mode']):raise ValueError('Conflicting check mode')
        passed('hardware',**gpu)
        phase('load_shared_definitions')
        payload=check_payload(spec['payload'])
        scope=shared_modules.enter_context(check_shared_definitions(payload['sources']))
        passed('load_shared_definitions',sha256=spec['payload']['sha256'],upstream=payload['upstream'])
        options=spec['options'];packages=Path(spec['packages']);models=Path(spec['models'])
        if not spec['packages'] or not packages.is_dir():raise FileNotFoundError('Configure a valid packages dataset root')
        if spec['mode']=='unihir_smoke' and (not spec['models'] or not models.is_dir()):
            raise FileNotFoundError('Configure a valid models dataset root')
        cfg=dict(packages_dir=str(packages),models_dir=str(models),data_dir=spec.get('data',''),
            gpu_ids=list(range(len(devices))),execution_mode='sequential',
            output_dir=str(output/'image_dataset'),books=options.get('books',[]),pages=options.get('pages','1'),
            limit=1,max_books=1,variants=['original','unihir_restoration'],crop_types=['page'],region_ids=[],
            resume=False,allow_partial=False,max_source_pixels=options.get('max_source_pixels',16_000_000),
            max_output_pixels=options.get('max_source_pixels',16_000_000),max_storage_bytes=1024**3,
            timeout_seconds=options.get('inference_timeout_seconds',900),
            load_timeout_seconds=options.get('load_timeout_seconds',900),batch_size=1,deskew=False,dewarp=False,
            layout={'dpi':200,'reading_direction':'vertical_rtl'},book_layouts={
                '1':{'dpi':225},'2':{'dpi':226},'3':{'dpi':226},'4':{'dpi':181},'5':{'dpi':181},
                '6':{'dpi':226},'bien':{'dpi':200,'reading_direction':'horizontal_ltr'},'kinh':{'dpi':45}},
            variant_options={},runtime_pythons={},reviewed_regions={},kernel_torch_source_policy='warn',
            model_options={'tadisr_x4':{'tile':768,'overlap':256},'difftsr':{'steps':20},
                'unihir_restoration':dict(device='cuda',seed=42,steps=options.get('steps',10),
                                        ref_count=options.get('ref_count',1),
                                        placement=options.get('placement','single'),gpu_ids=list(range(len(devices))),
                                        max_memory=options.get('max_memory','18GiB' if sharded else '80GiB'),
                                        activation_reserve_gib=options.get('activation_reserve_gib',4),
                                        entry_reserve_gib=options.get('entry_reserve_gib',2))})
        cfg=scope['prep_normalize'](cfg)
        original_assets=scope['prep_assets'];original_overlay=scope['prep_overlay']
        def observed_assets(*args,**kwargs):
            phase('model_asset_integrity');result=original_assets(*args,**kwargs)
            passed('model_asset_integrity',model='unihir_restoration',identity=result[1]);return result
        def observed_overlay(*args,**kwargs):
            phase('runtime_build',note='Offline compile can take a long time; build.log is preserved on failure')
            python=original_overlay(*args,**kwargs);native=check_native_receipt(python)
            report['native_build_verified']=True;passed('runtime_build',receipt=native)
            phase('flash_attention_cuda')
            result=check_logged_process([python,'-u','-c',FLASH_GPU_PROBE],output/'flash_attention_probe.log',
                                       timeout=120,env=scope['prep_worker_environment'](),new_session=False)
            if result['status']!='ok':raise RuntimeError('FlashAttention CUDA probe failed: '+result['tail'])
            data=json.loads(next(line for line in reversed(Path(result['log']).read_text(encoding='utf8').splitlines()) if line.startswith('{')))
            report['flash_attention_cuda_verified']=True;passed('flash_attention_cuda',result=data);return python
        scope['prep_assets']=observed_assets;scope['prep_overlay']=observed_overlay
        if spec['mode']=='build_and_probe':
            # Model preflight also enforces UniHIR's full-model VRAM requirement.
            # A standalone extension check needs neither model weights nor that worker.
            observed_overlay(packages,runtime,'unihir_restoration',cfg)
            report['status']='native_and_cuda_verified';return 0
        base_worker=scope['PrepWorker']
        class PreflightWorker(base_worker):
            def __init__(self,*args,**kwargs):
                phase('worker_preflight');super().__init__(*args,**kwargs)
                passed('worker_preflight',probe=self.ready.get('report'))
        scope['PrepWorker']=PreflightWorker
        preflight=scope['prep_preflight'](cfg,runtime)
        report['preflight']=preflight
        if preflight['unihir_restoration'].get('state')!='ready':raise RuntimeError('UniHIR worker preflight is not ready')
        passed('worker_preflight',state='ready')
        if not cfg['data_dir'] or not Path(cfg['data_dir']).is_dir():
            raise FileNotFoundError('Build/CUDA passed, but DATA_DIR must contain a PDF for actual inference')
        phase('pdf_bootstrap')
        scope['prep_bootstrap'](packages,runtime)
        passed('pdf_bootstrap')
        # Reuse this exact configuration's verified preflight in the same job.
        scope['prep_preflight']=lambda *args,**kwargs:preflight
        original_worker=base_worker
        class ObservedWorker(original_worker):
            def __init__(self,*args,**kwargs):
                phase('model_loading');super().__init__(*args,**kwargs)
                report['model_weights_loaded']=True;passed('model_loading',seconds=self.ready.get('load_seconds'))
            def infer(self,*args,**kwargs):
                report['model_inference_attempted']=True;phase('model_inference')
                result=super().infer(*args,**kwargs)
                passed('model_inference',note='Output pixels and manifest are validated next')
                return result
        scope['PrepWorker']=ObservedWorker
        phase('page_preparation')
        summary=scope['prep_run'](cfg,runtime)
        phase('output_validation');validation=scope['prep_validate'](cfg['output_dir'])
        metrics=check_inference_evidence(summary,validation)
        scope['prep_export_images'](cfg['output_dir'],output/'images')
        report.update(status='passed',model_inference_verified=True,inference_metrics=metrics,
            output_validation=validation,summary={k:v for k,v in summary.items() if k!='measurements'},
            fidelity='unverified; a successful run does not establish correct restored characters')
        passed('output_validation',status_detail='pixels,manifest,geometry validated')
        return 0
    except BaseException as exc:
        report.update(status='failed',failed_phase=report['phase'],error=str(exc),traceback=traceback.format_exc())
        print(report['traceback'],flush=True);return 1
    finally:
        shared_modules.close()
        report['finished_utc']=datetime.now(timezone.utc).isoformat();check_json(report_path,report)
        check_collect_logs(runtime,output)


def run_restoration_gpu_check(environment, *, mode, options, payload, output_dir):
    if mode not in ('off', 'build_and_probe', 'unihir_smoke'):
        raise ValueError('Unknown restoration check mode')
    if mode == 'off':
        return dict(status='disabled', model_inference_verified=False)
    if sys.platform != 'linux' or not Path('/kaggle/working').is_dir():
        return dict(status='skipped_not_kaggle', model_inference_verified=False, detail='No model build, install or inference is permitted on local Windows')
    output = Path(output_dir).resolve()
    if output.is_relative_to(Path('/kaggle/input')):
        raise ValueError('Output must be outside Kaggle Input')
    run = output / ('restoration_check_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8])
    run.mkdir(parents=True)
    roots = environment.get('asset_roots', {})
    spec = dict(mode=mode, options=options, payload=payload, output=str(run), runtime=options.get('runtime_dir', '/tmp/vtm_image_runtime'), packages=roots.get('packages', {}).get('path'), models=roots.get('models', {}).get('path'), data=roots.get('pdfs', {}).get('path'))
    driver = run / 'check_driver.py'
    driver.write_text(GPU_CHECK_DRIVER_SOURCE, encoding='utf8')
    check_json(run / 'spec.json', spec)
    checkpoint = run / 'report.json'
    try:
        env = check_offline_env()
        if options.get('gpu_ids') is not None:
            ids = options['gpu_ids']
            inherited = env.get('CUDA_VISIBLE_DEVICES')
            visible = [v.strip() for v in inherited.split(',') if v.strip() and v.strip() != '-1'] if inherited is not None else None
            if not ids or any((type(i) is not int or i < 0 or (visible is not None and i >= len(visible)) for i in ids)) or len(set(ids)) != len(ids):
                raise ValueError('Invalid restoration check GPU IDs under inherited CUDA_VISIBLE_DEVICES')
            env['CUDA_VISIBLE_DEVICES'] = ','.join((visible[i] if visible is not None else str(i) for i in ids))
        process = check_logged_process([sys.executable, '-u', str(driver), str(run / 'spec.json')], run / 'driver.log', timeout=options.get('total_timeout_seconds', 5400), report_path=checkpoint, env=env)
    except BaseException as exc:
        process = dict(status='interrupted', error=str(exc), log=str(run / 'driver.log'))
    result = json.loads(checkpoint.read_text(encoding='utf8')) if checkpoint.is_file() else dict(status='failed', error='Child report unavailable')
    if process['status'] in ('timeout', 'interrupted') or (process['status'] != 'ok' and result.get('status') in ('passed', 'native_and_cuda_verified')):
        result.update(status=process['status'], failed_phase=result.get('phase'), model_inference_verified=False)
    result.update(supervisor=process, output_dir=str(run))
    check_collect_logs(spec['runtime'], run)
    check_json(checkpoint, result)
    return result


def package_gpu_check_logs(report, output_dir):
    """Small review artifact. Private runtime, weights, archives and spec payload are excluded."""
    output=Path(output_dir).resolve();gpu=report.get('restoration_gpu_check',{})
    run=Path(gpu['output_dir']).resolve() if gpu.get('output_dir') else None
    archive=output/((run.name if run else 'kaggle_environment')+'_logs.zip')
    temporary=archive.with_name(archive.name+'.tmp')
    selected=[p for p in (output/'kaggle_environment_report.json',output/'kaggle_environment_report.txt') if p.is_file()]
    if run:
        if not run.is_relative_to(output):raise ValueError('Log directory outside diagnostic output')
        selected.extend(p for p in run.rglob('*') if p.is_file() and not p.is_symlink() and
            p.name not in ('spec.json','check_driver.py') and p.suffix in ('.json','.log','.txt','.png') and
            'image_dataset' not in p.relative_to(run).parts and p.stat().st_size<=10*1024**2)
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as zipped:
        for path in sorted(set(selected)):
            zipped.write(path,path.relative_to(output).as_posix())
    temporary.replace(archive)
    return archive


def make_restoration_check_payload(notebook_path):
    """Authoring only: select original notebook 1 definitions, without executing them."""
    notebook=json.loads(Path(notebook_path).read_text(encoding='utf8'))
    vendor=''.join(notebook['cells'][6]['source'])
    assignments={n.targets[0].id:n.value for n in ast.parse(vendor).body
        if isinstance(n,ast.Assign) and isinstance(n.targets[0],ast.Name)}
    raw=zlib.decompress(base64.b85decode(ast.literal_eval(assignments['_VENDOR_SOURCES_B85'])))
    if hashlib.sha256(raw).hexdigest()!=ast.literal_eval(assignments['_VENDOR_SOURCES_SHA256']):
        raise ValueError('Original upstream payload checksum mismatch')
    vendors=json.loads(raw)
    revision=ast.literal_eval(assignments['UPSTREAM_REVISIONS'])['UniHIR']
    worker=ast.literal_eval(assignments['WORKER_SOURCE'])
    mini='\n'.join(('UPSTREAM_REVISIONS = '+repr({'UniHIR':revision}),
        'VENDOR_SOURCES = '+repr({'UniHIR':vendors['UniHIR']}),
        'WORKER_SOURCE = '+repr(worker),
        'RESTORATION_CODE_SIGNATURE = '+repr(ast.literal_eval(assignments['RESTORATION_CODE_SIGNATURE']))))
    sources=[''.join(notebook['cells'][4]['source']),mini,''.join(notebook['cells'][8]['source'])]
    for i,source in enumerate(sources):compile(source,f'notebook1-shared:{i}','exec')
    body=dict(format='vtm-notebook1-restoration-check-v1',sources=sources,
              upstream={'UniHIR':revision},worker_sha256=hashlib.sha256(worker.encode()).hexdigest())
    raw=json.dumps(body,ensure_ascii=False,separators=(',',':')).encode('utf8')
    return dict(sha256=hashlib.sha256(raw).hexdigest(),data=base64.b85encode(zlib.compress(raw,9)).decode('ascii'))



GPU_CHECK_DRIVER_SOURCE = '"""Kaggle-only restoration check using notebook 1\'s exact offline implementation.\n\nInventory remains read-only. The separately enabled GPU check may create a\nprivate runtime, compile FlashAttention and run one UniHIR page. No downloads,\nOCR, training, kernel package replacement, or corpus processing.\n"""\nfrom __future__ import annotations\n\nimport ast\nimport base64\nfrom contextlib import contextmanager, ExitStack\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport time\nimport traceback\nfrom types import ModuleType\nimport uuid\nimport zipfile\nimport zlib\n\n\nFLASH_GPU_PROBE = \'\'\'import json, torch, flash_attn\nfrom flash_attn import flash_attn_func, flash_attn_varlen_func\nassert torch.cuda.is_available(), \'CUDA unavailable\'\ntorch.manual_seed(42)\ndevices=[]\nfor index in range(torch.cuda.device_count()):\n    with torch.cuda.device(index):\n        q,k,v=[torch.randn(1,16,2,64,device=f\'cuda:{index}\',dtype=torch.bfloat16) for _ in range(3)]\n        y=flash_attn_func(q,k,v,dropout_p=0.,causal=False)\n        reference=torch.nn.functional.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),dropout_p=0.).transpose(1,2)\n        cu=torch.tensor([0,16],device=f\'cuda:{index}\',dtype=torch.int32)\n        variable=flash_attn_varlen_func(q[0],k[0],v[0],cu,cu,16,16,dropout_p=0.,causal=False)\n        torch.cuda.synchronize(index)\n        assert y.shape==q.shape and variable.shape==q[0].shape\n        assert bool(torch.isfinite(y).all()) and bool(torch.isfinite(variable).all())\n        assert torch.allclose(y.float(),reference.float(),atol=.03,rtol=.03), \'FlashAttention differs from SDPA reference\'\n        assert torch.allclose(variable.float(),y[0].float(),atol=.03,rtol=.03), \'Variable-length kernel differs\'\n        devices.append(dict(index=index,name=torch.cuda.get_device_name(index),\n            max_abs_difference=float((y.float()-reference.float()).abs().max().item())))\nprint(json.dumps(dict(status=\'passed\',torch=torch.__version__,cuda=torch.version.cuda,\n    devices=devices,capability=list(torch.cuda.get_device_capability(0)),\n    flash_attention=flash_attn.__version__,module_file=flash_attn.__file__,\n    fixed_and_variable_length_verified=True,finite=True,\n    weights_loaded=False)))\n\'\'\'\n\n\ndef check_json(path, value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    temporary=path.with_name(path.name+\'.tmp\')\n    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf8\')\n    temporary.replace(path)\n\n\ndef check_hash(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda:stream.read(8*1024**2),b\'\'):h.update(block)\n    return h.hexdigest()\n\n\ndef check_offline_env():\n    env=dict(os.environ)\n    for key in (\'PYTHONPATH\',\'PYTHONHOME\',\'PYTHONUSERBASE\'):env.pop(key,None)\n    env.update(PYTHONNOUSERSITE=\'1\',PYTHONUTF8=\'1\',PIP_NO_INDEX=\'1\',PIP_DISABLE_PIP_VERSION_CHECK=\'1\',\n        HF_HUB_OFFLINE=\'1\',TRANSFORMERS_OFFLINE=\'1\',HF_DATASETS_OFFLINE=\'1\',DIFFUSERS_OFFLINE=\'1\',\n        HF_HUB_DISABLE_TELEMETRY=\'1\',MPLBACKEND=\'Agg\',TOKENIZERS_PARALLELISM=\'false\')\n    return env\n\n\ndef check_tail(path, maximum=2000):\n    try:\n        with Path(path).open(\'rb\') as stream:\n            stream.seek(0,2);stream.seek(max(0,stream.tell()-maximum))\n            return stream.read().decode(\'utf8\',errors=\'replace\')\n    except OSError:return \'\'\n\n\ndef check_stop(process):\n    if process.poll() is not None:return\n    if os.name==\'nt\' or not getattr(process,\'_vtm_owns_group\',False):process.kill()\n    else:\n        try:os.killpg(process.pid,signal.SIGKILL)\n        except ProcessLookupError:pass\n    process.wait()\n\n\ndef check_logged_process(command, log, *, timeout, env=None, report_path=None, new_session=True):\n    """One process group; preserve bounded log tails and kill descendants on timeout."""\n    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)\n    started=time.monotonic();last_notice=started;process=None\n    with log.open(\'w\',encoding=\'utf8\') as output:\n        try:\n            process=subprocess.Popen(list(map(str,command)),stdout=output,stderr=subprocess.STDOUT,\n                env=env or check_offline_env(),start_new_session=os.name!=\'nt\' and new_session,\n                creationflags=getattr(subprocess,\'CREATE_NO_WINDOW\',0))\n            process._vtm_owns_group=os.name!=\'nt\' and new_session\n            while process.poll() is None:\n                elapsed=time.monotonic()-started\n                if elapsed>timeout:\n                    check_stop(process)\n                    return dict(status=\'timeout\',seconds=round(elapsed,3),log=str(log),tail=check_tail(log))\n                if time.monotonic()-last_notice>=15:\n                    phase=\'running\'\n                    if report_path and Path(report_path).is_file():\n                        try:phase=json.loads(Path(report_path).read_text(encoding=\'utf8\')).get(\'phase\',phase)\n                        except (OSError,ValueError):pass\n                    tail=check_tail(log,1000).splitlines()\n                    native=next((line for line in reversed(tail) if line.startswith(\'FlashAttention build:\')),\'\')\n                    print(f\'UniHIR check: {phase}, {elapsed:.0f}s; log: {log}\'+(\' | \'+native if native else \'\'),flush=True)\n                    last_notice=time.monotonic()\n                time.sleep(.2)\n            return dict(status=\'ok\' if process.returncode==0 else \'failed\',returncode=process.returncode,\n                        seconds=round(time.monotonic()-started,3),log=str(log),tail=check_tail(log))\n        except BaseException:\n            if process is not None:check_stop(process)\n            raise\n\n\ndef check_payload(payload):\n    raw=zlib.decompress(base64.b85decode(payload[\'data\']))\n    if hashlib.sha256(raw).hexdigest()!=payload[\'sha256\']:raise ValueError(\'Notebook 1 definitions checksum mismatch\')\n    value=json.loads(raw)\n    if value.get(\'format\')!=\'vtm-notebook1-restoration-check-v1\':raise ValueError(\'Unexpected definition bundle\')\n    return value\n\n\ndef check_native_receipt(python):\n    folder=Path(python).parent.parent\n    data=json.loads((folder/\'install.json\').read_text(encoding=\'utf8\'))\n    verified=[]\n    for record in data.get(\'native\',[]):\n        wheel=folder/\'native_build/wheels\'/record[\'wheel\']\n        if not wheel.is_file() or check_hash(wheel)!=record[\'sha256\']:\n            raise ValueError(\'Compiled wheel missing or changed: \'+str(wheel))\n        verified.append(dict(record,path=str(wheel)))\n    if not verified:raise ValueError(\'No compiled FlashAttention wheel receipt\')\n    return dict(status=\'built_or_reused_verified_wheel\',native_wheels=verified,\n                note=\'The separate CUDA probe executes the currently imported extension.\')\n\n\ndef check_collect_logs(runtime, output):\n    """Export logs/receipts only; never copy runtime binaries or model weights."""\n    runtime,output=Path(runtime),Path(output)\n    if not runtime.is_dir():return\n    spec=output/\'spec.json\'\n    cutoff=spec.stat().st_mtime if spec.is_file() else 0\n    active_native=[]\n    try:\n        current=json.loads((output/\'report.json\').read_text(encoding=\'utf8\'))\n        active_native=[Path(r[\'path\']).parent.parent for r in\n            current.get(\'phases\',{}).get(\'runtime_build\',{}).get(\'receipt\',{}).get(\'native_wheels\',[])]\n    except (OSError,ValueError):pass\n    for pattern in (\'build.log\',\'build_config.json\',\'build_progress.json\',\'build.ninja\',\'.ninja_log\',\n                    \'install.json\',\'preflight.json\',\'worker.log\',\'ready.json\',\'response.json\'):\n        sources=sorted(runtime.rglob(pattern),key=lambda p:p.stat().st_mtime,reverse=True)[:32]\n        for source in sources:\n            if source.is_symlink() or source.stat().st_size>10*1024**2:continue\n            if source.stat().st_mtime<cutoff and not (pattern in (\'build.log\',\'build_config.json\',\'build_progress.json\',\'build.ninja\',\'.ninja_log\',\'install.json\') and\n                    any(source.is_relative_to(p.parent) for p in active_native)):continue\n            if pattern in (\'build.ninja\',\'.ninja_log\'):\n                short=hashlib.sha256(str(source.relative_to(runtime)).encode()).hexdigest()[:12]\n                target=output/\'runtime_logs/native_debug\'/(short+\'_\'+pattern.lstrip(\'.\')+\'.txt\')\n            else:\n                target=output/\'runtime_logs\'/source.relative_to(runtime)\n            target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,target)\n\n\ndef check_inference_evidence(summary, validation):\n    metrics=[m for m in summary.get(\'measurements\',[]) if m.get(\'variant\')==\'unihir_restoration\']\n    if summary.get(\'failures\') or not metrics or summary.get(\'variants_reused\'):\n        raise ValueError(\'Fresh, successful UniHIR inference was not established\')\n    for item in metrics:\n        worker=item.get(\'worker\',{})\n        if worker.get(\'state\')!=\'ok\' or worker.get(\'peak_vram_bytes\',0)<=0:\n            raise ValueError(\'Missing successful GPU worker evidence\')\n    if not validation.get(\'restoration\',{}).get(\'validated\'):\n        raise ValueError(\'Output manifest/pixels/geometry validation incomplete\')\n    return metrics\n\n\n@contextmanager\ndef check_shared_definitions(sources):\n    """Execute notebook definitions in a registered module, including dataclasses."""\n    name = \'vtm_notebook1_check_definitions_\' + uuid.uuid4().hex\n    module = ModuleType(name)\n    sys.modules[name] = module\n    try:\n        for index, source in enumerate(sources):\n            exec(compile(source, f\'notebook1-shared:{index}\', \'exec\'), module.__dict__)\n        yield module.__dict__\n    finally:\n        if sys.modules.get(name) is module:\n            del sys.modules[name]\n\n\ndef restoration_check_child(spec_path):\n    """Executed only by the Kaggle job supervisor, never by notebook import."""\n    spec=json.loads(Path(spec_path).read_text(encoding=\'utf8\'))\n    output=Path(spec[\'output\']);runtime=Path(spec[\'runtime\']);report_path=output/\'report.json\'\n    report=dict(format=\'vtm-kaggle-restoration-gpu-check\',version=1,status=\'running\',phase=\'hardware\',\n        mode=spec[\'mode\'],native_build_verified=False,flash_attention_cuda_verified=False,\n        model_weights_loaded=False,model_inference_attempted=False,model_inference_verified=False,\n        offline_requested=True,parameters=spec[\'options\'],phases={},pid=os.getpid())\n    def phase(phase_name,**details):\n        report[\'phase\']=phase_name;report[\'phases\'][phase_name]=dict(status=\'running\',**details)\n        check_json(report_path,report);print(\'STAGE:\',phase_name,flush=True)\n    def passed(phase_name,**details):\n        report[\'phases\'][phase_name]={**details,\'status\':\'passed\'};check_json(report_path,report)\n    def guard(event,args):\n        if event==\'socket.connect\' and isinstance(args[1],str):return\n        if event in (\'socket.connect\',\'socket.getaddrinfo\'):raise RuntimeError(\'Offline restoration check blocks network access\')\n    sys.addaudithook(guard)\n    shared_modules = ExitStack()\n    try:\n        if sys.platform!=\'linux\' or not Path(\'/kaggle/working\').is_dir():\n            raise RuntimeError(\'This GPU/build job is Kaggle-only\')\n        import torch\n        if not torch.cuda.is_available():raise RuntimeError(\'CUDA unavailable\')\n        devices=[dict(index=i,name=torch.cuda.get_device_name(i),vram_bytes=torch.cuda.get_device_properties(i).total_memory,\n            capability=list(torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())]\n        gpu=dict(name=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory,\n            capability=list(torch.cuda.get_device_capability(0)),torch=torch.__version__,cuda=torch.version.cuda,\n            python=sys.version,devices=devices,gpu_count=len(devices))\n        report[\'hardware\']=gpu\n        if any(d[\'capability\'][0]<8 for d in devices):\n            raise RuntimeError(\'FlashAttention 2 CUDA/BF16 check needs a supported Ampere-or-newer GPU\')\n        sharded=spec[\'options\'].get(\'placement\',\'single\')==\'sharded\'\n        if spec[\'mode\']==\'unihir_smoke\':\n            if sharded:\n                if len(devices)<2 or sum(d[\'vram_bytes\'] for d in devices)<48*1024**3:\n                    raise RuntimeError(\'Sharded UniHIR needs multiple GPUs with sufficient per-device headroom; no build/model load started\')\n            elif gpu[\'vram_bytes\']<75*1024**3:\n                raise RuntimeError(\'Single-GPU UniHIR needs >=75 GiB; select placement=sharded for 4 L4\')\n        if spec[\'options\'].get(\'mode\') not in (None,spec[\'mode\']):raise ValueError(\'Conflicting check mode\')\n        passed(\'hardware\',**gpu)\n        phase(\'load_shared_definitions\')\n        payload=check_payload(spec[\'payload\'])\n        scope=shared_modules.enter_context(check_shared_definitions(payload[\'sources\']))\n        passed(\'load_shared_definitions\',sha256=spec[\'payload\'][\'sha256\'],upstream=payload[\'upstream\'])\n        options=spec[\'options\'];packages=Path(spec[\'packages\']);models=Path(spec[\'models\'])\n        if not spec[\'packages\'] or not packages.is_dir():raise FileNotFoundError(\'Configure a valid packages dataset root\')\n        if spec[\'mode\']==\'unihir_smoke\' and (not spec[\'models\'] or not models.is_dir()):\n            raise FileNotFoundError(\'Configure a valid models dataset root\')\n        cfg=dict(packages_dir=str(packages),models_dir=str(models),data_dir=spec.get(\'data\',\'\'),\n            gpu_ids=list(range(len(devices))),execution_mode=\'sequential\',\n            output_dir=str(output/\'image_dataset\'),books=options.get(\'books\',[]),pages=options.get(\'pages\',\'1\'),\n            limit=1,max_books=1,variants=[\'original\',\'unihir_restoration\'],crop_types=[\'page\'],region_ids=[],\n            resume=False,allow_partial=False,max_source_pixels=options.get(\'max_source_pixels\',16_000_000),\n            max_output_pixels=options.get(\'max_source_pixels\',16_000_000),max_storage_bytes=1024**3,\n            timeout_seconds=options.get(\'inference_timeout_seconds\',900),\n            load_timeout_seconds=options.get(\'load_timeout_seconds\',900),batch_size=1,deskew=False,dewarp=False,\n            layout={\'dpi\':200,\'reading_direction\':\'vertical_rtl\'},book_layouts={\n                \'1\':{\'dpi\':225},\'2\':{\'dpi\':226},\'3\':{\'dpi\':226},\'4\':{\'dpi\':181},\'5\':{\'dpi\':181},\n                \'6\':{\'dpi\':226},\'bien\':{\'dpi\':200,\'reading_direction\':\'horizontal_ltr\'},\'kinh\':{\'dpi\':45}},\n            variant_options={},runtime_pythons={},reviewed_regions={},kernel_torch_source_policy=\'warn\',\n            model_options={\'tadisr_x4\':{\'tile\':768,\'overlap\':256},\'difftsr\':{\'steps\':20},\n                \'unihir_restoration\':dict(device=\'cuda\',seed=42,steps=options.get(\'steps\',10),\n                                        ref_count=options.get(\'ref_count\',1),\n                                        placement=options.get(\'placement\',\'single\'),gpu_ids=list(range(len(devices))),\n                                        max_memory=options.get(\'max_memory\',\'18GiB\' if sharded else \'80GiB\'),\n                                        activation_reserve_gib=options.get(\'activation_reserve_gib\',4),\n                                        entry_reserve_gib=options.get(\'entry_reserve_gib\',2))})\n        cfg=scope[\'prep_normalize\'](cfg)\n        original_assets=scope[\'prep_assets\'];original_overlay=scope[\'prep_overlay\']\n        def observed_assets(*args,**kwargs):\n            phase(\'model_asset_integrity\');result=original_assets(*args,**kwargs)\n            passed(\'model_asset_integrity\',model=\'unihir_restoration\',identity=result[1]);return result\n        def observed_overlay(*args,**kwargs):\n            phase(\'runtime_build\',note=\'Offline compile can take a long time; build.log is preserved on failure\')\n            python=original_overlay(*args,**kwargs);native=check_native_receipt(python)\n            report[\'native_build_verified\']=True;passed(\'runtime_build\',receipt=native)\n            phase(\'flash_attention_cuda\')\n            result=check_logged_process([python,\'-u\',\'-c\',FLASH_GPU_PROBE],output/\'flash_attention_probe.log\',\n                                       timeout=120,env=scope[\'prep_worker_environment\'](),new_session=False)\n            if result[\'status\']!=\'ok\':raise RuntimeError(\'FlashAttention CUDA probe failed: \'+result[\'tail\'])\n            data=json.loads(next(line for line in reversed(Path(result[\'log\']).read_text(encoding=\'utf8\').splitlines()) if line.startswith(\'{\')))\n            report[\'flash_attention_cuda_verified\']=True;passed(\'flash_attention_cuda\',result=data);return python\n        scope[\'prep_assets\']=observed_assets;scope[\'prep_overlay\']=observed_overlay\n        if spec[\'mode\']==\'build_and_probe\':\n            # Model preflight also enforces UniHIR\'s full-model VRAM requirement.\n            # A standalone extension check needs neither model weights nor that worker.\n            observed_overlay(packages,runtime,\'unihir_restoration\',cfg)\n            report[\'status\']=\'native_and_cuda_verified\';return 0\n        base_worker=scope[\'PrepWorker\']\n        class PreflightWorker(base_worker):\n            def __init__(self,*args,**kwargs):\n                phase(\'worker_preflight\');super().__init__(*args,**kwargs)\n                passed(\'worker_preflight\',probe=self.ready.get(\'report\'))\n        scope[\'PrepWorker\']=PreflightWorker\n        preflight=scope[\'prep_preflight\'](cfg,runtime)\n        report[\'preflight\']=preflight\n        if preflight[\'unihir_restoration\'].get(\'state\')!=\'ready\':raise RuntimeError(\'UniHIR worker preflight is not ready\')\n        passed(\'worker_preflight\',state=\'ready\')\n        if not cfg[\'data_dir\'] or not Path(cfg[\'data_dir\']).is_dir():\n            raise FileNotFoundError(\'Build/CUDA passed, but DATA_DIR must contain a PDF for actual inference\')\n        phase(\'pdf_bootstrap\')\n        scope[\'prep_bootstrap\'](packages,runtime)\n        passed(\'pdf_bootstrap\')\n        # Reuse this exact configuration\'s verified preflight in the same job.\n        scope[\'prep_preflight\']=lambda *args,**kwargs:preflight\n        original_worker=base_worker\n        class ObservedWorker(original_worker):\n            def __init__(self,*args,**kwargs):\n                phase(\'model_loading\');super().__init__(*args,**kwargs)\n                report[\'model_weights_loaded\']=True;passed(\'model_loading\',seconds=self.ready.get(\'load_seconds\'))\n            def infer(self,*args,**kwargs):\n                report[\'model_inference_attempted\']=True;phase(\'model_inference\')\n                result=super().infer(*args,**kwargs)\n                passed(\'model_inference\',note=\'Output pixels and manifest are validated next\')\n                return result\n        scope[\'PrepWorker\']=ObservedWorker\n        phase(\'page_preparation\')\n        summary=scope[\'prep_run\'](cfg,runtime)\n        phase(\'output_validation\');validation=scope[\'prep_validate\'](cfg[\'output_dir\'])\n        metrics=check_inference_evidence(summary,validation)\n        scope[\'prep_export_images\'](cfg[\'output_dir\'],output/\'images\')\n        report.update(status=\'passed\',model_inference_verified=True,inference_metrics=metrics,\n            output_validation=validation,summary={k:v for k,v in summary.items() if k!=\'measurements\'},\n            fidelity=\'unverified; a successful run does not establish correct restored characters\')\n        passed(\'output_validation\',status_detail=\'pixels,manifest,geometry validated\')\n        return 0\n    except BaseException as exc:\n        report.update(status=\'failed\',failed_phase=report[\'phase\'],error=str(exc),traceback=traceback.format_exc())\n        print(report[\'traceback\'],flush=True);return 1\n    finally:\n        shared_modules.close()\n        report[\'finished_utc\']=datetime.now(timezone.utc).isoformat();check_json(report_path,report)\n        check_collect_logs(runtime,output)\n\n\ndef run_restoration_gpu_check(environment, *, mode, options, payload, output_dir):\n    if mode not in (\'off\',\'build_and_probe\',\'unihir_smoke\'):raise ValueError(\'Unknown restoration check mode\')\n    if mode==\'off\':return dict(status=\'disabled\',model_inference_verified=False)\n    if sys.platform!=\'linux\' or not Path(\'/kaggle/working\').is_dir():\n        return dict(status=\'skipped_not_kaggle\',model_inference_verified=False,\n                    detail=\'No model build, install or inference is permitted on local Windows\')\n    output=Path(output_dir).resolve()\n    if output.is_relative_to(Path(\'/kaggle/input\')):raise ValueError(\'Output must be outside Kaggle Input\')\n    run=output/(\'restoration_check_\'+datetime.now(timezone.utc).strftime(\'%Y%m%d_%H%M%S\')+\'_\'+uuid.uuid4().hex[:8])\n    run.mkdir(parents=True)\n    roots=environment.get(\'asset_roots\',{})\n    spec=dict(mode=mode,options=options,payload=payload,output=str(run),\n        runtime=options.get(\'runtime_dir\',\'/tmp/vtm_image_runtime\'),\n        packages=roots.get(\'packages\',{}).get(\'path\') or \'\',models=roots.get(\'models\',{}).get(\'path\') or \'\',\n        data=roots.get(\'pdfs\',{}).get(\'path\') or \'\')\n    driver=run/\'check_driver.py\';driver.write_text(GPU_CHECK_DRIVER_SOURCE,encoding=\'utf8\')\n    check_json(run/\'spec.json\',spec)\n    checkpoint=run/\'report.json\'\n    try:\n        env=check_offline_env()\n        if options.get(\'gpu_ids\') is not None:\n            # IDs are logical under the platform mask, never physical host indices.\n            ids=options[\'gpu_ids\']; inherited=env.get(\'CUDA_VISIBLE_DEVICES\')\n            visible=[v.strip() for v in inherited.split(\',\') if v.strip() and v.strip()!=\'-1\'] if inherited is not None else None\n            if not ids or any(type(i) is not int or i<0 or (visible is not None and i>=len(visible)) for i in ids) or len(set(ids))!=len(ids):\n                raise ValueError(\'Invalid restoration check GPU IDs under inherited CUDA_VISIBLE_DEVICES\')\n            env[\'CUDA_VISIBLE_DEVICES\']=\',\'.join(visible[i] if visible is not None else str(i) for i in ids)\n        process=check_logged_process([sys.executable,\'-u\',str(driver),str(run/\'spec.json\')],run/\'driver.log\',\n            timeout=options.get(\'total_timeout_seconds\',5400),report_path=checkpoint,env=env)\n    except BaseException as exc:\n        process=dict(status=\'interrupted\',error=str(exc),log=str(run/\'driver.log\'))\n    result=json.loads(checkpoint.read_text(encoding=\'utf8\')) if checkpoint.is_file() else dict(status=\'failed\',error=\'Child report unavailable\')\n    if process[\'status\'] in (\'timeout\',\'interrupted\') or (process[\'status\']!=\'ok\' and result.get(\'status\') in (\'passed\',\'native_and_cuda_verified\')):\n        result.update(status=process[\'status\'],failed_phase=result.get(\'phase\'),model_inference_verified=False)\n    result.update(supervisor=process,output_dir=str(run))\n    check_collect_logs(spec[\'runtime\'],run);check_json(checkpoint,result)\n    return result\n\n\ndef package_gpu_check_logs(report, output_dir):\n    """Small review artifact. Private runtime, weights, archives and spec payload are excluded."""\n    output=Path(output_dir).resolve();gpu=report.get(\'restoration_gpu_check\',{})\n    run=Path(gpu[\'output_dir\']).resolve() if gpu.get(\'output_dir\') else None\n    archive=output/((run.name if run else \'kaggle_environment\')+\'_logs.zip\')\n    temporary=archive.with_name(archive.name+\'.tmp\')\n    selected=[p for p in (output/\'kaggle_environment_report.json\',output/\'kaggle_environment_report.txt\') if p.is_file()]\n    if run:\n        if not run.is_relative_to(output):raise ValueError(\'Log directory outside diagnostic output\')\n        selected.extend(p for p in run.rglob(\'*\') if p.is_file() and not p.is_symlink() and\n            p.name not in (\'spec.json\',\'check_driver.py\') and p.suffix in (\'.json\',\'.log\',\'.txt\',\'.png\') and\n            \'image_dataset\' not in p.relative_to(run).parts and p.stat().st_size<=10*1024**2)\n    with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_DEFLATED) as zipped:\n        for path in sorted(set(selected)):\n            zipped.write(path,path.relative_to(output).as_posix())\n    temporary.replace(archive)\n    return archive\n\n\ndef make_restoration_check_payload(notebook_path):\n    """Authoring only: select original notebook 1 definitions, without executing them."""\n    notebook=json.loads(Path(notebook_path).read_text(encoding=\'utf8\'))\n    vendor=\'\'.join(notebook[\'cells\'][6][\'source\'])\n    assignments={n.targets[0].id:n.value for n in ast.parse(vendor).body\n        if isinstance(n,ast.Assign) and isinstance(n.targets[0],ast.Name)}\n    raw=zlib.decompress(base64.b85decode(ast.literal_eval(assignments[\'_VENDOR_SOURCES_B85\'])))\n    if hashlib.sha256(raw).hexdigest()!=ast.literal_eval(assignments[\'_VENDOR_SOURCES_SHA256\']):\n        raise ValueError(\'Original upstream payload checksum mismatch\')\n    vendors=json.loads(raw)\n    revision=ast.literal_eval(assignments[\'UPSTREAM_REVISIONS\'])[\'UniHIR\']\n    worker=ast.literal_eval(assignments[\'WORKER_SOURCE\'])\n    mini=\'\\n\'.join((\'UPSTREAM_REVISIONS = \'+repr({\'UniHIR\':revision}),\n        \'VENDOR_SOURCES = \'+repr({\'UniHIR\':vendors[\'UniHIR\']}),\n        \'WORKER_SOURCE = \'+repr(worker),\n        \'RESTORATION_CODE_SIGNATURE = \'+repr(ast.literal_eval(assignments[\'RESTORATION_CODE_SIGNATURE\']))))\n    sources=[\'\'.join(notebook[\'cells\'][4][\'source\']),mini,\'\'.join(notebook[\'cells\'][8][\'source\'])]\n    for i,source in enumerate(sources):compile(source,f\'notebook1-shared:{i}\',\'exec\')\n    body=dict(format=\'vtm-notebook1-restoration-check-v1\',sources=sources,\n              upstream={\'UniHIR\':revision},worker_sha256=hashlib.sha256(worker.encode()).hexdigest())\n    raw=json.dumps(body,ensure_ascii=False,separators=(\',\',\':\')).encode(\'utf8\')\n    return dict(sha256=hashlib.sha256(raw).hexdigest(),data=base64.b85encode(zlib.compress(raw,9)).decode(\'ascii\'))\n\n\n# CLI child\nif __name__==\'__main__\':\n    raise SystemExit(restoration_check_child(sys.argv[1]))\n'
GPU_CHECK_PAYLOAD = {'sha256': '0a93e4d595dfd289c6d1abeb21ec6c530340dd666852d5de7a5828999dfd8127', 'data': 'c-qXp+m0hyk|6dg-Zj`EJ(6UQj61VfJCTu5*)fxsP-JG!bTcVmC+Q4&2kFjscgCe+0+_db(u1Id^dJG6#bN<r1?fc+B+Nh%pw^G{V6l4DC&(`6>KBrcmA%;Nt_r97x!AE|$BvmDGdqvpP2*(YXW#X{yUiBOD9(aW9N%=BNswl7;%8wTHOI4HeAB$`eAoCcjaSJyNWbe1zMCfT!tuQ6DqAIi=Q-hG87G<JN6-k>OTR?o9dt1J5?#Lj(Ytte`sUpG>HO;B#fSG!->EGp({QzD-iB#7ng`HH?Mw6}ngmk^zIe;<KA3x1>@EH4pq_?52MuR5itigvB06z<Um^$o`SKn9?_+rH_|s_^1<p7Q;9(pFQI__c`&!@MxqmU<d3XU$Jk<Il_%(rF(-!uJEnGy)RpwBSsWbC$1IKr=c-fo>)69vJ@H&iMI`JyQQeo<>A_5_pI4~|}>_@jgz6`Up#qGgs7@0GPgVX_(IKllgP6KE6a~dR{p<E9>(TBM|3g%8W3*hT=_K=2Sf9|}#yl`&)`6`gj@c{YxuHno8d58~wF$c5&P8v=IuMt7~I-DvzIsLwK(iX_Vf06*^Epi6!hEva+rZaM#U2M3Xz>hipm^B>uh)oYmo9i5xwb#@Q+))Eo2Qz}QqQUM_1%OQp!e$;s9JgpuRw@CaX+bgPI-1l31$s#UuaY_vT*FCWD&Wt;zh_cRCAo<iz*4R2AoFl4>Mno7XZSWis&9>ha9+Q2cDO0G;pn$B^;V-}5k^GjuJgUq0ji_M2JImh#i!2D`N06kM3)}|{rrR^bDkt|QlHi?0Xm1r<2+*ZCm_Oc5KK~Pg4eQWssE^3eR>JCZo}Zt@yEYk0bXc=(jd#i=sI=kToD&R6x;>~;~ZAm`~hlBR^wpe)&wR-VYcv>5>EoNG~~X(>?HB;JYpGX|5LIGM8zz5C{hBq)<SLo*mGOsID+1@dhP1XE7>C~eSbbTD~np=IV?V$A)av>7_cpz`EY^Eg(rLmKw5_VZoAzm!BM9z{sGF!fH?-&5B-|I%Ho>MhOQXk_$~tW0MJRU;ztt)5IYGXKp!w(SoDBI?75a&xSj-Kc#I>(oc;ws^)C?Pz$V0$!dy6080^R&-#B3;`&hz+9CeFK$@vlb^TjIh(=14uaRhTVUCrlB1OaI=a#lb@P~3@NRRRkHPywWe26Ql<uO`^%-7JWpg*yZ?o(Kjs^+V_z7*;r;<=k@K0XUQ0+h7da2n{i%wte7t0dy}0@_S%#$U|YfiRY_~hn#13Vi#FH@Mh6<U{L%hI>(*p^5U&5c!3joHj9_3T32$qh~WaV9(mdtPK}XSd&0#ANOr#OJ8gU}%szctx8Q0Q5g~wZZ_Z}Cr0r1B=v_F;fCJ5f@OqY2F%f&0gHrW*m}YfV7QfeB;3w*hEDLYBiFouoz<vsK?W*FvDrfB22n`{&1ryx*5e9-h#CsEDr&#DLUPauHD`y5Lwjs#`!5j>M+qT+rJpv}7KSa6c*1#al3%qP)56i&$RiS6^5=4a$ul|x{2BRAO18X0c(JIhy#U%BqNgU?#-a;t;1H3l@2|<_P7iJM`*a!I`5v9~`UF2>C{LUyC^8Wrlj)EMNsy}{cEmrd^T+Scrs<H)zTO6YVMOP>R$iEL#wc*SjmT^`W9r_K~^+;|#*7vdcK9QRc_nbfnvQ7LPN8U0HVe)|}A6%MNe8PFH*KvP^=Ib5(w}XG%_}6TA1oSiEW?`SJk8xwerpDNg=oD#MeC<?x_3zi!<*%i64N#En(Zt*%nAjn%s6iK~m!X757vu#jKwsMWOMB?Lup#H`?&%VHcHNHt(!shv04Hf3Lld%76im1iSH0a+Z=sF`Trg$^orxY%u}f4yP>Pow^%Ck!*#dt<|G?s)NZ_<}yLt&@!;0^7cSirb<Xir735>J;SmOe;)M`D!X;@SO=E{S{T^7Fa4|vNoCEgB-q^h0_@>EZOdYZymYjco5=FkkTReFYZIJcxRc(|p(M|=VA87>cYNbjiY@$Tr+?g(XI&)gBCeR8ER>8H#T2SX{^EtYdcJ_tN@!O&cig=udWZ(tUfsr5H7HN~fy*@(KdUvWn?Nyu2#Muwb*KUxA30V*e@28BGR1d7wrBc!m5^U_(uJ{u%3UE>(GO5UF2wqyw7Ynmps%5p|ka})VhZlf$~!YJ>>3CzMx1mcp>U}2DpwS2om12VQ5LIm#Y{wS?4fFZ&%=mgF7p?){E-oXo>Hhow8n&@AC^iZ#3B_9^)Ev=|lyWJt41C)G=Z0F~2S(gAa?blsaUtRZEjSLn<^{pxX2iHk!0;E3>Lq<ghhYidEc#E6cf(po=`{Q5%;<`6ZfI8$%;v6KwbPgnF%q#5$^Ke-%7JL}}4kd##C~yfo3y#&buufHRUF0gu8Xn0L_>|VMEMZ^+{6xqemWCCc&1$9~0+<Q1Sq!{F3!*5J-~wupsX=IWAot-rw<sRMie<@=;I89p6A+k)jX?d2X6t1QT&t+}LAf7_!eoAw1zGYvvw0d<wbo`%6lqi}Ys^|vaG&{85c2ED^#PDwDRu%pyF&JKalf`8Mq<>3fuzjRf<A#jdGD)gb;At$QJBUV5MMa<#O84w#4vNogE<^zeYuvDI;mC_y{Pk~_J8!DN=}Bd6-h<YFu6z{h^}q4N+mQDv`YhbGe7mSEU9yYKyP<`0_!UBNcF6_CAAFw)G7>M(O`--^4hHVO;|8^v2L_ySag(Y3Dhzlr6Yx>7TYxXvJd`^OGN8htw}hY1_|o6bycw;Yq&UJthAuOtoJkpy}4u`YrK+SUlzML83G9|5KGV^XpavyIr<7U=&#c3EXl+KALSFm2YjTzPc}d>rPNZ{Y@6H$IgBx0DG-VA1d}`wl@NC69?s1OnT2udp{A=wPId#$!<bbF+**1nD*GNO`#|n-Jg>71=h6fMLD%as%PXiDuN7DtmGaDwAFkd#vASm{doK4;w^Yo&#dRdxlJX~tAqK=)U49~Ih@?}{(|18)Q;KI^KK%u2AU8b{JWZYhK{Y}o1S0T>Vu5R*$p)yYVCY7fSJJcGqG~2x63~h&S#*UTQbe{a218i_SCbU2$yxxNOhVt&uWh}bzDlz}jkf=)qWDpjRMK8g35f8PXV^o!LGgJYNGMMM2(2;uP}2B?i*_*zCX-;o5-NU|Hieq94tX%&@B#j7tZH-R7FHcYtAC(UAWUVRSZFvJe!{t6LYROEn))br;!F_>)*o?)AXF6(CiL_bjwc9WY1<-Y19_UQiC}}KHu6#(19<YV06zG)emM7moNHolL1QdpzBZ5V<W8#ldI^&Nt7sX|S-=|;-l4!a21@lNK(^t$HdLI0t&^lZ<A1_|rP&Qx=MQma1@~}K8a_G@n&e>kWLe<rnn(I%btNMm$|llEH=V~Z$m?MWyK0D>GyQ#%iE=2~&BKu_dP(O;!d9QezhfyCw1D(R_9JX;wjK5qwrfCS%9hSBx^b||OJTM~bEG(RVcQWI4E!jL5cRts&aNC>2WiW+MV8tc8^$&$o_`GIfPL}fgS13mCq6P^rWYk?yp>!WKew{kYB7pna_aTYA)1=;zpiMhbrleS!rZssp1wc(?T3$2S@kF3Di!K`m(3E?9v>&{I>(ahA`YOOv2&-Z5eUHJj4MnD^tv&O)*FA7rlB9bny->N_k^qgi@Vme%i*?Ah(=Qt^71<N=(gB^H1`*yiSOL@obJH^Qr|7em16dg_kf5W-FVpyD0vpoCqV4`pmMRb!Yv_!Yk!$aBJl6Sw1O0V41v07MF34Wuwt4>4%e#;-oS!OS4-55Q}_~HJIQJe>j6mJ7X;)W0a<h)khp;wt-~TL(9&miz?ppcHj76OS&&MJ4v!pxhY0&Ht;Y_U;4?Z&0wH`)TFLG}Mz=nl9J$ht2--iIow6)m<XV~KTAArqX8BfNhUf(JhrR#^k1gUQ{^qtzj_l-NTzp5ylXSJH*Kf({giDO{S{(i00Xn;LUQh=;`Ga~vD;bt03uEF(z+{c#NJAbmcu#ioz%?4q7}ZLtJQpiB2rx2<amnvt#7Ag%;{i4a9^+6BwLslf^?1keT#wfp;QdgIo<^iw6XU$s<1{AU9;a;nUYcJ^#%We=5I`m4MB|kKLD@jDX7xZ(Q-q~P+VXwPD90z(Ux9<&JWzS8+?mE(P;P?=#tC(<GhA9?2NrVDUAZW?C2;o|IKrO|9HTTxnzL<yW9bc8fvYA}fCEdB7$Lr3+wc)HJbpuKhMpKOs3p2nToQF+lZ66P^pK5Ck*lLHL}xPXL%9RMj=NYb^k$Kud8#fT-m%+)!pLNcZAlIhOs=IpS~Of}5GowX;>fV^{M1h}d0u1~W0aJ^n8mggDx2KZ3#Vf$)1$c`AuA4nzU_i2&@jFR;Si-2g)htK!3|(x2CAq+ks=@f4s8<;teBGe73V=HYXNXv3?^EjtIR^N!oC=bbuRzH36T`hOwA<5bP3~ii}ll`AK7yP)%MEfW$(Z(sXq!}B81+9_``;HE*O|5_2BmC*RWWhY)`s+*u!ncnjP3~U`W(TNqyXpW5PG~Tnf0Rs2qybcB7Za4xmkKJv=r(hLQcy>}{w=W3sAMm~@cBr5+Cx>6R0cHh~E4iq)im1A0Z0k$E0<v^wb&&{QgAoJp9l*dbTKJ?!5P@b5jW=7%mR`oaVrh$A<UBzL%p`!3AEeJ97pndLm-w}*~$r%6|H1a?q|@-NUTO`YKL2s?76v3ex;p@7bK0y^vJ4c%dBFW4M5#Z3~-)a2&oEQ~RDgI=%MDfB~z1BoSiiNZE1RY^7%aBu^NbRo{q{1g=r_4{|AAgUj+_Gn82^_b2;-<s>6s}Tr=V~{2Fs|Wp@g^NHaIUpX7R|)8|S&N<MQl;;31DA0;Kfe!P(&9wa!A)yC_tR7tR<9b)Gz^fpQORip%GOwGTl`Zu6Cfl;8}z$D@cm4bS*?iCm+beG1xfkx8@^EFoBE*Bp+!9M7eNp0>#o!M0sr=_8LTC%NO}Bu@WX>StKY@R4g95g&fik64d(_VczT0}lA=*q6w_;J8Ow;KCn2b(NVM>+HTK7|z`F?^>h7R-Y?BY2T8O@owIVQS4IZYoN9^s%d$b4Q!W(Y=NxLcfNy@gkw4N@5u>?9q86fK6Ujh&fECcv4wA7MXr`c&YItMi@0)jy61oXZAp;P~V{>wl9!~X?X{^LLXkFednbZS&$@yCDsFG457z0uM}uw!%x%o2jZwT}zoO$$E3YI2J*9T|RjvyGJJ$SUh;fQ|FdDf#jwBy@7Kpo)Tw(6G$guzo#U!hI?$yKX6#R<AWpqbT&|yy%M!A9WDlWqkzq4E#N!!!nR(6@UWv6zDFP)Kd^s>PrTWOi$>cIR$YJw)`bX1rdoVlCh%ohGq+CyGKMo69hf=w(~cd#AAITb~%~8j>juJQmZ5Wu~m1S8ym`!&_bod8hp!pt9j6P^5D@52Egf859oZHq5lu6u#sM4XyLUNnOOT=5A8aOpf*%B=_oQ+B+Y_FRS{Y>q8uH;4c!(tsY^?+(*VyC3HVTe4Sxk?A@sTj`4$E-)XRm+SnK6%1}gw2{ty2eSX(v=Vki5<zl(OS=kcfoR0`9Q{qZ0EZRU*s_z(Z4g>#FIJ2$gG{I8(a-$3R6c1bw;!@rK5`5*uBzX`2nBo>T<sW5k92twVP!6ZfIa7<5u06zg`(vWa%0+A~5J59sr+8A<SiO~uxfnyeuL)U<{0@4TsObC3QP6)lGcqImit>j&m7$zyUU_HZGM)p-w5syhEAxQ5!8hnXRO9+!)1Z84tfZr$ZQPGy+8p~KSU11iaK%qgj^}86Ih5||61^k%8Vu`OTdcCJMq4b<b(dSdkIs3!^;-r84hyM*G^^y90s@c4V2Pa7%XBJsa3Jh$y%DmiewvNQ#hJXCWf0;QqfB1LgNSGyPkuqAr-f@5U!@qInfB5?aLIsP#Maeu^(2UR6E6_GvCUJ~=BCTH4n!QnAkyZdQ6gmeMJRN@w=fV3p`w<0ki8X<*YzB+&!36RV+j`PoxpUTZ4JIvkH}#p|`mNV?T4-Q*$zoCSymzN(A3pv~O%W9`*PO9}8vY7M5wX!9{yrAiwV^HHQ^xvSrNN|XVXMLA;i5U~8)DJ2g|p@6oeHU37;z`|@}N(!P1d}c{($9Zy^wTb*dTZarkB<NPfM6jGkfnj3<Mn~!BXo93_lrzA%4O%r3G?oy1`>K$wY21yuH>CH@V%`UNMT9S^{>J*x7I{K}7y*5X<7bl;u!E7MJS|Bsi2{$v31<%4cEO8IqzY2U20z0H#XxuM+S#$n>8;sq`nNNDsPPN{(3b09Jw&tEm|JS&Mq-x6qgz-t}_jp}-m>!0K;9AYPI?04d=(AU#RF?2}Q&j*g-I*qO)I6k;+7Zn+jfxkNn)I`!GG9Y(Vt3DGfI02MgPxu0RIki}G>KRmjHB~p*mmgt?f1PzF^r?AFle2~nCftPN|;-}4-WEtr+g-4C`(3PeZ5ZxTI8+|WqczjMJS|aHMr7m2l`sJ_Ki%-5->)HM0^0RaCI&~H+P%K6Pj5q~!0WPCBih^tQ)sc7=Evln=h08vUo{ef}pI@JPKV5vhc=h(&dwu@X#o75sSwZ`8sH%pXO-3Zuq+x3+aTb&<m0P5AKZ9E-q-$%3ste7iuVhhEzW{@^W~XM$7c@U18cE1#D!e~RW@VEL)Kp>pQPh#)E5`WC@YsJ;-+z@DT>W-Xj`ksm5K@8=5TnJg9D|H!<dtFBuyQQ!RxVxX58}n+#KU!ro=ZV=9Y(kW;l~ozpF|2;u>w3d<F|-u{4I)aq*d5DFPT<-kYBC=j(dNM0XhgiVimM@JOX`i@Q{;RRpdHROHp*+Vk2JNHN8EzUWAkU0-TtOZ?a)9mwp23Q7|V!(IFs2mWsnDU4l$86a!<AO3VjwDv4F&FmV*nl!Os7+c<fs+l=!POOOc=#Jq4m0pUfkXXKfm;slftK3@lZ83aEE%rSl32Wlx1DWCIMIy*;E^`Q@A$7#UaKyzOZG-A(v5IkUSu+8FBrZrwo{1zAU=!B<^t4i1d`&<T4mDalrF4dpbuuktWgk|xPjkEpfJoZ6<zTdfbTWA7xTd;B#INL(oWjLfPK_~=j5AOS61LcXn0Sx+%2!QYtP)Fm7p583Z{JBRbofVZ}Dwb%a1DHd%#pM^l0%pW)2XS0%2FiKZR0zm|+UN>D_Lu%BM31pP5p;Q{x)fBi0S+P%Q$4^+pTz^j$udEk1kP1!vRW+Dx<C+pRMNIIoN8&I1+qzf0HOkj*eC#?D+kD4%@wLI%`yC!DFT#YfQW)>Iogy0<A5%Ld-)hd;}~Uxer=Uak85nTgTnojHdMxFT{D<Y&^|#7b3IAx3?1<C2~^^}WsZ!c?hcw1Y@mcGS{50Qu6O6(LIHe4*>4q%*{_C{%_Mi&ezl5djbdq9jEo|B#CBdf6baKLR)j4&!Y`fH4J;a1<;d_<_!}1jM&rZl->)csE}lAdmS;ufi6?<+fwUC>7J+jl7=@QQ%hhNOo3jyMh9v@A?!bsZrxufhmVIa|glUIoJm&_8!i1MSEN2*&BZH*bogi5)vyzxK=sUinJs+8#1Wfn?016r;yEg=(r1B(;l!=D(D1~s~D&qiMF-Z($ix4l$DaKC1hUNAO>OHn!aKc3zFw~6@B>JLAicSJTz)btdps^`*;sXR}vyzUXVF|bIJmOb$BUylyqKSj2hsWRw%@sKrzxR-zVFyp!`0Qdm?vNO<AvxCt9915+0>uMSO#B_<w$G0(E|K@AMPXx^Ij{UQIHyNDGQw~shT*jgHLx0N0YU|!0vc|xtTEajjY$ODwZ`+nPt3xk6IyRI%5ifCFlaeh)-7H@f?74Dn#KWH=KH=PZn3S)PgUD`g?LAHlqa;1bJ!$u18$1lfFGfmr<x534$k>=4Gacm+)=E)cGnlc<RK8trE*|}6~k16Kr*Q?Ro5+|-V~KcQzUlIx|#zwKV7ROiiR4IVv(B=TSs1`H2-vZb#eOsllSrR{OlvSpiY2?fl}>_!pKj;pQRrpy8YCmV3pw6m(J@@t9<)99-l{_P~gH?T|n~){=0kZ{JQ!2;?>!QZo6~r{1kq2(3vm-Nq8k418PWi5pZJIByr;;@7(Yfw$>VYLQX(Vi+(wD=gl{#?=^6ro9~0n>9o81FP*;%65v%Vk)gLI;Qgg>p4&P(*cQr3Fq(rV54-HQh0)=LOF`WhwmTq}WZ@_1%({qg$?N?afq2mAI;wd`H2)HlSHS)m;%d3ZwaNJbHp9z#Gc<t0mo^@$WnNQn@kzN~8e#Jsr7c1WsMYb@p9QkXNr2FX5$XeD56{L@&S@eVx4)0$Ie;lW6Y=}m+tXj4J8%7w^Ohh0ie}huNgf?>m!ojJf`7gHF815qqF)iz0^i6-*|XenvA|`BY}64gJJ9kL-S1=f`+th=_t$hk38EPI_hfsVV|iFRUOMv=;C{G_;Qjlz?=+^b=MT$Sfnw07lp&t47XG)IWTYl&4@eP3NA0z_4IIgH#=RMgt6cYdN{0AG%|$k5gK!NRPxLK;c=HuN0NV<_B@C~&hGFiHy!kg<?-%Dpf?H7m_UwP-p3#Vq#0%e<g#L9DW8uxiAAMsWM$ZKz!5EF`=35c~H0exYxhT;<vK|E7C9la5%!}~DJpB3FElug-?&w|sH@B3AWfRXz{nagS)@?zt1R(hRk*hl*02i<_k8ulJ#Aqf@tNV?@(Hx-t+J4`2b!QRyQ4{K|(&kMw%|K$3fp(~JYkX7#G79G?Gx+mgx<ROhHlchKQN>@{pAroL>7vO^J%b<~M!~(eNRw|xdk96)<WRGDeC=Fu7i3_d__mj5IzibU4+(|+aw`bO%RdE-?>=6gUA_c$XR(0AflK+?M_nF`3aJlz3M+7<^+a<Q&!=BYx_=FIn|~S4u^T)*_UBE_62$T`V@N6&&@Wg$7iWJ(OXtPS4u&q(&j%O(d3X5Y7D^#QbE~^$>@LUORJ=jE<TzXcCMdZ>gr+UM#l1AdA3PFR&uO`b>&4<*0f@Uy9EoN9=G{Bz{M}1uL~#n}tEVBRd{|2k=&gni(GDv)oMyk20epbRv!BopISUDObIkn&%{$b5!--;N`H=XF5VYV2w7dDBe^*bf8QqRSyQ|dDa=#YH%OC;dHQxbw-=HdnhK30$-M7RW_MHf03q~<UGDUH7I$zyy;$p^Lx`>k{EE7Jx{8kgL#6|~w+3=VoT_sb09K57mbrLvIHGBr7*DI*_&3618VCb$kZE_b5$H4s(SmP{ReJyLD98I$z4bxs8edxiU#1eS^fZeRw319}&1Z@Udb9|1G;0hXOo`1YTOTpFE>sM$ZxZhvSSLtTmKl3MHntWx)pH5$gAFrIV`1+?XxZ8ISkh60K&uBB4scV#se3>gR78w9G*iK+MX1r;Oj3Cc4mq6`L0MW(kce|h9-*@plGCF?3*~HUHJfCr>=gpw3qHq=_Ru_U)u7fB@WGrvqy0-S?=P3O3#g+5w^v(HO=iOV-wAVIJz)m~{a6GhC`U!3mUuihJ3R0LXe7gt|=k=$Lm!M`WGskFn^Gq=;zt99%4U5t6_OJj;p2T;53Se~0+d}gV47^p~<ayCXcyL+$@~Oh*1=|`r(_@w@JY^<l9}wn1mi3cnr`<;5cR0b+BzkAv2w@_}2>Tqx)BG|}HOk(-{&c$g=Jk*79d4KpUMa}=IZ%N_5sz2tp8zb}Hs(Q-6Jm&Nz7j~>M?58+d@BGyUc7$&l>l7}ZT;H=HJW3jxUvRp4c^n2=;P<BpU!{def;qG>g?QmbM@i#C7;XeolH8T!@Z**n4FyWhe7A$xHI1C9tOuphX;E{<MH(5aOC^PCr3f&aN3!S{r%~oAB@|bV6S`JZez-bHys3EFF5hLqu|)@wD<i<dw+jCX&>xQg5YS}_D_!YyItQuIGS|FC;t98@B?^vFgl)`_y>o6d(xijmPY<@rwi{pll?t-^t=A(;P9|>uy=5B7>t7b<L>yV3vIRi_V{GF*8#XsCPydZlY_}5I2s-3mJW|chyEA<8MpV2jt};O{Z4Rr5KQ(Dra>ndjVIls<Kq+f$3HsogT2YLJDQFUPL8@q2c7m&aAdS}IGyaBbdI~-{e$+g@9$5hQ~zKbjK>H5C^*{hv`_ZO2ZwtHhX73n1{a(h9Zru=_KrK9(cW=DEs+_0f7CrhM0F;kJpg$+n(V_kkB_F4qwaXWb9B%>fI;s~0nG0B&_6snJ__1hzkS>}-0O5ZKDD%0gGv4T{`9BQi?^rfF?MzS@zaN^(@z&4-h1!PKmGc{Yo6cv9l1sZC|f+%QcMeu4Pd?vRN=1^$^*pmSV|nb_JNX<11n1cO2ii@4;Xyr_{n${W}x=0l3-WpMj(KWR_N&D+y+TF4Jlu)IJE&@nc*=ZJldqXTEfmB_^83vW?8mOd%L^WprEftEzq`hFW&$4+u!WI{p(Kx@F$WX{mCi{vh7FlK6YT{gOr`*VmT&I*Z8!?p(K7<R-<HaWe}^8Re^qYch`!1J|740F_dTwqU~<F-`+pjZ??M!&35PH@Su6jL-DgX^-$Wmet6b!mSGezenWK7aio`lBZ0~d>7|pC`K6c2(#H;TX=qf7^#mjHXHI<ML0`aED1o-v)VGIOQN;xJP0C4Mf_ESvASW19n9lG>7dvcbGl2Yh#yRNOH((Z^u5lYY-~@qc2%`SellEdn<3{G8!!SFYYmY^Ca_=j_9|dB{icu>D(y${Qr9$$ipj^)%`sF;@pI?LWmCY7?sS*h30HCwf0AW5x_pwzn5;<=$EV5tICT_|pg{6}CHXvh2MYo)!d|zgC=OjbA7N$OlS4#uDhDf&q<C96utn+#HBPel3v~ZAbwQsavGBUeMYgi_%M#go+>5(ZE8w1fCouh;g9D6CCgA<35G{l)Nrjv-K;kE7RbuJHvIaNwVriqX<eIc@CvK@A<gq1NtDxOHyEu;M4cA(0Ne3H2{_LjB)tdV;ZePB?PvlK=%h$u(|-)+(Hb&#%rFWuK`99^Q5okBm{JjMRhT+7K<`I9`V<~YLUsYO^%*{oK_RtBPlp<5DP$=O)t2snEad5^`bEJfRbI$9U$<m|zg!Aq&TIPq2yXr8QGK`1ApXc%6jh2eG}V-1Hgd_4<pagRXXqBjQ7<8SpMr44G~1fvfHy_H}<cmj(iYV4R!{6`LxJ}BzJ#W3*%^Fq_bV7TQ}DWBuK2L!i@z7ZxxuOhXP*?v0+vp;#znMaGua1&a*aFSnxTsF-u7k*3I$u%WiGuDW5Bch5_ep-9P&YpUYbPd1@rvs0w%cokyF8$@Z$Mq|$s-Z965gcy8Bi`me_6W7_8u10n=DnUVp!afGaEml(!}goP_VGRJ#;`kL@=ENeLEM=f*U=*mUtv)4qgd%Oe;zL(O(v2cpEl-q{mcoE#p2nu1|ssW)N&D+Hv#8cFNMJ^hGmNKgg3>`a^oC8Ay{eX9j#Lm=gQqSIJQ`YhNz0gbi?I?J+($B%=)-P$aos;KShsT3@yR5cD9?IoPq-TqlL^9L)kFN@C;ihyRytzi)f3fkk>ZYb711u0wzD#7LF+<f_ZGN*if<RQPGpO2$Wr0AqwjTK9NT5cBS*b`E+&aoxMH%`0?WG^es9R=Y44kUM(d)CPj}A<$s~vA(Tsk;TBMNJZ~J28z<J*nXz%4n>$}E9A?U6LwR;*9i~|aSN17U<;l#tqma_WhY~Bl;%Fq5QHl{{>I<~`r$R`{C8ZPRNaQ<R>DNF4Vj5mA{Qh3M#h1=d0-S!kb==?tiv7KAt8M#duv{79PBQ5qLp9}N(eJdn2MtFj>g{(AZ2twSdq+W*l=!i~*J(I6&>tp=>UX-w+?jSfsIHLr3!su9iW6G;00xbW&t>{u9v{6)mp(wY$35wDeghjFEbK2Z<OO=#gSOOJudHXE((Afj3izG(I|sTy%!|5O^!E)A=6-S=;B`Ovij$4=`8#)Q%)?>Xf%~7hJ8^JciTVTMXuG0Nv)$_CPof!WN`ERZJUZ^2bc+z>Pl&54%46NMzXu9Z7O<p~0zMj!JSX><j!(E<>tvT3=A&7U(@x_2QerzySkG@dVg*zky>i5o?i~8Q0bcJ@MUI^T!F~yd>l|!28l|Rgf+Py&8k_4@W!d57vl?kQa+CmeuuQ{w96_0Or~J5aecMD$27n^)f|wPtqMaSdm~8#3;q2@P4)$b4C`z`53BllJd2M(7HDxjmK->W#StV#*TX*nO%f%+z;tTGYZi%D^pX!vShhpG3PLZS^%Aqea)7EcYL-=82>6pkZ2d6g#wU=}2PPAH#;I%OVnFXL;v2$8GpxOT&r`!Sf1~p^N*NPcH3?$q&I7g8{&mIPN+~I!?+H?cmo%7F}E=G2A6ndH+O06S$x~Wlez0SsvkJ>BaY{`h`oJ<@8T4%4-!md%AqKnN2%}%2;%t>L&*d$S;MO+KGgn{cG9Bc+p`QfQ4>a<#zr8Z`>*b4A;Z2Hkq4(bZHXp%Hg?XSZa`Z)m>Bs?MAhggjaq6pn7aQTBmzfOB|x8&QTyH)YEG#jmhM%&GkEB0P4$7pcs*38h>$A~y+x2cqcB+0b|uDBVi1>$6Se%CKWCFUFcof1kWNWDbM1}Eh!V>lMLLf}&1P&o@+C5CaCeQwEWL`!DR9#%qe@i$r$DjkIu$c%9q%s>Oes6aCeaR;*87d{e~f~_6)rlW>);;JFpv&=%Coa@R`B>0KxZf_d*E9EI_uvU>Dz>H!5-Ex69Je43r%9i$&mJ#S)kvv)a<j?Ka(Lu><Y6nDaWaF4V3lG$<8BvS!N~Y+8Syuw7u#8o$K9)|2DcJ8>MlJb3lIog8Yl@1ZSjlYLD|=fCmtL{NcGF$#3jJVrRdZ!KXoYr|GY^s4uD~-TSwn9Z2@de_aNokET>(EG9vti)Ze+7(<ZU<&GskMeVE0*Y9VF0Qt3@qV@qInYiXEq9Ufit>BbR5Vj>PtQ2oSXf#aU?8%7DNr;uFhuEkKP@2KiVmX<1&k<d*CYW;u)XSfy+|M6#c@>u!%AbZEs{9IYlGm+s0IS=27$JcgE2xj-$ohrv27ZYriGq}j~c_4E$CTR&pwq+5i~<eKGbR@JPR%~f#}`I*{lmrN8XZDQ{%+eD-yCwX#r<KFpNg0F>9@>$2NR6!&&JL74vIs}k?VNcLeDeL*=cBNuyzl~f&(aT|X&qAD8jM3Y%HgvlHiu)UR_7>vU*2~AjjwzoJr&s)a*IG_%ginFdDw33orJ5Yg<<vQlv=}tomNYH2aY=3m&*2yhV$&Ea_qi(@wU2AK%c7YPuX~|@TD)cj69EgI*uJb>q|P*#x1S{bLlZMSkkEym0pWt~uOjs~@Y59vl*_=sajrhnDajp(3PktTX)W^!ab=fby4jZ(Wm}e_WA(PmSffkW7Q9Zk>;qZbj8A3e7Ww(9c=#@|;?%}V^r7bTg~EWve;STLktb0kGS7&udIzs(!K1E2e8l+fogJrJS`XbhR%zdkv89PfJw7U*hoi(#9z4viF!nP*8{pnsMRGoB^3E|&U)J8|888eY-2)!5TVj`{$QVxqmeF-D4O91a?F_i%+phIHS}m3jm<%LZTKCz|wK0B5t1Mc!$eMDvudchs=J7nj=%|7B+;-&+T9m6EI%|#Md%kf-fCiessE<aRWD+EwppQNlef8n$_4$=|b^hbq^RrLV0{}gqC#8)+e_z+8+Rghg>=qy%v+3i-zoN4y4`7G><PLrDl`N<&o@M6+D!hyGg0Xd}A)~(NUivBMJ07PY>aCxZdE9VjA;YU1R*eo+tLRhPl${u<R%dui!8|7R2VC#C<ft5{887CWhc~)q14KSRt<xJ?SR!@NJiZedr{=MXU|!F}?^)>PvZrAuD0kLH2Lu!m4yf0=PF;YB*H|`a%PDp$?(o{!Ykod}UDfCQs3AD-CXB1g$Ds~plbQLj5|GgL+{$&qB0pN}_`YT+Fb6#xc?MD!=1~!BTxY{JUN)-K55xlYh~7amT`JD6xX_zRV`6V%vLEySf5R3ic6il->kfx9jXK)|n2vC=se7)3$+(xTFq+CmPU%bWyXeX_WCv5GHxpf39f)fx>hf*Fsf)f~_@YtDPDKosQ<Y)9|9;2Jrb3CyD05<_TGB(aBnSC<Ky^^n&C@zvgpt4&XxxU&fXI|C3Ypk~tJzt~WAamC?+PcaalCwx30?S39!{7;b#nYhv_VVPU_9Jq3rf}71`*`77`%U-dZ0Vl#U#-UG^sJ6vIMLqdbILOEn~cfF$A&Y-V|&=PQyD`$+ozIEz!Lh_ee$@R?`~_o|y7nLF0N%FR%tQQCF{AzSE-Lb@fL9shjJy*{8pV3o-W%=Ozf2lW>um8P8}*Ch>?z3PO7wtGNX(MrC&Ck3mo6ZQ=odr!(-C>NmYY_Zo*{-5B>-1Fy076a@!PED@twT#ywN`sDnD<R81#ki`Dbyza=NBdZgvFIU%&XvxhtC1+7=ZRr|fGmaok8k2p}Q27=m#Xj=mlNERS@{zL<$|I^J-6w<Vnt;i3tz6nYoc6^sd$2hs^Tit3@oUDsihFzH+}hl?!NwfhUySY@p=v#OI}eq@Oi^{*o_36>7Yx3vxQpYU!42IO-*KaM3>!|lCyLyCa+ARghn@z#?oe)kis5>MD-R0w>8X{)lnBukoqDQdpMGOjXC(q#(}ibMp|X|tb0LjW7}E>{T@htTF5M|Pj>#)LjicbQZK2@=)2RsdHHr%UByPLx*Hpx?F64!&2YLrk7;MVyk*zR~-`IEh#uI8mmaNcHsJ8qlxbqA#j7DBhu3_h=?O$GOP$C3u6pqp7hD8FLWI?aCFtFE^slMns&QuMR5&}Lxk$+Q(X<@C8jAb^E83gb)sq0Zm|1Bhpy5PVTzxB+@%$PgNx_W7`Q%YQ+H}ju^Bu-g2EGut$vzeA?@`^VM(CRnaor_nC>waGV!fc?M8fX;mtvf7LDcC04cn!K@ZtM`ZUNN9M;r5I?p$J{pkh6&TN0nr6fZV9C&YE(xkWv-6JX@|Q%5djMFC7%-3-%fmm^D&flv#9kL?h*vm7Kn9PXQ%<umm_3&JFI5bf<P~bPGb?1p&2H(3lk59SZk*ncSW9f^v#RO%I?Fi;lK3r6H{6u{vuXFG6)0H)p!=uHL-zuxW28GWDe%ZT_jpcZzwxU->ZAKa0o^WIe<XLy?JV8cqV!JG$aJa5YXw(pSQ4IRC{dLdpl2;w;YGdGj@K8OcC-Ph=ax`-45Yd3O5XQ8F;#y1>>!JkA1d=0hJU)Asx;PGCw75=yaBzR2?G+FJA~Wev$^y|T8~&0pU3A5m$ubq)+{V9}pgKs%qTG`tQXsfYXbQtF#y)`II63RDGk%C*Em^_Ijyx+>NX0P;(bZ(5^{aGf<f!-D%Ub1o|Q7o-f4Dl}-fx+f=ih2+8U5&r3P_R)p+d#7FEN-Vl*;@{jPa%L0CSy~t9HOBB4RMl0}i&L$|r0oeS$GugrTk&pSIIVUS50i4-HfvizCewuC&^#DXA0G^n&Y*$5@obbRm+sc_f>k6lB9jnDOW8;&v+y6h8E`d_00D3hkD{2(yc9lqz!$P2W5}I>6+~-c!CO8gcvwYeRcy)w*jPHNIwosWoQz?L{rk7UZ7}bfQ<B51$n@gPv0&PV6)n;zKJ;raN+1NAc^A;Nh$om@A-It&a}fXw_b`He^x(mk8{c4(g-GBsEk{j-aojLN@Es#4U4oElnH9c5S7iljGNHwUHbWopt)w0r#Kd4w{Tg1{StQIkU1DvEe4=iyEJbpNJT<g4aF>H#vkSf86^;!7#h}~EzrkOyoj1GrSKZ#QECvB1aze(C>a!`)RxHlrc%jS%&AMt}H-G*p3{Y%OrZPEpE#~G$zwBgkhx;-FZ9_dUL>q<wtpyir<&1ALaxUOq5#OABc>CeXdwcq-3sOw=3ZU~(AL%`cJxnB2qp-JV9M1c-X}n52m1}72LbwZj)VUZzituqoFhK_>9(m$^E%7~{)TI0xXFjYH;{@^OBb8qvYdGhaR}%N%eYm>(wRisZ?ZxHC^M+G?X3ED9#yDh_9d2D6Og^07*UgDylyVnmZ$DrM-0mjLVXc6%{x~x)c;j2TV1OasMKZ~WgzDr4FAXjnZoYuEEskOGt9t7Ax4<iaclf0dKH|zV>yX8N{?UqQ+d^qV^hra@6{$Rhu`1!Bq{^kIGD@nTi-|>hyN)tN8p@G6+s#!)n;tdXgjTrmoPekZNF82iR7&vh+FS8*<cb3gwbDzs{y_~Z3g<x}4|%9K@l2-y9dGN&j8$^!8JC8Q<0O%y6WJ(sgsg^$1-%=jOEC)4b4}Ybzi#A@{0*3mEPMtEsx?yrph78CfIfo@R;(d*kI&sRAl>sCHt$sL>v}K}0$%u^etmWR@z>s~i}$BjzX9ltcOO1|{JdUz<J{K)!{q6}6`lk28;_|}z5vrEHjs(SJUcU|%nIgFelp4nHr%8mn#q0)HIYRpl2U+SW1jNBBZe-CX}yA#-_jHoD@5VvZCE8Tm*mp40@Q(fY9Q#PLlz_u?C?rW80BgX-%)x+Mb}5iTs)y?`Yt!QHi||%Wkn3`L0LBJh$7OsPPrYV6)_hAwyhbFZeY8-cq<EBV4Jp0Pz1K-LIBqXCddF(1+>iVATGF6Ar&e*WQ*uFWrZiN6uVNDuL|L)br(+PVt4*EqZ0sa`K&JE{GhuV)C%4ac6d8s;T_#4B+&5XJ07g?flE<K@I)&@QF>h8{?kjm`ZLBW!4`bsjEK5c&SLY2FW*VbT#xLNK^_XwAYM*h$2^`jsgA)y$gd63eDvCY(p^{|>Q)(LIfnh~9+b9_vSZ9&I%uK5n|OURUZt7Z99}wj8JRx@VbdQ!kf~g`nhY(&G&=Y^6@3Bo$=Wwbqa_1)b9MTgyb@&jGV4BSZdnJ=rr<V@HBQyurXW_%n2sr-MB?)9(|{Z2yY!&BL1#D|uHh`>nIDm`C?l3VrSB4pl&Y@UY|%i7={PXEO>OO{Fo3pvJeqqx*5+a~z8r}j2A+kl1^z9@>camM0G3jHdpZvaAB(6+mYSMWywZ=OefgTZf)$as1}h(HnFgz(QH@){D~}+oGtnEASAD&I{7@2N;W~d_RmKdm*fC`nSsay0`P@azligHvi#k?+M#M&$GG_V75c%6Oo;XUxfe!^S({n*V8xT%mKb~;Xgefdp!arS3L$U*1F((cP)({4^LmMThYzfns4uzJHbr{f>Wi)vpnt<RT6aKJli9N_eHTyuVj0h^?nho7og;H&KOhzKO_L*evVjb+LCJP5UGM;05)g&h<3s8&FvW6%xFzYYtWabJJz<Q}!K=Tbn=WivC32r0#AR7sdg95Y?S|<q=2M5m#eiXilTkSWMPFYeLEjsO4vP_v9Woh80X<EpHN<_)R4PU@@QH@n9i&R5*@YL^x72pT2u&Z4w1hVN^Zc_F-Rnhz#I&0?ofo`gr<Pi0omd}equ`8(F%KK$))h{g!9wXKl?MjytTpx8&k2@mwR6L0(9AjwhnsUgKU<VBoPZiCs)pQD==BGkVx9}Bx;grm~xmVoHU^8KVRoG;7^GzDuk<dZgYgTblJ%ZI&RN-(DvTa(tC^OivwGWDu>{>&gF7s7G{Q<L_VX5`p$kpIwVc^M_b{b##HlmYARPszRMago)AY~iu$|4&}gA^O(QHlSt7$s*^Dpj{_w5ZG&22!50k4oG991Y`zZM1Ux%(s9?p#BJNL4_Zt*QXKvh%?mQxZll<)uW;-g}wYg(QRys?!_LSjHDIT>VoZnO$COs_8bqS5^$kC1lMj>6y*D3p%1bdw9t~kY9~YG<`hq*-*LSbMdO1;z)C9qS{qGurYs3|hI?lfp*o7v$o#=Uo$4AyD#;r1#uo0oVP;3{8#o}VgK!~3y8}qxw<N`R9*l(SUmj+wC?=|GXhT6?JvC5wlQ|@TOGT3aXCL0s`9u<DG9#o!ZC^fMFJe~Nos0}MXikE;n1>@};Ndq5khZUbL=;2&2TgjA%5v%z9@7SBmGo0Yg39wW^V1o$tA2;Edc?bZbNQgCn<YFNpj^VwhI0w}{bihn_m@;qd`llv^^E6LU!se5r*F=^*QcLOKc0W`e*AFt?(`Gdo^P`SCzWm@$Z3$(@@0QIzxsIb;XM@Y<UgH#czy1Dym<5e^wa07b8K!D_=oMo&T+>-Iq6Q0PbP;G_`Nr6kB&Q?!(;!zZ=XyLk57WbgOm2j$#^t9JUa5bo#SJ_8;lPoHN^d2DC~WS=ns?pYX<O)QpU;DOUtlFVPAcx+ZJ5OAOC(8rj)j_hv`xfE`OE9!fH-p0UNJ)eaZ$pZSjw2MkrJ)N5Gb+%Bh|(3n1EVCt<=S1=SxeL^hOV^O|%K#}sBMx)+eTwUq7cX}Clg`Q>omtm=(d8GORy{{ig)L+5WfRb-$3NDJR};8_)r?oducI+QtAfZ1WL&WAo_#uK~(G)BeeB?$XssMr3IR7$C-_E;Rly`a@@ABrJ>qKO697Lth@#Yg~Q!%FDp{$e!od(J@LJvp>I>p_=gYvp#s)c)|EhmNHldct~M9d8TC`7q(D7LXUk19Iv>cu^o8j5JWGRQ4$PnF<!DhvIJz+ilKELuCpSfsGm84T_*ITL&Z>i6{Xr3zX*oMZ4UG9Sm%a)>t^w3jN3{hul&lyqIHi20pVD8T*cn;J8F1C4Co`l7_%qOL(CQ(8AAX=m*s;OP-Mvu+T@Xc!4m6$r3=;YJiuNIwo;cFg_K^uL_GAR8#q7f&NjggXc`A^wB9v2vVL0i29W)%}5EU*OA1pRZ?C8joF#353afn+I}r_YUbWz4oB6Kec`Ho|EYN$sk#dmHsjps$L2X!BM`yg`j#Jx_+Bk>FT}WbCD))FQI?z19ExXTta?<?nIJzOQh8RB>mr{Z)H;jFPn4`wk{>naUS55Sqzq%*o&>GG96N9H(6`J!qjoMy*dsgS5yzry>q1Ix8BPj|e#5z#I2S4=1u9wn61AM3)a<_li<Xo92u48c-YTylY&geMc(1!vR#m>ZSa+vbCunZ_LriQZ8g>TV#$MM-8YLt*Pvzjnec#+-=tYjcI2kd1P{frN9oA463?B0aV-2?zf@pgkW_98Q7}P_rkr!??^nAB?i+bvGnj^jw0kaR^1P_HND<8EODWBxZA}CRFq<7tlxs_@Pes4`yi>3Guid}>UXP%#q!%)VZrocfwghWKYifaL4P+{t|2GVB_FE+t=?>jH@Bd&*maJ4UAV6SO`dJ|7j%gv_E<5~eqR0&Gq6S@NWymk)`>y@K&TeILErXoQ9(0uDtBp~Vn^$3-^luCoxl<Px<duKCJG`La=RkV678DT1-)Zsf6dNy5+=JA*^y@hPHVF8Yd(k@@;z<P?W2Xu{Ug-8$vN*SrJK{prgPX*SDc1*6D;WghfQ8z9&Sf{}hL!;M~79hnj;*XHzF_t|SmXh{Olt7!)Obn@4GC9=TF0oNp?E`E==es3DmDUoiZ{nRH?}_)z$&~M+NW}}|Mg(o3DEWrx-7r}X_{n`L;5CSD!z7N#F)!!)aC!08`n`MrAi~w6i=@JjxwyW^+LEp<ul2C9t@PJa^;(Zv4gG3e{^s)aj~Accc^B`0{NVCPF}TPslq>R(%_tuQsKqVOhd4NP2kqfgE}1$)mmfEhKg|sq9_EzoK`&DNEiNlipB2Ty>i_bEGfT^&D2er~SL^*dgzBp^+aQK1iH&L!Ap|YxQNA!;J}89KvJ{knN#$A?fHz8k8N<qBiLuqHiBy^JnH!T5uJLX|=1eKd3Vaf@Wx$%N37^U7Kc3$PI(8H|QlHAT@cuxWm>m-k?br}=Ke`DVo$S)fV(wu(%u~NXI`)@<eXLgsy-e_erWl)m?h;HsYEw{yzDfYZ$ktJuE+`mGYjFe0Or3woeJQvP(Jkc040wi(97x7b9w@0O)JM+lA#Wkh-Xbmo8&xY?ENl5XEdpL_H3V|L0Pvr=KjyY<TwpHuY$lec8VH60(nbRxN_IDEBYVoDk~R^E8F5~(_A{T%;H5|uMF$KKhUh{=VpK-Dj1AzR9%Qs7hA~`AMPI&?WJH%FrK^Rr2-Ah1jd>B|AbZj-MtbYWV*y#VxNk^|ke)SDM!89vd&^A2%AOp?tyhSIiw_D7Y$Ntz%}5I_8-V+@%lB_;oC_LORm!my{X^HnzOXohKumIpY!y&d6f_F>Eilz-Egqrzo9;eB9Sr$-qNk_UGP<rkbM2UHf!0aNT&_Y&!pVlNdPP}#&3Z1Ue_cqEK0HuR?9b@YL}5MJI!_xHnwY$*mBMASSWxi#MKh0q)=g?DES)v3i6jQ{u>(nf2F0vhlrD*h*mW>5t!sid`9syYMIImdg~$2LVD6rQmRuVaez#r1;tED>RuhImaTc=Gf({wCU!uRrA1D-F*poQEu{JGE-u2<5LqfN>M=o{zX-2m+!GDw0QpT@hW(1HJE-zm1VsRvOP&flWS}mO@2yiOQWMF~}ViBunY{}bhIz%>gT@mEk8jGqy3n8dUg~ESPrBQ@asXAoAd=3-X?-ybVMFC!sqgp$!D2ngX`3jb`<-<YmF~G};z!U?Yj-zaD=r;HhPhMdPxMoAVd;8(+ue|fWasJl){oXu|1bI3K^8;F`jmFVZHGe1{SOLH-!1?^XAi9_6$yzI7J3!f$S%y&&3+1nu`R2LVP3!3`8^l$*b^@A;5WL8o$*M>8O8&S$N2F<GkOr1j6#jmNDi$6G)9u{SPD_@Ndb`nai^@96nq=elem|}YCt<t+;%B;TNZ6P2x2r+3@~AW%JE{qm3L1`aV2Rh~d^o$>rF*F5wS^JvxYiQAYdE;=VXxx6S{%*di3_JilGb!I$>j{^>=b&ZGpA7IBY8m;jy;H03yd-h<Oogd-gg?#1N?j6g}+@2g5~?Z#QDf5zAwpD4`iMv8cO)8=D2RR>$<LGcE9fcP26`muRYQV7KLI+4Cv%AV+xY6NRvM~3Bq=P2jDo$?TznaKZ+wV7}Kq3<T8`)a(fmtf!Rd65yddy6a4O$Q&7jD-gk1QXi6;u@X9&eoPbzCsws<ZA;rVHh7-dkJNK7tqr=d=5JLeoc`D)*hGY;E49k+Aq*pw&rApxh+-*7H_WEU7Hj<c9Q<621w#X<eS(I}>^5*^n2%s`rgD)7PoO@We-?rD}r9N6gb0c0)jaf4(N_NgWyGR855=j@D12j>537buc)J6UPYiX8BQw?#Do%nb58M7g%>)>Dcv6anMi&5l<uyVwGX@%28Ind%4S9IFC3MfJxrn>d^^!?dyKYWymfH-65JC8TPLk~Dq=4YU-Nz}nh*Us`{osH2gQ@L|6y09sC>Q$3BlnB3YeX2$Giq<9M`C|vc<g_FxumkUX{GiaJzk%?C7mKTl^a_S;wAD-dHHJ>Q3V%g|IZ(7{&)6TW@NF`$4A__H*4c6H40gs<ho(a+U9gBT^Ylrc@YW2sFab;POthbOEhxfa?1*&(B#^y)6D|Y$FiEL_{TfCI%PFv&EH&tNc&#yKMAZ$|N`0nSXi!z_gCY9;0^vYaoEFU=Oxa(YL89~F4!3YO)Mt=1J1}qE^<0Dy%Xk4K1Mf=%W6`oJU|GD)ELbI+3X6HkOTB5sY;mlobeyMqapg<7--!`OFgwl<a!O4n4~yNeC9#Us-&}=Zv|^qlNv-pU%i<W;UG(5xuPBP{r6Zl=JUu&R3zE)<)=oe!CTqHhVl67Ybc}h=N7m?*ZJUS(B7ccMpnWXgbdKHfPn@l#d$3<o^99r#x+-lP2OtbKtcpi<aT1!|EhgA=vL{|%8DG3CM>AB;HROhz-e}cS3_M<<;GAVKdQ+pATyMA9d$4HzncI-*Tln?K5x<sz8=e7}4HR=XLPrD4kL%yT81IKbdTj#hL8p)~dgSa^b<uqjf#Ffy>*ByS#2|uO^!z3Fc}WIS7RIG?Zpw!^UYG*_gtO<sK=1}E;z?MAVB7_6=~}7EM7rBLNTT;90drE)m<GgGkIS(S;4O4Psk`NcqIq|vTu^I_=_fn+)}iC?9fmjG1$=t(tS+#E)kOxRpNQZC#6?%)0y^Ab2PxMUD8=QKm=*vgmeehD&K(1M_jI<bo1|%=Nl3;0R$~%SrYevH84~|8c_wBLsDfwjz?zs~inbjJ(}dYEP$MSx9>C%T-QNAMVIuDyao1yFz=AJi&$#tk&e&mkV=@_eG+{JF6y0j)Pt~cWcK2NYQ%+*^$W(z*9NovU!f~Qp=OezVov$9wOW<M#zNfsWduzK~MSQJ*=Tpa?#U9GMbr7LawPCcv55y7T6F-W@Pv2a34dr3Zfwi!a|2gec!V$&;1_xUV35b@+FBZ70`6!jSdth*5v;)@^I`P##%6_zecofg8kd4+#k$3t91v^^OVb7jX^z2Y!1PbePW+0wXWDq+Gno@Ca@(yGS$a7ftga4zj=vw}k@Xb)xK`E_cjfRj*h^<$NTdvHDj&5^(TS7Fv_KZ?#jx(0$A{8yT;^KKz7aGr2v{8s6xiq6xd}u~VVV`oc@S4zieKcqf`5>Oy0k-FIQf(wA_}ne~Jkv>6uZ1#Tj^if8=qW%Rt3CC)yrFl1$fW$l>aJb}ie8CZdmC~%mUYIkI6%X;xq*5rCWs`^G;M~s?M`ct2;GzSNR9Q{dvwdSab0>(fS)r4^kna*!9tkf;SufcLS=@x<jV0>;&Wr!fd&bUD&d1&MkHQ!Zrst1(MWv)XxK1xnvF!bRF|(SeaF-lC!kWw&;V2~Tlk;4sAMF^)wZ3jVU@FJ+Q)IiHW_@_27*hIw-xw|q^Wj&x!vIVuJPVPGL9sn44c>UpiX8LLCB)@Ch2aV2l4|5!l=UcYka=~@uYqy-y(8CYz#FBOcdA<hm++dBUTTm;e5``8rVetC?Z_JHf@m}zYa9t8s@!R5EO*AUgweqXXPB1Wv|sx8K9ULiyCUzAO|tbJy{2xZE9Da-a6kmDMWmGB~wI?zx{OOG=Hd-;6kV%d~XOUdaz>nL9Z@x$~8a^Jv8?b`GFOB>r)mBXKSiRDl8!e?R@xa#=nNf+Sb0ky~+_%aiVxSmlL*I6dLe_oJ@EFI@NqNA^AWSC-rp&pA%GH%nz_~LJT&1=BmOSw4hyLh5)?YwPvfAJ?t8u!)eI=vc{&UBU$uj3K*n>&8VHIcQkrr3*rtFY!!u|)VmGS73KEeYwv9zIiBMNt1$MFUBl6gCV%FzId#ewDN)v^Lty})m9?dq{6cvvmt2n_nlOmljDV~eUZp(;(>$BgG8jG;b3Q*-J4Bm~trUQXkuAAfA5~RenEMqFqu&dxt$N)~xI(Qq?7S=DZJDj`)y3Io_}8_^FMucxzsC2d_PD`3GOSc4l-)U5#&eZnmvY7dVZn+MT!Uk>Y6T?H3L<yx-nwJ9d|IB|`$BpiHhYhQC6XZ)AyW&MfVZUvnX6TV0ao&S2|?0cT7KmCb7Viz$t{e7Jhnc7^m6vo?)%l^5_NJsl$97*rH0>nJe}h1P;+^O0072?1nQD!(?c1V#)0J~-E)qMw>?+nquG?mraD;-i=yE~r1aN`5+xXG@*@xk6o*M3&~+TnTO5r@p=D)e<zb?@iEY1ura$7f*mdlrK_Iv;iih(DD_*#GS42G@L%B_OX>JM7Tzi~fCBF=0NMns-8gVbmM>Vcl7cyP5&h=vj5SK27GI#qj>R(6s@;+{&EA(0EiaiKe0UjK4G6Z;kgxh-HC*zrZ*X=x&(WkzIyqpl%Up-!SmMke^hI{8Wr^-f!-1)Ncg>TVR$J*t6-VF8i+Uk|QQJXX+J|W5Dtbj1r0L8lBFJE(L-6LtW!IzWnxm#RKYTOq0{_Q)>q1xZizVGw3ociYO6mRW5+^e~NLVi)g2Btvr@8_5!IDJjjjQN4>xPQKr#^ww&VhEM!Yg8fc+i?Xmy15q37m6vA1Uak7<3X(=LBLCtQZQwYC60Mw%8;2#RlO|UMMpZZpHfL4xC*zBrRf_s2JMFqP02MuW=|vE9zkv56s5n;!wg%I>LFdxIJMei@kNKOnvsKj3WIK8qiMO588>d!(U?*C$aZaTaa|n+_8jcW-waQnsQAEAY{(JKfMsL(wI^2}sbs@K2rf}EE(+zB4x}~1_cAv=*|BjZoZcwHUsFMd^Qx#c0WohArXOYSk*bWH08N>Z3_F8tUusRmHP(?0rWU2Pm14JHbd}c&jXPM%bT*!M1tHB~v0Py!>nHSreFlXGA-ghY#}e5{!A_%du3(=(<0Tn+_!>n}a)HdWp9$l}4Gl*&ia(7#%P`|u_OpWJY@54Luh`I};OEq8Da*2Y=wj*F*70JHs<h2;NK3(#QrW!{rUe_s^Hq_i+U)b0u1q>S6XZ3Xn>>IOp>u{y&7e-^tNRY6ING4%p~0vAfaM>yEML6BfnYWHk^%<3c=#iS6KS&xQ@@6ES5+Fchtd?I0W&^hmOV!6aH}A|pTq=7g4tDRM+%(I7?Zi1lyoB<jVre(8g@x|YpG>`Sa8m}n0*itS1nvH^)kY7b@37y%hB;c5-vHN39kSeXOm42l$j+Kedpp&cv0mydUCa8HW<U8?WmK4)mZkN*M(%hgmj#QOL!xqN6?X1#-ZUBTjTg461tJo0$zv=IS|H(>(MRA02cm3ZexXSI)Pf`FVk6^rGmT*MXm4ybJN!aDcFNdq7;TnH;{=yOuZps3mnMH@eVbb;L_YFB4~h13%#zaxJaL}c6%s_ZE~w%Fm_e@+{!4PJdnFq&$(p|-e9jDf=I9)9M28q0F~(TFmT!$tBOa&772KKJFsf3-`S*BysY6!%UEIAsszlm@<gOW-l<ltOg(cAM{eyHEwMCWTyXV<l$dz1<d|l{fvPzmQALOuOD!S@e}5z`{fZC#xe9;7C)?RYbXYVIY8jR{Xo6|I@EPEajb%wIMd$-Q1EVf%N1{iRKj=d1o8X}+uB1;ib)B^nb-EaZ*Q<DyitHeP6em;mTA2lSB~c(SvTJ((a4)niQg^`nD+FbY+h19@C5eU$3jF?x@w*jmk(b^ky5jaFMTs$bR?KXjaR>~mA!3yHc}JeU^;y3h#qmulo{56`wN4WawPb(lcj*Mm<5C@RbfB;yXiHXFjpkuGv-BnDZ%S}EK06Y_B&c-6=zz5h{7g>;UQ~uu349)-Qt`5Ok-_Fv`Gim8gz;qV8iJ1DPCkEt7FH?RA6w3c2!3*i$bWVwl)2TIy~jdy7mk8E$7JtvJi(;poMW5^x(h3w3@tD4-Z;^`Ia+xTkGNF_=tvC3d`|izT{^R|)AN`?L6{pMAE>}@W$qO<IfZiiOD>&Gk@l#`U0p#&3jkQ54B^FgRi-$_i+Dmk@uH6B$=-4@O*Qqog2b>5^Ge84zzryMAL41yQ&ooY&h)v+6w9-MGbNlZX+(yKp=8%0SGxlWA|VxMHD${{O1`pbLrRK~CwQwZy6ax6JsDN&{4nULO2hntzih0@exF>y?vdZ+BWDB}lxqY*-4KOcI~Y(FUP}+}V`c@abXml*W-5iaWjt$R!z$yAQ-KQ>iEC3OJLiRLzmwT*jJHp!-Gt`bh|EWJM%T&IQKoAxVi+arTX<<$X&`^2GH-%wBoz~Mg4{0(r+9B`65pkgcdo>3RFd5A-^V92609<cja{~r8@d-yBtWp`%bcq$7FFS!z4EjmB<@l#w&G?{f=GQ*7GJR`i-GdknO-Z$3yY!c>~P9I;YaTlwHIUKC{YZ=S6iUvXgZ@Jxo=9<4g^1PSg0}9T(?^4ZNeni+!EEta`c}p!&>R#dX*-FGACCSFjkb~MJnP9rW|CTc2c^w)h(d8d{Z!T3K>*5kZcO}QBGMZCu4e@4-eF0O1~rySTrPJ#&fh5kC91=q!W|r>v+7P6}7HX@sv_D)uMr+#GlqHCWa-UsGRBL288p_&t)CSiD^Acv}Vz)Co%(Bmen5N-*|&buz)uBJv?E&M?N*4_*!^R7Q;zE4jL13v(UT<{V(*dCwJ`#ffxf2^eIMMqyXK0)T4jO^N!84poqyI&uEJ(k>QQtavKgip@3W6IvOsh1H8cO!4j4+0H?a*I%_dENvp5QVDc=}h=fpC@2lH-j@@21TWYs2oon1U2f3YcnDec+ZESRYU;prt@Y67G)o}hQ;LBp%HCV>nIZJ_^cswxFz~>W^LUkjDCHe&lib~W|m)$lnscB!WtHSc>S1hVDL)XdchkkCUuEA}2*l`s;T6k4eP+vD!3=w9+%?#SAN>#BHMZuI$jke6Z3l*#0sLULoQI-hI#E`oMCwQc_+A|ialHc@O8hL?t=0@{W-CNlxZ6;GX76!6v7($>l!~F7gBV%*4%||K;r;r59ftB)nLs&JQRg%M}r)^*y6mr5*68}dlWRG{3twQ^zc3j_j7}o5YxMk(oqRs$A-X59o2DHax?|4t^=gCTaB1|;9j~cjNu&s$ks*~j8KqyO*Q{Ar3`;ETVZ-w&wN^OODsL$RC-!dTEZi3G~Gks<w`E%4Y`<t!x{Ym#X)04AdIF_%AqDB#&;-{DE88=sxsrz6a`g%4Sj)l~I_4a_DwO8qyg3|BQeJ;%#%##ba&tuBDh0<^~le?-i6=ncR3sh*RGBK9fLoU@%>Eb8^a+E0)=J|5rAbFjDjK@+cXdO?!)S<F{_Ep`4%Gj%KqpN~a`pqIESLwso^9H@!8I?RUS1X_3K0dkY8<JHGwFK!1w@c&IhDI#k->*p{TrO9ZJGX8*ih*=al>S$Sd*b>Af58sPF*fD2qQwO$!n0%SsHIzJL3u10=cWlZwMa=^yN{bT+U>=E1N|En%I9j527QSxjD2+T2D<q;{&`Aeo?{l2N&(~+O^9tVZ}?odF@CcRQd{8opp3325$W{%h1dq<e&z86>;5UYoRqJL=a3fq+OFFvX)G87#EJ}SFluD9#Kf~oENcbem~J4lIgTnQ^8`^PIK)7CgFJ0dK6Z}1YV}7_)pndbr5iVg>e}#ZeJ#t_tz!@I(1nauhM7?t^g(6Dq_()QZBt;&Og~EB^{EhEgM=2&@V@REaZ{U2bt1r)6d{UeEH&YI%5mNDQWS58g*dGeE-pOlWI>{kl9DB{T?L_9wN)DGRVZ#aB<cR)C8ODPyR<$IRj8Lo(GYr(g*<38N>qO|{1Nlvp_N-#KDbmD%moxy1*=WVBh~U-qPh(UzWsje8Bqil9(33t(1%2*^>T2hvj)flLe<9icO-C`b%yG2%9LBDMN@H^iBIg?nw-V<y-eb9Kn{6Q`hahx9yIIr@qB3T&W=RCWM!!q4$`WYl%K<+;cd~Y6m%?0P#!V-R+&P2mh5T?HYM`u({t9pX@)9Wi(Ac2`MegQBrBNrW?%BP7-u$KuAOaDo#oM-x@0z8l3=!?sddIiI=J0^)SW&_^P<&PdHd!G>VJy62+cORBlQeiR35;vbSB-aiZWTXWf>2;3Wj90$qF@nRI5>T?|Ru~2e1*d!xTJ-6kB?ScFS5V%{*-NSwSJX={0mA3juAIJu-D*7>M1I8E<9yRX$3W!(OSsVZ*6y;E!THQ!%0XxV|Qxay|KdFbD=rdqZ+W%Q<h+JNDBWYR@#Y(IMG>c~ltdZJ>!5y?GR?0<q<bbtisR+l0Dcu;<h=*qkvxifws*3G+Lj(y_ls=_63Xj>n0&E`<82Y%+50k9k#B7tE!DC}&`$#g(@tD4k!vQ?69IkCjj`HA$DM%CiDd#mYo2y$#rjtHNEvY{mAyu-c-9FU@-ffc;9qmBiksBn#}_AxpUOYUZ@h$sqC$D|9!o@Nn64bFL?S^a(0m{lX`Zpp`CxYICxE0t+6|dH+*mT5~Sc0wsDb#pu3jN~JTBD1n~T=!9UWCno?D@&+B!^K}OsVo(+<zB6Ky_t_O85uAXC)qL&XRAS#YhosnmwDtMpjFT^Z6_+XT-qp^gqG@Z)gFJg1nwOX@{AE2`E$4Lpij`fRMNquh$^}*S(G&?tnVS@hB4!%FVTr3<$>=O+Vj+KS@Q0S}J_USAA>lg6g<Ol{nP<sq`Acar2l}Jvf@k$<cll`biK^&R&w0Fi`tqGtB8u9F`whC;y?-1Jbb6|x)$jI7jp_Wag9XT`-?$dX@^@0$Gw;oj>?A*1#oGXQ3pNO_VSDKJoHe|(BI5^z#$oab%D`P;Hxe>=^J`fPV_4TEA=zRV&9T_6Fk?AJpJ61A6F{2}4pBl%L63Z%9!8jl1vwNldCO^&ruD<wm7Tg?CbA|2Ir^<@aOhjrG?_kD-)61i>yo9)uL~>HPSI2d_RH<vl+0E7dq-lp80XBW@s##i8iEzyIU*VMU<)7{I>e0#QM5fQjh%wF(Hjvcfi2NB)N;x-iSnxeZJRd50A>CpOa&d%nQU*pE?NvnMxfYe1sYVPFH0_cZP_7;uqY8vDw%Q_JuPE6mA!_L-M5Id&wu2tao0ga3D73yH46DDr%X3A;qe_WWO|c|rA>FPRL7c?g`!~fT$GLlr3%zp`r{k2GnR``FpOk8Gw#OXMfux{OYh^S4_D`}oy+%c(wAaA2TqKU2-z%573Y{Jp`@1Tn>r?#aw<s6uBZ4pT;jK{6Wun7ghHa>K}j!fImWz;Hwq~HsmwQ-Ws4sHRtlj=Wyt=iXj~?W;i|BiiMEW(r35zU#I#%l(-)3BU-po#xBSUO+8bnqQi&_8Vt^aMoa}~Nx3o*4P{{D+8`WVkKDopTNd^(SWirp#r2wdaPly!d@YyaSWZM*JMK`p1YqeG$hpsbsCq|U9T<c#U#pL(MuaRmh#$!^N+5HfIVt1OL8fw=n60w`#7}?xIfLC!j-h6oE%<IeeV2;?$ouR^MyOdU^_Phaq+{lz&o25MW8aIlg5G=3t+i;2OMDin*%#^R@0)+QfNyUK0Mgvt3_qkz=Yv8z*UNOR*5=G|C0nb!#8_teM=dF?G<|UFan#Mqc#?UU{jquSRvP8>3zuq}H28pyo|K~Hub|J6=Dv~mhz<TsB!T=iH+v^-3JKulrbPkL5kBx>KJJXUC_$(0BA}d9T^HbO1%DXiY2wig2Pd9w7@vjq?sF7U;Ehjjv@(aeq{EIvG<n9FIK%1t}H4NNy+RGaCipNHR6j5HQyDc<cQbN*gl@gNv#coV99#M`5%)ul#=`ba63nU<slj%zYBD=?C%%4uLE>7Qn`iQ$4--=qtq*s(+fF3$S*GRd5uaGQ7HSgo)`PoP5v<k3Xog<~6e!BSZ-uv<N;_c_FbD;gxPv;-m$m#=9CVW2yA3KqbLDoVU1drFFEMTg_(VVs$Kt1Em)3vc6?ihT1Dof6sf?p~`l#0+1juf9Gl>n};?F-18WQhIL@nmERUdG%Y<CUERmo91M*L2w^J{VtD%OHUXRL#^bWuUK=qgNUV(BOa$jEj%Cxw4y&T_yf)p+{MQ-%nysF*G_A#!JZ*unp4bUWz+Z91M<?iluWm3nJ7IF!t5HoD8o;>aXAH?8xo=tR%4`CTqk1MO_xk732SFx7nge`$N<0fp5RbN-_X+wvP5U+B1+;9`&!b>J1XteWv$WE2R;>DvplIix)$V)Yb)`F{~?NC<4^Vm~EW^rZkIzp4M__blaX`=QpTGaG-h=cL3@ksxO{WJ>0Dass~~kQ|<V45hS^ygI`z?bVXmEsgM7aLIG~gpttKXkM*TCp)Si*bY(<Xn*hPk2Qa6kOchZOhV!DB%hx1hot!p9)&tYsJB&<ym5*duvMK;gZw-GH`AU0g$T!mYAs=NdWh~B8Kr@<elkQ!itHAJ!#Nn0zxsuj$Q?fDd(CZc8YYZ%JHx&1WCDyXIHhEyJFe(WscTHW=K*^f`1@SF1FDVf0uH!p9&jWVEedNFp@tb66Lnh7nqXfopB(hTq(v<H^Ese}R((QsWNvQ><URf?gwu{aB)Kiwnnw@o%ok?zk2s1Wrm7@v+bpchB&cL3gl+s{Qif@%|M<l!x)ACYSARWprfbSfDOBT9({iBIhIh8PjBD1AAqgn^y$#<<~R;-2R-_|`LSwLq`VomS_6t1#>LW+OJq@LpSMVKrVMkfTF^d?*`0~AO->&UkD<ovBuEm&=Rm!64aDU%eyWllMu_vD&PzeQXRyhD*67kA;tL&D})*m3{lM%+J<1y}IcuQ%S7Sw96ji6-&lw^F+-_AE2A0flaz)ymvC<tF2#AW~C?5>AV!4EqAeT?W0(?U*+q#f(ZpWi$w`s+c-0N2=toVL>b%gDsKRAoiAET;|SRS+%5=+==ochgIHAAve&O$A&sFZH^JEw0_?vkKcAvCo;}(YO*R3uZ*fZ&7koXxhMesGWRT*dXv?k_35JaNJvqZ_B!X`A=w^}$*WE&gcM<H8)jlW6I|n_OY?P@mSs5E4vj`L>yVejh@FnkI%(<iNsF~>E5lZiYDy_BPfJYe`Mb#Ft>s@!K`JjJF5!yQkRqT|D==(`yi47D@<QGrv|KMRl5R*{R?3Y)Yy*OhlOjRORGQ{~M0*ik^n>WC-XZZKba|n&se9-X*tL-UNM;9X`<$X?4onl2`CyJ8WkM(eTsjL@8mjIKF-70L0+g##g;i8`_(q8}sozi)%AG0w(o?k^d9cQ7UYx7(kNl|`+cWgF*K%QrjiJa^*4_zkY4T9lNVY7Vo?ZfV``1y7w@Ff1BU?DIKdc~Dpj!ZlW3FLgWGM-~t0<g>iNz|PG)d)#xqb%U$YM=#k1ml8^qZ3CD-wgl`U1f)%rK;-8AjzGu1U+v$4(`@rg(7t>@1y<oJ1<0N5KGlju31Oe%>pYQw0KS3`Qz_Hbxa0`d2#>7$@ws#Of8f@*26^cvaGxVamaikOH7`stX3k!J`mBl_R%`f6&ndI;ucnQ(8!6p{Jn*!nz~=HL%?D97BjP_8P7J8-OWzSbCwfMkXBQ);I~S{m8q6(h25>;^gXbr=*3#K}o*I#Bj}j@Qkj+O0dyDA@&+uQyL22ngSH4Y9U=tHgVY3=gl{#?;X`ZQ#2rT3T}t*iZz^YDo`R=h^6s}c<+h<NK-17GOj+-enaS>WuWV+_o(ote;KHH!ZRQ%&UTyY!w4MIZ^ZpzO@}WK6Ze(g5BB7{&aim7Vx8gVv8<MKhoW)Su!nR#4?J<f3Q1B%1?Hz|h*#hsa#{X_R^mjIu6OY61+&h0=HvB-LE@2sP$SU*%5?Wd4ZOOO2es?$(<NT`x!r0XY-7}cdHnS8%8~jy-%Gg&Kt2Ypxl8W<4&NIx&dSJQ6OXmLV`P@u^|K_eCtAm0jZC3jyqLN%6<G%89z2&5P8a4}U0YO`diiO4S&xeEPVlyslw!n7B)zgVWaNLQ4E%WE{XwVOa5~-N4NF?go(OhAk=2iGeVtVjnz@HoBmfJTX3pF=C?kL4J|`7<SwPFKb;DCbRBwpyVp{y^6wSeXr=Tem0zey9s!DQSq{=j7hgBD+_$61ga1n7ii?6Ri5vPP$X9S|#2;=_d{Z7zqpNI^(_(f*I=iVC<SMD?9)^QBGEvO$^pfH^$fvpepdqszQ%c4HbiNKFLJ0`Ak5xA-^)5!Vwq=R;;E=73f<s9b0`VEC>#1qijl)2exKljC@xC@CDPhiE9SK*m8*A;Vf_q577>Kzf$lQdzfLq+tWl2S&M4Nl7cl!~NU=C^K@!y7W?8T2FwmiRlD(pbK>1I_5Ni@peuqI_-B>lUvFWltgA5K$PGuu7B^(oqvZ-YI-LKTkTiV-9TIlJP6PC8Jl!ftYiqET$(ALbIpipP#FF>0y4AXCtL2v)?H@r(1U8tD!s;x*m#TxDSQ<+lsPh`894lH`4nYzKauf!|p)8w-Iiwnx%QE(UqEG^~mi&tw7dg?9=nV0@9@Onn31UCX>jnNTaMQZ$_SHC!SLFW}&S35%5(Gy0T8s`GBMJ?3LiKk6lR@-Czh3%%)b7IlyLAMXr=T(pkq~jMH7C60ac~k|pNii-A#WSUTX6auuh}94qfl{Y5w@u>JWpNR-)ZL1PROnC}tm7UcU;968^OXt<0}8GZM476x9hOv5><_fI)8LNFy6jbP3#M?IB{0=eVn4EiatP+o<GQ;2OJVTySj^W765U&cU;Nx?vmrmrlPZoI6ur6O^NoaArgoiANf6y=LUbS0RiK_K!{f9-LQIx$$A&weA@Rn3Ti6-qNKVcS?+7o}sEiesj2BM|a)|IGq~jNEM{w=5yXN^=@))^$Y86^h$vBoi$-s)baTj7@8KaibG*1>alnd=aC1OUrFi-&j^qXU61~hBAX`y-|i6&PI_G(oo3s6G>U#37N7ZS-Yh^nQq98!{=VvWoY6&j;-zhQvPamvP5{p#o>cC0Zrwv$uiGSIpb&;Rrf|8l-Lbut=x(2l6mJ+$<R~?zY#;P4R|^uL;fN;+&0+ZuqS}c$JjHM8<-8!&ShCSpMIgdv#xn#L#SCvAhdAwWZyP`^_M*<N}<TVw9K;7kVxG<8Lmpi<en@~y~3uXkF(N(#ZjNVsEq&G(|Rk3O;nS9X?j(%ZM-}iD6TA8Wu_-u6DX>A+o|RAH&_`Lw7s=Nisq~%h-pb?Pq}zz0d2wKRz&Au;>HrGAQilB<f9KRi;pY-nARjqeX{LL{oHSyKEsk~eauOilqfF+SzzJ!$&b>+7y`^FQyve3+WPH$tT(*C;t`Y)*DNp#EBPSuB*TZ%gTz4}1qJ1)t8d{%Cu4*Ma~pt=y}(Tm@D!%~+U5H<&T5HXJ@N}{m5G1nT)lY(y{=N!WOo7gc;%>r(UCJCjhD_S0K&w3swTo2i)CBsU@Y6~!&+MeK04e&rL)V=g7bw73L=&#Mbp)MPJzd}Z=i^*tQetyACTQHiaR*xYvUK!JCib-D2v{CUZ|qMDazE@FcM^GyvDso8ScfkFv2t~Ov``)M?oQOXf#?f$`!*oSm@7_9p^H^ORRH@K}9P!oSC!X1YKmxD(+G$jimr^t#=RkxoaIAe^NL^nTG(VD=g*Epfu5gDq}d0iC_v`kaEEoOGvF5+bs;=fQ%%tLCcaax<!Dj8*WMGA*UAoQuJxgUOpqVK37l)IbAbfW-c+Nke-YNS)HtE?zvr;C6SYewHQ)mD#GepW%6@2sA<fu&1_g5Br81P_+%{4r|Z&v=6=6sg|Jk3)i4%JkEwDuGKtl4XBGA}#0{DP8f$A2kn=(ON+gJb2#OXFN_L6MPYjDU#?@QD)8`2@`RM913POf9?U!85!m$b<L}E+vtb9oV+N@<(O)*@U{avUKKUY&?=}uVLenqWqI+RrNE7S9+>fRBt4?%PFJ;ci$<zFWZ!GAifb_ZXyyZgoZbU77e>s4GtZPxOXORQ2v`D`<bD9b5~YB^8c4~hw63i~`lC{qOV0>v0G!D-k4gf&#Qd%lYtLxgJ8zEA@H1@Qy;v(O`mE908$7nfh(G9^&9M_z~hQa^#dh?62$oPJ4kl4ADq=7_?$`s+vSN`}*~DK`=N#MpX?%9_8*Vvth8u``M9B3_d^I}hW-pG+u*7tdo0G8jdt4IQx?^XFWBTfD-uH6gY)p=k_)1c3vkI8ptj;myrcPPZ^SAew`u6_0)goLHfQ9pC|rcy}?HU&mRk)FSs?<Gat7A3t54pT6_1&VRbVTwEM$_BjfFeQ||q!u0TH>hDeW4%>%&qv_Gf=;YuiIP4rvPdW#qgX8^)f3UxIa<GSb-B0K5Uw^prK7ROob#|U>=&=Ufync7yT3|?C?N<Z-0#Y6<Y|z-}Ad=!WAY?8{{AuRQV;F7tvq<Zz4yl3s6P06<L$TNnBK+T_h(4^tBH3l{qbB##yzRD@5ioyHJHABUPr_StsJ)K*U%ndyc7x>0cR%pAKXVv7@$1k2%z2G4HJqRDbo!y;@S$ALb3O+1Y4a+W@@;Rw4)HZj(}e++d4xxSus*g=fnx98zJ15N40d+TqialbwX-v<D{Q&W-#YN=|NWo;?%)5P|JA?$KmMox@L&H+{oOlzk3~cSzZ#sLy>+_n?xFL}4*>(SEL*0%-Q93;-AZR+FrTEYFy0;c6OhQEtcPWrShjh5+I@B2b#)yd_*n$xG>@+zT5)o{i)CBzwmFZZ>sF_Gu)phaua`ck8XJ0}GEMCBRJPiDwbyyodG&RDzC+n*L!VTpN&o#gj+a66u)BZy<Bvv%cXxL!lEPB_-Wo@{{A>5;pYNwXKRoO*8OU)y0BoDk%7vPHtxmhScXV`k`s$y<aN_=w-v2p1pTPi={@o0A&m1gVh=wlud>aC3roo0zMXBc5nfK~9&F1?v`1iH9-)c9nlOTwUUdF)x(3poebmHL(b~WeP>F(&E8I7CKWEW~m0N#YzuU8|=WtLV#aScSh8tqb{-QWK9Z+`4_1nT~InmvFlp2p+QpBr#+jTV4mmlw3Wv-6wPL+Jc==114xr>o@;fBt9t-Iw3*;-m9_|L@LUt|Ab3-iE8PkCy=^<-1-}8+MyH{CkZp;EUhRf@rq#o!{U)`27|>-USg9jHv!U{oUVl?VTNl?;~*GnR7M+sREZW1skGyVd{Je#<Peh7#n(xCt+^`-ZhC`{S=bl=~=u0E{-qX`Xj-R@{peY9&?6~Z%jB^5%>7>Xp}Br@_!v0G-whSW4eqZvN^*xHj5L{@-FZ?!P<Usc6J2s4O2R#n}$h>o;S>p7O*vhunOpk`vim{Sd!N(N*RVrLuBTdG_$w~f1H`-comGewX@?~WX#l4XX2AT4>ET)h!YE+&p{qQi<#I38#0nM6&_CLF_AhbXu+P0CL9zyI3*(&Aa*RqBaIHIQvmAO-94gsCppR~>Uno@a?_G5-ueIj`+xOso%g|A%5R#0Ct{E7qh|X6#_&&n_ph8!GrSrc4J&{<b+@eJXHeMUy*oQCXOL%%Td}tx)_eWqvud@Llc_tb0{TJ*f%D~>bXzn?0ttW#unStaBWD}7+$5mc`6<RNOWgd^fA>E)A7Dz*X-tsRvUKlsobQ1(IB7D54|)Rz`D+=V%Np*#U%rE?8GILA%Q@LhN{(y>njj}4Xr$9_|1<9U``x9WanW!RfZl*vXEM+fn<<<9a^C2&#Z+$0>|ZB{J)6PQ!W`1oh7KNg5{X7^5dwBchho%crv<ntDWUmh72^HKZw&!POa*3!#eg9p+XQkwOw^?(Vs?<cQ*&KN@h8Pe>0)2M4uga;ARxlD7!qJIQqdh>%N$IgM<F$(GGihN!jn}rA@eP~qsgQT1Mr-5;j^$g$1{2iV~451LwaN3A<>UwK-t0{Cox}A9OBbFxP=`Q!Q4YIPxTr!#2^J=G{FE0T(-Q(Cz1t#WH`Q&2ebhu-=zyG9oRMjAOAmlZ`vitaU6>Nl`?SdWA(76X8}P;bJaM;MiEZ{2mugfduC2eO?S<7X?jCf_bjC7L)ls=i;`%GT1aiywYAyzlCm$aOi`qG&ig5T7y$gF_Y+=3<WjlS(mg#OC=Db6T~(Qpk&%&+5s{IRm?#9<5km{`bUQVQ8!!FsVKiv7L;AGz)H5CK)A4~OP325KFL@BvF*?LP?(owJy11*L;Q6?U)d_AcI}mzB1ISKrj%Z_lH|9o*SVAzH!hEI%*ZIW%EHwO7MCqUA&_xdYj|cO63A)a^aTi|C$6H%q^TZSb59;=q>d%kH1X^6fM{T!39L&=u8e4dngP#RH;@X<Vcg8HM3~wD8zW^`Vb95&;if`29Gijfx@@=)>L)m2}$rX{Ut|8ROtr0Ej>hz78%`r?2Hr~5QiVG&Lnz$I_gvQO`6c{wYenZ`CZIE2uJW$tw@B9J>lZ802*I$q?Koh^3^mgE%TkN#R7x44^{Ji{+ij&kwtsPecsqId<450Eb-8uj*pO-sO93xBI19q8SJh?vXXv0`#pn(m_J_0)J#=2i$5D%`pf_LE4|M$~>{2EYhFaXi>Gyr6{0cuOjOZ6w<|DUKYFD<WpVsQ~PR1MU4z(UN0SxvCTu)hAH_nC150_7HFyU9cXNbQRqR1BB|qC3)IqObxR8ylNZ+J;}Ct$Ix=SEj$#;R94?wlWJ6z+?@A=)Q<5IBQYKA8j^?xrHOI$;!_6JhqT}%tC0OdEw0;$Z|04lU*#yiottW*l3c25clSX;&=?Ul-5W4BNH!>)t5lE!tE5oOwkaC1z1U~h)@}5Yw2k0M5`XI)dZNWy)u^_t~48UUUnm3dw6{etRjU_sFJum_s839Z)1L*dDby!(S{=bpQw8`$0*7;r<x?c_}NS6Lv({&xOzFfbotpU7cbvxEOL^2f`Pp?`U9%Y{5<G7)QT7HM|g-WU3ltNxU?KDo&7|(T>t!FTP+Ntjz!pePpYreXg6j+>I>rhzVbQiiYI8GjvZ>GXv@*M0FYoU-RrZYgQ)XvNSGTdVzpzeP6~5NT=6=lEXw#r9&lul`3MQ!a{ryGMjm?J>7@v85cLvN0=u9dAQzH05D&LaWh+x3>cSdX{fq0*+`ha{zdsk#c3ig`)5f`X{n;y*uAKMIgN`@Pk`m!IA}MNfb01&yK6Sx+_U8F(piIzmDaN(#U8ChrV|vh24$#ra)&+j7Kw01rW$Z_9+}#B6-QQA{Cb1G{cq2PqWMrSGyd+>Mk$xf<jr05k$f!$f8;gErI)l1+tYoXlM#fbeP-7#Hv+&~;8X}=W=IGC?rxLj2LW2Gz{(W6`IcI5B`kw)kd`o~bKErOxsmu0SRbFE{1hJ8-O-iuJ)Maq_yvN12Pp4~fDsZjPwquy=P(}YQe=XOS$xIiJqnlVuvsvU)g*FB5F9M(vYeZJz$wP%9k&ZVh<VT`6f9X?KiDG0c&u=rmQCFU1*{5&a<cgXyWTY9IDwV8qp4G@#v4pRahq0Y+;+YSEag!ViDcK^QP2)`VNO3{w=wR;hPy;fKn>>VsTj?0BPUo4R-7dvO(Nxr=4wvj;=a8|D+8vr&ERu#|g5#`5G9Nrx<5|>AybokfL2d3Z{_BB?OG0IcJ}B&Sj09tz2UtIt_p60GH5RU%%=5qt<5HT8#;Obeg><~x1<e4BMl_x@?L(|a*gL@Mzg~PT#jLR?D`2G?p^Z`Z&a(<6p7?<p(oiwh!i!<v_YY#g1ZHNG;da162MokO>YeXFNmO*11Q7TVU$9emQo6G4ZbePpZe<yF(6;9~IPf)|i$)ewDub>a4=4|ReKIwTXmu5&6P>jgjUqrBbtz2h1B(?oR8)5gaz)Z(6Fj0k8~fe?^fqPoc-rpoc?ZB)2!Q?_#Zi|Cj2;~Tjlq|JDi5XgC^DTYUL!uSWDzDWv)i3W;fOrCxCX}<G%{w6oZrIj2tq&|cm&Q5Oa`DML2o*et(hsynYJaADPW2n4ibuvQ=|s5!&V<d%Ct)3lp8N>0}7_ed2;j#!;)Itr19+Upv>LE>Zw|VWvWa1)StT842fYe@1*CL9}*U};la62retuP6Y_vrjN>6&-&0zbc*eI+F7UGGi)6Jk8R49z5uO3Xtjv@Ny+dvQAoSl{xgqj8Lf<a7`+dMYPAGnxHARi@pkEbLTc+uZ(T$L0h}sYWVqt?q%In^hElF;&qInCb_{7`^if}e#q&GGo2WLaX7EyLij5*ryge$2dOvf*vD8Melx-U;I$n49oP!msyR&(q|l>T4HaL3i4{#-JkWsohM$}46!-MXpOfM>G?yqX`O9~abq`a}~P2#1pjCpk$MwKTKzv2cBHacSefUi{7m{CHBdd0ubwylfG9mo01-SXM{{cV<JZ(Q|XxKv5+B114;fL=v8MX>uUa5^{_(M9^0{iD>Yp-B9;Fhs(>y=Km%c1UOG=%M{aTQ{fH5hv~9N6gni=9P#?(`2w&CD-jnVm|l!>KzIs~V-W}i$q{6(j|+C9^^oSURErGVnYzV{6P{Q+Q|e{vGsa-pzTizcd#@(F@jl&w5;dQ><@p!Do}tN<wreYvVHt55XASrBK#*W@aT)$Q1OGh%|DA>Zo_xVO=dPhc<D;xXQEIZUs_F8%q}RfoYM(38-VGqsV5DH&?6%0l1*2<W)63=2$Me1{=YBbp`{jw;FK2VVJef}(9rtV^JZ`>Ow%(k{C*5VvtK5{sEgRhL0#4kMEN@H;QzoJSCe4dPYbJ8VY?S#lvRDTUC)j~$igvnSORe>~n9>|Sm?jy|vlAC$T0>XR22j-F1^E46um0%2UVRb%?^#k;`>M)Fyw@)31nEKVg#{V7a$1$c|M#tDt}_Ae1_GnD;0S)12*&xF86JUReueOi3viOYz&iLUugS)CVU0y*WDt!WJ<r%F;4=ILy8>cgUP{F;gUUAL9MxXUsyREe@Vuj{WX}sj+IE5Y&C)rF^-^*JUM1*VMj8Iu+fU7ZJfkk^2&ft|<)ZFwfTTpNml>ZM*{O;yC}(6%62otxjNS13jrJHM$%f=k#mh}Heq(zu+E7I50QV45=A}UUCT6av4GP5vc;t;@Es@DutBW;lgU<h_LR$ozH1+1@#O;Q{K9&6<?NrW%xj8;laLFeV4Hj}tT9BO}6fxAG@!&=gsq2fdc!Ghs$lNQ@5LG9b#T%de0@MvDkM|eO?JorfOTqqfaIhT6T_aWk^!_iL+o)&zz_6B+`Ha&SI6cN)OH|n8OLFB>h_x`@xlkedwbRE{>jG{9F=r9Ikt!9tAB|Ip>+?hqjf9NIS$GfeRgL2TCU}dWB2Xr;*gFO^N)=}oH@k=*S#`Ftt+h`r;@&OO$6MmKJa=8=>RXq4=9x+?cL)9fEn#$XWY+R$#<*EFV&k4AB$~|E5Cg}#_D1ihz_HMnx>@3J;AnM@iWSU_r5vmz?4g#PK(<qcAQ1EW@BH!YZ+SO?ySmXpnAEHL>p)`Kf6g3w#D~`B9OsW!PWS#$)8Bs0JKx;t_4hh)Ydco_c2M)e`A=QG>I_uD$i5c(P3BY302inrGj3PG2<tz6>87*Vt&5+%J%8!yvlp*Mn+v=kuZHP0@~R!g<KA|&9ryO4g{;QYNqVW@T)I%+k|@b5UW>P*r=#8{KKAMIs#)2Z%KSg_c$^Ulpo_wrqdKB_e1Wj|P=?8=#*DPpix<5Me~lqZ*DrZb)EB1;O)B#1)V-%Dv!47g#tqN`;{{qjKyd<9bE~d~`Oqdq9C!9UpY;ARZH`CvxHYafdqP~w-g^JfuUVMKYPHxN=^4SzkXC26IS=-4y%*yRl2xLJ)F6y+<Ak}GFqMS82a_a==7FAWjnVhRyb^*US??3R?<8YY$oHa{2+1ZabMX5wL7<}tBt~;Nef~L&q?!kL*qNvB5)#q@i42fVux#U>hWGq&MoGTF+GBA|ygiEW3OtGx!vVOOi~}AUhL8NShzH9`$oMaO_19ELgn;22;D<mk2X7X`npsz&*&7*YaIlb~Pd)(%gT9Cx(AVOaMxTE=pHlRj3@2dh5pkUwY`5((hUqrJ5_-1Z8H29AwBlVwglQ)^t#K4!Qa~H5=bs;U*h}<_d4l{nHiAqnjF`Q@1$OVHB;M`^-bIkyfJ85e0;Ddg=(pIxNLEjH;pk%D%9!)Y<6*~>$V{@`&Fi=NNmhSpvA*>2C(oW)*j;i|5$KXqBD(*v<)tU*=i#3-OE19Z7o<-57{3~JL0mCWAJB!o@;&|)!kq1-{LAS~zTr$c>s=7S-@7_$)samWC`2QrR79PG4v<pl@U0t{{%T%CWL}}TZD8Wzig$j1sRibj>x*o{t;w^Hc;!g;Gz8%N?i#<2<i?e&A`=X`k+h01VQ67vJ_dZK_j>hcgb_68)ZgmF`=NXkaQZ_!Jj2}b?5D&Ck7Kf3=MXHtVH(qSU${rZPK-f!<g4Mo0emmSr8mU`SG&-0GU3JLAkAo;MjZxaGw?tf-3gP{e&Frw;=i)25}-~6ymJj+gdry2p@Td`ynFpbU{?{YZ`4g=tg=e==rFkZI|I00^Zu_A2!QcoO~;l};KkG6`nQ;p%Ep2<f~dGK_=|?;M7?I9#q6tz5Rx3AQA~Zy8_QZUh&AdCSSBI%)q1&>9Rf`P&j;?>#ynx9)zw&v7ibF&NgwlQxp}ar;ZBAe9>qXMINKY<i=o!m9z)#VpTL8^2ov~sogKiLUs530eGnOURtfmJH|H@-Yi7HwirzYqOli>3UQRO%mfo0~%53DJRVEH@1AQ>%tJRHKebo2)eAPz84Ir<cV5*D{WKFRB{Z;l8oA>=r07_3%6C>Qw$J7)tXwNp5YU}O>Iaf*n<_n?bD*`8QNeemzWX#Gyoe?!n7!EMI8mKcq50#*2x@k&s8gg*<n!jw)w4y|+Ec0F)R&)Lh^Cy5DZhQX?>%xR4wPQ~{uIafC834?+10Z|^>@6&Kg8=`(Cw|t9N{sMvf_D)4VCc06dw1r~wf6>hnwZv-iq#4mnqLC2b92kUbj_Mh_9R=0IGMoe4)HKK@GK@8@3Tt|@eFfFp7jCkH~ARb(8kuc!#$OjcxZk<1YQW~U7UuX`oNb~MrHLQfKVS9l>sslLjg+d=r6_apsNJ{NlM|C_;&0s>X7BE=H>TF%I}HtKmqJL9@-R5t*)@d?l_C{dh0zdTdLJLJ$-}t9CTIkQ)JGgM*ht{1|Op%&?M-Rw~&iW!0vS}Dk5mLh5jUVM6Y0|moS}aJpUwv=^S_)(B4FVBu8wHn>+Ey#)iEI()bRX1BFr`w@@GD*DAVz?o|T1R|Zt!T?6TgENoyk5ht_!9C4<#l1ygMNk;p{$|pLt2II9;l5u$kFV4kaWWof=K!94#yn<k@UFS~SKBmtc2^V!j;nK{N%^Z$w31(p#Y}b;9EGX-E#8;F6?}8h~Y{!!CnP*;X+&jTH1cf`ZS&Ar%R)Eft-awXxf_~QP6$DRPI2gW?viK2J5f@i#6k*q0#NcZlbO+C%!!nVPMGTZa3X;8^i--Nx_cH*20t^<|7Uscda8~U1J92QWb-W4fdfAdkX>pWc6-iTbls#3~<BO_R6*WDk4+ve)srRjExxq3BsI8si^-rm7*`!&;j;v5`9>*SNkL!|feY}qhuU|-OCQa(2Vbbwy-f8dd+h4!?_SYW#^4k-&NI$V9Rh?v)g3lIlMDg&;zk2ZgPwsv5=Lq8HXtK}1gvY@c_K>oHFB{PnMhtJov&rF_0JT~<ArxQ*&{bj=Q>kcvhMi<5_Vp6_)MUAh>-Yhr%2oopvahuFw`N_IGhpr1eA3$j3CoK*v~sPUCf?b2p-!?55<T7@gsr$49r(9ErP$^Yp1o?oS}(^DlP2h&aU`LHI)>LVmkc>i>i7?yzXuiemcY}T3z%UD*En3e4d9fk9p9s8ratKJ`9=W*s}2nru>^3gppLM8DDe=f4!3IS?hZ8DeGm$GJz+ZXxn=~7Qeui4Y5qH49ATPrK3&FLHeuiuG84bJm;vJpX9fl!T&GO#cp=&9t@Sk{`PC`|x0*b@gsHj3)4GX1X47ULYXok6&5>y!;Pv%oKEXSrwwH(7o6NFOhnm?T*?hhEsSD@VPGLw3`$ACxnB!sX6qpvhl<uEy9-v)Axxmz9?V2Re>>ReYj6fKCbv)^GHk2fnTfG<W`G`$}iefXU$~Ll;>Uib}4|V&?XU_(n_^W0?tYbi9EKEt990CYV3%|B{X1(UXm+IzlaY3J0m&z=+sMVq*O$lg-qy7#^QlI{TZVlz2L?9aULaHlAO-n25*7fq36IbM?HKenWc6}EO|6$FN%vSRv|Hy7lTkR}pIDaaDL$SJ>5eo*~=w`;8pfG&$3rSaOxT6~<yK~b3t2Ug$6u8@vYR;Bu<9-J12sn;bc4_NsR(-Od<+nJKz8SET*3dxF1~X$%<a^HVSqW)Z^;7mnIJ879KEVW2kNRl9<cuV-<aB`4uJCkOP`1AnX6~H|fSlvNa;dkb1?D2MARo6HDJPJ+&?+=&s1Jc^F{zYl#CfH3LNxS89iGz1hWa2&+0s!GBG5Y(7>uL|P*?<)*<m<;9|TT6O+!3fO;&dUD3#tB!}^3IQ<k`Wa`ZX;?E6_69TfK+69VwBqv}#gRi*~y)CG;FtZCViocjL0H#g@k*3U{apR$);?~krv!|2?NTP$U+4!@@E(b0-dC+;8OUfkqcZ_VB1>;brAaDeehy+P6D9GgM{X4X+hp?daJl;0ZnxF-i&da7EbE7JQ1-><Pa8uov7qV8I>k9*CRTifxqq<5<gb@2Di7}XlgorVAUa%&MFj61R6TVaKH#2p5LF}afV*?|pqN)yyYI?K_bAV25;-I;<v2EYq+vS`#NrxGbtQc$JiA&Zk12?r1a3y*6h*ln!2c|fihST-nL08|JlRL`dO`Qf&<j)-RHj+fescbme-NUM|KfdXX=5fXd(OvzFKB|Qxg&@kzO9<D)3k*cMj7}5d|rI*Hjf$rC2$B5X4IBcr`gR3-1@&J(}O5Nx$Par1{jrIvq{#Kw$?jsAeL}en+w|ENRT>>7ULw;IO+}*(qM~c$gJTTO-6=K1a4L1kZt!6gr@+>8=(8h*Upnl=X)hpL8pTD`WQ8(JUvL$#k_1KnHz=G{vU=YORTemkhf}+pi<-a<A5epej6(h#ZSh>qw{4D&pu|do&FK9G-!XmBITt7DwIw!=9YD#S|!b4?&7uUDz*awhgb!N(Bh)^q02WBZd*`>FHafj@7aGyEwSnkoN17rQU%;gx-5inU8vGH5k7?D@m@;xdW(IOy2RTuCSqqdTZ*}CG<jZBW|XMbU{$x%bj1okbD%Nx1lqk)q>CkMLFD^yWKQO16)E_+%tx>}dQs3Ci56%k^lA-ilA;o_sgd+usPX8+6{%wqcVZRj*H3uipi$OvJKZ#UYBLn)JdHny~-9dJxFYB3(Uj8dO1C~OBmNp)dQQk4sN)uXK_B+y5n6$vO4;}Ewzd<JeoG>th_*)JnNf^xeagp~e~Bh9ZZ6W5Zl+^eQ>!UQv3ONOnnCy}SsD9XjTD9ayn3JbVIcax~HK%lZhZhN^fWY^q4IAN96vPwpbPJg%;4O^UWNq3uV1;S#`SaJ-v^1@uGnTG>}3hH84n4gFrrvvFj2OV)TSVEFFCsw=ch&zitr-OeKRNurH!I)+QH!KxMH>t-KLJ}ABNU_Ca1;ZC{;6!XX-^vj@n_b#hFY3PTz97P>Q5cpZLyVYU50;lN8pNyf%PZ?8`v4k4i^@#0C?!vOAN=BbhhKi_-pgOR|K(qP@ZI+x{NZ;EA*b&B_~-ZE{q?<9|DaSry(uJ3hXjqUHKPeXREuQEq?cb(L+8O8KfC|N_aD6RYlh2%Km6hFrQZmI?!W#ehS0q?U%mT>udA(zT+5osUC7YQYwz%#SMR_2D+cxOcfY;+{*NEL`9(Rhu6Y``=9?-|oT4PJ{d@oX;=P~#{Qgh>7j*XC58ga{<rnvU|FZ|b{oi6Cc<_ra-uw5z)0qgV=|rdIeNtahOs3GuB$l0p0*D5H>Qr!+tVRt+hX=p?@`Lxknk{A@y!OiBOTX1AkqOT^6`ym~s<<FzMjl1{+!|>5y4$JMxJCSsiS(lv4`2D_;a9$K|7ULizp{yP_w8?F1jAAN<GU{%{_vmfzWt75=EHBj`QY#0JACuq2VZ^b{;$4q_=h*3#^G1r`Qy7^Gx!VUbq;@>(`T4TO_)!=feG~8!&m<C?%Q8K{MPsGefKSxzku<*Z~pwj`~L_NI78>fch~Hzc_g9FDeaogR9ZJRM01zD409mmy_dgu_{Mu5{M!#ezO3aZbkCYzU}820z2Y_5sTdUE=@RqR+B#lZ@U|B`sVz)AST2pq!qe7Po~knqfi0$*nj@POa5QB%P;`4;+Z6<<=0lmW?}Sn>2!8^5GmJ(2iRuHP4Z4TyMiEYh6F<s{ALYc4a^goh@uQshQBM3QCw`O@KT4(!qakhjAZ#}YMY`8q)p0tq8=zb!x(=l_Q}2XkVk)8~OPR?9W2+yt_$zjEk6TR6)Ha{SG(R?z+(jo%33k!PSc;BhW9PZL@*zG9S{$!IrtWEzOt~!-;FW3ABzD0mvxN;kDd!V^%~wj7k<{?$mh!-<MKzY5oP+;=Vs3fyiH`>s+-@B88mo)-vw<i6T6Z?zGD|siDNWH)F}GOI&0tuc4Pz(`4S-ZFn80EEi-sBnv^RCtM!{g{+G|4_o1EtPFl0Z?g0+%mU@D^gDG-A?KI(@fT3fZgD^}?3K0irnRh3ze@~U1J1$>$Y*^EHdw<SB&LIsu5h|AMV+SJDjEnw=GDpRN@HQdALYt{I_HFsSHMX-8KX$KL^(lUZ~TMIvH#T0wYGk_Yw)Z4@8Aj}C7UVH4S5)#OvGwYU;cl%iwcHpgMknr%=nVJ!&o%B3}h*8J<bPWM#C!DjoI_BbnJrNL158)PFVv!v@RMhs&0AWpQhJmP9Qz~XknNd?W6I@kAo49H&sf&{fyi!PCQq{0)vGV0C!gS3uh3GYekAV#3nvtBLKM?eyD~B2C*^w8P0t=50@?Ur=P%SAB{n8;&c5qXkQr7fV%pS|3Q6@($i4HCSm%ZE5!U6e(<zODNNWZmH7x8LnS^=59BeJGty1W*szHgu6SQJM|=VjZ1z6ZIb+ga&$P8@ulh>1~~2LZJExP%TbvZpp4xL94qQn2Q(=9gPv*G8(_rz0?%o$)Zlw4=>kF>X8Yuorij;9fe5ip@4&SS*L|w0HlV_wT*<%X|OyHh;rvpf`OW&K@a1G>G!}9TcSlJ7<<_8kAL~Fdoib4!o6h2AU2YX@FgzB$oHS|Mds&{m%?bmQ4)z%CsiDWf%%lNSv<)cGr+kcbD;C$4#Dh@md7S)c*1%pF!r70^>+dFP0KKl=MLWU^5TSZ=ti4;}Lk*0xzFnTAvQz`}W<pUp##Ids(-PMiv^zn>wDIB!TFB{}*_`pXA_@GPOwb16`vb2uEK(j=x1VBye*VBX5_rLrcI-L%NO}d%Bf$fGj{CT9@#^zK#K&^c}Y1;cXm5tZ?C(o0l%%3~ye3>gwf-xARU$0Ttf--Q9P-dHCM1AAIdU4uANE!@vL5AK!iX@Rcup@Z#$qeDOQ?zy3e(|J&COzww{<fBB{pH^HUs2%52up^v<oogvC>?d5gLFy|mw>-9Q_yLCsm#iuE|?^^vZtd<Q6Z@wO-I1PO0))*wr$7{MsPTS+Htxn8NLW?<$0psjgZ{Tf512);0joqKa-+YBp_wc2!9DeOT?!Nub;g7$M`?6pD;)6HdKK#u)hi|@p_w8@o`|evGy!J1DeD`ZG2p_!ut$ROu=ib-8bN6@ue)!5a?|uE<d*A#y4$hCi|H1d)y8Hf*?|=6-`0>H(uW4S9wnS!+|H5ICI96X1@Q@qpDKEm${kL^Uyac}^h98a-^-zQGMFl!I=Bg!*cnD#Rvq>wUPFw}NNJl`K2liZO(b{WbwMSU(<zmGusYvZ<NU`rKPW)uNdQz&1{L~wT2gcD><xHKRi#8WTy?v#gLNJ+tgi~fUSvMx!=d%X(Tn#Awl)zYzRzJV)DwpY#w1!FqgRGB%g7)JArt)3BY+FZXIq3wo>Cr5s0GAj$NOq8YZYvX>pNwQv*XQjigEs<6(#ibA<}d^9I<Elph-t1t$ebq@`yz~XF$`1>h5DYIMN}1YUqLj`+N>3vN_4R!Tm8Os__c35_}R<%U;omB|MQ=Rzx=xQ*WTfeU%B^BU)3f9Cg{?t9RACf?!WRCs0DK9{#*a!@ZIlD%iRjhm%g8&@mD$;#mujD1u=U4OCS8(4-Vh?#T2swAbsn5hhNP^F6(5@oZyVKD#^57b&_fQW;h?d@t%$|R}TN<2M^wQi?0;_`HQ=6|HFe{ew$GTRuIoQ7i)S9kH%a4R4U;U<W#R&Sls*JTlc>A8#yk33*{|%@$TFI%CKhxclW(-+<)&4=q~TukH61)1<cQuDH}1@#D?O=)X)iv4Eea8R6wHm*CvW8e;fG0C&no`__?Htj+?GgNFr+*L(&^WphzUWZI`;R*@=2PA*I4>EY{D67w*aIOmm_t6~M7fXSo0CckaLT-UomGD|k<sgVu2W)n9?0m!Wg__rE{<ho5BuE(Gjbe|PV@zxm_4ubb}16;JMmdk_Bbox?BxT&RjK-Tc&rd%ybG;k)0`sV8PX?HPMGHwi@o|I*=4{_Z{w*zXQs`A3fWkH1f!R1kX_2%IVkh<m?z>+pB~y9|Uv-xZeAz+<!_ssLxO;8B3L__s^0sx%TS-e`=uTUgb@m)|-3=}%G4{Er`j&df-FYS}OT4!*)08aMu*G8y?l58i+E@P|LqX`B(430JA@p=2OJyN3Qm(K}J}P82;;kIP0#a^sxIGB`<Sesg!BUUN<r8#PB2bO9~~EKpj!4)o?_?-?k{Dn&@S@^t7mgRCu;@z^z0nDWhLL1H&d>dA&tOSUM$nwc126@_vddbUZ99|db<N~z3^L7%j=7>d-i#^3CL-~^?avYnS)7_xTe-FfMUhrfU4{_nqY?<YT!i<Pjmzj^r0pTbL25Wf0WCZbsu6Nefud+e*?R1UIB1Zdjxqb3%FrYn~lSZD}ro~~=seG08NPq}^RH!v7XH#3T;!!eEKS|UvfRmu{ZQq6iJwLU!9E-sJ7<Ln{{bE+IWkCjv7NmqH|SKk;X%FWa263m@(oQ@R0(QuoRIOm;I&pYPM4wm0(b>&%g_$}V!Z6}>%Q+&F?l2@}T2LlXU<X<i&%~3!%|B3IEVq1J>iR<OX(ZhhQ8lf~@9o)*JL=<9Y2yYu5s&{YT9Sp!uPv9whAwJ=ikALBi_Xc&>YZeC7jWAPtsC~iB4wkoe+=}W+8p_M+vV=8C0T+n;B4O0&7&zbs_aLNFEZghVTQOdsoAfB7I&_T3K|0IQy`*>pjEE`F_Z>i&_hIByag`SB>N2V-2`+Y{SkM}O%56vEG2F)FX1mEqEn#N-$OeuYu4lKns+nKpNJ9;C@YRiQs@>qYX<3yy#tj8`<-6=$L+PebUv{ElbXqN^cDA8+Jr5j`sbLe&mAeLM*3h2pd?b=qVf$b}1T&MWP^4Wufi;qW={4|fphdOCR-fk|YgmCN&8_VaZ!}_IJeX?D{EBN?{zu%6*^N3FY_dqPYU^qdlwly3!6a@cTcgGq9RMmaqD0y_YXOITBl%!1G#2Z5Uuj!v?bLRszlq5i1Dk~6PFZ8&#+^9Eq~e}sx$S~+?b&9zB7b}xR?TdIhxh2E40CEgx7&G?Naews|9SZ4??~17U-$p%t$W|#E8T<le~32m8?VD7+K#{bEt=I|eTS7{@1fTJ`{LgL5P`&aglHps5Y}=+vBWn@r<_3{>K|l?2E#KbDLTe{IvDov2x@Zufw?eI`C+`(i7DF&zcanz(U{5WtIGh;)Zl#Y&CMC}NnsOpW$_ha2CPAO&2^v;H~(z%P<FEeMx5Zy>0!vT=Y$~&<}l>>bEHBD%kz+RkABz1&EYrbi{Hxb498?b{I(Y<IK5l+RSD!^5GlQtxRu=34hlY&T``Wa8tS~oNtKR<nh;^95U}mMyT}bF?YJ|*;y3tB-AUdx<@%kw8B0&n6Uh*+t{4XsL|2}4VSBy!;&*u*W5s*2HXW*xl_sOQyjnen%-L=0Kx=G6xi~AUIR(DiB0ed?j7dw_qUZv5c~zc`IZit=Bd7!HiV}P$<H|Di2Q_bk2Rf;F^~=LZDn%-9U8uX8z~i8=+}VqJ%a}lNyMy=AM&nT*h6=D8YUnEO9T@w~8e;u~ccBBCA>Md}!NpgjO^%{fa`yc@O=WMxbY?d<wj;doHl~Et?D*j30X2H5tWmtN8&RBJJUhR<SeLM$5tqUf{JjY8FQ<#F#9d+;K0$VsJe|Db_l9}=j?vTV#nU6tb+;SRaej;Fh&Obl3tOG>KD_R4E+9iKT)lGf^7UJn>w3>+U&ZB>28QzbRgO?yVGK-mM)AtAw20)Z{7Zh~q}LzF3d0!V6ncuy)~8G1*;jEsuP56~qpg?)C8l+)C@JvlF65Lwq+27onEj=S7>lvSi|j5l{)B}p>Pj~eP_W;ji|5HhFLkRBPl)uyc`<oy$X<`BsOsBhTYZ8xIn>{P)!ijYjLAJ$;r+Vej)V3?7`En)*3fg#=GE9gz7rt6pyFzMQ9?fUlC<sbuW%aIy!mtD1dHwr*!SkRNeDB(+fQ2lz8S85el+S`K#YC+kgyiC&LdY1QcieEPN`1%qI@!rtB9af+&$>kKQ-)+2N;J-zRhiXN*iaEnO9OkXeiJE50!g%B|=28SY}YJY~>DX;$pwIyWH|M7}|ptHD~o}XC_c~X95Lu=+B41x6=Ys24AMq=A+*Z3wVz0|EM9wemhjLyCc5cF+S{w4?D((UGZU;e^58W0_B#QJK7C5qA@9;{5xQNoZ9r7UK@%7=9}J<=fm%s*IYYQ!>Hs@vOVsPQ{TiZgRwIej9m@J?o=>{sxX#W4#qk=rWUsPqY%a$Gg)SLR$y;2A+q}jOomK5WY|a)7)FE~I+oiFq;3QEQW`+&&BkV}=!D>Y32#W5^tjHa>_A$E{eI1qQ8(kX7mto0r2;O;k4JEcf@&LjA>7^2zQSUS>A_iGruxNf*MMmhb&1TS*~?6=4UFJl%9kxFhPrS$BhCsp@M<T#*}b>z5MS7b<w^RWw^+Rs%UUONC`ai9jJ3rwf6=gq(|LrahevAgluolqfajyP<U((;c;0GL0k>B6jpvNFd~G|7zL73szm}}xw+%U@L0q03;<5#CT8i&WOcbh170DsD+J<1dBsIw6)}Bs#y*_Bi2i-2%tVvT0Ez5b2sx(q2VEIE@eK2x>N5DAk?8d2!_ovVeGR9x7%vo|pSkZP3aqEQctqQzFC7fOiiN4Z7ai6x4TkJMQykd#x#=u32asf0JO&GGLY&achuGMk;?BL;y;%w+|3Z4n+Z<AQPq-|e62-P6_meailssos)`ehZ9Q=40EWyCg5$!dT#Gf&U|VJ!Ws2fyujWm<FQW`SYSsh^*7H#I3gpmfu=xtUC2ZS#K8J{R71!tkx-l;)W7psVbKSv&(w>Zy2`M(a79+0np}E@p?G=kq%!!Gydnw!^WDc|D+Jh+1^dPJj{56f~DhM6E{lU>3vvvJNr!#XV+iSc&GW*Svd1JVX+9;$3zfEK|i@Bknkk0$>Xp=9e<{N((6%Ht`5!*4b5#+q$y`Gb^b|NqOvM=<Xor$(k!y9DNcIG^*aOFik^XuPZ_giDIpm2bpzQ7Bq^P18b2D3)ICh(7&z%d<5_xJA<GYdZC|~(|q1ra?ynVXAUa>ujn44Yp!l9vqb*HV|Om)lVQ(jkH4*a65XWL>NxTmwQ{K@3NWgLXgEwYs(R-nRJvLXfo5*3!t(6n^3kx+NCLESSumLcrkW9xHCx?)v79^^5WF_`x#LhI+@^i*8o<I|)s+A=g!P$7LspYb#)-v@G$;^nDZ&_gta7O9emh;AUs`v~s)R0H@f{$LBQZzH;!4gQaZb}%PifiJXciW1eV$&hWI!EAWdcXgqFf|k|5#y{0jG$991|~_)&&LlJl0E?lTkA?$<s8kt!dZdI;~|1#z)m@9m=V)`7kq`wMaiMP^(W=T~Tn08AecO8PhBKd6UhWDb0R&FdoH<t`%fT38s^j>vM!#Vtcr$I@HW}Q)?3{SRM-NW!f@n0;Rpsy8m2Ao_3~EYio>XA^T;K;5o|o#(Po4R8Zk(tJ}n;!inbM&|JoYJfm$qC^R6<H5+~mQ+D5H-r{}yw#+Oo6T0X0k+V+pQL)G{9u!(8@VTm8;#6IrK4%!5`_f@Hsp&E2wmblO(!(3H(JeS0)Q~^%E_R?ZiVVCS3!Q#2{1_NTX%#*iia29|BP-y~l`7zQqE%OCK>vQOnhJMp+FH2`nuUC+Y8~WGb=iUOAm2VHGY!<b)eQr)RzZUmCax2#n97bJ`4DyJB-%!Q^rxb6nkG@Nnm1lV@L+;#esTdckoLH@6Sk5rJE+?*2Dw~Qm6A?eTWKn?96qf%{P$5NFiR-c>2L2wLwS)pLH3^uFi8@?Z|F_C_8d7eOc6V3Ntp5ez+bGN1wqUjt62w}SmI^C2RG`WrU)Uz11&w4QwP76or_ZdnIT76^j5VY9<UfMJM3%1LFFNhKnu~leY%T_lm=^4u^!BrZ*|G%g;mG4dG4z%3c+e%eItD(E5M8cGhu=zw?tw6ty$+e3&rQ07+RES#e!18uze&b#a2`Cov*ErJ~f9irrWZ_n=2em-Fst!#@LSf)~cZd3o12~x4K+nlc1tw@3|}&<+iWRAs(e;mDBJCyYe|ayl$fDSPP!hk!1W=QFdbE&tW=v3xUV&w(X`>hS#*CsY9F;h$_*v$R-3n7{cNdQot)T(!+!t-6X~P7r1W+)WmhrhkGEj<K`%CDX$3wj3H94bw<5|10YR<ybAa-gJ5YOgoD4ecIs38k@vg`;RTQX0_F7pV<!mlwYAaODIhGSe8^tZKi^NXnl%f)D9vil!@qOh$3CHeIaMpfN$J1~3Oy=grqs{U&N(RduqY!NpJ-%eOa<i<ab&?L_Cf5iQ~&K1G*O7hltouy7vV_~6f@|)FxTXQm16nqlSOO8nZU7|#?KIb+_6k7Eq^>`fjE=5D6B4#GW7)ff4yW$SS}F%OIAO$n2E*unJ1T)tRjjOf+9;xXFuUMIC$E-3WLFJON!M48j|)p(S11>H7#{&Hk24_i22@CVL|w`#M#+&YorW)O-ngWe5=YJ^dQ7_Ru9u1(s?yMZ84*^ITFX-K+{XMdzR6hFZ0w!j0Cn3!V~@Er`qLV6*}5toUmCt^)Qau$Wu0if;i1^0Us{*ds_)~i5_G*I(`%OqTOU9N`D4liPH3NjU6qf-%s_27eQT!I#;hrAnfpt4&@fTdzNlr62R1V1XCg%hLnsNx8^vVA5)N)=cd@p>`_70cuC+NlSe!sGjY#~+lacqaftAV(%PP~2b})avl=vE#B)xrXy;9`BPn@V1{_%l_+vL34Lbc%C)tFy@fTepPeCGTyLBzX3d=m@PE&v`&A_{jaVdfK48NW!zyoRh3`=pUJ09}W?EoOu(iz+s;++F{mQ4(`U9(B_9<wurR&{<9jtz=NHtdTG&#rHE;{7bvTmWbeW0VEV)6<R89rd759sklB6srWUm$IXDSW$8jW;NdGP)nHPqn{>(chXA3cptE30c%x?EAefCnhGV^I-u&cq6vCEN_OLmIPqmE_YhpWdIKojqoac`qWp~c<?d$OYGE#b8yqF}h6JAVDx}kshP|xJP2U`-iMJVdlN8fJV8Q@$mDr!gy_WT<)9HqceHyro1V+AsyiXvrb3Wl%z&gOPn6T`$=G7Fu2y>Y(n@wT~T(KDhu>3>3Q?T(d{m8x;j}TT}Jg^^k#^K<AxH=p~y>@nXU%xvnMh3DI7Z|ApjNAlsb31!@^t005s3>e70+Is+@;Q6RDF%a;6&u?w@ND34v6l}vv{6<pd$5Yh0mWkS!D`eTTx{EX=epZW8FGHXnexVkvzrTeAKV`u4*ZNq(4D6~=ijVG9HQN=MsKW}DKw5jFP!$mHPd}VmP<Zj#12`}hAI}+>~#F#G@c{hX9*uRnGW(;&ZT|KwCzN_?J-?#!Lza;0ohO1?uy!AF>kX9HREPr&Khbn#24xdvv@lJ*(q<$V{jFoR=-=Dm_!S#wsz`TA8ewe=K#=&4uCJZ<bres8nokxa?IE@@vB@0SvlPy%gbd`*m>Z?HCH5V`BNsJ!FXkh&+=N<f(ixBa%!tFhjSAlX3u9{JAYc6sBwGCiPv+k0FCSnB%vmHNN)NZYR}D4u$-gWJZgtAU(#{6!C2*elWS`(DA0{%;3T$Xcs0SWdf$n5;8lVLu$FhpFf`yP6$3Y6YDGNRPg!39u%)T^u0bxw*P0WsUODH4ls!U_8=^rg=$bj%3RW*>(!w=F0am({Xb_!JOTo!4wd@rM?{89+O5fFhO>!Zy^_8ubOTY3_&Zj`t0XJ4j=cwl^@PJu0?Y_z-S<^P7*c=WOf~iDO9a7g>S`_zKYDA)^nfFqGQLtLT)J8~karlWX++yz6_xLq-i?zw>Z6;C5YdPHB1%^b5`Tj7cQ77(gqr;F))>S%ft6q1h`Pl}TO0LC{e*sZfcuBmXjOUBUE8J_v`*BGHwoV~JQ1GfNIFf@@2euew55;>+S?JM370|VA!tk3$4rRJD(Q%Zc016axq8vK-9pz3_SOa4bmg56Qw53N+;E43oG^3bsDsXf!^!uHBlsSPq(IzO7L>;4gMy7)5l(mj7Vzug5ubbP0$|&f=JQ0)(3*hsPt0Px>qx|aHNWo<0iJbIPkS!uKGS);?BF9C7T?B1NEWQ|%sp3d%L|!E}Ba!pg3kO^*`O{KN&>gc79)0*^ioOY}chjlapLRP`@=tunu_Y4OznIobe5zZ?UJZe`G=GUgRz^I14l6*pgjkA64~i8+5go4pi4{f5C=WkPq@JhkN9@jZgbeFO*1<Sxws~oU$1%zSn<rM)7M1~jmLFgAz}sub!`NG_FH%@Zn&l^<olZya(|pp~;$@TVUVoTJf1U3+rI3K)LyEV5@vGx7*%YM{#0(BB2c#_L!Au;+Emwr+ffr$ND4J+qc1^V#AytBMbh4Sx@nGM%q#3c~W%Yb;dLK>hpoD0m6Jk6Wn^bMUnD#mRhmb0>pF<405I5LxV)7`<T@Kb%B7Xu}i!Oqph%^C*u`b?K&W`~!@m}WC%>vrez#awTkO%tN^Orr5pUss$I(L1@=9iDTU25z=Y|7K76L0Z$SPh+|xzuy49TV~wgOSHv_L^c36A?)n`6}#XFE&e-(1E8t>9tGi-l9jT>2bRd@1OUYomfZ)6TH0`m9quZk$jQ6_^9$e7PeaE<4`?nY_p<HrMN`#jYX%X<z6%!5wxS&B1f~Bj6P(yLXxZAH&#K1iCD{TW!15it>-muWdU(7D<a#Dxm=q|UA6~Bcb1wPQ@*km28wPUDEfPlL!(x_n=~EH8<c7Mb}j|F`-FpzSoY3!i?g<$b^+tepKj<r&A|Iic=hu22E!rnKK-or=sJQ-eP|SQ5Zb!>%*9WKx32unWeU&mLgeeD!-D4vu5spOkTPuYBn010EHdn^nr=pL^LeQ;&fQd2CV#Oe$=F&uaf!&s-o+8N7Bd#ryI2e8EY+gpvH+!t?4dsDbHE@0v-?%G8@vT)iH!66cI&lzGK#x?t!zOT&LQ13=i)k{OyQ;7Ath!>8<sxf<Skyx?RshCC^3Qa%!%k;Bm0?m(K!3u$bJST99m49%u(kqSIbpyxDQSxvUJP&IlZrpTh52a?e^UeP0?(rdh5B3xM$gE?Y=I&Pl~SmaOZR?^@|O1efve)ih5OzdS--u4TO3Gy2EqHz}Gagpiswh9ww1(vLCA}fwzJu)dukI9{#(I;nP^(cWgvZmP07I2Y=U!pz{d-q=(XG?NqZp*t;`-uDv(7)5HiGR<0(S!YgYB*t&cFqn1BUxwgg+!g08T$|2K|<iUn$<ZIuBBs9&$KHlfuun!<OfY(ZNI|)XlLNyCi0N~njo3jfpfIP?RHC@H|O`8=PmqQBx=dDV}N2u7WD-bn$Kb~I&Y7ljcJI$uC)^)l4RZ2jPic`dyTTFhwEm1}qbxTbLQ?m&mI`p^`yNUwPG)EKx$e`+?jCKw}N<@#8eEtQ8nB%+ED{c(%!UL^ELnG#I?ZaVtB0Dc8w5;7mHh*23dXU@Tdd>`2X;U8$T9|@t5`6ML1o=SlHV7vWYg}FGfO|>nL}|7whf%R^W6UpRu!iv*<U8XUsxeZi4L9`1pxgo0aJ)0C&CRJJXP$o19kSfdI@RLNFhN6lFjNE)-@WoJy04lxx0SwS6b<n>af2@|^G?Be+PnVD?aM3PZa8YR_QTOaYt+~em*)4wMd0y1yROue0gh<EyQFD5N<&5fYmGRk-K3UL%D1{a&rx3G&`{d5W7KBP1n@B4AHZ6vYJb@RAfU*Vz?_k-WMr?~4kAfNh@su|oHclMtiHS*Mh+}-Rg=G^Y6;m>sqerm5Ckq!$&9>lZiX~mctsMDcdThLrS5p?FiYFPqs<Z)B@Ca!D=(oC^kdi`9Qa0~f#tglJa|8Dvb$PN^{|7d+^I|{VjsxXAku-{LCakmj)wN6Kv55p<qnlNwW$w()zMO^O|h;N7c00#)7lZPai5`S(Ho!k+H_Zf_enZ^!x6G{c3=noX*F4Qt&cE{LrN0p_~vecb|{L0GJK6o|Kzl!Ku=eDF7)DUyi|pI;GiDXo@ZEBgFxr!yW1gM;h7mj#vPuM*g%4=c=ah>N~GhsQgEXZ-510sn#okFf(2G2FFbC5UM~P`@RzV?o-euIX>IK>P8_^XNfN5O7SB=-br?-6R8c0DY>c`t?YK6XhVx6c0$?u`eauK_4og9$&0!pwNk&}0IVV@!GRYVTJsbs7fkZ<ba=ac)2pCE#va`!PX)MD?e}K}3mD27T)NZ^p(h+OUa5U1*EHi2mBVfoXONCfV+;LY3F4O1pitWS)%$MdWYb1CwgdKF65eIvZw2@T<yE&Zk{Q5P8vi!HwYv~gaRCW(TEoafRx`RtT&N-H%1M4#=Fpo3H&U;jmsj?yteJC4J$qC{pF2lZ-Ut?9{m{km;1{+blfXDT9%qJ@<JnpCvn-yHGWvn9IDX?nDt(jVJynrB{Yb&RSb8VulpB;n#xTm94D~~`Z7}#Q1oNWa?vLUu=+7*fSw3<b|$YCn2kj__GEDtk%(4IH9l};H0yLL*%n6X_y-}nxuF<ojVefrk2uy)EAcMGKPT?g6e_qI!5s?1RcTr&zMZAJJg3b7ZigzB&@nUdFlRj^VkvR3LIPE7z74dn|?ZQNw3z?R!ty}#r;P5p3t7`5z(&9I3l)k5Y;A%YggaN-5l8Bx*|<*-|Ys7)CG7gSg)i^D9988PFy(7e$vu4N2+Zm)B-iP`MT&&H8xR$V)Vts^V1ox)>drwra?_FoR-g4g6#k2!v=JL7nGV0pK6+L+Ug)gc#`DJF{=arv)oc~u0#a0Y6!$f~DJn}+cmRMdPLxl+jhU~po?cn+*oe>C|`EX^~W#5E}h3_PRl{95ef%b^Clr3fn2jSo)Yms>_v`_Y@s%z_flpA`zSW6x;?&EIy8JCt$9-o0aIJ7=rbQoDl^qo8F3l9-6)*Eg}#muvnq4_t6$BP8Tj$SjNzer<FWTSSceBWsG7YCBg@Et7#fG+}=$6qgdOS5dS%G$MXZd84SSr09p~eG1hNru-n^AJc=-%#!oY*X}A;XOS^6O3viCHiOhg>7H|csTMfiF4qFLbIdw9<}9b3cmh!?&zP5-i7JU9$>asOvsfAeU&TIfp;#8~A%`}5Q*$J^LGO`lU8zi-q}9dsYT?6bTSZ&7meiDtuc1<_7*njz&yY<}&Yni^Y9;d^f2Nnf5tBhLmN#XrQd7!>7;n7)$Vdgp+9L?QYKm|SL@?L^24OjG{0jMqDo)@`s8qYmqp&oMhv2x)`B@v<)<m`-yMi05dNndRlz~-3gj5ZS2TT@0a=S7Nc!nFJ9I7UW^5|+6oMhmtxL~IBzT}OXsWny}vJyg^IB{m@!~v!{J|E6(JuK>}CG{j-wPT(Y8>sG8a-1G2_gqy6uTP#Tu>bVERns3H@iAysCu!}WogF#mVE)Mi`Hv99KV<;_biw;G3EZy~v|k#qU!4Y%1nM6(G=K82d^;A^705}Y>$Ff|>VNaREqz!y+7SU{Pbb_8b8xal0)KKo?z{+2#k*~&UXwbvr}b<98M(Br^nH(%Pdkcd<kJph^d+o5D-}~k1nSw^LSJ=-6c43AE`N&tY&n43Yv-iGiPQT}!RdVzkGIJs0vYwCs}c@)GI~gE@;ooOl_GEoM|p9&zgqr@f2YIQpp04l>A?FOhRJH3B-i%Ris{?NxY(!h21IvPjnaEi`g4^_7l~fc7Zw3pv*(TFr^h3L?TV=pPJHAiKJrK2N6zLaiUSI@L=%=xQ?>@eBQA25>|8et<O~c#LHii1)-)ir%Hnc4r!-pA?x5zUd*u-WW^St~EifR^!UlxtoQ_4!TFrHNt4Gk=09V|JU>IVn01VF!yJXwC`~5Oy{8B$mwN;@YfyR{f!sX7*7k#XHfmWL9fj8Czw_o2VRO;84r<N>l@T&N<UMs|@+OB|gu5jY*oR7d3(aA2x$u36GE=Cj=UP};(%;VE+Z72e%Xyc(MGZx*$;kI=r-VsppNtNw_*qN;=@5>z5CeDYx0aA4<qsnH<$>vA-<_9t%b0lNpr2UaECu5y%T#85WF!gIS=KM7)QsSr5npCPl=hB|h@yxSLWlacqCb6yMWmDyMQ}8k4=Y3T8N2{q;u9ADk%va&``^-M?%<<)%c}V$YD(0Fo$y+_cOeXL2?V-HfB<5OB%a{C9=S`VWaucg-r@g~({ovk9uK*a9eCht{U%LPMw+`R=g>#>Y6Md@fC5gKZ>nNK^4=>@0wTI+5Ev;-T;l(_s0%xHjE1_z_kwUG#JORuL=H1?hKl;^!_h0+qt1ln^=HDN@|HFH~`q|-6zJBkUzq<d*SMI*^m6NTpS!|6>xU+TSowPsAtuZ#Ot89%ikIn=qyr<oYvfT^$_Pyk&WE4_jm`OILoQ;?pVaK=U#pXu-#uut9K04ZK6YNuoQTp@Vtjb6`duOvUY;KS9Y<87!;6K{?Mdmh8Kzlm{J3+w&8$ZllP7YVGdqWvd@`hW=2Z6DJUP|^u--I7HFU*=Ny6vh~oumajOBuTb8tYjUIO;A_0VI6~DJxhwGaQ8<{)W?sDhQ_BdQzBr(#<EM>Yu@Z344M&w=9(J<);viWe?=x>|)H^bAK$`3^Vhte^fUas_hDtY3*TiYrDGU?o!o@qP&7~eb%81pXj&8zpi|^n$#n%O?)&K^huXxnCanhN@RpTW2vm5ta*WEA)gi4kBp4Y-KOPLco0hQ5wnM|?Wm|im9|YM-lm*xuk|=jj#b2?(kA#UH%)V{!UUnA>})6G<6nM4OXH;Fx?kZpDS^zYFRI0GnhfJ!f7lJXNiQq5m|Kjt8@zNhr0cxdool+|9ShS<u+Ssr(m!?uA~g`Ob|@xlbusXk*7X}35dw0p0hAayZAP8Aus<x>kVCFac!8k_>j*b8DxeG&l3+4kDg<CL*Ccmgnmux`-9+7DBb9Ww&C%~}R~RQspE{diQa~MXlo+Jx2MOyRd5o0q9Ce6j$h9ep;Q*aZtV~)>C*BX04MWvye%d?y&A;FK!JBvAe(~<x|9be^zaIYcZyW+A3)}1@y%gZYbO<xhg;pT$u!80sS8{2<%u7D(+dbSDdpLJ>TeO+zDYtOUFKKDhjqvo-U?2_p!;v>=ZT@%stDeTA5YuhPei_CLxUMAdSkw`kETcQEuAfDrX82;h+)>nNQ1(mtwpMa$nkXp;Kt@oarip|Wsbxy2&o^gB-Z@Q{Lq$rRJ@P);&_66WlW6NAJM<dHo17|Ac!(PEMa2}ycDo>asx)R7+YRN;Ybf?w@fJdtZy0Ea^mKXT<*6|7qv+->OF3#xvPYexmNoL6Qu!c+j@d9BdSz9rXYV`0trxcLR<d_&E0QIf+f(cyPq9ZmlZ|IvlWXXYB^AiXF`ivjoSK<?+BJ+>7!=iT@j%hXD!}x(g2Jzb;7k_C`%r?D&J=kcs&vd|lhP9TW2x(IPRZEN<z!G$K5razzEuEak7C}(B4DmDbi6Od#~wk)`wCrhk6%%NT&@t))-_)p`>RZ~!@uWoWy3fyMF9XE0hFWFWF1&zH}3SC$>^Zuf*l3^xM#+kk-~vjv--x~0<^wdgT|RhxMq7$3|JK$D7lNX=|2Ia@?9EYbv{NP#&kjK$L#I-kOR^u^OsDLVrfFR%5gYDX6ekBjRRtKevvXe_hI-$9yL$KOxzDM^NAd7gl6Fj`LNs=Cr3*7pD0b^;YhA+^>Hr&n0`$<1Mmi@rAHtZUlgiY&L}PN9$rlpZ?6ZOlfE-iD7{pgW*#<=kWZrJD@4nmgus8)L*Si3?lXyjS3Gp$CQ=y$?-+NxD*T;4ORbC0b<NE?huVNS6=j*D@$J;SI}!gD^YEljnA!MVj^Zsjio0gE4wgx#LWSv2JSAMw!xC%995Hybe6lzIs*e*+A*oj9&mb)Fr!j#h@P5t2>vaTI)e#(JGxcsg5_1QU&Y3zHb5@@@893!>#!C+`rg$e%yXV*7@goVKf;q?oxG?C(#Kv$(z2P~}-z68KTFiB)DdE6+Ec|xEF4kp{(0B%UCyodB8;{5pfsm-_fY|Hm;8+J5o-a{b$bR!S9`}~w`H$CX^=2pP4t&R63Z9y20P$-DWMqm32RY~E?RQoahq}U|^3vl*bv8B(gL$>23)j5;Q%yH=?e^qVJO&<`1<k_hL<^5hwNM>-+Xc^jO?aa{@ity<-t2clj)0xr-;IY})af8QgOJ5*h68WZ9;XzJ3buFW!0Y$muXH@Xkc8CR%#IU*4e#wGqqf(Jc9T&)*(}i@oR)cg%ALivk_wLL>ZWHNW`*)<Mgpb;;p%{j*kTwZY3yC^kFH>FHQwnQw=NHd{h`0LcItVf?=Oh!?rD?<LnzBGkDpAK9bJxbMs`RqS6_jt``*Q;K4qJ$?&HS*+I@I|o$BOnxZ<M=rRR)QSQ8vpumRbcUG=M<G2*An;e`+7yn=&Jxrgk`TbFB9T_Jht2q!%}G0E{XwMV&Uoiz5Se(1;UVO|S%$q(`L(?=){ezv#e9@G6DIrFE|zNpO@#ntF2cHG+@wJF?N{nYA*?9|SRx7qJ^U{Y8%&?xQ>kg(%n{yr8Sp%t@Rgp{khFq$CZVTOzL<kWXYR@V*LsNT0{&(o~zUOR!sjhgwS+IfSw?M5|OQ+TJ{v|B?3kTa)&R3(6dz(&bVTmd7^vd{7-v0F||6P%wW1@X>`MNIHJ*7C)Gu2DAtxNl_SaiK=py2$(GszH*$s$k?N%ZW$<6^FVmR@2<YjRlgJt%w*BK@5|w6l{y@E29lkL9}Q`)2jGj6bvjxNyW8S%XKASC~2-r_G(_Rt@xs?9_MyX$psrn-$p6*rWIc%ZTQr(pOg91^4Ha>-n41GXj6F4j^Z_&fw#<+cSv`Z<&~JsS2jT!qEcp3D?2`$Achzu*56J>X%1U=x?wwxT7IsV&YpVNdg>alVSf|b(yydS`&%Q-d9C8Jpsu>Yj&zw?`m)F`m>PFRNw42cqK;2Zvf~pBue7mLtEbR~m@X2i8WrzoCK_%VXsd{Rgk-fUe{+oRRi%7j9FR~xt>STz$MHm(SPPYpB2;pncf!$BZAME*fn0&A=9|Z@lgdSgYyLMI4yj-{H%@P9N~7>wpwdX^L==wWVJ5ncd0O*kF=tj|$wAZ2VI1waLISdsr?w&pP1?CmxoN^E52|N#$6rk_utS4a%b3xIyKb-Rr>T65148zW)69`>1vGS;?WoYcAtEKu#rmmF?W!Rwhrhi4#QLNa2656Ij2@nCq0L=AMJ);Job0sv-ba!qJs2)jON<MbT1j(ci3c2xoxt105!A}l6@AdqirYcTjp#HKH|f@LU5}b`WCtbCZk8*zra8+Tfb%_97H}e#+6Ao@Erb!|_*T;C6lMs_3}s(jz_*ob<K|Y#f8s*J!_qp@0gQ{|79~2jyKe9%zf_5R&=8ASEzXIqJC<;ic*bzIc~_JKU99^u_enAmmwDwag3NK!5TNpkA@qi5#%cPnb{ebn@4B-{b}3P#V3s-uzL7iMUF(d>@+R44#`PxPLdGXE9(N?TuF)OTqw%QU1eU}M#p70lnTg|GwAqPUOcmAKOq(Q3mY#HCV>32!LGG=+ruGghuGyKIi))SDuPGRk$y<vXD{x{3F66))tL(?R2|}(Tjci5nilj|G)0t_qCuO$Ej5ZOI9Seg^!d&F3oQAtvG3GQc_HJv8q-~5?Y-jF<{P~QQee#Is(#-KCAHl%SG^IxN(^XLFO%3_Wc`i89of@jO=yLcds&m86e`svYxQ%*CdKIiG(pe-0N1bu3Z_s;nC0JZ{*5rmfGY0{yPktfkhR}~MoP&oN>gW0=VsQS^JT}i2cLoyPKNfh8k%gc1wih;GVeKr4$8bDKI%$1y@Yss?*lF)#e{e8Nw%a4mZ`Qo!#pPwtVVg0ivoI&F^qO_DRnTIOwm^GIpi^lbT5!}n>s>eiQvlUa@9L;UH5cfDE4%Wo8<+lS{wnZ)FOBD~fHF10+X99w-o=aFg}<J^e$jh^PMe>m(zn}SJZ+)NyB($8W*qlC=nma4;B6jIvzN-61#60soL@XUzr0wNu%CfGB)zD^;O_zaVcdzM6hM!`79DCB1$>USV{dER>42qAJrL_4EQ-5P(!n|Efui3YjRxt;!ooJx8E=9v*<IM;v>nE29KpO@xRl&p*zEK-7rIf>Tey1V;^phNF4sr<qq^RE*<V64v9LsxU?#-^r8;8}o}Hl^joR7^&|(J%9(1-h$X=2pfk!&bNDm^rL$4=a3k5C48zP&dWS3@4DoUQeczgM&>#VKQ-u#q*sPgh95A2$FzS+lrJYk7@>Pe2%-p!bMWJx_f?Eyhorl34c51bZ9#hY!^jlyNHiIX1St{e$r8zo>L2j2EDX;ElO;59p3I`QZ{?!9J9nQ~JuRl&O;djqkZp+I4a0eqr`AiJZTitJ5Oi`D7IX&awdtv!x;yM~K&8z}GXdE5OMr{^#oS+$uH0iOde+D}r=F~E!AKl190)#boj(#Ru!N-?UQy~!bUh|C%i-UiVH5DcpkgeMuNQr$b}X(3Z;i_j}D1)C_{F)hg|JhO95t15)RlkKk8f(*dAN(V)Bs|Rw}{|gwYpmzdmey&M=_Wql<j6SAC)h{znQKELXy#AIFRfR{G*_ASUta*8WMeK)2EmcDbb0{t|%wRzyp7Eca`&{5%T9B-UVE3j6yW{|yL3WH6bt7QSiqTlAWgv`!VHe!%L$XwLNZ_*^k+oCwKZzPEvmz7%Nq^=o10`BYyo~KvyokKZIAG!{jn`((hpe^YQNWvjEf5`n23GJn>zam25vNe~ZoD07^1~blU5GsSL`|!?&`@AQ`MLpjsSa<K7U9p_9D9dj0V5eGLkXQl&ul+c?&x9TuJx0lDen1Md#oF@=FfFe=cBO*SfipMu}gsUU;zaau{CavH15Gs=5j3Qp%b*GR8)#J2|Hr6)KRRI9a9W9<A4;F6i^B{@`y|9ZGJxRGCyUguc(j4g4*l=hgz8Wy<Ytqj94djPq54Ez$3=yMMEY1RMLalvKu!+y#aa-5c%GSsW5dH97><S5`Bd{-cC7`8hBen=J-Cy_IBw@WBIIZi=|_*U;J8K0;?Gg?s~T~U|VOm2;>NNw@hb5uYMJDp=jvmwQHcanM7=-(~5h-c9N@~^<5u$8ychvV67hIAqm~g`=o?Hr8q9lZaixD)hH<aOe<FOSwt&EY~mdaNx#(~fuaq7HmvyP;xL98h&tvt?I$fT0c46W30g7*{#hYsyf;EAyflBwV{u3*i?0J;wLtMk34^LC$P?PH@nROxTF>zqvOl-mz=eSP0gCahVI=lV?WSqLKvh3mEueLL!L3QVN=B0v4+PI>*zcrNqYHf49e1;l5TJ58&!i9Zz32JYQ7S3@7i$sF&|TXiKB*lIPb?y;KL+s0_W$DjfjJ50(VD4u`rCfiH%e%y2Ktb-+*m}HNvna4>FyGt?!wZ8A5|t+?(|c$zo}=&huef8-c*O%dFu?Ek4O;qtMpdg<xt|NvVBc{v>z7@1X&aNuDICHi*?t=ctW}wuF}>aXLl=E(206sG>f{D-{{so(50MBW($vbgkHHtl}kr`yJ-|6LDUoN_S7vD6-Lj&{*OkZp)Uj`W$T0GeJ@5&1o}oyq5vPD$|kOjR4Ue4C1)@eI`|0n>^Qs%$mQyjUF8*ljnpbyHpGzW0s<MIRdJA<O3KxqErr*v-hheRqczO23_4b<q{mcE+`dFfp*(T14l;*wM@+b6We$LF3pFKX{jX*~*R3U?xx^MZzQ!=rBv$qk(p=Wc^Kx2+TwWAw)(Aa7dSauRH-Aohp>|xWC;i*aC13kcE32VVXTrMB@BA8^IkfKvlic7E`Ldj8l{iJa@krixq^<swYBf%X&`tLTv%1nS=GD)h%z~PU-mBjb`{us2=Oaa+pMx9TJ*0Qp+7&`+m4Na{u?x&^Q=|wn>M(he`UcPHAF;ef;?v$0_zub-nRV!?09g?OQi@k%^8nQt^ps^nI?sQR=bSRh9TbgUEtmXt%z_4M+TbAc`C5yXF*zK4S*j+M%)<Gr<xN?!CxG7as#;tKp8N^HVmVKy#UnK>ltYatQ{%X%hGTc=&R*PG4(V`Y{?^bFXsLLgfuf-LgK2>T$2U)px3>Wl?i;Z!qY->P&P}581KeYtUw%}!h%$VRyg}4vv_$91g1Xn@*d&_C29em;LDnQe8OyC4`AU?bklsGpk0R+e*jO<8<lfN10F4j+@oJKSekszJ1s+{+plw1`+HYb(g1tIj`$+pgyv(gjH!k2#Yp(^<u{DUoop^}nVRI|Oyo&cj3y%KQs-DF$iwbn!vlX`bqmVld9o5xsGTLz!wXfmCdq!;3DkHE3q;4Z51vBv`+>M4E5I%^EQbkPSI!)iY+~TE20Yyl9oVUy81)@@P(`zs(^cOdV@$DfnTihai^^Z5N-MWq;TKdaASUU%o%>xe6V9+@ThxAcMRQy)U?*<FB8}qGrtuNHlMf#4Lm!a}3iOEJTY#sEXZeIlOv7i)`-C*~26Lb`xU0Tj79W|Td?zlsNt&W;cEPmoins^37Nvpfx9l#{?_0Q-O2ICqVN}4=^1AE>;KXtS}kKrDpaStzOM8Zr2+ED%#viK^E<U%pj{4pdy66O`si0?6Qqdu61{bB4V(BJYRZ*_w)5f0<VdfmG+LTF=@Jy4Z}rjAg=dXdU8Y}`E{c7I4-3n)xdY7jS>C<F6(jP5hK4`B;*0W5^BD^?K*l8nF%9uH$^R5pX9A{|fOtdz|FEgE+~`tKzmqz{s~1C4>6&`r==tap*Hig3$CMSlKc7v?wP5p*6O7qoZlz7Xt+G>4&Eu#48}*&=^Yr|jnEp=;={ZpTAlJDMr08!$0PY+|zF8|>17X4FIHp`bZ;^eMz*{y|vdZpKjy-wVy6-y{;VM)MoNO{p&O0BFrH_o=vo`}yIv7M{A>Z$_I;J@q$|-e@E6=K6dom<uxc1FlPFmp*aEj$3*bA;26Q9<U89?wnLOr|w-sridBif)=$I57|!U7HwdRJieJr!^bHkef~@@$Os|+Y_L;3s3>jF`Ftq2*9LV0q&^BMgWDZ;e3ee42(h|+$#m*6h<W=I<fp<rCc1G8XIHhBmzNipDl`o%LQ%U`6m)63XO^u-ugf9h9p#|#?hioWcD9D{&CQ{Sf|P)CjOmC-WYU%all!pQ2SRWH{IYW`Q{$%!=ZwcCV2&%Xc-rodL69kOAQ2fgyiX1LV3W4I&)}&RPpSbRH@D0Q;6S|LAofO}h!>Kh7*22vk7jvKqmYq}c>cAg&x5g;RBCEzBOk7TJplg9`I^^<^20q$BJ!!voEJ+^-TPcSHqYdMT-=R^e1)J>Wb@dl2TOc&hmyG?)gpqEy$cu{Y$FMg@+5IH>ML{y_~~c}8lje0l@93~c-=Va4M2US);ml%_l0S}h7lJi;w~6XYOd}*B?NB^mJ8G)2`}91UWKyusX;J9v-Kw$?kBtT{&0I?(Arv9KC^hXzI0~!iL-Thtl`*$2_zG$h-EimROft&1lc&VyivC$UDO2aasxWN0sZhcgr;1;-y3VE(xfw9JGJ59bKviOZ@xo*UN7^VdIGNJ%U{qOGz)~RiOI;KFSQ9syF<e6&9<c~Ou~;n`Ee(H0Fk!L<Z(<E;>yNr#H}#k+AdkVF(xb$lMzQ&7u>>eF$`(P3eY0EzW}kt`r@(^FVuWm(Xd4ah@@hMWC3QRSc6uGd2OOjsI<$Dn22Y=mCV_K5raeA&{T0Ac2fcq;zoKox=Zo=;*(B%p?=Z<B#AO?6-MA5!!da-%7m?k0G{|JeH>cH16M_yh6=XLWgrt!SRex-&u4;$&tb9c0`Ecf9pP+W;9>4k{H2&jXe5zGqOo8Hm{HI4ItP|dtP}5oW{P`C8{EK1(vUMO83Nkm8YOYK2O6=?vSr;RUt4~EWk<ofzy%f~j!=s6;3I5L*|}9D8er4Xc%VOsLx2XFZv_s(P5AnvlLt1Vls3wfeoHXx&HjzcIs3=dEE?X(rvApIq^Y;-P-vLd4$B29!%=2D2q?Oik8{C=10^F{!Ld8&kC`=#M+pca&#`fcM8%TvK8RD5!GIh<<sI}fVTKqOlZ_k!1s&T%MHOs4Sh^WFsLf7PFfl_cX|^ovsLp{IZg8290d_@FpgrZ<zuaI$3tigaQa$&e6Um(M=KO9nOpvo-{D^LdWeT>~MW$<GC<J`M2z<^J6o>_XF(EIWxJ@h`529gD76eiuUih4Y43_Fd-RKNO_A?t#YY|WpV=tFmLRV@>(webo0+t(Cz8g+n&n3@ig}^7-AUP!krf9~4k&a!RQIj(zYmbx=Vl{3OTU>dim(zB)pN=FnYBd0TM)m`SW1lVW>?n!Opl#q@%qzo}w-k%>C2rrF%V|;4RIH9*G|Y4Jk=J&TR@4jgrh#Lju0V2|pA;UV>8UadqrTb&N1QqLWwgda7W?2rV8BggbYr=Cn<>vnWT|X|$j)T_%VQ>(jL1*uQ|YtSP4!s18f0k~BVi+W<DQ})g3_ppd8P_Wft9W4`UDGrjC>;IG%DG2i##&qJlYbz>+$wB29I#f+eu{fr~rM_(S+2+RXbDD6cX1xxnV|Lu$W2o&HfnmtpjEvV$}Db-(l;BkB9c9Rx0FNPTs@rKq|MTL_V{e-{rK^-+P!{Ca=TscX`R#W6({8BV`uhJ_EZqd41ZhPd0;)Du@GHP4@v;|C4zPT`J^2XS_BBk&W<WXg9O-)68GVHg1?Cj#9-pl+!MGf?COLqD;nKuA@UKxa^`hlIKGvI9B*GqW2!fgH5+oIH;YicDHmoYZTs7LP1gdgZc)lqN<#`kZpsx`0%mCRwqFhEyuKzP~68}(rZCyk8V?uw(D`Q9tWFkH>z;}X46x6hi~Py<N3EfcfKZGBPWtzEpzE$8ylv(gaN7Q(p`3YNixRgaL7G^qyj_%Ti%b)F@&Wvuf`<7W`8v5cSTiEizPfHk4Aq$Z)tnoN$gD3YJrSFPgy0`inA{Q0$GXDNX%%D*{7n-qyut>`ARs9jg5`L!Kf|Ot8?ehdDh0TwyVkaS*;ySQCo?1VeU(|6OxV&I<a%J3oGe1ZbRD`td+oLRslm=Y`n~W>_SP4tX4zmldl!ET0P$cLnGC9_NsP=Z38t0wK(DB-X?!VrFi0OZ|xLqD9gRio%jF`0fxzTuRn~#Xq2V+!(=U(9ahiMc52;--V-5S1^J4y-LMAR?F!u6Mh%>lLDBY3gSKyiqMew=ndP8(o5WGVbi(T;GJkJFIB<h<M{9!|sX>v`vw<$nAjienSc>Phu1fcxm6^}sGqbO{$Xr(q?wAU4JWLJqem#5nu)!PO&7ri!YhziXjU}}j27>u{30a7UZfwCo$?y>lcUd2`J3B;a3mRdE`2okB7@*KZzQmlBSMie$g$3M&U>w*KnVZ|$!*dMo9JywQq<Ob|HfOS*rVmJ+J8FEM8O?7^BkPTI3d)*R)*Dw9bEY}oxvIIg+l+TPUB@|@{WJ^Z&CHzlHr>qmYP~V5XcJg{u&zRrlUe?<((^cGwd0+ILN?E{pUy&Jx~mT^JdLSszcMQs6^aYnlEHdoR<VVJ^(Oy1Ir+;h;*=w<m-bzB`b2jI`*L{$t1ACx7ElEbX3<n09R~}!)L};n`GXM#l_hg42<^tH#uK@%n9bl6Wcy7G@A;h-mb;Pi1!b8ymHIE3_1W~D5P~@VI6$r(jt&&mvyyf!G~c<x#GpFvwPKhlSZ*D0nS=&W<XM)GFioYB#d5{k=*l9G)jIMj)sj0hc>P5=ihrp_pt2#o5T)^Tj8bM1l%E5e@dl<yQlctlADl^T#VMet&>-Lk7+EHFS*!YuzFJ~{L6l{|^jT7<fW1^t2Rt2kR|D@72J}%8MjY98i>H&yzAN|SKApkC^Owf4aP?fF(mMOikM%8?k%BjfM$!iJ^&rurBN0sN4)`=pTKfv6V09HFDqiy~AT<a7@jX*6HB7EyvPFm%H9nPx?Ze{9PK@D-#9^rmQ_Tf%XGQ?scfoA=`!&4bwy-O`UY*6fj}H7|9L<ushX&?INK0|;ob6&4qiA-ERBu$)H=$OW<{F?Zt-1r4-5uU=;SCtt6^8oRKTT_=>|^{U3-W9t^STxDZj|ouLfZY{f%AH3Q@L7U5%5~oz;gm}UDm_Jn083;4hEoq-n@3XR^LtHJs+=SqEe)|k_zc*3|w<x1)?{xQ@IIKUoCBn71o>EondoTX-L;uIZ9Ndk6JzZ?na>5&n{YGxkd8%r5OFdiIOYYzWBoDJYYc%$^q`!u;l6?ttf`dmz$VapGE;qqoUiJWS^{ebX1+4yoVp_^<g}SVbxor#8D3Zpg=n4#8PdV)|7-Z6U?W4Hg~+8b%)AZoUBuM86}PB<ak>rcPB7n>@Om|Jf)!`bQQhC*+$9W92tHET20*s2Sx^zO#7(a`6Kd=>XT8Og(@b2u@<o-hlGc?CCzmaK+iej9TX1SWHFmH(_*@|c4#w4IhEyavZGpQV>LUsRloeyQ&%otxqSV0c>eZn_~*(q*Vl7*igYp-9m4VpvaRUc+4qg?sdXvc{6+T_K<cdnGXSG=S_z;6IJulLEzp|tP6>;n#f4Jps|<LKfYOIjz39RE?wDIHrBc{M$Qk3VIyr0Jyi+ncjM98fL9^{#(170E@hC=_4MGC5ib>0_(xPL7SNLmU;FeIc!E1asaRd4Q>W9(FZ$@H`mQE3y5LK&aohz-xBH_aAJ5lb9KbxI?bBAVD!_Z?&^3G#kV!F~KTU)f5LQJ^4PCVK&5Af#_+<v*bNGimNY!4L)rwUpI$y)9#RYA=FPzIkW^fjekDNu7}F}BOOjX>vJTselI+X-ZwgW1Jf`DNVq!>EQ{rJBZNi>}huJyC@hG|GCKvFWIpe<5UL5EU=ma&Kl$XZISnRvn-9Ti(zy*ub-B^rK`6=T57SygfXD)nut`B?I6vj4FVrV*9wj=r+E29fWz$mGNEK8c=q9lGc9V-hu?8U|SKa3zF4y9hS0h<A7%QV(;)fe^B$x77)C9)NwPZsUdXZ;kG@TT5r`o0Nz^l>7UbH9tDoCMa6;RS_ju%Y+MzRwPLN?eJZCz8D47H>d^9WD}Yz!PP>fNN{k~ni&Y|Zxriu5Ttid4?>OYF72c6npnh>A=)}Hu%dc`TD!*nf@W^V>m`;JbQx$T~{lO~uGF{i5uEkt#?lfErj!DIaHD+d!-G*a`>^qV&8aLF3Ot2?C%!67`aYwDVQ&tf}7g%9v5pT~_1R~w0;}=!gp(>`t%&Aut&QP1e@OgNK4y>APJ-OtnrGmAyVbAgsSOtq*2F;l1RgAU3voE$S*mAzszkRanJmXzw14FzeJKT<YGskpxfL+VhthU3;sZtSkK@BU|i6ntufoHJINhT=mRBz5R=$B<%w*r)=YR1e;GC_m854*vBNrMGsvk{_ZC`|PvOyYRMBxW8Gp#U0ZcfIK@w5$EzcCOP6xTk8Z3T`NaaH_pzei9oEW%ImN(a4{&&5}7-tLz^*#`d+;uJ>R0%7b72^WnE&zW?e^@4fWO;Y(jReD}qC-sQod#@p3Ao>?z|!~{(aB#V!%>wFoe58nUk-M7DS_{z)oU;ombo!w%N%15&})CVins)4rg)<Fvq>pQ67wtj3gJRkj?Nzk_e=Oe!}CGsvyJs<tuNziwJ&hNnPlpUzxMx2c!-1aTb-tX;lJszbMLy6BK#8=8UObfYTqVjc2+0|_O<pUVzj9?~WbGNVyhc9%Fp9NL<CSjH!Tt#B#Z71Z{J<1qXM{lwU>L^xevPjUXR)jy1ur3q9VHFwd-k>?2;i!t*B*hDc!co6y(Qpk!m1XC+!#cyjJ&C8>$Z~?4!B!S1IdA;rgV(-%_w9eU_k%YNfAp&d@4t5U?SH-d-Z$>Q_r~EDU%&g#SMI*^FL&Sh_Kd?pLpQ@5TR^D0y_1m8BZP#8#~l+Io&<%S1cjaig`Nb39&1o&SRpDjJW5z-SQ-}^{%Hn=hQ>TOjZ2<bi|F*EScs~`jD{ycqmM+;XujLCu(i<LoMpjU^eF*bZY|~R<yJG73fj!KOZrY<kfZHV!z6ciHav-?{qu;W6`7A3Yo{<X!l}UJgo|}9YUFeRv+g;|m|J7mAl@Cnw)PmMV|Z+(((#I83F`fj0(&2NaBpSuR*CMNI>I;8!*n*-#+->IY#_4fH{9<X$|@)Y_kQ~G2XFq52k-yz-j9Fo-TUraci;Y3;OG0Ve+jQ`>W%LG;LW>lzX&hyR6OA0YS`Z4G73z!w^!n5p8*b&?Dds82_6zYR+5AV@4xoJS6@E-@^25{_z8^WgI|1+ZLE3f&YJg`Pe1GK49To&47TbJbL)8|iYEkW3E6dbXR|)#cFE-_)^K)<QU@h!)f=$!FWxt_DdZU4T#u`>J3CW$#|EF@nVlnd#<eHp?9T3#-ElRL-=W<jcj#Dl7$43q<HOr&R4%N|X7gvfPkV>&eH#{_!<WD3J?p{L{@_P1f&`nI&~gc+74Xb*?Z^_MY;#VlHY0gab^FVPQALInpMxnTTxCgAyoFYUXp>2$Y8zFh1Uor(b#m(J<kZ!)r>-bv=<1{gq1KOj^lG-vPILYW*0MVu?(n|%ox`tv19a0rzWcJbvx~;n@4n97oE*_QIimF_9??4O4Tec?<ZrE=`pk;=eC}?~3xW4(_-+k={Q7M6RTI+e``W46$+@tTb72$uKPNpqHkEhHv2Wm1YD;%Is%W*2YWZqy<$aD)z4p)gB%PI--3JMJjS?mw`x!dxppJU`b%VE|zH}Ut!oR4$P{B+@>pY`p0_b)I@~VleH+UTjGrH}P<xO@Sud`9DbYhJ_gK?vTn9Qt=laomwp_559Z~mP2WyPG{Y0?@;9s5X0es_hSG+5Wli)yMox1dsXqfR+|IRkUdb5XuwJX`thvq5O6F>twQ>nlV><PT0G|5M3Yuk>FI%j1q=(TMugxYIfCF7|u9*tppHC_|ekI$UF10t&?7WVNdHU0<2(GA9uPf702wP_X4B+%J`J`Vg0~K#9C7+_5N^dU|zBtEwstbubM#aMP-j9NbV0<0AM4@-DYPAwUtg;d&VqYqkyJIVkodG)xOU-vO>MbR+MBbr*}9o!ze|ya1(Efk$n*X|eg#E5~#brfe0--&iaJtE?!KLzs+8`WjO*Rb2+qwf^l%Y#?)WQg`o!(P7<&m1`tvWyy477uG1H_2ee&wygslsCGlvG&WcMwK>*7t`ZAP>D?@?Gbm-I5&3XzIRkWFZj(pQT&&c_jQVq8nw})ha3{?;1~WJ;%sSyrI#Df+g!kYVU%db7uVl!T4Av;Xr95Q3#9E&yaNKG(KWaQjaQT%JSWN9k2#@MV2tN{zgraHG$or53z@mO5??aw)Duik23G8Z|TfK0>>FY|qh46SCD^m)K>8Cp?$*yE!RXsb^<iR63^hc6cipDn){iYLaNA(7VCoZuQm)OHSRC6qTH+AEfAz$@S#$=v3_g3{O+^MeCVkqN5NOFgrWG60fE+1Us-gl*LO%W1nHlDv1UZTBKJycLT$_bnB#G`uRQJuL*RSHY|E66`j<5Y*O4?)2(AEU1N?zmmN1wm$-s>uVzX73(8qFkTefjk>0@&wDoBYmQ)GQO3l?m84y99&-3qlq+Y$0?!GnDkk!=asuS$r7)y;lwB5sxRZJQB!!<YF3N%5d_!_w_Yxvgr9w6HaJd#)lPN<KI%IHCxLM%fpP9JHr;l;cyG22VYX|Y7jQS-<_=pE@)U^49GngUOyvD#5damSIX%0Mo)wNyvCkD+J8`?7xLyAQ+^#3#jwjB!lW@lmKiu(%Zn9bXvL0C%$wS=vaA-UaZMWl5h+*Zf6UN11D{10c(C5Vt`zh;mA25@B<S)?68y*>1#dvSkx^31PM8i=Mb$r+NK#c`$*wL(XBKHNi!>AQ=m^X&;ZM3W77QHO@7*)B`&-Qslv}B+MCvyv7)>O_<E$f1_elyxM+(_9wv-CKPk%c`vjC-vQe__sTUX6__qmsjWMC%#DBW3rRT^1{xZ#5ivF0YK)#hMkk%vuNVHnXid&|G)mN#8k!%)5t<3sY%WRlzviXOz)gJ|dNdQEyw6WT~TA*I0YC_574BED^@+yko){#YfY558-iz1oJF8zzh;9y)k*AkukWo=3W8zhk1(px)&$g?NLhUUnm8Sx0Qfu?hS_hW}Kz~%jprs3FMzR@;*FAUeYQz4IR!p!}QAhZv_^d!+Td5Oqn}x(hFkBJtvb~=azHV(rK?bj6qxW1hu`)jDXH~o3WR+qhSm)rU{xCylcn6G9)XRxJtG2A((6@kv^Mj@|rH2z%|NRR`Hq+msXop$&#-!8*4G?xj*MS{h*dFa5e}73f(XY$pFP-&%aPz9>QuK4@bG2sV0;%ZIZ2AM#NgK6jVjboAHN*1UdQuoJgXyXsL`U0qV@8Do9X2ecbPnV}BhBAg-k?bwS7VR9UP`W*pE>t6k9Vs($VXeBBfKyJr@|jeq9m)PXN1bICk##2@DwaU~nT69gSkA4oirjDE;r<kJNbA3Z2I-?bc0ppKO6PCUB4=_;DXiBtL`;*@^as*dgE$YHVLReN&1#K)ujxYPz2<meE$rOCQ%jmqMh0*%V9t+G-{#fmD>Ou{vf5Vkp6C8&INq^fQ-%?3&3O_F2TC^^z*$rKwVQ*4^dWaH$>n<o{cn{9<MIFd$LzDZHKQBgg1x5{qCF&xsEa*N`Kr!$VSPch458AsZzaNlt^H8a>77&(f@3^#};5Fj<z2EtSQ;YFH?SFaUsBZQ*_kl`*-Yyl<v>e{IeyP}}ZzO0>E*Y)(-?L6Sj>d1gk+V{vQIoXy2?*Q}1UxkiGL#G4Uv4yaiL}}xx2)O7Wxsi!NFms5S8WnNz?nPZ%e}_?5F5UPJY<YYak6;n^YYrK?c8br&wNqFT>$n70f0%3|<{=h9dci+<k!~kK^XpV7fdl|i&*Pi1D<D#b6(a(Zd2qnkqE1OXX%EG(t!k~TQ(1!w+wq944$8`GQ$#f=iFdROA{}Cy2a#V5t23+{A`pbi+PLzI;;JIc8Pdkk4JYu>HJ81#aaA~6D`~gDH8l>GKjm~jQ|#;>t8j`3<z$=WWSit@+azpO$S5r0s)rL~>>y~v@opBe$&Wyq3_cM!xlg=tC*HV+>W!OVnm^<~OJ&;IUrs_R|3uF{7fHs)0`D<mP*6-3Hlyvhvydj+on!zyE%14LaPZiQ_t<IgBB;PaG!8t!S@V_`m!I%%w`1?=@iu6{+fRX74R%Dc?u}wlngOn(##!&e!6<G;Xrx{pwd!bPEPySLpuw@Vm2?uiTZ1a!x^d~R=C3AA82@<wN()z=tt1|<c;^Qwzvh?gi&&1z-fkx;*eRXZYe%WK8OJ>w0xaM1HV>%LOJ$9E{T`t>zj$_jd9f~GKf^VA27eFW593Y@QlRC5sELOPK7qFf;($EsTjNg0>oDv+92tTY?M6ulZw`lL)oYJNgLGwKVH@g<H^IW_F0>gfw}6%lbf-2fRb8-y7B)Nm&4n)LWeZoYT)ce!*5$h1ciCG)GO=((lv1>VI7)TK6G?Z_ACA~M0*_n6ewRIX{7aA9;OvWnQ9WvolHCXeM=A=Rzj%B3sq3P&V5vG|Ccf!79#Wl<y%aTX(bKa@3ZE`gIS=R@+OP9~i)SZX8JrmK7lzkB0L^Y{5%t}7w@A-~bWYfa`VQo!Nial%6aL4ZXaEzoA)^t=W57p!FNHyg9R>PZUgWKAs7+CiHrDIjl@S6RBM$(Cqz7gz5CM0HBHl2o=Iv5F57rsl3yPY}{tzuLtVVn@h?|VLz-SnEfs}apPP|2y43%IORs?*LjN&GUml&Ft4Pq(Lq>jFvEdVVVcVIs5B`|FclDGqnd1=yx*)lXL5mqpLpzm1Kz@EQlUE__eZuk4!otR6t`AwJ?^Mh#AY%e`Izr6g!x-Vv;GW~#E7mBOZ^bTi2_M$#So1cdgFu=WbJWPP4kkeK-fCWd41zGWpb@I40qaN<miBW@|kB0q@LS!zs>)uVYk>S12Yx_+a8rEojBj|bQA~S*V3p1IDE4ZHrvrx4u)pOPU23T(!U_SP_N;el|0|&@{?8%Q??wU&pAeB<-WON`#M+_E^Eya%G>p{QY5pV0N%GVpkAaq*Dywy-Sv-q*)iny$uQo*mC^8BP8*8^`Wj$8BGQmA=yhY73P&g7uzii)lk%vy@>l8x;3ad(rkGXo`PRUL)QQe(F+9|;Kw%wxcijbd2#ln;}VsM(Hsy|`13*qH*nBoN*%xy3+wgBFu&EQD{W@eWRHvq98zSmj}y_B&&&Si|;-1_ZpW84@5Rs<pQC<b;h8?7BTgVLgV%($R3kgR$8VH>njgy@9(vopCrgpd~*9aot`!wc*64*YC}D$lL5?3%@5C@qGCU+6Ejzh}OiA$aMv;rKA}Si^CQ#6$mV#VN=zGFGKzXf2Zs>!4atAK~Z=Gk9WpNb0^#uWkA5C;{mSWajR~H#fxxkAQ+vZDH@-6e$hQdP(SHFk4aX8l8Jp~a{$4kGd;@Yg~p88%xn$&1CZzy@K~%bI<XKCXtkkfiFjnQaxm;~Mw_7Hj*|bMy?1|-BRLNP|CJ&T7h8?3shRG<bHrW}i^nd4SO5fuT<Hem>YDDFnZopRHM+XzK@bkb$&x6YeDZ~@hc9#?+b7Ekhb_yp&nJ_j6yf_<h$Z!t?oYVyk(rgRs_vc{V3#5`NK98%KEBNS^2;y3C+`T+#=R4g6lvGiUatq3u&-PSKl<pSz+wC98MY{JxR9!CY)jKJ5J%u+`iLQDK6U|dPKOpBr1fI*3SA#()>;BC8b=TkTrdp-m=2*JX^F{!**+J*QabJQ8X7empl|nuI!0p-7M&DU2lORJEmmJKrQ#KpN~Wh{a&WmM?u5*5B`glVRHHs?kf&)JMHe`RsOg;V;yE5Q9j)EA^W8f4JSF6z(Or+cM?@D^E%K7{Zu@%VCB0oQEsbg^PZ#LQ^{9y^bYaKq1>kMwh`gjEf3XAt7WvRuLf0MI;tpq3oHuqAcbS|wzAE<0dEWV|spZrkoKaM(Tm5u}D{7&upOJssaM7!ub|GuE&BuBXyIgzcja|j34|08rXy@V|>$+<B7<KN3Dg*2M08WGaFrws~`~2rP3XtS=+iURO;$fS1Cs~zLHf3+HHPLzp{vzG!s+6MFueLaad)NwAv5-MFjuO#YJ6?m&A;JSwM15G&uf0h=NcKG)(}-HWlKcZbZ)Hz0LDx`tYa=50_BJY<Gl04+=E?`naSb%bEni9gu>dqx;Vn*8vZ0_6tJk)m=@%uwhC_vhLP(TIWeGF^+hLG}Ji<s#yJgae_<$`O&uQV<Y{5a^Y%ec?%p^Or-CeZ9yF}2Tww`{p=2xI`U<LZoTC9g-*~9UIJsis(jxBYnJ8s@TIC^dD8`hpE#j<V1aV(H1xT1=SrIZV*YlF#!P-IMoH4dq<h}&#&w&ayI?jj;Ch-7V#!VH=L#-I$L2L6PBg_R`^4ht=Tf$Z<6!`<*iL4VSQ&QBQTjo^gC>1?gEw}aKypd;Zt=~SicGSz4VB-&6`PJkkuw5w4V5RsKiA}o~0uykmp{4xp-B4x&`;Jh3tYvf=s%cO{l3=RfTKA>b((muJeU{+IBStolwzGSlnG(mm|iw8|$Uiwg<x|c2ZAWV}<15#BvsHt*bRT1+fI*Viz4Wh|=l`X~+vHCWG22ti>;rX~BW#8oYg@*ZGD2<VJmsiEBcdiS#x{GTb;PRb^^6*DxcfyudFzTbuAaq!xiuuqynd^8EILE8<=D=Nr*aS*3D`E}9QUAk5dN8+?a1#eU^$2b{mWI^pU=0JI=CU$@<s}NH0@qY409)7UxuGFtnCNNy%9W0{rqE9M>Y4^NXz4Cpu7Ty*sL3s08A;o&_io3Lg`*akhc1t@2`nsONHQ%FmXlcFE)TP5Dva+e!yx=(1ItY$UQ`rDe;#u+4+<j1uyz6PMIC!g!Z?tw7ci=$yq|TSKmPzhgFhal<|Z4YFA?|#NIzFWWK6h(DQFCMcsQF9^94Pu<LRhUzxz;0>wa<2Ym8O=GbSNJn?eBo^Z6BU9%OOvSXBY7EfgI13eTsDeklAla!q;nq!m(r0whnGtnl%FA9;Qm($cLWn3)<ovC2woLE|iKBxTJMh8|#BE+Nw0EPd3-@KZ9n{jiQwg^gpLw<&8o^s<14$)&|g+((_LQmJ-AOUK&8Pc)fGI!weu#)^wbnvjEPFqZ1i(9A}+*m^LUy2bF$#N{?nBof*8CbI*NeG}b6=9Z$X$3kt*FXc#w=Ai!+n9X7G5>0NR09`zc8H|yN*#Q=WnGzL{JLY9Fo)j-aqjJl#GBax&%&Eb)JZ7DvnmYBs>5uVIo*jqAfYr&*f+i7H8h*<|-wS{BaGoZu0qT!uX|_HrhmRJ$gv<QPV{3v?f`R`t+UCn#<SXcw+t?0lrWQInI5sA&u^XqDuM^K;4p4gGzZKHg$dGi{hW{^*1(g04=xbz1I$B=qJhhtDMRtp)dRxE8+GcRRYCLh<*7euOJIi(fP|a_$5;lP7i76M=0=7V51AFHeujp^`vRgeg6gy}rR*`9Ft$JuEcF<5v*!dCP-f&?uSAeh-?|MX_v~P-^G<Gwrr$-K>Rr8+07(;%EJN(7o*DDopvWsF6HvHN$ukKK2+sQ!OZ!hY(<R}?hZetx^W3G0>BU5M$c-6o-c#lvo!yyp?hFpKJd21N6yJTIMllmG0wT$mx<hnOHTULh)kQ6N^3es~p__@;>r@KlaBM}l`P&haHcBh4MGu1dPe?o|Af6nc@bFRMMbHj7F;U>n{6P(twGa7fBhDA&6)7t3EM(M8cRDU8abRISxbGT$(W{PZC2d-0>{`J*5?lL3fsde0C^@v~OVb6*dakXI;8rEvTh3YK1P#7C`;Tnq$VmUJW)sodO7!TBQkO*=}MoP)qLx3(w?>HcX5OVyY-Tvp@^lij7Cd=~E7WR7!1zEA%HR_uCDAJ}r3hHT7^M5+!GH8`SopgHCI^$;K9i`c|er2}hT$ahpV@X+HjegMOUGm)~^$IOujY<x0>ORezULU}hZr|=0v<5sjK|blcuzhh2T~t3PJ%|+8)i0keOq#HEEy7$Ebs7UA==rq_9`Eh2cMUT1T5r-trRNt%FPFbQY2gEE{F${8cRlZ-z+cZl_-^CZdI`WHA;p^10gHv=B05)vWE-QG3cT5B1bW*WtW&q~p_b^)8CoI=>%eq;MwpHrnC7+A9C2xguADkzDZCO}*wN@~2VuAN!0P(XE}g{=9E-i0$fK=Az%@xq=#=wZ^PMsc#~K?Pd!6IdF~#VcaAluiwKN-sx*|M@h`jRrl4oDm+76%Z-1NptXN7@gxNO0%pX%>4(0YAr7$lU6!KjN#q7O-<l<}D~vi~p+lLcn#tOv=(og;7OCf~AsxhcMdaNibw8%0@p{Z6v#_lvU7{DJp|#g20NeK|OkULW{dIm;HXR7Ei?<LUHaz9WllZr+Cvve1(U59Nmk$PPyQ=k6pu7C%L4MV0Q)Xs7R|liQ;a8oyOMf`>}@?g|A=eLg+ywmUcd1u$?)(l3U^q#JgwU5kP>_+Kl+U5=mr?%({;&wlGKgQuT-Tw9r>_-G(gzMQ2vjUTi&usZzc=aW9>`}iRE;s?JU=tXo(ZQcFX=XR-$V}pVE)!+T%5C69}Km5}mTw7tF-~9RSy!rJ%;}3uT*S`ZV`OQ!M>Ywt<mV-cY+#A|Jp(+gZ&9DF3Pk;1J;P;z9{a*rYR)U}W)enFA!+!(ie2vjOaAR+N{BM5x+y4z3IP)B^+JpxU^jZ9JWUOis=*OZSyu}DSS&SY`N;pm@WtL2`-6UmPLvt^ISt5EBdhQzsPE``^UZ!v!qud9E_VwrR-}iTl6Fh7uQ+oO+#6N=%TT#=GxK-87FbgBrG$-p&+0RX~7*3l(Ivj>BC??o`*8pv(7!q&Wy6sa>c*_@E+E{tgj)GJ8KU(CTc1U!9_T*3&gX~AzW7~Y}n2+mDY(x#P7>-q#i7+&K!|nV5Znm1CN!3}^kj!(Wnd?!IWI&ojSpB?2dW>X9uDXoomLKj#dP6Nc+XYraEY&Jdj5y=sE=#e4sIU}-h5rC{+-)3)R`xg_O!uQ;pO_0($GR!&5oVw*U<jck(NWBJptT&K&zjaC%spDRiT@cBH~?ZSTS3Ac<3FKjzGkKoQ*vn1UY|bDZF2m~;GUjM@ET)JqOs)t=?vzMjK>i*;Kvo7h+s89f0CBJ!l|rqiWS6^HP$$hs0u$8>q1sI?fAN=9mq=8u<|p3uKx&NeOO;peXOPE33@0d>2SP{wtwLji&jPkb#1wJ$!N-_$STwS>Yt{%H5mqU`r9?mMPH~!M8P;40zY16;Z}_RP|a~v&1f{IY^N&Q@f6*TFkZ^A>z&6)&P&&rpNO%i+wlg}^$%X1f3<_3Wc_J6+8h3S4d2wv%!X4%ji?EtvdH#kVb0uNGgT4vo13!8rJ$CIPrR+j%F_%Z++-KdCL)6?+(Fl(3+E##Sd(5^{=*G?@%z8|=0E<ynOM<i8BWxMXQvnrgt1En#20_~pWghlKYjDt|LUiI`3Jt5SP6KZNjBO8yl7KRF8$`Oe@Gl_du6><+b7g44zh%?YEAF=cYpV1fAinp{P~Zejd``jM5^O$!)}IWvWfkC^Mn8M&A<C!-~7(+fALp;`Nbdq^EdzU&lcs+FM>B$$hHvx%s=_@KY#Nl|L)CS{O;T2;42K=FwjkJ?xLA~1JgF<Bbb3(8wOfIuYgy7|JOfgfAL4Z`m-PZ2%gvB*Wdl-KmGf^{w-gZI-+gN$1w-JHkL%F#qKj|;n8$Y{yyWSr&>`oA5;&E-;LbXZ~<gv_}eaIED$w!LcXr~C|r-s<!iOPJ?P6Kd(z)ERWin^zpLX~VSa~sIi*<O79VpdUd8XL-%9ge&PUIR(cbNNI^8uLrD{rkJg4O5rz@M~{^@At$(kq`%rDwrqUiO-%XU0Pw;R03m6RsIhMYSim0_VzLjjt3yc{HbjFo3=NOT+Y$?1ke8z;PUaLc8usai)LI|WQwK@!mYI6HwdWvJ@5Hfs1KPp5{YFCWs_BzJ>@ji-}YhJM#o^O3cqTMbS6rdHm1Wp_>Wr&9?L&%HLtdA;h3CA$>OHPszY@HPaP^TiMT*_+?~V<5#p`|*$Ytl9`($tg$AdzAWX@VxfNR%f+WKO0Yjd-MlGlcZ&kZ7c$W-u;_D`OiQ5um3i9rMi3#0A-$T-Mfl%)v=bs%cpZR9v2ga1sbID*U&fArp4A<t*z@@GBo$_#;m6)dVc9=|M5@X{O`XFht?nc`!9a$Z{GaiFMjf)AN}OVzsFMsC)rPaga`Lee*CvT{TF}scR%{WpZ)ot@F#cLeC|XhM*QlJ-u(K1<|X;b4}T+gHOxk#Z_usRz>WNS9Rxpa7>d;eP^3Z&MhMBuF=o_Y!`asJtZTW@I_kL-N~V|jh>DAgjMjh<wph-jS^<?(6M12Mg@5{g{F5(!?bqM@#{cb)9dG$jdT3++FdZw(Lw#)_0|;;7&$fqpm$RKIMLosr*ibB{r?QTGPX?h9B+05V)~h28e#o^ibf-BpR(<h*{puHg{BL3Me)=E&6gk^h2;FO9b^qYMz4^@_{@uU$mo+$Mb+2pT^3i~%4#PyyQcK0BXI1f?vb?g+(&h{RM$s+&x!o+$Gf~^XI7_hiy?pH5<dKW(EngZB_9VB^ycxlpcnThzzLBXFPi-+njpYEo9ECTvlq^^Y+Koi<Q9*nPMf99xDhMw^nf9mfB(TU!0C&^ja3}2_NaExJVQX!B*{Ly_t)H2?Vmi7k4d9sos{u&cN9o|jtehq+SQ?Kp1n^FAl1VjAo%9*j7nfH59DKd&E@`bw!I}?JEu_Q#Y>3*0yqJXmMH_EGCvI5HlXgOK^MoYl^c((#X}~&L8&|j8@*OJQq4FeaPdZ$_<0)@fPlnm<R9w!uZ*0-VyVGP^j76za387Q!B$)^W1>W%(dW6YsZKs${i$hWHR1UK<pDB}Fd|g5gq)9dRb%t}J9(&g)E_hQH=pa;PeGv>Vl>M!U{xHl+s5)HX4(EWKPO`L3jp0hJ&=PYy{EQRY^3>^wIvuXlk##y;$C<6!h=Jz*J``=N7-MRNRLqhw39F)An0)^6QLsjAU$9nlGo-+piK$EEm-?O}7tZc5J=__j!N~@UbgR7$_=2iJePq0eDf$gw>q58rCOuSTN={o^wB?>TSolR%IEU#2oNTVcLa37@$2k|Jw({+oQZMvUFSeUGTZN9r42@oG;*sPLx7k{-nO$lLR<4dq0{qHpQ|Gj0bJ~wNAG0(->b1V|3#$BrD!-u0FGS@RqVfw@MP;fh#}+Bdi4?tP&8cLO9EN{xErXo#$miCvi55>%ZM8YANuicGjum4v<}{)7!4`xo`Y@M+4q0piz^_}qHQm6WGA$fz9$X2G#@B%c@ZAf3_k!MS%rWCX%qiBDffgT{6V)Oc?K$meo03SWhU14AXH7@()}n2_ptiVv4aPUMRd->2tK)wm$FH`j9RETvd*)J(-%BoPMLKiFj`(xR_;ae`&ne^2=^60{_xgv{9yu@m@StfH&V)ZF)hfJo{5k*lUHoC7tMM<3KPS~LY{nmt36E&WXRihC$8&AAA}!#naj1LEY?692oA>r!u7%IJtiB@dX=~oC<>%KnXrV#9T|1!#=M{OQwWxa4G^J+Q({|HjBJ(6RntI(dmsp;sQ8QSXH<cc9L9dOB*BV9$ms+(Og(ET42^aLv+3N3%CW#t#V<8bsN5~9^;w@D|!N{at5v##a;bI#ndPKHx><V753s*bX_X2D2v01kN2mk7e-}?cIjQ{m-y!neCzWGmowIM8kUeTY|5}xiW{#zr_l9C(=^SmoMd0R-uIO3j$Q^KSX_TE|y>)T$dwWbjJRBvv+{GqB{H#>U!?M~3#;vCH(PDae?!%B0AVC<(QQ29w!ip*cJB-%pp^<1A$$9~~(j8>_03`DPV<a}P~ppmF$`e&Mos&@T0qq566Q@jIvh?MrCAyVn%%P<rcRE+L8xUNr^Lh62bx5A2W%)o_+emO!>Dy)usDzmOix3}kFgVq^JMJQL1)@03nyM+6EM&4{(6;bupH8>=y=4xx<s6^{#HGWxFPtZ}(b0iyEP<$Xcu0+Re+;=zm{H|VK;g(x1zYxOHP~A4F?f2$lFfYEd++NPK#pa9EJc9KztDS!K7r*w!@BLTZ98A+^M!_Pg+Zw8?ojSMy6M>X)=489On5|5P`X@gf>A4gI%vrN?1}#UGaj|Bvqn_xS!F!(o3yIMuMSK1eGv76u^KCHXTL&kVByYvrKUe=SXt(?%9NIWNUmF5k0Of1qRD~A5)`0BF+!rS@3{Rlt0Zj`0J#Fdp+pQ+KYs{xVx?g2eXB#Q*?v|Nrl+$3jZCP}6{c0|)U9|UFYi-wBLpu$27HW_k7TTVM{g-2311m;L{C9<$^rBQGP(rj-YpuQ1@>LtfTD+<$-wvy9;^)Pq1sags34dOkXN$Mtua1X{W5ar)n3WUO^LR?A**Rs^&)m&gH7i+qt(m!OeGvTY|M@RJ``h1S_Woyo^WT5^d%pu5J{&D4?bp$(Q~c+ogTFd83_Yf%%ty01$KpQ}50!p~`m*GN<ys`@t`)C|vWUPagg~KYZP`gW$Y-Ts-PDps+FI6DUC3!h!M!57iYoBbt9f-lbnD7jKQB%-<~H7rf_YJfE^Y|+W(PLRu$u5#0uhn1<sCW;tD<ChWd?kG@?uj|^*;yB8k<;gd7FE0Gr;xHkEJeg%U|S})%J~`!PvKgDYCj#<JBg3v2oS|(%fT3-#Jo|M%_IG!FTD!bs}0-5f+%Psz)znq0dOVPE}3O>xvFwYf&I1!SAfC)%u@!kA~`Vq_xy?lb<NwxV{;^R8wm{o6VXFQVq=kj)CW1baSe9g@Gth9e%sS#_uT+)s4I%1IBs0^_KI}GonzuJkCa)Rq2r@!cklf?f`4a`tZ(L=PKng{Wg>gHna4Qq%8R6Q%=ZzKkaA1gHb<*NlFi6N_+q8$=zRGdC1x9R~`&7AlPo6O*VqtV>B&W>5v(aiehn!(pwdz;#Z#BOYVGn@6NX#Jo=<A!ToHS4^_TXx!#XOcGkNYN}RJ%|MXEYy?vDCLo`yA%bGk)zjw-^TjZ;}Os242BOo7>GD+<k^cfRJ8eghBR|&&`I54Ap<LE*m?*f;M{GJI-`14B}!PfIjIHesCp68N^y7c^#fTgCwri4_~dCfp*lvCKw*NiGOuFI1&FS9u_N0$sC9%(KqTOOw+av%D2(3zwy-$0Yyw4B1g$O*_A?|uH%I9y|uzCt6F8%{3gw&mp{#(KY)+@YxE4?i<Kmc)0?-0S;d1%|51Y%4TH3v_*NNqA{@#U3=3`%A}Dy^6bhMvV$l6^XKr_hk@usU_6pd5bNhwv;|OM%{P`UHyY5Y+YpF4Ji8+7_*7EHpP|y*~lCDz7t}4zgWoY3FdEYF#$$-szvXG2ZR>LdbyQQbh9MEZKl!tP`-~Sic&f(!xn7LB!LSByhVcQ?%pWiR+Dm^^}7lENs^kA)U%7;{h`X<t~9gW+|ybrY4>C`M_`<@uuM~4CPfU~`U+t%NT6At-iSzbl;}>T!k%-!4-1+Tg$ejf0YAlu7$dduOSQr+E1;f6!$Nx;YR^p!KjF>&yhJ}8@(eJNbaRu_0{FZFPJ6dNu^XhpR<A<)-gX>3Ag>?FNDo!>Q3?0oDJDV)Bx9&AS=#Rx6UyZ!(qJOq;ero>bh0-)%*dN#kiiYr%rl+>K><FIa2GmN-EjVNUlwvoppFfwoc70X!46LIYzUnNWqt^RV`2@Fph~AOR2}%08y~OiWK$SCKCT+?w!{7DbX;z%uI@9E?*ftozOA}NTXD6o@0c^Y^8#6Xn7;jNl24gRZT0LW?QPSJLjR7POwBop#U{m2n=CfbDdXTNhMk4?Qg&PPX^Oei`cBA72XP=DA?i}`SfQRTI>AxVPj^_C7WVSdv=;?e3PG>0MAZ@ixb180TkF@I6XCnq!VFgAG@xbx*`|V)2;_-_yNDMV<L+p7xC7e-iVbqijSIsH=qM|B1;r#y&~mFp8PXuwIo0Mu{?`-nH}gT+qk{tq9`6GI>!(B9@fO1S)5^~s^wqUn?vU@w?5}dL++qc@F!yN6S3kGsPG_yXK-Zr=eA3jf6Doafzw53O$|JR8VxXd_d_*8<Au(kXZyN+F4D<PbP>luL7)2h3Ge`TN0+25VA0{p0TtsJvPp8inj;n?Rtc2p_elY`DrjY_TT80_!@Zgh4F&o3nZ)4V_KxHQZ(z$DI01x0@&t$_eDhflS6nYIlgQ4WXr$}d1Cqekxr?=4vZ=tT*z4~@7=%L|n-);p3l%E_U1Nq6fZ_Cps4nBun>*R|dQ2`$khuGu_MdAj=qJJ<ha-tdV3EQdOMT`b}IE6U#M0Bg!*XAACXO1cx*fQ7C;4n)^<DwuU0<?riDOFu!8S#uF?1I&#?&9FSq`d<;F_2v#xPYPuTTr%iB7%;yNp%FJlau@?E+%`c<H7D~XMOEj++Ocoy%xh`3)eo#>9{5*bb0bdJnI54aYB0QonGuxy0i~Gat{XFL)}jglYHT-G~jm+6P6EW&oA{ncn<hoj8-_KXHaDz3S_%MsD7bC?{oqa*^--akU1L=LzKEXEEm~?H?H6G!UrwphqJ?=%wEo5$7krdvqxM7tQW3^z}iFNe=|TewHa|`O~NgM`A!m2{{vW*o~!||wRo-Lg$tRVK{^>w-dx3-;j{oUj`}Ie{4i)Us#L3fS-~iuC0251VYHB)026Ky{mEge!ox2BZaZ69yY7V->gPiMl4gcaVQf5L7{%RFoCZ1oP+mmu2zxuQlbSFMJ+WOZ1Db%u0ucytz9KYY56i^w@E(I#<8EK&*t^v!0At6x5_p_6YNLc5^>{QqbqrvJ*-?tl@Vh}zCMGBofv}V_*w|dtJWGy&DK=4d-f+pkyMDZCps2Ot5aUF}M8-JmPtVU<Q8a*NVBi4WZlE<_eiy)iHTW-H^Wp$1fdU{_c`=a4`m}g*PlLkg7LC3TO~c;Zyl?jFVQ7d_Ap3~?a9Z6TWGg14k9)xfL-6SF=DiJEyVGJOVr`GIV*&_8x|w4WJfM=<36N7A!C*RImJb3%>|%sByIu`}f`J_)quMVH;ppb(pf<<kXEH-5=~y$1tJLr`wm#h>U;tflDzcj3B0XMO7*bDA^;(M~qB*mjm7{c$BWA<;(Yzs)C0xhuFt1&Vh;7Hn3<4|2cmn=mA}^l?r*NCi`||NLos3jLm@0$|pKD#rJ{S(u!*m@ZKSX@ZGkOG2<l3v_mcUiqk-E@0a{`w)xcv6KxL!Lw-;o5LXoJF(QVOmN=whBtft1`IF(HI%;#TZn<%2pW$HM}smTZlBjews~<ACKj;p06UQ8)|(LH7ZB;hj9*N-QppxRc;YotvPXQ+YbVrC~+fAVqEO)G8~Vx^3V&s27-W7oRj7qV1_849~tk1Z5kj(?g>5gW1FktZ$mN@U0mIHNMeS%=0->s@foGFxB|#kO?*;;uDHh3O?NwJav{A#UUIFsk|cuvysLh0@J9Id1i$b!K%)5bAv@NjbbC#bgIyFmxMATKiU<iS2^3;%g95Uy7Nvl3ROs0z2xlTsa>&Y+7!pZlXQ|EW*9631z{Oe^qpdc{MIR75izI6xES&|5*E;w;z}i*^M{N{566nRojb|vo%$h{!{Yc|4w<A5Z-2;b?}&jnnM}1^gt_0gXyn^x9r|)R2xkRh;M4R3U=6=shtPA09Po^v&46TM`xJQqX)fRM%SvQfViprYiPKV(4b}8ZlAu9;lxv%DR4eGv3i_<PdN@5_@`PiBU!eIuPIf2R%d;I7p4qNWyJzM+8#8<yV5fNUpAz{8_6>t6YRtJ0*nPMzzI$k~*JZ%Pn`1f55eJgvd^CW;p8rlo+-@M64MekQv326*bbSWt@UxnJg7Derw_EZxVj>aNc_JO`qi2gtxRFIgx*fliq(Xer4v9x_ssK=cR*d6w23QI6T1*K<a0yXW);iwL1+$87=J<AZ1_Sa=cExH9fX1MtY$4H#cPz36x+SVhVm0^UNwJge<U^n{EGrRh^m@JV>2zPQS08=!QQ+wR43#?Tge@xst+tl`iN%+!2`Q(i!_2Gi!b;{Hw`=GJTXWjyTLA+t6)!6{KAzK~%xzVTDKv^Yz25GNj8)oIWYOWU8py~-2@|hte)&oSrRZWyiNJ(yiuibtouYX{lJAX*NtUFT42!f!{fw=1#+buQ$*4<Znj?K5RGjyQWLA|ntjpT2$gei);--w6)H_{L-$YHCnC^NfYF3kYO4vepvqUBD?MeY|G+)r#B|&P`B=qc}NHeMtF?QS8iZN6r{v)AT)GRX-LFB!ntF91CMKuzpuAT1IJwDv#t?%|yI^?z6G3LghT3ri3j8jO4C(h_8i+5sPwqT)lF%P}A_8oK8u3R}dPA7ZH_TEftM;Xg1C2m*4VK<n!U(VjGLkznCU2)nC%*3%qL08+^b+?FJ$yHT$-A&`GWGffDL38F;s_0=fx*qhgt2tGy7Z?`JZJE47wIuV#l9*?C*F``(zaXA&7~&99^ASe{h(t%*sb$mUQEbXUuhtRjD_S>=z9sP2wcggC6Kbcr8tHP8$=1MhT>@oSGE^X4gIxJAL53Pvzn%6fae7xHP~Kwlj?^`1m0i^3e!4DgappqO7Muh5+b-lIGiZ6{d2OTCci9d~8WBy<EoKDAJ~D|DN|5XoyN#tJqBL6j_znr=kqOTb^HRJvuq^%q79F8A+fjz{1mNx9trBhHwZh}PYRs?Dhax_+i0u;9yrjb4>ZLd8Qa{TAz(Q2LlSQnri1=!sM~)d8XXdnai=BYf_HytXj-GdslP%UfAEce4#~#o$pZu~+0iT?1qM4~-BBb+zg4;eM-+G@-J_<G>WmXymn=`=aDEQ7uVYEy_iTzlQY|Ix|Rs*UJ0!kDejp{5CD}}eZ7u>$H*}4D7Z@-8WVA>tt!B<gmmxbSV1hsLi?q#D)aL}UWC+f{-`AI%%Hh2`aWVVg$$7z`nyZ@NJe-4-VC%`#N<dG(`?$_3`_LWZ`Ke?AYc=TX1x&NI<cQzk9e)O!l3ER5UupnR9?IIlvILuZuDEe%CptgqZ8(u2-<{`4(YJL`qGXti~>xiKh*dyQSNm5Lb>?KAH>LhK&7Yfr*`8dhOL)b?NYq2dr)kH6D1g|XWYhkV|xf>S1W<-tob-s$}I-?yd+x<Uz{NNGb$K$8T-N$$6)@N8>5=BVvJbrZl!6!~B<*TCFBua*Yr<yUAG0Ci|0rTTS{P*c+&mKWpmTu9Tm5-<w(`Hu`3st`5Rzof^jYEzCc~LUhcPA5&-U%LXfJ#7ZhHNAmY+iea93g&eOebU?G)_95W9d2w<4|gZkm(~Fr)8*t?k3}5bu!?uv=8Y9%vX6*N1HQ2+0o7D+LY3IAa4Cexp-jd?TxQa>1iPbKr*Ct!?{*M>{J>T$6==xqsbSAfH=sq@gP5>t6OW)u7fM$Xql8RC(}@M_Mv~eS~ZEMk7XbqXI$DOSy{n3Q9K%eGRNUb|GoSC(yJ++kfGK~OzzRG*Nm61yl}q0sI5he<2}j1%-v#U@p)UVL7amNX-f#|WFVcjtHw1S&0bjiV6>PW0h{AAD)>VoS?{(z5(`{Axa&)M0^^1+6WB|RW=kV)$cXCQHK(+18T|L=<GYW8@I{dptqs5iVzrU6@8!F_U?&?E$5BuyxjtSiKfdF=(J+H)r((iv69i-3t&)j<43$U3kpOP_oME3|>iHJFN2cH-xIXW&6#2BeX8>6uwG7&Q`$UrPVp7cZSV?5iMy*pW+EWe;ad3M$WFyteseuLZWRQ;YH^9rq(bf3sb*I%<?G2-vJ})5JwPT|~mdIO{F$@cuIOi`78vu*KAOClX^8^Jbo|U&mon+|JM%_-b+j(6(ErRB2cRshENME183CI`o!9wCHRo~DviiQ?y!52G-Z?wjO1;=Tr%u)bD<qgq|BqA&d>u#{Mxl8n^^0a|mR%n0}H5B6#twqx|IRIz`Y~j@s(42qK<~C&0E<p=qO)qI9!nuphd+sY+<^)*5pz_fXDO4j{pi>EeoPI(Lut*#y5nSPIV&U*ylWY%ey>P$SnW5JVtflN8SG)oR4uld5j;1X6b{eXJHi<eCOx2#LHW5ZziBjSwY~Uz3JGBz|pp*k@W^qM>Y%3?Ep3171&6)6s643Ody~=JegaZqw6*`>p1zhh3Q)lc=xWrC&T8suP#9}nGEy2SSTa!dopCpW_L=|Tf@=da9=F0CyS)*}G#V|c8@&O!rbRvymEIB#q5W7<zqKTn0*lRUs1C<!hTqTB9)Tg<HQRZyn7+3kIZXZ%+FPt7HnB<wTxQ<rXh%JEVioZTgU+5G)EAlHY2$>)}vQC2?xThcRq9C&6!_;G4S5M8v3r}o4UV*PSkT4Xb)wTRleLY$`bnu!kKR5m@-%KXvO+XEp)s%db@pe=sdy{mam{Rvzcw*8IJM9vjHo`*GlIbEb_6l_Csg;jTbx&eJuxvT_9dzU&0<|+N`Uf7OpJXTFlx!$1^0`Iadax1EvA)f>QQ*<Jya5Ma!nW47G1MT29n@XnwQ_o2`ui=>3G}A=F@|v;)&VFd--iR3^em7bM|-dVW;?K94|V_7RsCkSItHeDwR5X;^IFTf14)iRPGM|uaAqm}iNsXwZw&VV?F$8}Y0)mx%YO~oj>9qVoUBKCeyKm683dEf!3gKl*|g}V<y3FCYxKeP)FOAeoA?J-5IObGS!X%H!8iSDBfqy|nnNg)VQ@(|OxfX%qU?RB5K@;B_uPu;(z@b=Oz#&671V_lE8R?!`AB?)U!jA|w+BZ!*8^evPbThR7VpIY+LE?}344m;ny!o|g*5aqjr}4^7$v{@ER-2s{P#-Dv7(=M1U>%SP}r4gK<g+QA*l%NWRi|V*pB=9ymAcGeND4uKOLfLB!ytF-h?L#jrc5oTCi9MKkEL$u(ZnVjYK-JarRT3{gbUYj$uQ?#$xKQ(%z;3T2kjkQS(J9yA03)C1Ga&s0z(iFrM8>+rY4M%?V*m!dgLO$dMNpljHjo!l`qBcaxIV$fW!%Q9&bPpxW8@vIJplThn@*%nHkZ<pjYKsfes3OTr7|C3&AB9YH5hz@@U7=tf_*6a32{?D0%(QrTW>gH&GJ4%gSUkq0&9tKMR3FZj#%kyXr4sxg&lA(X7@S`%CQ#NW_T&FJ%Px7X~dg>L5Lq)2l+8<G32ZhfOK-~4z2UPHd;&p$AlA^!;>j8S%yZ0S2tzVIoT9Ut%K{e80blnvkvZ|UQuI1DcFw91r)^b~~!74Znh$aqYK$Uhe80FM|pa5Vs--Tjg=k3E93lk||3lW@<s1sojizz3lRru|e1?>MgI)(&Py5vm2nU89)=PTDq^XpYd6o58VqK4@%?j5U0%k|R0tVN>$VwstFU*9TeYo?8nIPt7VjI~5gd=A1s`=pVz=YHVUO&g9OYs?s|)Yh22m<0+9;gK+d%uC8yPR*$X~lblXau^qhxNgN@A%G;)%vc(1=6c6f<jTOY90lQ=q)B!uM;7gXLApi3c4r6E<@0y{0*1g1B8qk!ia!^${sHt*bRS{>C=q!>=EO8zk!qsJx4yzMMzW9v)Vo(fX3dOTm&j5uMQ{(l`usn-t*i2hFA-Y=9WKKFE&$FD{!w*29Ua&&%A$j(MHW*P&M^Uhor)7<_Xf}RJ4G)UrQBBjHhTJY2ng(u_C3<X^v1Lv^eBEADRq0cwnb3z?kD<93q0v%HGPhzzt*_^1P_uECPLqQp$)3t*qn~mk&}kK%N~k*AK-<h#u=0`dal=P)^dKkNZ2-v}D?qthTESrkR5?6heiLp&{H^R#pTdbiFvdcSx>xY1L!#JHk>If;Bsy-KO)We#L_MF~qNj#$d|Cr;mqv9*lT3$=76TGT>EuKZIAqimpBpvIc+;;g`*hzI@`NUc2?iT*iwc{Am}E4JhE@TMYnyc<oGG6X51}>Y{WBHf1({3bwL#bb9AFhl=0QnHx?d?~!J`b%j1M@JN*qA*9FJ^{yU#}O5ST^YQ_?Jmj0dH471Y5J!87GuO-IQyN-3xZ6Dprya?a&>r^tqGxf8MY8#tg7P5BF(M=E`nJagvgJ5*<Aqa%M?iOyA#^Qm-qpI<UVy|INsk`QtvNh}S5Uen~F^{Z6n_7n^~sUOolmT|2}C0nD_h1Jg{6J)R04P59BCTuL74u`}zqBK(|;5Y`+6QEXQMl38$^n_SYAk*mohi>^AOf~&-=rObb<4&Pv8f)y;6Q7q`AiH>BX)Q`GVDqlgE~nSkRjcL2%)GVE=U982MJQ;JN}g8@M+ak0gZ1tV3u_!*({5dL>b3CJEwHN~F@8gApNY4q<FoWOw_<26*%fQ1S|}H_SJ5A>^}9HS_i$6xY*if(4u#V#`o5MqUdDD#gKB-kJ=l*utwL`@>}~DcYwsSxR3f;0d#W{6SG{@NiW}`JsPxN)S<(vbE9MRm{G)B<MP&|_YTi8_7w+ijU3FB{bks0C^B8Uq``loN)x?8(U8^vghIS<C6>NhjTcqVw!^6Cxs^K=MRVKr1Uk#B*w+b`05uk%IlyqFplsid%5jGH)16US#m%3>!kXihe$?zL9shUJ3fv|IelNOJ`(IADQr<Jvnus*|Eg0EfwQbkR`Y;SMt^lGR4UK9x7OyY9am`lStIZ3KgG0YlmFfTp&e!!JBYaGm}!F$fEb5v8O9yt9mKFYJ>P#>?k8zXZwHYt>h;ZzR|J$P<X3-)DAg#LJ;{h7z0?}R_Ex#6ba8m<R3RgnQxWC0O0>YS8dco$*wluBG@M0;WvM`;J<=T3!LVo#@7y&&w|6;i<}v;|ySvXe~di#S9sW{xX_z%V;ey$P$CL9MByKWr22N$P+ll93*Q4W)}hR*MlR1s$*<m)4Or9ZRE0I`_Rm$OaV2iRT(tN)=LNnL>Id)G_HuY1tOcvF+0uZRQ#EogA4P(69EYDPO6=JV<XGN=;^he0*pIrk9{WG7m)+8>agpTTSW;o{YnWJ{WIa<3aF@l`MV4G{SI}JJ~7(Z6Fq9Frkly0Y+Vg$hw?H#+V#kHcOF$gViGuq=jTH2S<XD`Fh~qeT+4#1Y<ftwKtRo0$8==8BSeWCbu(6*c!joY`vz3vtcdLXymdJ1?#O=uu6V%F*zQXq}4<zleaQZ3a52cU^`P0!;4MbD7+zNB0^b8AH6gk<ilGJH`uj$5E3i=pANs!S9iAUBwW^P7>^(7J=q8`8>8AtqvwFOyW{mv{Y(QFEJItR!?-MVr)XiP7m_-1_)BI4re6t~7Z~TcvN2ewLvi7=IvjWXSH>ui;p(`6d~&)I5kf6QGCR=`)ulRGGlJj3J0>6NHix1g^9tk6c>U~Y9rv`5rf@LPmGspTAiRc)$_zs#*Iq$ixx;)tPex--u-<^hzQmO3a&K15Oe3B829(6rW}b8ptR`r*8(ye=ahDhOyL)$O-i9J(fh)n*dWr%IlMV9k6ARx*PeLWr?S!VR*e@YwXJPU|)b={(LWwlvU@m8uh*@*N4>DUr$vj=p?7k<3!mQ<xb0OF;(E@Ct(=wQiro{}+B1LpcbfrPN!+e4vT=ywf!Lh>Tks*rIQr#T0fBZBUW!a$Q%n;;xkDQ$hKt7)B01*yuKY4(l*z>?PbLo)T$8jpnVUTB(Itk2-P-d0!k!1Fe%{vw2JB)+dP?sqcD%sH~s7{hj=L6f0Vdy6szR=ura44mxR^m7_)`b~3+Loc8$K}kzFeNju7tdcI7kcGP-z<5z3)^QBnjCKg&lDw-Y%pF)0)kD8jgM3Ow75g=o;$O{F{>~-u@P1Aa!~p8ejUCA!aE#IVT$p}Fx!PAaG395ELS0)#KSTY0lALR)CA9bz%VEw12XV3t+4MuS4TUWh)q+_wdF7!s6;FTX$8hdU4)T07CcY#9W>n6a>|G(wREeJxjy<CCN~j3H+<Hpchh3!_JfD%&MI}aqR?#>19Srhm#fosZ}kW}Uo)rZ<LsF3Z3H}S1ar=0T+^2c>jLp~JCMIa5%X7u4x_!IN}?XGg$jYe3>89Ya6Ca35ph1DT!2>WMB>n#v0+xw*L<8{QOPrjv|IDs$iw#{w_jY5+j}9|dm-6-A=!H&*?S?`dm-5`hLEhvl1(Cz1o>QOYb{=r_gF&|!fru+@rgY%utlHTyT6I7;EbrkF|m46kx2%#hGCpIqIA-4idaI~7g{J&tUVZ7S{5TS;u(cH+m%@u`I=?fxMgseEORuQqw5{nkyo9?!KcU>)75H}8QG~WD?kq^{GY1TzRprQ4#rK^E11&gbV?>Jn9QWa+iWJ1i^R3jS6rc%x#Nt|QqUv--*7$N`8%+;(kUwuzE`0)$}|uJj2I+=;oA5{Ro@%W__N~xbvgJ{UN!{l&X?*cE(DMVV5b2$zUQR;g4^RRTvkfF#F+ls&o2QFlO>`ehhZ8BANmD`ptTj(HME!`zOP~C;$%D*+1i=ywKi)Iw#>4_QsP!E-bR8ipqnJ!c!w!_NYo1ANE(8{A2DwZ`6I9>H`6j4Vg9FTS87~{+Ag1Gd^*zJplwm@JS6gUQb2u1U(Yqtn#ZhS>>;heR_!@pa-egnH8GQ(ss^oi3e<_*AB03k?tbAEfi%q$L@dacFVAP6ja#0_^M*eEU$NO5<G4zTPnbL3j<@PvVyoMh&R=*MvkWXhGB>|3q^rU-$C23yF{`yyJ1QULa9VQ3xuz_+F2-Bfuoa?uo~HqwBUkRM$sz2od|$FtDC+)$hI_YFP{wK2*@>ues=DT*UAT(yY?=hx<q!=GiBER{sgNh#_yuVZ7E@0Ayyz3Y{rSB|o#giB=DkN4t0lSf@b<H3_nv7{mY!eAQaJpOgEvL|kt*N`y}jYpQDBwVK3&FLP7g?N`Id1fsy75jXN(K?HS&e+vV&PuifX~F_(BMiBff9O&dhVvrI?_|hs-OII*-Jp5*QJxtoNz`*4MpXs=IB^M(QYpT|>^O6c!|-+`u!GP9Tpw*9sN?p{_BBf*%*R8X(*b`}WfN48sqHW0j=g;S*7(<vNeyE~JKtPwJY%DqW1=uz%9nSqqNsG>tI<wB^32KK|hgvwa4Si;1>X6DYvmo+IR*U-;)s-0jVHU|XHuNABM-J*_R_ys}JBod`^JfB@G&OZv@gjiNI?-#Y}$%@-H6Vjz~gC%vka4>HXuNqBpW?2xxN(^X&Bcw-G=JK;+NrGY{IO$l53w0h#sEVaiOQ>&81yg;CHcKzQu4c6>@Zw?ask~Vp7%yypNLeK20wI_Y1F-w=8Pe|&JgJ)T$C2uTy!E7|heCh4DeY~2ACU?!m-LZ#>tIcKX9n*td&c-K`8%IIUN>0-wKX8^?3WUN*R-n;I%y9>#GPN9M=qf(u1ls=mmK5N}>8lkxxyKE`Y1rG;yx27qYuPv^VnB^W!euSJI&zB{r~EdY8vQg#;d4cMdip+bFmwKtBYMIt4ze-k6Beom)t?@7r3&cA+5#EPO}>unt{Fy}=Ot*(eCS!lV5L76!9j#}p&Cp&)U_!#b*YO80dwf-nDi(;S*E8Y;H@MuRg?|rD}!~k(&X9mCB*DuZ!kCaGn3k#9IpCs5QvH_Y2q|bgQyBkd;i(pC!7kK2*x4Mjbb2jmgOVFzA`{o+|vhe)P<f|+a>(Qg`~^}GxG1U-E<q`LHpRSM%CwKpjg9Quk^*J#`n0yx)NbDxRJhzyB1R#Hl1EwvxmmBr`vopU-%Mk5}mI37|lCRy3U)Zj!bveuc(m-b<M|U{<iPViFVr*FI-{OhfP>@Ik-=8j^S1~qASW`c-e@QbqsvHGT++_TtpVHvDj#3hKg=rwj8;#hPiSE+17_$cAPXfjN4$vK{qtyY%{wd!K;WmmT2PH=Iy7On6cp5=F<m{KEY{mV<`ybgkg}l)w7ac^tR*R4(2Nn36{s%ge4GzqCczd=hAeEOvCcNMj|Cq9OgSjYSmSgxgX@Huy+O~EN6Ynriwu>;2e{3_%c1h6c)6&-;?JAAF0}Y0NOO8GfY=7X48JGO`Se+;CzN<5u}043L_3xB}&;2rIFb9R3s>v?5$$Tj8%>nv<gM{r-#GKR>c;~)Q<4M=6=6Zl9^?S23(VDZ#GOPs#~r@2h1ef<*cDNK>6LBYMeD;Ghb;tgk9bc5G++$sj$(VY#-*2BZG<8D)EAW&BBR4-0EopqTaT1cz)()U}TT;Dx*E|K&lhXgf$z+W)?}VBKzJ&lfglek;K~(?xX|5Fcx|&`Z*0>nPnVuj!d=)-M5lZN?b;NXg9_9R_`BM*dODjNMzpjY|z^ln9bR61vrdO0;pK!8d7`mo4(vHp%MgAMpNX$*?yQOm-jWXlausVkg^p)<z(CfG@dP`H2@+#S^5^^xQg-br3W(AizZ971|C|^3I#;ADX!VIa>2;t1tO^p%-e2$LZQ2v`;p`a$Oz_jDJe#4j%2P%iNs-l@$w=HC!-=M$HRP@umN<*IX4{%Xq6f(DG%~-LRmXS_)SWALt)$y7$;q)swd&n6WLhMs4{$p%Zcz7QCeS><jq?QS2ge3WppVnNnZtm2~qB$DeR?boenIOO;fHqGZw0}W>;?7&ZZaE>{92OPSY6A@^qMln8JioQ6dOk_<zgQ$cyUe$~`akyjGO^0q_e`_Dp(nKlmze*;^WckWt7;B?}Bh>u}3$WLq1aeuh7I^yE96&l(PKTT;YV06nr|%^laet(o^-XV&EXw%BdH#8UE4Jl9TRI9_DB6*e9dS1J_XNP}+SYl8}S5$wdC@oDTT!X+?5LQ(8YiAD*FyRF{s*|gY1un^^ucR|^V&eW)nF^b#=S<e@i(O)4+&?6#ApbDPBEV3y*KyH1G7ef|Z;e<W7TYh6ZjN{e4VNsSaL6_<Mie>dyBY0bO0UNKqpJB^zc5%FTVCK-R!HA@bM~LCX!Sc7PjJYz?Or$cb$bi-FZNa{2Z}&FbsaOfx%o{SQOy$H}xq|x!$C(YTT#1}oYc;iGjmB9w<&pN>tFJ5gwe@p~V-I#cl0D%70o>16y3tgO&=5suy#b$?i6Ry~5s#BaAB3+c%d@)kOZHrta62ET9Hd!zAxUJCTxdjvRMrJxTDE#s=D_ik%$0z3%n}K!xiNa%9vg^5jKD=MK+yPvy_tAn26+lx#v$M#VCojLrGWPpYoAU|@*`sG)17j)eXG3|x7V*;Z?}|m*2!9cn<kZ;g`%)ZqBcbZ|K4fr-w9HZ&F)>qu)&he-wSp1^xe=HV)rZuK_eZ}4h7j^sf2Wm$Al+`D=t(;O(=#iVGhz2#|oHZJ|$JJD@b}8>4ECfCB*WRO^YpbhlwNykse+~W#rWw$@QF>gI&L(%)w2zJ~RZ|nsj)$Te%`}Rsc5r76tkf7|n_=7CJHHRK6ub{?<tLd9jdJR6>5?UWMsT`@)EQgn1hjTn<gJU2vsJVdqr4CgGZ(5D`ehtVD_Z9$C1{UeRj|&YYf^$DQY0v#`=Wd6s_jxCnTUvT}nGu4ID^XKST+=hO0}#=Oegfn+RkUjTeFTbr|WaKtjbU~dMjOGQnzga*Kbv#h{KY><ss4~`iA)-O2#DU1JJ9eU9Q7WrU$YCEth3NVlE!L$ph0VM87d^?&J%Qipq5Cg;2#h?pPbBN^<prcd_xW-t;=OmfR!KgSMNk7(MwwLx#eG*ZzO1>qm?$yXc3KJpP4(VZL)E091GT24+RVl*T<7qT1JclGv1h}-}IqAv(7*@0WDh(SW^rFWx_B50I;I9eoKkaa4N@*R@+R+@sJ--xQg#+B#AJ{Sx8B9B$RJt)kxwn@eWh0g-Ny%5M?hI(RdW88TNnmZK28}JPT<{!mY`#mDVfz)?3pQ&cIGsarACD`aVr_IP=%Y5@$eG|0_4f;!_X5}l;SG(&f#`~3i!6AAj8;gN5dR%xJzVwO*$}UfiZvpIFKXN-1t9hjV8woM%&ylXcrr0SFyE|4(&Sx78Ro?x-gx9eW1(&Opz?Xr)M~?`H~^Xts6EL~VxMitLi*-+qibHA8$F*zz+1KkP=mH(#m~0{5aY3pK+ZWN)6!my#u$Uqp*;*8a0mCi!;lH5^ZxXF(9@yK$F-JCTpZexbOqyKihK$lf^n@y(#2W>T%;uc>ful~opWFV)2lCX5)R}NF{MZ<-Q8u5nlhDgAphXORQHi*zppA&onj|B0z?HkZc``^_a4Wph0CK#XPVjR>V8fo;Vl353B~=<LA0pMfw-|sR#~KBxS~a>6`P7T+7-3o?jgs<Mq9{21wsR(%eQ;m^;u**cD?6{4&7`7Tw@8U3r3YGFuY6%#o@IRgcK!)mg#Wy&SR0vtf$y9T|?L#I`yN#XBTQi+VV!K7Ok^l?jfwCZPT#j8eqFNwM5jlu+}5K*M@cI(Auui=(Fj*&n~*#WHs%CLB3%7+{&+4wQA+h=2p5D!Z_(8>J&g^N@}@JN?k^!{{zCrL11<%8D^C~%xg-F=xc|wKrbd{w<>u=Z8~!Y3C*_3y0(Nvt+}f_2(5V;P%c9swS>|_y(W#9{py0&a%Rqo<&-AQ)rsga&HeJNo%PzdLAN}Hww^|e3y1@NUfWZ-$@KXww$(LtEQ|BG)ilOxfzoF-$tFwa?~I@@TRTU=$7z{q9N6b@7C(VSGVGgOX1wcp=QFd>m1~-twIEDR@p%k$*YVo(7ONn!-Y)=5oo!Uhd&TN%zjdpi=B`vRpEVFO*InZ&-!&iSHLewQ{T?e7qX_ZQ3jUFJt8VpL4+}))yM*a9DDXS8Z>N~Hj4s>=q0~82m&n}eJw3nF@}$D8KU9>jq;#nF_>I@vW-Z}dBtm-F5==_ojJz_O%^2IBBy|$hzyK95V9fDqt<Z_si{seqT-5Ts({W9nr^b<bAfrxwd?yG;z)nahw@0-^Gv9Jijs<2;KfXDo72vQ_N^x&>6a@r<i#XoSG;505;2gElXnbP+Zg7{A_!1Q1qPgmun_(j&9dvutt@9$==XBxb3m*v!QP?B}orFiOa1yT>Qx#waf0UIFcVOmQ^rD<dKB9!997#s_)Z9>cO{7$d6mz_CfN_{6CmkX8HYvsvlxZpC<GWZxmx~MArKJ5b&;ANMtgu+ORlAQoktcNO=0hg4P}k+|D$?opV96wEx}X&-GdN%rzi2HgMXV6!_i+CXTQ$4EJOI|3{Uu~HjFj&#b=a=~7P+8)Xiks_#oOLi*|5#e{IbCdQja^8*RP6ErfZE%OGvop4M<xPvzRsn#e*5yeDD%6iswZ;GOD_+%QkOg9*jod77FU;Q@hH*8v7}l=A@)W6xwW-VmR>v{3&nj7*CcDfgR+@fUdSHw1RLLxK=HDMwo2HN5CeOJ+@O{V>Er0N7>0#86e2>#$5+C(%@1;d^6JJ2p$&0%jJ8vl{*6;&a+m(2quWm9+Fv9Hu5Zg@VA&?c<H*6$(R~t+!EE4ZgdUt?2Ni1O}-#CM|NGYxxOC`uT>9slgmfL4vQAx_UafDdm)YW2+y+&mZ#2=tI}P5iMnT9vF`F~*j;ipyJue9?h*^$=oH*F*LrJh+e;c6?>AnWXExhuprCDkmvjARJhU1qo!K$)3atg);1=+GO9ul>`Nct=rT*PGlDw`!jimbwWSiz)&bMa~j60O?u&VC(`YpqT%<YCA;%Q-CPW<6kbSabfa>jRv@QBXDBeJ%$IDx9q$epIh!hvn<_{fkZxB}T436xFQa9xewn!OM$O4HTgZgg#Q$5iLu$Yra+dIIlQVe5;@9U`$0Kl7@lqCln)Ek>W^CwUdgKAGll2@w)=a*AO8i6L9~2;+2io;Y6~>Ke5WTk&*t<yuY2x%wbeQNsf;Qp@?Y86L%uH=o>BOUuBXQWvRfa9eaW4iZ*D=oST5EQJf9za69m$J$F5R&8yKl5P)zQi^uPQ_bKj2FM+;-cIwPT(ExTo0E*Zla%1K(u<4Pqckvb_nU++GW@Z+$(_fK?mze>x&QbncJy8abm1zXVY&kc@RDHvV(X1$9`#|dmrrSX9AqN#I|kN`jNm(Hmz{Vb@ECe0|6VZEPLqY=>hNm5Ju7+q?;-brOpzI!oQ?MZhSA;(ji99uM1XS;D`O(zD3Rr43`IT~>~IsuvG1nf9CzDfJiXFhTN6QAH0we|Y|ws}&jeCSNUK9>--or?o>h2q`69sZfLd(rii1Grjci+$y}NW1pK5)HMbL-k8-&N`*2OAVN<gdmi*hF&_Gjof9dLaG7+c`F4fh1dWLDf;>tPUVPZPk^`NAIQ7QLkAN-1b;2DAPc=ZAV_XDiVdPPIswg6-}p<}MOOliaz$r9+-(9z;6z1@WcE(blp=gW<#Mu$Y`uhL--mh>KOCRsM8(8kDfcXh(&%s${`}=|)PGk)CW?CVRTWAH{pJ7U(Ham(ubHyLXbJiHh~M*XxZ>r~8J*%10l4<YwQ&=mEuK8W=8Ek;5XJtz%0D6RMT89lXOQw>LjcHXq|j`rUg^H}Bm|KEL<ilTSCFwX7DWvU_ON0B3<FHy?lN-lGTq^S!4n7gFFZ56A4M{AmVDgqT2P$#53*(P!ctfW0GnHBE=)%y*c5!yt9o44-l&y%%tW+;Sx8zA)hIKEE`cI;vD`;<CI`WY!0n|3`?`nX!~Fg*G@=I3EQVf|luJcYUpP9%H7n%!q}0A3%1|p<C=6!Mu`I6}I6pOGn9UoQQTfzI8bUj!LrlEDNdBN6Snr{5N{={P_ombPH95In&z%+EB7MR5ewzfYrVrKkqFfO!?(IFkPth3{n)Te3{^!58-37o25`0#SkoaB9Q4qAqK&}@wVRb#*wKm2%%3mNK(&gPkae+kIH!xRD&@#HF9T&cytCitPg3rzL$~TIsg~GtNlK65T-ejMnlG0;n-u1)q%qsWdwLO@`=vp_MjjX7I{PI<i?btc;`TD$DDR}@Dr)gc}3)=^jios30>o|Y|0oh2kMH4?T&(tZEJYMs&zdwMah}f2RRT`84P6IWv<asj+)zuYynSX^7>1Q9976@*-<O_P_0XQW7{vG46#eCC=F6DswQdS@7{<@{@X|BnVBwi=SC<7c)J4ZIXhO2D4w%(eLJR%7ZpXy^8Ss9OY=b#d?2qUt1pp-S~ZSadKHgPKiCL9pwx+{oYt?i*11Zae8_eMn^}5Dy8z#O%BCCl;hcu6PCpK&@D*9e%fYiJcYk^1A>2PkWw!EQfH3Xm*<>TQJtp&wPP`^Wq;$G&YLKw)NXE=K@p`d-1Pku*MOkQ1MOpoxjdDCEFj5D+B=(aq8t%l3_Z&`U$^wrW)YCEUQ}J2eizD{sG&-JSd)WzE-b}=MH@&1WujJ&ScwY26qrpajCv<?fHKPHvC}N={|Bo$<P{qR^$b5ufhDkC?53?i@8}qx5@7(@4x&Pqdy=Tdz+n-sX&Uss7<R%PZE}mnW<x5q(a{y>BE45<$BA%X1xmXqB{IlD?oP759?!AXBHA|iWugqOuLR2FZTv3c8wzl56as8%f;K|c_lEF@%-uvX<FISOM!f)go=oR;U^rXapCK>*FME@S*Kf@vZJ7|4>>mMQWHz5uX{~5n}_?rJddcFM(`R2p#%Xi-ozQI*Wr}T$gRa4tns7853f5)wqABNxf8vc0J63@2xj6yw<FV9a>x;LJ7OpKy8JEv2M{GSMPhqh2eXul5q{x0HT7Yu=aLiTAk-Cem!m03~_?EZMn(L>~Z3WP+HQl>~Z9ddxBI3%TB1+Jt32J-K#!beg@p4v|*X`iBrQJU9rMlm<hVQ3fS46z>M4eim6peR{d%AzGDBtp+12QCf^2Um%bt^j!ThLP|+_cD!A)X<>FM{LtZ=q@wenjtVa%*uBjeG7+<e(ms)unyhe%K~GGk;ErIfHy&V?PGCKM*@Vh7i=pVkv}AYL<Ss_Vg_%neM|Ga1J0CeNYR-z-ODszY~wPIvHQ?Bggx##PzJ?6g5x=bJJEXQTQpa^e}hd4Kgtv+D-;cjy+=jQ6a}vgcCzVl25?unz-kkFo_+YVW8Xv&u|l>sD^Oy6>1%L3jROAtkKs3*-|ZIu>y-Q-|5_{hKmJ2=*+19H{=a{#%1W`LL^}j;2ly$;Q=mzlw3)+WW~XxO>0G&T(~vg-I_$8;>j>JgnV20obH`&?-*CrCp{G7!n*i9zE-^yAa$5&-tG_=9N9=VpqMXa6gjujI$Vr9Y%g5mkX16O_m8$W~v;@zLahj9umei%ta|nBF&_qE<xQJrb)5D!&7*Hj$sl#2W``CXDkBH35;hNq2mF+4wPzIs=!J&kY(n*fkxgvrk2pnvAk&t^wPIjuSelbWzg8LobBS;m$Pr_XM0N~OW;5au7Jq5HDr-MNlszxHJ(K3RJXdU1mbFt*CI=WzW<cA(Mz(3=6fT^2KW_^sk!M>H1*oBnn<S&W)7G~=x8wMZa&nE!E%Ds^Y2U0D_GmOAMvfEFdY<4s`9Q&av!PtI^Yk|=R7)XjjfaZt+%V(JGjnm20%Ay%iR8E{(ioGBS1*geCnW#+ZjD7;g!cRCS2|>z1B6c$1Xhy>Sk{xRSZ_qb2lrEzLLrb_9YXV7I7}+R@G#8|5#Fh}6<s1<qhcjILw#C=9e658j!5R(=oS)$U3T-{Vqy!695X7xpH`Z=MLHp*7TiZ2oK}*3xO?BG0I$B#&xTQr@&CMKt;?u8Gfu88v3l}Q`W2DD+gd*q^Ka15;>q|0eoxz`A1jyKm=Q`CItczx=MeGt{Z<T}ouwYJxwKA}$MfBPjqqa`UDpnQ8W|c=D_N)cORUZ6$d??`f(+S5?qevdI<$71%ikS|=9&B1SAm5V|L!A~@&^?Nrs(ZR?{PrR(adD@W4Bd1}(SWX3iDb|#vLo(cDp}HFKfmPb2Te!!SlCfuFdmNbqhdTT#{gfyvw45zCdD7H9d4;e%yPUc+zu+RV&1M4!fSn1#2H>Kr<9zo<oLXL?Ur|`j5rvh1C}Q*GVZT`ACBo_cbc6{zyFbM^vI1uixd%uT8)F-g3ptYYXpm6ttc>U_+dx+P%sHiF_0C<e^MH{9nN)cd*_SzeYDB+0qzuK?Et;-8}!tST@VM4Fy6p;JOnNi*=<%s>D$sksD`f3Hn+$iyHFX&@!*2H<knPv0O3y_(L=^6F2-=uY2>E<Y*!<1%$t-(go*}@hA(i2H3D`Mn;T6-g;x^9je3P^2A!4M7a`{{re#Sf;0+CkBJobm5ZpLd!C*EKZ+%E8$+xhba&Sh*Cz@8wM479_l(l7CXtLpCDD|1+D8Z(h=t5^q>TV^M!c&Jn6xg#i1vP`-)ItTyaEO6t*K(d=YSZk}xTrGm*bRCH%95iYg=bSctw$>PQP8V4r2G{+2;U02HtB^MtJ%eN96ZLkInK*0A`pC2-xX1Tmgu`mH;!Sef*y4Z|F@-=@CwB=5&pmy<nP_7dV8XNs*)eU|F?T$4&iD~LDiL!f(XYfQ@oXMO7$vytV{$B3#=B`#Mh{f?~+7lc&TutXQC{di7~?^%bFY*CIz+phH4Ep8!Wpjpdvqm5IHA)S3A%n-_h80Rxk1$FkRaeNMU7j1Bzxh7n`sCjph&tx0I6tY{O9VkSv>Kw4z6Xl~hr^xanUKgj0|U5-JIwD+a-PBD-qYO7a`3Vo@-}3+-5dA54rV(Z0};vycblkXJ^pmXW^Zk*jq9ZTdV-Dqw3=y4hlb$=?28viXC_<_xA{1{(*XioLzTXkF9|Mir?UOvMT{4n`GwdxO!ss2hwbQm>jNDMT>_kTn{|Vgfjdfp(BY5>K_X6l8FpvbAscQaH1C0q^UmyofQuL#S7BEY_u2K{t4Hv=JQ88F3J?_k-SIoKLgEGHkte0>JR~sw?kVKwm^KK<-h#&qu!GYt#f@ZITLdSBmegFc)&YSs(a2<17((MmV*$y=$VMFm;O}%J^3Wp~?XVF$oy}mW|-ZCL<%Gme<6O>U8N(Oh_C0yzD8cMbPaAYjmL!rBNb<qW})&GHDTt(FTAkg+)|sqs}u76CICczFc6{lu6+(yUcK}cs7Ntmb+X~Ym$x*$bU}>a8*;0kUbCabWm?wW<!^QM~^q}ZD_IH&~j;Sf){Zi4%;EM0!{+4V0#&=D@jMgS`cLkIGOQ9P;j-G>24fm<v3<u365*J?I;UZ5yn)>g6No5x&svV0hYvjn4|?}1PI74K_mVB5$lSKZs-yq%oyPkl^&2m7S3uq!MrdSJtQ`2n1ZK$5t^2Q$-+qB4ssZw*m0OmDQqTRy3s~RWA@aVNH3~auRSrX#wi;z#c0C;s2@5CN!Q$gIS!Pj1J0pa0q0I(P6qivrl@C6cl?Tdc-c`PNn%#;zOGmp!Ttt4yZ0U%Z}62bCs_khdz!#e5o8o*(4Vrn$ydlG_snzhADh{)c*%-2vFocMSzVEep{!0G+p0<Nw$({%>t&kw&8Ck|(sv&#>U(Kdw7FAjw!ew-G_uL`RJ$TnW;g8Kf(G^ZXzKBDK_a4`TUx_s)({yRkMu>l!c^*v=v}S%4beRvFti_{XX3EMR3&c>Y9Oo{yznIaji{cj@sm<Fja1ZlOjZT9sFHDtqV}U3$S~m8kF8;X&z8E)Yeq%i>wGo^d!={0j}l|t<c_{Fr%Vdn;4nuP5n7Efz;Ab$9_|cM>Sx0`vxuVMhls7*LcHGY;#un94n_My7-}};J8pO2fI6@_27qdc6bq;~&L6kwc)}hhVt%)5Q?XoA@0wGR2VbWRVkkBZ!Oi5fjw^T}bck_28e}J-fHL>7B-u$H!!lJtvg+EDqln?4MP`CDkvW_9`O5pIt0{OuLmw|{JHhFD;ks&w73S*O$Z*jVuoPRLSIUz?TnH6Kl{^*a1)U3~;H$Xt4@IqdSU>CrowLq}=t>N!v5$5={_*~nFYH!CA^F+eGS$4n9JLqSB=vEu>ZpnZ(DO_1zxYLwkDL+?O%s^!5_R$RVkxyedv$uP-btzr$lP+1u**i!YKCkDV#kRL$&vD%HAsg_S47qzS-cAy9k|v|a)h2<D-BJTAmI7*EQ@m=+uGT#Oix(d!C%7aZe=rMIK!W%<C5Zhqv?;*xmiX!5nKHrzh{cGq>A+8C-=~eDdn*A0f+T+ulv<pN#dd2F!1V0BlhB?6SpBkvzEv2ziRv95lHCTMATxqNUOlbgiD^!waftH&mFEo@zz)ig#r&Ub%C+Vyi!W{=VF}^eOBDa@m!VS6Z<{AP%Lb#_4ei{N7l=b(HL{2eHL>>$6w<djkpX(m7Hd9x#5X2W^Sno6M_z!81DVTr^7%AlD+U&(x#-Yh>kjCGwvw;X0a<bWGI|Bd5UmWhFLA`A}yUxX+)=P5u^<1WRjkSTaLhJtIwpTPoxLRZ@0u0SF|UR_c=|(<=Mnj($VOMBO9vMWXMqB#`VYJ9kvmm)=$QxTvTJ=af9~g5U^w-q!tTz>nIiQ5(Z0U>K1y9t@jMQC0)aDDa6tc86A@FQ0lDAxHp_^qp~QVw77u$$PFCtXOoP*+H#efQK~ATGa652Rj*q-*_%#W_1(3sUB8+&?0Vk4#rt@}#CIFNE_p-0AfkhQyh&v&qKCyWzX%dTZ#kPy0rwQdy@;%R$k3k!J{tbRXX(gKs7lTZ^mfC&VHW72GTo2!5;_RYp6X;)L>5B03VydTc)>=3@RixX>z1ZyPXqPd<#8AT{kBwH1*DJ5GJA}I(3%*yN<6=`0_+G3*#1PS3Z$xNKkTS_w@mklx(NRywf)i|HC!~dm({vFzjREr-!JFib!7YT@g$#SA*oshv%_&|NqF$I9!<L)l*>-Y{Yg6g!lnbtm84}q&xNju#?8(ePFQH-C(K*DrK4DA1O%>g8|SxTESkO`Sc^LvJhoP&;$S056bVcu;$(#oT7ndzf-xj62dKI^+G=lm#Ss*{7Tf$<SMQ?kVr$Bh>w*Icaq=@xE2kN&!6EUgn#<Q4bWNZaDw;&)+c<)Y`D~gWWpQnoU*>^S5MfH&P6`Cgye3oWZZK!fYz))(wb}-2Z|DL&v?=nGw3b<2f2(zLP|)9z5C^yir@ob{i8L7`nHd!@$T-xri*2ON9|I`_Yo|H5#Y$fb<9T%d4q`stQC-kNZn*N*T*x6X*-gvo&w*FqqE38gbJkn1B2AX8?Xg<E51*HD@uPyVyC_YD>t8{iKrW`XyH>KYC#xF0C=bFKe+yiLux4%M`&~T0gi8#h!*aj~!kx5#z?r%<{3k8sm5P_Y5(vxYme*rLbokzH<1Bs~VskCzyMbkYQT#WScdo$MIje((a(`ksi+AQ=(VVXNtj-n-w_4+PVbZ*18r`ahM&&562fJssgC+@j)`ZonXe;1;IPebIee6?;2qLcmNpO*`ZTTWhsf9+L+f(*w7165z9yFsexwe1!z7})s+uwU*eD%FCz8u?by5$dx<D{YhO#iyI-iFKDfjef|8Ub^@nWV`1{1r0^pJTrLd8Q1pQ(SRmyD7J`b}rz4*C{$~db)+;IvA~?%{P8YEzv9WAv0{`DyHz@mx6+r9?p7fChDjpveW%ZG27cGid>pzZ0tUVIBf86_bC`GrCw+34>V(8&7vDerW-uJw?Kbyf&ShC{X7=v^Gq8xn(KGNv8iv3TaJein~!I(W_)Lsm2XKU!V)Hj5PmguBSgGa@|=qM;&7SRWoQfYvAeF-uTODGHbd0DBaAEay<t9HmDt!Me!B=^n@4uG`72NEC3il(cjsFV9(__LIWweIa?8jGzMG+C74T=Lz{}n~O7kIdvJ!Wahw1lDIWx0-m6wT>695|Zvm~`E<%qkJ{=WCiQI0no^HUWXC1EHN7UaPBY{<FfsV#JiH2Ww{+A>u6&*;5K%x!Qh^4lhGGELvVOEB6cwksJ~m6gDNLR5dkUyBLwn3|+avse_Wy!xpOO~YKu<k`Gp<Z&U)8JkOPs4y(VU+MN~&h&37*`GP*+st8$>8yk>+iE4fqqP_RD>TgfFx^8k;4G{$)CX8fU*CqoT6^FtG*(H#tTDP9fij5r_lT7frGBF|eCB=DP~d(sp&7q}cL9p5MXA=SEve8Arrn^0a`-Ho_Yn@8Ad=8lBOQ_Rz9Jj$H=SfeR#Yp7!9gtvfK4>iNo|=pFG)0(!ZOVkdNEAXu+3-W>d}b!2JN|C*ABXwc6<d!TSGz25nBXg{;(UV$X8(07A^7Zs%n?`_jBYe%c+*UpX_{0-a?A@_&;m3$Da^NAFdcwd8(Pf!UGN-LnLiUZ5`54jb&0=yeQZk7CUfD3Q=p2&=n#{kUenea};o^Wcbug=nv}weY~fSvb}9SltpzsjUup#0fr|EEX95G@UN6=q>gVj$7XP`Pn0@HaA|KdAgj_)+Q&d>gP}I3nG!{6eaItJACySHJT2fBgvu7TK(QO7!B)>gX$%VTfCTI$h=uxm(VkLn=Txd7d4<RtLHaDG{eB@-I7Jm^Wf$**GDt^Bf?>*6s)epefehuC^%_tV6Q;=SC`_h|jA(#P%Fk&Pg{5R?*V8;34#<M`5XjBMszgwwN`Ym-F5dC5q`GadDE5Y#*bXZwIb0c|<hgx)rPH~(U6;PPxmwE{Znf#}n2WRX0!oxpSEs4!46NvFOM6fhP|6ik_Wo*$Wo+o+X~s$%d6zEwv@qOheJ6F1wi<PFlmX7o<YFoZ);nu!YtG5@T{9<%xjwvfeE$yaB0goj0Y<aK9bU{z7a%m+wDMw_^Z=AQr?zZnU#8Y6InxUsQ!;82Ixr(v%3ULOj3>zaB4fXC{iZYWyZIq&=+a?6J(UBOc*@y`Y0dHVxG0A5ZETpzO*0@=1KhI<F2mZ5&VsO@_M>Y4`K2Jt<1CJX-7Fidi2P)&KxI+JDw`GUltYQFv{GW;!~o}k+jfC{q!wRf1r<##4nn2H!8rWq1VS|yaATyzg2PVxu*?);?iF9!?s+g?zv@-BL38=x?66YriQuZjG7n=@5#2bXBOcYkioPK2je|`%YdQ2YX;WD-sfR+wU;|diFJ|RbX<9;z5D*2|+8s0jX(JN>e>-beZ?-(!DeX^%nh{I&WO{;TrQ=>UoF(H^W^>_8*{@SPj*8LBkTOrgAvVpA1T_?$1flu`eQT~7`o_c^$-9wKOL3`4LIEAytRh8P<S+QU<loee1bvhp<12W4Im`P8$(}3&bXY~XijAyxnDPXr=Sen3+#p}wCJfG2)~@^2zvD6=7D5}lb1IT*A-+oCd|KhKq1&o<(O^=H(b8u=9Bc8K7Yf+|9f~S)SG9vM;zPQV19i$x1)Gv!0}Y?QkG*~814;0NBD(SmG;I#NCIE}{d-c}Un%S@VRwe=~#qNr!jBkb{TNaWVb0C4++@dgucHAUkOWAfgk`ep%WtvlIIbVr!bm}wL*vy^5#yuTcjI!2;$xwD(uD!J|Of&+vhQr7&1DFJnsJ2xlvOX8EGFO__XK0jn0KVN9G6s!%Ty#2^CF{#;Q%&+LHmSg%uH?qXgG-R7<Akev+2t%EkkjcBLmBxwvMz93Q4^okMLsBMVokbj=exx>SS4ys?(X%gk@uYIdJnFsmdtm7h+mJI2yhq3Yh+j-(5;bqWMKu^yYukDlPb0{S*=RHF+7u~5IS0<xj;vw!Dm?(%rjBbw)pCJHXN>YuHLw@emO78873~oYsgCb`g*5x^TxIHn>EC|+m33;ZI{StR6_#k>y3r&OB`*m0fYV!nLy|dkqJR&d{Wl^p2$N+0`2<D+4@zZtE0JH(7g_HUDUTLxlDQ2-3`7<HXGgbZ4xK3a}9snt=m6Ns%ew==~}y_X6{l-R#!dP#aw2l>+UZtY7Yrr*mhM<+H9E__c4=J^(^vClVGTrB<M4?Au{EX2D>O@v<ZEfA7tAa;-Jd<`6WkOb!o%!k}cx|^J`157?G$e^dg@~LQOf>gqH}WG#FC&mqhy-HUeLZ+?OdFnG(KmgMrKuD6*!ZnJFGWL~#s-t0BYIpb_D=s&j>+-&SXfIVILWRf>9gLZu`gMB)t7gS3i%&3*z77OEOUQn{a&fFcuKWg9G#))@Wi-?!1n3+KCf*&HgYCQ29v?8mue5BpRKfp!Brbzd0(JpVw0;Q0sJuN|46Ro&~Eg`vqzz+YGCYqYpz!nPBGR~Xy_S#YwEh>~crIFJ3#=Y}Q|HWdg1Z8t(w1EDRXv_!AEmhD{&Up0lnQsP(ETm8HWU@hq1i(oHQ1p8hN`CbnB7UYl!Uw=Oxp$>EYVm82~8WH_+3IbAe8<HGWeJ_MNxV3#U46Z!M%3?U9J1_Q6b3l!oawE)Ly-;uM>q~Y<ZNHaozL#yDQMP%JQZ>V-$TeG7NQ$Mm@z!K(23yuz0<L#0U(<U1ib~jomD8DJY+te*O+R_M>yh_}=)$T+UUJ@TU;nbiZOaLon&d+NkOV#kBhfWh*As|a9?D&9Nu5DgoI+Q_iL;7y-i~zXy`X40L6Nv5el<l!f}*|`7X6|Li<T1;NimPQ%3pxU$S*9CHl@ab7}k_$kP%hZRy5gSN9ufw;*l-U(OVE2y%!H%jCe?sa>E@4alScR!QGs0iD-~l?AnMq=1o~iETx~cVO}ah)h|>A1$`HQ==TW|6mg7&3-^7Hol=M#wQnx^l4f4d7QdLi<ty1zm$7$!4SUBIFr(eiIUNsq?`x=MEenm>W1WIj!gsYkkEx8+&SI7+1c_dM&g3K_CY{$9#wscfk40|xf+m<)$Hz4hFN1WHp$1E2_SRbRqFj47u7dN4UO57WNyNlEZ}7fFFMF7LM5E7ezGI_K%z3SdNMYpR{qQ&dj<=_krqrT6Y_b=dXh6o#4#j3lBCSD=FrfvqST4{eHLn5NEHoc*qr3rnUIV&8;}-*-Zf)j-I_P}midMi$m0%CcgWy-L7=?TwRAgh40Jj31AT-sM-ZArTj^2HN7YhyW4G*y}s%&^ggkF4}IKjyu7@ViW_=d>M1H6;`C>t=xD~^HEw(PekFVdt^*Lvy}R&M#&oZF|w?>ot7Z1^^U+oM3nm0*6m@8S8J=<aowi%_Q(+V6N;@Rg%w6A$A_FQ!w98fPsEOfRkJie!KH-8gHB(DsNm4<9_?2(RNJAIbP~xG@XpJqMNwT`$m-r%-mH-+B6Q!)H0hEWrx;X&^Vgx}(VX>U5H2s~CPxfnfN9!7Po_Ntsn|`o(aV5$C2{#ZdY%%y*Q}7d#q`jAn=9Qw$b78donTX!=@C%jyk@3}u_zB_n}h$hP-)@hKS;$0L+i$kN<{@*t}|JbCa?z9xsp8W*-ER?+kXH5kaq_6d$W{<3HAcSAQf6dwU9?BEKYCPcqmOS{8qQX)X&3Y)ZO4VOg}rwY?l^n@A~l0%KIQk25g5!^S+3HHBwS6r^jRmR@c<7{V`C)p_3X<tvEVsd@`M(65{o1J(x{+=Gn)Z5Xu>u2of+V$l6+O@T{_3LYkbhLi+j3a`I$@QDpZrr?i?dl>OU2AvF))7?1j&82qytQt2bXnw~Y*;-iaK2d`R<nAobK_dOX0T>YXPVZboYwX0>sPPcyk?K~5ZI0!DfJA<Up44?h_sToZ;qCP2CG!~biLiavA%xQhNmp9uB{z5O+S|qz+64stZ*ANG1qS1xPI-bj&HAhJKd1pzglENYr3As{Rn>++D`Ciq1|w;(n~d?tgmA)#?9?nM+jf5{v{L5R>6>El19NKFqaWl>!6rB@7+tkpcxj*9iUumYb@_(C{fZ~HOn!$$Sz+wy=#kIhX-5Lhiw&vMAHPduXC4TlKj24uj=8cbYD_k$Y%`OWaa0mmAqA&6;_hD)A+%9gYPQz%kxW!680!-11eAr_QGqZ;(^6fGNNN%b9tYx@#@;G>&>-%%Eoy;`E`wLPfzA@EHWZ%%@-b%S#Qx%xwRU`CDxMn`1uyIeXVnb#av&zdWOZk@lF=gJE8>^(>taG7W2lr7t@S%!XZ|wB%$H+ODr-q)L@3ZLf$Vmsh*(ieGKlNB@>Q2hly5%SgcY^RiE_ChnIuh!|4W4#$BCSng+T$0ICt0Ju^fyJV_7B)%<X8Re5Qw9%PddMPLe)wv;8Pl_Q6smg+^!@?kAu;1d1q<Y~EFZ(L*XzTo~^)t=Lp<F#UrO~!C4CP7Z=CMf~@Vz_8!S9XEkHK5k4I#<Pd?hK3mLGmhpz4{Pae1xV9YpZ2H9a5t5U3*-@7E~C;K|ReVqU|o)1;Zs5c1f_T)xA)l+Gm`s&Rd?W_Sq(@^A;xSv*Gw-dRSY%i{d2sTTz4spgSCn<K6z-!7dMMOU#>u>>bQm`z&+Tc{g*`UVhHr3ZCI5`%^e;?-tW1lVULIPj9O;7;)}&xj95xJf7mC_yD7>y?8s5IUndHCf5P^ozF7qR?a%lM_<}JGuUUE>7y@creDso$(b?w<(D?o4EC93`sJ52)5UCh)_I1iU)oF~*k_t&1++HPi*m*ufQhlF*dz-Q;E$)w&SJ4M@cp$t)?5Ru4$m;g&u%}#S%)b`_YB{l@WvO5kBg6_YS+t*`}#P~#QD3cnKi4oWiL!sUya<LuhUxg?|hv$jkAWYJ5Pu0g+|%A*ip6@7-i=?qg;k=`an=Jk_JL~vG6oahzA;%I$SnlMlMJ-rYF<YdwBl`C@i@_74k-x8n6NWS;ZmPokv9B@9JCvibet865<J(AVIv>78g$FBf4<W#1b%R$L!)DEzQR)3{d+_lhk>ulhi)TBy}!)lI95`njt%fC}MfI7M`y&iy@YUslEJMb>78XwU?c%w}4km3Ll!Gya>_5`2k*dQqM1TI3KVcp~Ip8AAL#l>j)f{1o$1#ut(UiD8Mhjq#1Su4NC(2j%U~rGAs)2c|;872l)*1dsgv+7g$HQph2;V)}~3tO`L&?uIaD3MKpD{%*6^j!%K*9VbU6>3(E5FQ<Gk3sX^cMW|*ZD$`hU)j<ZQhNwU!Jn#wNHEq666_VQ^W7G^Pkcl4S*&{#ihslc@+4Lx}EJ{+JkXE7W^!6-YHjv?xH0;D+T22cjBLgNsMwG5LRfKOs9+g(5#7$*g8kiM4(zLzh}m+aIlub}F+5j>xOf`8*lR_>?cEL>?vbB!Td?*webqUC|Eu}EoI_Dyw$0FB*|nKE2tvv55MI#I9|1?`q@IRiC;C=E*V1A&=Q8%d1Fy#qssBDXZoh8VYbDL0Dv;o7tZ=j+Gl<87kEBK^O`wYAnjZ18z~Xl=k2-GW9i2@cuZvmY4Q(1%c|Wo`d-fDsH{CI`t8nc#%w6b)n8wcDZrm#I7sJ0A`5qkMpdx*UxPw*Wg11V;1=AS|>!5=JH~18UV5QI)d^tn^!heyE=b0jT1pl{!of0#^g`TV52{Vs)*B6pK8|-uBFyH-fh_ciKQMa`yC_fp7kZuYKF|XY%`RXa2N-T;%-ee1FSNNG<~{LJ!e&Ku-3DDhvYl&+$b79t=h>+8bh?jOipl2|f&tD7UN3&!vFfY8EB~5z<zRlOa0vOOJ6hM%Us#v_69Whg;5qM>g$=+)cF&qdUpV?pn*5^`*h)nLG#3*g#}bn#kTfUU2z;<WfyBB95|2zIz%Jvk6e^l43`YQy?2;QzU@d6y2}ModFMl@gxHj$`U}JDzV*cG9*V|bU52sT@}#=`vvg#=`<~~X<SV9a2s~k+Ba6$_`mgAH?H4`UyS#Zrz3aTW&f-gqoRk>H;e+zKsS_&I!g~Jqcz}SdfYO3*SzxW5)khWX$6uIw37`)_#D~=QGi1kO_8JRqQe`so{izxs&<Ozz_lx*PEB%u05guK8#fhGF*DU49IV(f*{aKK0Og?z`1b(MZ|yx<)gL&nzgj--qe%C-i_@Kc{Ml@Z+5bKRz8?b&+J2Y>Qw;kOF9p77IEo~YJaYJdOOC`y7jIeDR*lOWhic}iWi1GGM1?B^{3hf5s7UrEz(usAovrC?S^3gvz7()#BBIy;HcYY$my1kBJz{K5=(baeoaf_NxleAJvmKH2d_#G8Xim21%DFmDVQN;V$60#7rr;E=U&K{FiiZ0{_+zT%f3e-1qKFjjMMY&%j+`E+f8`g_=tZKpjd7eDuP4K+ug>Ama{n+|Bo_(F%&oJyKHn2JD;d@zBCvozlkU95>Bd)$1gUWqOwWgY%XgT;sgnaDH1}0q*bCrXm2+)%hDoj&h#L7_0`685>mCPC#<%ORy0Ms(?65dukpN$@DdAHf?%%F`dA&WlMxV+_9|I0?YACI!UxR3H@~Wz0>UC&e9vt!c-Ooxh5-tWa#h3wj>;YlP1{?(mrVwfN6jwxzXOXLfvn0BAwb-Clya{itl2)9-v7<s}@Ue_-B~^b~*G88-&B0hp{mLKMUpF01H((+TM9LoY@ul0VpnH|G!eLq-Amh{59JUK23`JH#Zy&G+bdE8GranimIqp*u?WBv}rG3E_ZD`G)6zDh~O!v*#uZ{lHNYzxD9?u~~eAKznYUEwp-#M&yX}9_r2^vBdL82sO7gKrDV?IE(NVe@lQvqVZ+qIsf3b3yFWvs1aeBgYV9Uh*F0LdJ98xG8}Vr-JOVr|fuzCRenoUeua>4}oTkkaAGD4R{-U|z+%hCq4XWE^Jv?@B%{1}>*VEjREwN!!Ei#Nv<yR{rTHA6FL%EZ;dlq&vfObl?HCGY8bp5}?uqbBcBh>x2EWm@*B<Ii0CQR@5i$nBoMI$WfW707xXJG?D;sb9fGl4*{GW@5%)BBvdMa#~YJWRpcOpXUvDf+4>BxAcZOlMT2xNMW)Qy%w=_8>U+0~-n2#rX4nHn;k5v=<NO46gb1O6VCiButF>#_muU?>mgls%esd}K(Disui`UwnrCNmYW{Yz@xd;)N;MKcxVZX>0Vx&+rR78l;GB9({^GnP#tWE)!SNP_DiWdQKSN@17z6QlJf;!?#Q{l7317ld~az{N%U|JUkoSIC9m@%XDVz5@|Tnng&2Ncp-KOczXWHJ?dG_Js!0#{H@LkbA$_OCuyJ&q^H^FP1zS(Z|GMzzssf<>^Q4IHa?>f=MGS6yK2)=)C*G$eqoX{Hi#6w_=2ZUk^@V=PHn8RT~;6p|N6V1+R7I-UK;c;fSUzn=oTmz`i7TfMaChT}P!Hza+YMMy)*<A-W0yY|#afj*$Fceaf2HFbUYe4Tq{*jaKJtdVbU$rx^Sg!o$<>rt?=zHL+}b}HP3t#DZ6gW3jKWM#OYl^o!fa%<>u;0-7+j4Cz1wYCla+lbr$(`ycv$pVTs43-7!_==7eC;?p52h|qyG}4~iNLy>9+IPl<cEgkmXdd}mpj}@?2kYXVod8pgmyvP^EWDaRJ{f(nY`3;nI`OUR9XMOzNBjB>JX^O`*5b~MHmnQ$z13dZ-i|!6VJiW`q7YMvhYNssJ#M$JUI&QV@vWP;t_p~6thZ5@6<=S!dE<g0=AaN4GRHU9Zr$WLzIm%HAimnUN)TUduXQd6Vve_QArRlVdFvKI+_`@Jrh@p^HF&ocuiaR`c6N|`p+7DF=5~DTX6FXMckSkyoaI~XYlO+|_3P^w1@Z+VM*CW4oj~rue9O&u>)I^@`NqwgSI-WzH^RwMFw>lNu3l{ufY+~Il>pvYyS_#MuHOIv&q@tse3lECM_A^y>sM)+0W}4bYn@v(jn~%KYNpZUaJa8@IP4EmK|b%)iY#cOAO)TW6CPaEXGQWU4~ycU42pxV*{4CB;8RBTwxdE~^tG_AQY$_@803>MI|1e^DGvDF=70mo*t7HqnH-j}ysDOsVwAtCYyY*W<6^-}34D(phrbrM1~@9Qz!Kf7N=t#DO?@Rr-f%{xeMC3~+pf=49Waa56fyKLW-7;Z<1LdU7_Ki#s_j7$`cPmZqxHqNMHJ;H^hF86E(cFp<}_dhMmDX~@Z3lYH{(RcU=EGgj4LY@gl0|;A?xloup@eLx}{)>1wH!5eu0w3Zk_x(CV6Me$U2e@QET%EX^DtJ#MpXT%z&#J1jC%9bW*_(S5UMTA9Ee1;PpX{n}gJ<VM)kvCrXVdQ;Xoq=_a-&0EL}MiqO+4#QqYH8Em3C!zpQta6K^keX@%h5VM2;@$>x*prcJC1RmU+l6DyJCUcC!t-`0DaXu^L$0@WRodP++*A{pM%UA(QtS+PZ<Dvp4^IVE~{{Q~?PyQglu$RV+ik~90hJcd(B$Sq;RcT~uR8d$g*7Z^JqFyOQpN$ogRjTpVMISZ9n($tL^KJr=vXrGFx12};RST?CiFJ(^hqgRLHKYt@d6z`_4|7x_JQlIvBTlPQ6MHju#%sjK{1|A79fI>?BrW$dvZ;)O`s{lw%WvpGy;5z5*1E&GO!jsnrLG_ux)OTJlseFl$e@^TasVvSYUtP)kL>foXSYB3|Ji%jE;o+rQ1q|#A@7L*64NsP9};H?N+^nyCgJOn<JTHafCkW*X%Rp}=*G-IGP7hSw)~cJo$JK06Wg(!<mCF1ljS6~CENeyghNT6{1@(iR8?11S2uu}p&kys<RQAdp1XGK+I!c2T)la9vvqg#^34}6-+7^R<Lc#`wZgA=H(!t+ukbijB1csTd)q*XTfVuyah+b?2@?#VkD2C_PPGrnS~eppm9XV$k9iyU8t*?^17aci{3X%AL^chuu_nQ+mYMiOJ*y<W!z&mxF5tCoWmz){L23YVlQ_RkpuD6bgX64)#+suT&1QB+gdigKP7Nm`Zc~E2AR6Eudd(Srb@BMBbXT8iHPfERh`1ZqotaOpi<Ug97vw;>5|8*MI^kIJc*7i-8@D7|MGvq%6j>QNaXcu?jfnhVJ{6*NY>1X9=_DIrJe7ErU13f`mrqqsp;wYml+wofZQr}XIXz<7h!alr*_qjy5Y@=rGOJ=cL!P$BalG|36k?W0lsKsq(DLne@pO%QGz^KUU!cMBgFGzmYEck*4T3)4Ap-^x$T>4H?FFYX;q8l4$?xd0<;zO>&u?7$82mp!fvk+!LRfQ}gvbHQNo2DtnvNXeqEq6fMA~vfSV&D}iy$&lW_ZR0+M=Y02j4E5PBuK8kew@Fp%v4*5$R7kBk7zBm+ks$<H7|8p3(S4d&UaEheel2pSo+ItUjH~CewPJ$iiifl`I{q<T+-23V(x14?5Y4m<e5!Zm5nHMNmGvscV^*l-3Vfn=N|YE}Qw*nty>b4D@F^4+OfH$ktJg&ss^D$1CutOB?~7yGU^81?Rrv8%#-sMlvn1oqWws5?q2T1fUEA?Mz-=3Vcc0dpyv)o5O_10!ypcQE|fM#b62WSRRyg;y;dtVOvPf7uepD*v_IkZ8O8mJ1;(8WG<I83t<BR{kq7o1Xwh?s$vw@J)9G?N_K%3XRz*>i~$rQQQmR#t~*$1D4u^Q&)J&utScOjNt<OJN*Gj)aTCW@_f|bP8fpdPxUlVP8?P&oY>PXfwpm;gyK>tF1%+crZNS7dHB!4+F1afv+q4y&rFnl0#;?i$07c+`GleM|-p=Lr+{Zyve^7=H{A73Gmhzl*vZ?*f%q#F=exo29I`ZFSjD9oQ5q2RBQuZgsb~e>MaCbqhNmIjfM@u?8)U@#3(Q3vRGz~0w=c&!DOF=qY6U>fob$Zye&{j(uG)LA{3MFYf&osD<VzF$z?vUZ#^vvEmCqvI1`_`shjus8d#b>LtsXVE26Pgc{RGB7YxmF0cncBfk?SeEAiWIHBt+m0L_B;RD#+|#HW-@R*8{I8xBg!>?L@G=a7f{1zvp@Q~?n=?5uA3BgYYrq6uY(#oWR!$gh)JDnjYPHqfC<TVK%Ui5S}xKBtFC?TD&4hdzDD^mu$hZtr-LDwRLbKuY!l8`-`nUry2X3|q~{zJx7P{N{SbJwHIi1)kTXnmS(1Gb+BS2yj6%jF>~08^sAuf1-3pR<{TFvVR~us0(KH(<A<yerX&IcIFRRHQfk5Pv#luyMI|Mf0b17PU@78EAb9B5L5cY@te#j}c4NX--XrTCK_v$aG39$~{Mb_U`j5rp+R8-8>PdE<r!sW1Z7EU!y1}>p)nGdgJR1i6aVlK=aItkxJtd6c6usD}8IycnR1bw;(<G~5-2O^{!{d9%L9EcuIRyX;pvbiLMLi@32=IGNsSMzv>2?N#R>1Ky|dCp|~AKtl9qEjRpqA9{}Y&=r~5KVE_UjYFR2EHzGDZ;TOH4*L6*he5uf?00tTvHpUzj^iM-CK9AUBC4So&(yM!r}E)TWeVzgEZiF+V{DE1)9J8sm)t=uDqn$9;b8KCKH*3o4@(78@E5D8pq^-WsNTdm_NS!N!9Sn!9jV;P{5~4tHKU6{tHmXX12H|M5K~!;%4>Qh3F+TOxfLuC;G>|qL~y}nx_(!Yu`I1)mE*3G)X}0<&hT*O}Gpp$5X+Wl(I5j&l}e}JTQN5r-PuR&+C`1E{#rVo<4%|F2&aRSi%In5Rj5h8D(I!38T$?FTD6iUsioMz=?WOoK3BKCaBlM``k&`b9Ac`hZ;>kK^VPDkmAF_o^IZ{d9}EwqPCaTt~zuOb9t3{3RkDh=F|(}>8d^0Iww=^SXR@fq=tM&!+N8_UW=s2ZYhVCS3}L(3XL?9SEj*4NK;kUJ!z(}61>0o?Hrdww^g9l%3#D#B|x&YU~?2;*-c*drf95gy;+IMOpIE4E*iH>%XUP?r^VtPQ{!p!n8jBZw#rUS-FQ<pd8ejN(@Zp7W-r1KF`vC1`_QdkWI=mB=N;Yh`~BXjjS;maw>WY*-i;}6jGa)iF_zY*<{2yv>b9Lt>pUAAGm<+*N=SSWSNC(Zt7g2ZVpE<<p}gD7e%4CG(x&-XE0GAB`a`WyY-?seYemvfQ~jzHDQueRSIwbVX}Ug0YSWZI^a46iPkWaKgUt4c#G*tPmCKacxHO6H!qqL)5UU~_qA2>fJjb~3b`EzfJ)v7s2VIzEb4zt$8J3CNv9-3%k>oVJv(-DBV)^^QWQ3-IF~&%7Zy8LjjW+dU20V+W>;pELkrF7=MFay3OE8nqJ*lxo-b*n}1$#%rf;{_<q6x%-1bcDRGwnErY=7(CbQU0UEGiH<`^t>b=@TKkGQg%}kF;PgFvd?d7idgf#AeRtj+=AL3bkc+A)UD5u(~mrEx~DQWjA-*HTgzd-E9}*t78qhOCA^RcW$m}@pv`IzRd%mnR`=p@#FaoVwKu1fR1ZJlul3Q8LUYX7Fu-meV$QvAGGwGR8@0UHfi@vGXDO}uV?}#R>T!C2W>2Jz*-f@faeU)8kz`W^O|jCA>^CnLGCWhFt?^SHck9f6PhpfIyd!dMW*fY`F%BMmrfUKb1U3;bW0Rrr8`8o4)r8Xm_!;O<CMvEH&oWg>Xq9R{_<$1+rV#5iD+o1@S>bJ%YiQ)XD^8@vmV*@**79UalbS_+1$u@?T<#vl>W-qPekeNb@E5J9w#7}Uzxx^S4Zgt7J%Q}xO)|U*lqY|Y#|$J0nzAA2gvdDgAs=Gfm4ot#?!RO%pYAn_5>FrA#`Q4aqXtFw89h&bxpx^Of&kH3`uvs*D{0dls6|m*){J1iX3!{-q-9G2(Q~5;Hbu0>?^q7gl}A;ygOsew<qByJD}YVVT|UOw@c1OAEESR=}|Bv;J$isH%TXms%77hNn>j(=m~JUr<wToVM@CU=!Rw+?<}xG;gKdS*!STp7|5&9s;|kS!e%~mpcoHe`?dsa$PbjgXVA(?H~06AU}=I>NRaR*zCtBxGWt+>keY8kP8RuB%%slDcDg5<ne3M=rm8Bi%|uB1tAO)SJPS^HFNULV5&#b-VFqtoS8m<Bw((-?+O0dS7j9jl&?-7Oh@7>qyma--pWME+adXosxD&@agK)LbLC*-Dh<;e3t~j4tl~f3a>PW45K>a&fjEF1K)cXUEBIz*zCC!cd{fJ_L4e{H=n*wYG6!GdQvYiaPJ_gB_r6%DZ43NUEcy~UbTE9V-iyjWThryd<tfO%l14;FhuHR3le%PC?{@Kf&>Eu2@JzWL*kVsdxWR)fqpffH{${+XonvD*=Y;}A6%7G!E<H0n9O$P%~cUN5B>rRJA0HZyO5r<3EnbL(;ANtW^{As6XrS7kJXX%vj!wGcdt@1<i$|SA8nN_V;(ZWw2UNZhT=~tcwNUeAc?+k*!io!X+f6nVboASLrc6ol0sAP*s_fAQz%`m3O81T@*CV2I>aCx^Wcr_D#-#`dIuS>Xh3eRi<5^-M&L&IQ(ii&8|jVI{w=$pQXB79!={vg~72hD~p5Gy?RI5&rK+z#yzbpPwjyeS!G1fE)CK(h#UuQ?!Cz%2LJE(m8_kA*xCWRuXl^1?Nxh!+d3H{q-LFM>EX9ghK{QNA0*`|u0!#|-I$#x)AFD!70MbzEdzEE&SfDAdUpQhk1mCpDO%b;Ce#Uu=aU0L_aByvB69##1Chh6i<aM|#@ZY78*waAUes-}Y5_E^N&gfNNC{r3Nq<EC)ZQ_|qIsl7S6pu1KbKnQOuyWL*q5sa8ofW$i6<M6gw$n^wi>T;-}{JD?aBLQyBPK5Wxg1u^@Kz1z%kcxcj|GCG3OemKTXpwLvr>U&CAeT$wclbhLE7wbWL(WnV~X&4+dJ7!pQV?(8VWzEd>*F94$#};wYv~JB`qlsp3>i8zeo{<HHEu6hzszTOXddp^1)N6oNiFQ-8na8l+B-kG@o&~v;bnO`8<@d6U%UAh8HUW*b#xh%b7v`+v%B{EWiT~jfaG<Z<d{DthjHUA$Lg+!wQ_ldN2Y=eAw)N;rVi>>zCQ&bxtunD|1DWRW@DT+An$du<rt$zn*mUCQsF(1>*rpy$7<z>Ef-Fh00*KLr$%%gV<H>zV8af!Fb02r87)9X8*$u*1Jabycr%qclrn`A-^D5xVE=LujtAFTi5(Ohv4Wn8o@hDscT^8|r0!r)yb@}7VpCpHF90dnQl0&2U$uj^!fD!r74~NF9OsooMB!ij+amLbSop6sB(@~G%LQ$v5dSlv2Fsk4v1$+<+IP&z_jrxMRlVgA80n}qZ>SFgsxr3m_XhjLxFiO)rK@aFayG!*W-DvW2Gor)y)@7`7+o)xs1_IF|9IwOa5F>5_7sRNdmFkF6!9h_e<92Ue5U~Qr4%2)P1wpPZ2TDakt2rVmh#8|CaQZ@rnYGMnCTbkn=;pmsEHKiAo(32Ue8<E;i^0t$Yo3rRfr^#NouW@IPY3qMF*9VRh8t`+g1y-k-RO}G&3Y)(7FJ|=x1eUQx8tr^;XZ+L6KxRVm!Y(TCu%!MNxjNd4aU&ebkwKK<eHtI#>-;FR^E7N4-ghn)#hw7E(%CSWu*ZOfPoYC!KIcOwt9BL$|>N?*8FFlzI3s1;o`;m+PU+MbC;fe#z-oYpNVVy^Ke^3Sc%^SLz>gM;DOBEi7L_^plt3Oa1^S^qm5H>TPxIjjeDss8WXdRA$0j-(3r(*`jZh%1~=;<>`q4(3D9yl7477C+c%GVX1DVC9>nAWPMbsiJ0~}Qe7Aq3E9gkch^JVAfS&-*7L7AupPsH<-?+`X1Wg6QD~z93o@4pja_i64sA#|4k?2wFq{Z6chyq$Tda1S&H#Z4cexw+qMQDDrM8xv=k#V`jBihn<oKl2?m6x5NtTLxI(MFQ7`5N+);`AiKfeAspc7?3zM7(JpCn~C|x|_X_fh7;YAzR35DM};K<M$Y<ZT#EHQUx{LE{B`S+(n;vCne<?G7ZSG%p{j$XC9J42oOT-#RFs3Z?_OuG6gfofnNp&%+LS|vny%M(-XNkTv<hNY>rE=@cMW`svG+0>&Vm4qaNwQ*GEpvhnnX~ZUR-y7|C3twpdD|F<uT_Idq3WZOlZJ76HnOE9KV^M@1&D4$xN6Jj1edl;-(r#cOyBW^tdSv9SEkP^x)nt&ZJIhe3CjYgCW6u$GSt{_iDYIgKDyxwXyu_r`-Mi(ztc<UY~nCd1F^9Hmq}9(&ERmKI%byT%b$dx1ru?C4>Dk!J1nuyso2XXgrC&R`}pez3y|f<Mb~ki`8Ic<>radSrJ;dO<426;W{61L2Qv;YczF2mKY*M($P{rcmS-t&?BP-zx_>>EtE#O*Yw*Y6Pu{S;N?;T2HiE`TlC2@B7<Uui4Tcp@EY;sPP(LnPb{2zw{xE0{UV{ba1T0n34j1xX)q&#gn7>$x-~|D1LGj|6U!%8u5K{5-)lZFL(eidhXf>;vi1upM>?=oDC+d60kAaRRz&)uD~Haf#t-TUg}s`TdmNkmqf#85KN+UW*l2qROq?GWtLChU4(hri>mwN6*}`?ObK=|hbO#>Cz=UrTjwU*q(yIQSlt$W&}g|$H|9y^OhPlv88*xLijE4VO~9EqYio`&jn7HUTAw0M+JJ+5)>vspCX-xv*2ibi9w%4T(@wUnw#E9kxBZh<`#e@%jcY2Jvdx?ftqPkwiCUF*v2~Yw+$Os-2YPxE*pu$kIZOMjLh`iCA!X61xo4EMyXTfS&dxbyO_6tzhIKfvm)Px@RUM18SY^#xD2-dp(;-_ycj|n|ULMW1*Yk!yTsFrM4f<R6w~=kWFB2?naoTueaGWE}F=|XDO#VwdA@B_8ga%Xd?pJ*^Jlm9+64+u#WJ1jm)$_JCPG$C0wRJ0_kq@w%4zir8MpQN)2D7B=4s*O}X)Ld*9g|5VN$pBwKI1cABmmvXGA3%0KtU8JBf(-WrJ^c5Rs=39oxO9!+v1OzV9-SKDVX(MqgToKPAK2hrfAymn6>4k%m_SbniIE+a<1_f`yZLp$QxZvqTbw6S+YF3YBFuNstz#VM5sBkfUnq)UtP7qTlLPIq3h3lGjS5JNYKo+orAt8YcGI8-AcaMnuSHxeBiR32n`ooooT-xVj4Qt4dtz@$+RG31LWwT>W2s80FxzSs+4-QTJr$q6Kr&pW*)5CPKrj*-#DgQZfK?ja#O5^U(*gz+enB#R@Oa=8c1#{^t;gL$LSkPT#+zq2?m3BpL~<GF^N+ectbX2_T*GOInV@wf(CzRSSurZWa2g&aKb=2C{R4G?INW6l|B#*+KCFapcjnA#k5HOCnolCgmz^?m+SyXc$F>mbQ!el5LBWm6m4{XtSIYJIO{Nx0A|tb2i>p}$M=y{44}VNXfa;x4C2meFT50n=X(8%-Tt{tQ0~%&M$ox%;hFR2pAP!<eo#Mm@xr-_jbK%fa7~Kx$FtMd>rbCwxn93?zNVQcE;o+729G4HH=c$*8=c_N`AZl3ouJo$ItaTL!%lGFV)#t&nP(ckdU)=Fti&Q*@whInTyNCtE?m*Gof~^ygEn!ATlH<bti7T0yVhqRUbL`gxv!iCOxO=*iGh;)!ql-VeEWp49;v7fI7_5JKC;20gRnOn1;eN-Ox>unB=5qj(94gKmgu3vK8nWFo7U{G0Pjcq#+%U@iKYk&74tn18;jI3`6L2?b>_o!I!a!dhT*GWWzA}ekI&rVt<){pW4d=aDLK15HMc5Aw>|#XSzIT&uT0XGjVQ`ZpkjPEb7sxIP%Q<C^#)zxaB4`4^F6uWS*ODMK7hOrrzt?W=AEmN{p{S8gHuY{jRyl?!|Y1#EH<h-osKKnGS#vbsM=+PIYlSxPT~J~i)=XScQEfXa1)b+52})be<`pu7&29VFADehT+G2)><+6H^Q06W7UE#k-Ps%xM6K{(6{m37G1;o&f3_#8kvEhOD+duFOE;YiV#P>{NtVzgTQ|L}Gb)V94)AFFSEr?#DBU!=@<s0XN&^G|r0HW!R_4IMsF>5prCrlI(;m(WuDXa}(XR(gcUmJYToNwX;iA;i!lkSsCFmUMt}&B|7P@LYo(PrsI4v$f%@vMpZkMvA7xT|DB|YVJXT?u3=RBVs7tLV`?tJ0IYTIA|^o8~ixd(9mXME_eQ$>~$uV15`X*?x+{b4waCo?*j6;Hqh&EZ5`d(LJ4ZJ(D?S)hD+3&;g-YO52zvUtKWA#YA0bUhyJnED{So<5}|t8%X2B`!;hptD7!Hwj-6w?-6RK<1BDEJ_vx0g)jW-NhJ`tkuZ7qZ=qy5JJHyWAj_^Zfm{ft!Lxib_Z*agDL98EYx)F%q#)3eGca_t(Y^c6cJ3LvN@i!t}e_m6q8$>B&zE^2Q_3#<HmH5Mk{zbhZlP40+2}cy?alo2E2`gLZy5vo_aS!+{RDAkqEoV7Fdo<?NwKb&dwv-AEcH`UCU;r@m1!TN+0c7_)>IX^(lFF)vk%uX6-Zf82*Rd-cceT_3xd^#xoRwk9rZxuBc<^%(A;^ud)LKpsk9x*S*)w0X*RAAXTHi?hNk&eTtf@<b!H?93&2NCc=gpa5-}ZQxE3?&QIbodp3x5%GOAHavak4$s~sl)9cYF3|RH6X$0pfwf2tP!Yjva0X3ROXk)oU(7IGN)MYMf4T>L-JGqKayqv{_Ru`lFVI4zq(k=^3-JQG3?YydPBU4wbb6^hdz}t+5A^al(7rOK``ehxRPax2s5U&G>0`yg^1Lc)zIGIWNrKZ066{E7g!1qJD#{D@pNCdL#>=o209-RKjvZZ6%=8mqJ+tej9IcJqUoP0+XF!aZh>r9_mYp*4^7IS7gx1{ZrQA^tA$OPQOARSHz6&?OHy`ijo6#c7Jc}mDAiROW#cOh^RPZ7)h%bc$OgEp1w5zm>7t&p=!cH`-wCqnJf6~=4Jj^#nk>tN{Br1+$9%=*Cs@xM&kRg@ImS-1$c^@YH>C`fZ)rE$)KX}uhb1g`v49yq~kUxQjGY&@Rza=~N_$#XjHqS0r@ji#~JAD?R!?+{bBC#=^hdLhz%HAx3Xp_F(}k>~a~)Z)a(#`399b<qw5A7OPkZtDC;&7-><RE4pyA)a!b2-^f}+nY`6<f`i4+rr9sxvgzdv?Ffrw$^|9s2kfH$)a1^nDpG8ZGu~Gz#`>-rsb$q7ivK2z>7|iBDIUzqQS&fv~N|XFsK!%83^l?4X<G+9Yrx36-#1tTM1f)Jc`Go?bIR3nDSzg9zeF;4JVlIoFu}w)kGVEa%E<vy4f&R1{|MBS!g;+<7sy{>~UOx7?T%F!dIrz1jJ3ex!-|S6t=NZs$gKroL6|^N2xaoQOqX0RtlB^4RtY8BVD^rJA<g}UB11MWp}ivN#QfYU?9@d@a4Tn`UBE~bpvr5%NJjnkHTms<~J;KOG#SYV44I2$<{6-S`P-nFvw(Ue!Px<t<_#Y#y-OdomJwu^7L?4VVk6H-g@EIx_4L6f=M`x_bAX%H}q(1o`zKM#Jdw;8R8k#nGVN%8Qt*fzR_kDWa?S?rlQs0x{g-{VL#>T6~qk&D4sPnK!Rw1x0xKp6EPvgYs3kKZ0^Kq8V^@EOdeOq__{TAIN1*{&Q33sezr8yY#7fqag2=nXf1?pv(&1zq6QT0QDzo+*NafyiT2vQcPA7-*IgIVF9h+*<&En>XO+6jOsrRBB&XFh*je4f&eu#3Js<7|6A_A(P}PK4O+@E;1rLAZOyQWgFMn6UgKjv^V&)T0tCG#8SFZ}YP;}k!_9rp4M0`g@E<n|{B~m>y7(fLhrj1)zlsT&=TWQS?hGs^p(d`V5xi7MSV(+^MT(0<ah`}PY*l~p4n%jn5ipGX)oEt%&LjC?q!KVe4%FY0B^60UAXp74zzt&adX$>&mfv#~s@RVZjK!OP#jJlYLAnaopvHj3{8Fh0=xUj?fVK`>z86a8_I?vcWKM~+hLV%ZN7;qkVw#%Xvy9W>f3H>Of5PKoU1p;iak2M2f2Eh!Kl3S(zR@zYID7SqKG0j|#L;=7Ik^KuA{*#XbrS9&UsArmraWL#*#*F9_S{1D8Dgx185RI0{hodcQ=?V!!tA$K}+}Tq3D^=)H>f(E=H9ZU<LuzB>-M)Hl6Hi7sYk)7@r;{(0ixzU&HRAGw^vEZqY#2t`i3aGy0OwW$RI*?nXH>&*QAJDR1mvUvM{B~aF<iRp%=cbG6sh63Mxk+?dRh&12(+eqU;1SU9+w4QCLW9rWWN>e!8&5}tsP;v70CGaVb5eHn5QSsC>g~2y?B3=P5TcxDmgiJCNc8Z<oE&n3-C`p##21o6)dkKEN>zxjZSc;AnIF&gd!d@O~67udg%<;&ZVUsE#T)+*zk8?I)Z5$<M>+P2)BwOqzu``1A`l8gW=lvMpfS#Px*7sIQ+Etl4Jn|L!|<PWj?~d<N(2VHJ4_HBI)b~i84eNH(F@Bj*)s>q9R9hWw@MLWC~haeo;gAQ1bR9MGf;2E)C>lYe&mC(B}}Cu6>h);V_YiUATQ}%zQHzlN>S-QJ4xf3nWDXeWmmUbiA{E3TkQYsS}Mj@Xy59wMeuvpd{fx?M^LO)Tmgkp+oe++*}27>2zGt8?5Hc+6#zvd9_lCov&J`?kI}xTPc}gt#Aspl2O+3+DeTxx4$y?V5cYHlclMjI=L(-QkG|xpL(>*fh1%KLd^w)r4mEIlh8{ort?}<LnSPI^FmCy+SqgpI<l(kZf_i9rVvXgx^kK=8O~M)Y}d8UoNA>P6EM(SGMi_KLQM2`6t$RBA;x+zjsxn_fu+L2KZ{j+_l4V+QQdVUjg%pJTXT@YG>4>IN-Q^4?1L1_KP30kuz%+69Vww~3<3Z4gqlg5sI2`7{T<(HbNr0V7-H1I5T&H&;?SBHpAV#iDF%9Lvx<(|iYN{{mo%BfrIH3QmfVo+4(}+(MrECUd)d9#_Ih#1H+0f}r6|sWj|)T|h<qZ)A`U1{UEBI5c#4vI-Fpg$n*iq=^_8csXv+dHaRS7B2<0{Ho=t6sco_2;i|$?UK}vx|Ky;TLJ91G)T5YLJ%|dO<ykK72vA)fFr@~+dEw|cioC!QiPYqZES9z+$0!MeO5Q^Y}lRCE#o5oYkQQX{*&GM7kxeuGUH??xE>51LO6SYsV+J}xu5pNl-n^!a~4^Rz|ZpjXM3PLu?92|tENp}H_D$p<>iz3+5C?2gC2GbPFiavEgEXR1?G7(G77(@95!FWbps(+&-nTE+~z23O+^wYk1ikPx<xEFOpUkb|iPIaff0Hh8)uNSv@C_}UYCbA4=N~)ACzE#Zt-zO+*0(o#Ck;fIKXkxa84y3rNYH``e7btNQ@bTAHj8L2ON!TwD*=PjPV9-Mr4f8#W;6y>K%cvKLa7LKSW@m(DZCwW;m*#2NiB()%47n4b<qix~`q8lm9j#A)3TMd9BpC0Cut{(@Tz2*j68#<p->_PtLx+1o4>lc7G>#7sN~J!_KHw18n&~;O&e}A`U4YA=VeymveUbYXI%lMiXPR^rYZZ$HL8lD({@EpzlE}wzlwgEGy1Vh>^^M!D%bS~5Z*Fefy4kvN{qo(rSMO5dG%1P*PG#MeEo!QStu$z4rLD`otWE-aW>(DPI-$^m<NGe%xPE)C!xEHP#!KmB&E2z1VXZ#0{G29i%d~ulLfdx5ez$Je6X$K^YbxvmI;{=EUW8t26oeq(Si{v=u42t$l3y)zR4kGHN<CrBSb|M8<zn{CS;K(tl63(l#WlpPGQK-?9#_#pFr1K@{V2Y0M+zjeoDT*5A?+5FE+PgJAzKbXW?P_(2=0*^4XI%%Qo6)gdFPNbf)B?kL`YVv1#feW4hLf#uG+wJ`)(YoZ>4D2*csuqQ8ymX-WNScQMu!*J#;6v(l8lzNIGY%&s_K{=Z(jPyr0jm@$4vDHF+nmJ4LqrQcdyV1F%eU^PRhmaogg8NSc_1jQ*_Ni-5=L%KTh-jR!NE?BKoN4kt|$UeozFH<leorHw0RMe-9iPNXqxb0%fy%s#Oj;;p7f38_-PkLHj}dPvDcq+=jOCYsu1>J`2%@x8lYNcsv<q3umaK){^&ghh-&*(6@TQjWD=m<Ev%8#aejn2fkLCDztKL`w;?Mo$(b=`~xSZPL9;8FH7aN|sWSEU6+XJE0W#&X$3%1jq|QxUuA+31oTk8jDRr#+q(=X3Mqw-NszqtZeHnG-jA`yP0ND+<CBD2~I24b3t@ZV8lBK##Tv1(4Nd3QDUcxO&us3*8pQOj_Jw*7&45E!(m<G{gG4}a%w*Q4U(n^SR}d?JaWOs0v~!U9#7NYJ7k)yAPGf4HfXya9guFC^J}WP3E4TJG%zT-x3c8D>=BPLY}PH{S1TD0B8mYAJdvG3T571xcmihd4p~O$vfL%?&~8U<ha7cLX{AmQ5k2oEc$Za??0QQ<MbE{W!NM$q@=y*=#3B->Con|ti9?f<)ZOM}aC80UDo24SMQlV_lDc7qTdNfW!pogFGAu}?C~3KjpTH59N|{jifDFqDgBNyGC4pb>x40*}y3G11lLQIV`NG&jBF~$*>W#*n9Ernxm~lSbz|})S<Um@KG)N4HdtaoNqKj2~#s|S(1Y}+%W)dA4h9cGF>fS#_-MwSfm3ojlwO<V<abi!j69vg)749!vVQ<+AY94l#^0fv5XxUW;S^8+82C;ZfMQbNvPdFX1al)003s-EERGb5mGBUj^tYAetp^wqRmxNVjco2<fU0c%ps-!IwBl1lkNOBF#fri167y+y)_gF}jMyZD3g9zczoIPf9u2%CgwQzh|BQz)<^YykWcH5=3)r@HFR$QObiCv2)Mxy#|P)hM6f;mnQF`%x>crVGl>wp7vT<DPFNC?scSSi)y@jPj<!EmkQh$@<YMJelQS>)AfevOR;c!ftvmP8W2<9d|cG!<j(qGU@~#N@`TwN-=_S}|=$3EV?$wYA%KH!t7W#02qoH}7oRd=Z5YgCUZ3L1+tM_{nXxEyljR?R!^{v1RuH)WtF9+>5)@%(g(f>ySHC)D_V^6D{}bAnHtl$t*KUU;!G1a_pT4Q%R=XU9>^cd3`9A0KpmU)JTMSHOuUc+7&&l)Oa$b3lTk(NtkwhP2ACEE<=y1l=fTdu88%P+ZC0Mo-aB5c(St!Gf!60;Cc|QLebrHI5=%ptYY?Cf|Ha&D5rLcQwv}gJJUfhQQdMKl4&M*<tjBs@_QcC2(=;7%86!6Dtyrf-ZaUg++(AZE;x!OntAi>U=qL@g%d+&6w3RG3{B1)XnVW5?y=zv7nn|XB#a1j-=N$WRl9|Gb67DZh=MW_3S{Sf^pjJzG>8L+K^pWJcO#lHXXBJz)YuR$-L*MjW6Tnn@g9wXu>YBb@ILN?Rwh%nw`HhWfUp{l2stWz`Vp?0lqojh=G9Mbwl;3w{@CW-lC7A_&!_@kNiARIp|xl-Bu8q64b<9kdK?iJRVGBr{6}$O6afygbLIn*u=-CL4n?F4CcU3@MHGJ4XrzqhUs2D7ev~@$!N|b~OCpsqfK7?E+LxzkyotlYQv)dwD(Q66q(1r-kuZcsGGpxG|J)T?m}z*>YC#oJDv<6jaSU?n6O`?1A_>jL3&|gDSA2hU2ZQkgfKJo<6)V72fvR@d1=O{oxy8JJEi0EAAj!yvky<1(Dj1m#!mPH%=*v`D<SmDTnp`e$_1jyRD|)-Vo(IK>S7!>(sFF&+oH>Jg6z3WC&YY>`YONL2k~K=99vHf78}{n!3Vy8?gt<G`9x7b|Z9ir4Y-NI}(jBG;Gn7$z0l~9{5b5Q2LRu0hH7g1|h%2?-E(NRVK9A4xla0~~sqS+SddwPnrH^;!UoO?oyV9o}e@dofh?tEr2ctuK8v4L<te<llBFhorVS2pL2Y3ulQzV00sth2Ix`~lD9t4rdcu{cReSG6C-k~mi<ms_;m&#3J<W;cZ(B_Z8+t`h~ojTg#hR2!Fj{6A99YlDbk>qxJ5(Cx1IjK_(O*PDKC1=!=E%~9|l11^7kABxF=eKmGh7UERc)}Gw;fkMd#ZS26k1|*Mglk!vYtiy@9{+IfTG>0Fut!hWqmMd!w2<w0Fi1`FZ~5Z#AjNGdtU6dfD&7^?f{UVsaoVNQJR1c=5dMYjhbaw6LEl%2s>$Dpk%keeO_xEAJ#sb6#Epu0(Ua6X>y{~;6|;abbeRtq82&sS<KsC7?;<Dz8RIgDuLIO7TMlz+Vbe==poK@Ck!&Id+3I#~EQO{>IS1+^+w~eL1Ku}}=7Ks!nYD|mX&Ze`X)+)F*a!-6gtD5QNYu;*x{wjEnQdVijv*JKN%u!^+sDe;4Q4f+`k*sDqC{H>RBhrEkRj;q#u!=($XPEE;VH>?KafmW*vpbC^KFWw<4xhSJx#*?bb!~a*#3SnO8HcvV>=#r@gD5_-FTnlVUFM_HAW>kb0q7re(SRr;vI703IhW?RdzW;9&B_bpf_3;Gi4CR_orjLJ)1-a3))$qD3AO>h$=@HSB@;QoQ&9VbY||uZzO&!#_s)M?pzUq%BQp}H+AwHRiPEWE0$#boy)R1pM6NTamf{*$GYb&Nu9;Aky!5+7)+4`e6@h_Z5mI|v+-EgA;uBX3Y(-ru#4KntA*2|mbHZ5oU3qPu9s0?`=VDaF~T<vVXYh@wH&M9N=G^e=ZN6uWI|llx;6-`7iV3T{#IXQ>>ZkT7!JX%L9mZQ=3Ky`p4=_A&{Mi7;J*PRCG*z`O3w|vF+UDg;pVL^kPx6KWSk8SV^)<9U^2~xr&sEl)--39%dxwt=ZJIbSuX%@&CcrE#ZD@vyW~haBp>HXXYbvCW9W#e?9p!D!yUe5IZl}!*Q_a1M!G$n8akY0EkVIXrUAxk)O)o&t}J(~+<-ESF~^p7>~|!p)CBqWD^ukhA$d99T_by)Gyi6RW31F#G%dwPLGeC@G;U>nVYV~vl76&CCN{(KTyLl_p%#YW%S9<{WB_Y9!)Z0$Q3|qA{Sk;4h*y)eg(i(44*>vMX)Iiyq%xq)noSP%k3cRQ1l^1eQ0-O3{-^XO*4Fq$l<o>U^_t<9)y$#WNoB|^Rzn<Fe^HuN^Y=~dp)J`Q4agnpG_i!yCoK5~&yw3&Rb4jk`J_9iJ|zRuV&EW-H<z_@?^M+lJ6R^Z){?u*fM)rdeZ3e(jKRD~T058$wz`1BbaqH&xm26#SIyY4O*>Dj5h@7)9Mahb1d#MeDNrO~`>RZprx=diO!hGy4Wj#@86O|D@OfkC(;nvv&gZ6DC=nR({uFbt_?*ZGY_!9_quy#8cm9@@rz_n_TPi$3GU>{P?Q*qQ(?k2|aWDy6=X)*lW5wc+t$25?aJ`ZRvV<4$x>BZP{ouUWiwH>5^|Ds1LTPHhOM*A$xJr$QHe`Xa3QpFik}H|pCYO_BO4`eNr;;@4-k;q&<z2mXm-U6hvAae+N>wI$UhHEmSt#ZkXK9Raxm47rT;G_78Og%>c1bP8YE_IcM->|K5U815RzX0qfn8tUcKX;CpQrN$h5R^jS*&4HatuvljJ49B57y}Y#kNhpUU6gx{?EyBR*d^P3@NOUSsCFMsV81&qT>dpkRVPPq&V~u47*Zep2lOJo00vy9is4S)gd2oYy)S+110e|;|FA{>Js;ji;7bPLJC5Od39qLx{Q~vdD_$b<&{$n9zf&qVCK>8N1&ojhiMZcGmWS0r)|Z-!dUBk#%jkeRw0&B$(rl=9bwHyJHGAI=w^>eQI8@~p$XH4JCP9{pU6p1<fIQyPP%#!pg3>8ZnxXx84yAJ-E+@9m$wx`H%pA{Y1oX`Y)WS~jt%9>iPI?##DSeS-Wi0e92I2+L#C`ysMGqzl}6({%L97LsT>7H{nJNv#I~*GPm37`BUrm(a9`w^L0KhC<xOf6Ba9!2Yg&vr%a@tN+S65Z=GljzjG!{IBN0BgnN6s&b?;Q*jdx?{jIN`)(Bvr$*loP+#XI*-ZPyCMQj^#x?%m+?ICai6V}og^l{05}NH)M-9G*O<bKk7GHb*7v84Ly<CS1^~D0Z*v<-}V+T1Pyr$1di&j~A!7U`dF_5fci9%a4XUVev#980VN;V{YXA)*CONI<~|9oRIREcq9^_j-xs@nnXkwx?>p`8Fz(D%amng60sR%c_1LXYs}CM>a!sY89mvukgmR7Am)e$n<H^i1+HX|mya?KbvaH-HwOREb1!A4xC_PAe4-)#{nilAQ>z--UWiJ~q7Uzve(%4F%FaA+<|^z=CQetE<!k271*ehD@l|P)kANy|@}W|t86n+EkK!WZepO2g6D;caOWV#b$K>T$CJZ^ooRW%6BVuv^@7Wt+;(@v|8hega%8WO))DL+8l?9MTQ~Qj|7wKF0YNQ`cl9bXoxnpm;I6hZTClUF3p(ymmT4=eT%@DahU$7bEMy^S=Kt6C?VUSj=TcUc&rJRaPui#ECnPap@oHJS~i0eD2%Zwn}w6LI-7N6G{+}9m6*S?*QgI`?R%}n|GuJ7g@dt{BbzTsuuU8pNmvLo2T6g5|86&@2o$?XWWcZ9fODz)-5t+Rm-siIt;lCH%j{g_Kt{zUHjh{#=7<9(O5j-qG2y>VTtm&pUzR*>>1%rY1zX|jx_TKN*!@-%9mu39U40@~`a9Es3YXI{=HqOG1z21Fe(yx{ukYJ_0KWMY5|K?)6|6lf68y0KP&dUcKecka^D7ccoQk9Ul9WspOCXpok}kH_JtBEgS2!Yj}nMwOj}!H{_u;Hvf=!&16!HT?<?tx5lUv~|>dU|pt@51VFMr{}iVq7%1f5vBVO3c6)<u23nkjKP4L$*~!nBeOP3U*sQ|OA&&z7BW`g4_#Q8;ou9b1u|>BO*ndmv7RddZoXwi;qYM04uqV;9)osbUPil1N{SmL@_R|<c=+m=M0I`^`9fG2@rG;j*u!PvVYEIMB6_z+f{D@H0#MPHOWky6FZCp)#!Q-?t4Ev1y@v8^Ela||MvG*BgS@UTnutiRa5MmB7v?FphhB7{+@f=o2UXD#B|o9!bKYpgglsOKYKX}xYsh{(r<hDTGA~C3y+bg`DrFJJRFyEZPBe(pS-B04W?z#1oFkJM{Jj3)5NEU~<E6Tiyiz0!JU+kTW0rwV(3(byQv_^zyg#vbd0h4`xp)N&3mj&&8*Q<cih>|!*^7ql>bj@w1htD@70;+MvlXp~urhGCrbA&@PZ4ihFp7e6gi*+7B#LYqp-3L1oen<Uu&9S7TM^8P1tL}ZiP_EvXttyG&Y{X#8y%C^ZN*?(eBXC$4k11Hc^c=Er(D9!X^J~9)QMSG3>j)2BkRY*5D0lVz>p+(*+ZTkQP`{MF7|-R39&-rwEP3!dU!Shl^CUSf^R*!uzk!DC1HnkfBMAcX~_#&TcMx>b*t{agEZ<qCZ(&=8lw(oAwEaf735eCJrU|hh5tCB=JaGZ78CBrWLa2M_df6}nw)eZT+92S_5JYhJx>(C$5H?<%J5iwOXqkja<??gBQ=0eIG*1}jz_3UES~3yHoaJzUc}~@kbJC+&M~i*c~kSnr}HBDgq@(W@YqiM&M}y*$D>p#AFmdCd2yER_wNE-N21@aEAuDJ(+A5uA%nnKy>$&z1NBNtPO;QL#S*;UE>no@ha7>mEtOB1>Z$D|5k=8=>AqxYyeOH^-{Gj6H1wKh@_NU0R9ln>EHf6c?FH~2BWaDpNlUJpG8eEu<x#SeD3egYwqYFgDt4tRz#kxr5j`d{ig?@!=<+8DlU$oXj4J(mr(UDhx7N1dzjeR<KOb<6R373Qayh_+zD7WX5)a_P+%xCu{-yuvs*}efsQN_Z_k;m{!T=wg0e+$^JppAYOXnh%xm$=Z;H^rQYE?1h66S7>UUeI6S-l#MtHvWGzpjj%jhfeshE0qab3Y8n_}jMLcZ^RpsXZ;wqA`;$tS#*7BpOD8U=pP>ytm>5M|bMUP9UPMc>=~K)-Xti(?LZWPtBZfTB-VJtQYWm;a=1Yt5uniTTfmd9*lD-rq9~O0tM8~aSB`0@Kf8(gL1w!9gah~SdxoWb-5ReQ<)1x=3<7PcBdU6lEYPI9#=3VeE?8e?SMeYsJPl0#GTc(df4kcQxDERUBB4BbS^mmOeZ{lt^+@Ue*JvdxYX^z!_$4eBJ)s_WEEAe%v{o(1dHmAXQ!_>o_X31VLA&HcLBdEHb5xu*;*B|n&6@|?eAi(>^v%U*oVPR9hzPd38+V5(!%Isb`R(|)qs_T&hv@yp@QxXgYkV?%)-T%%#iBrjJ=%puttMHHlk!YtZ)}{V^nhkptXlu{xA$im8AnQDP2X5VXN6gt$Br8q-M~bnnkm!1s-b^9DNO;D?QX;yfmZS*=;LFv}Savv*G|xlX}T7oH5yEZuZ3$h^@C_<1)n%j+8poocAB1oB>c*d#v~oesc!Et<pNHBFmdga^x6N(ieuAV8OtKdRm-dBsgtS9|8B<<*2FbW<!oOUM27b6s1+%xQN#xuer1;CL1U_*uEG5XG8~KR-{g-dZEAL6BACt=FKqmje#%k#d9s5P*YcodCakTJyAy;p>#t<iycXF30EETxf=u>G&5}Ket7)OK3lJVFfP(O8QxP)3q_+QTCb&OXF}QH4P!iI0=TFJ;BxkfP6N=Bu?_@)h~>weDrR5@7AUgMc?KhkM(i45p@$@|43Wd!Tkkm#VTmR};~PC(hjeD}8y`AL!(-~EjZGL%oJ#7}m0LHjZM@jJcIyuIa|~CfB{asoARl!D#A9iGQ`Y=CmuQX(SfKOb2{Y#;|GCcXA$>uzGEL?EzUwqvorF6RM0mXP#|(!mzoy9XhzTG?-XGuh3zA}LyOQ%@dK+fGHc57yUcK!Fp4BO`IXqO1^K=9Fx0SoEOi?|CUxA_B=v)1^UF3k8whr5T^H(PDYUL_sh#JqTmes=JkCbL6w5*jLvE@qk<KF<}>@Zmh98H5kD+$A1<y;lXCXgf8Yp{LJZoGK&)}5=ZJ6G?1?E2<MLt`!@Ebp_vbr1&>fidvf+Wmc7tQqDsns(rLn;2Aq+ez3OhS)cm8x@V(<KbYYc{lVh->Oubk0&8g9@0XZbvWt;X4OhWbmi0wI`N*7T}T0<WnXt#c~OslpJH+tZhSnMs}*%YV9jh%Gms-W0g{xGQ|0H@n4ut64Y=%4D0%U8KW5tEP|;KAu*XMmE9xC6ZG|69c80-01q4<<O2c8L>eTPwJ9S$&!kw(+B?z-*g3?931`)a~M|Vb+M!O~N`sL@ZUT@Xs==7hc*WY*b`VsZ&|IJkCD)XW3g`11v0g~5VIKg8B1?QY(T-v5M`%racePoSO-3)NdB10zf<TRIZQL`d;yNfC>1$(2RSzd=Zj4?qq7o=R56HKlt^nmy#90i?$yu}>{lVC6i2T?N21u+uRG5<hiub^XY-Jy)!B(NQ?C<oz?q?Mwq4|+9=Gh8o7WL4M&vKtvI>2!{SS>`(fR*?k1ikN1PTa#EYh%w>t#B+ZI6s{g=q$pGlLLNiQWK#=<?#?SVx1%9OA}a1@i8XB65O(G{1I%spiA>(Od-Xaz<+T1sSah);WaAPK6WBu`ruV4H<xy{{OTy6IgKRgS<!vsYN*8zz<VY$JLP;SA3t*xWnON}6U?fUZk1(k+WSTRl=dWDF*df<xluJ%ZUxs7p@in!2vej7Meyl2Zr})35E?#KV%Ok)o8~JU1#J8tv-cvH$yI|*6Gi6vyT}(Z-?mb2M<7X4ppm~k8#(DBOcxk$`10}BoaN=x?y1ti!Xy#M4`MbAY_~goUI6p=p9<Zto!L#mN9tYjsaHZj|3DvvgN(qpw=o6_Kda+I#^%|^ow=>(b3^yC2d?$omOoU!jYM&Y_YxL5Zl<tem;1={c2}o#bQCe8p*z?FtT;oE{23Vjm>d-_sMvO4|8XAO+YD%)Z5gkPC(dwB`xN-}{FVzBDR^5d<X}B}1p<EyxD|<AcnuqX}KPS@>Z);h@*{Lx|)>2^z1ZTuRHHV5h8+Ce`s<117l<8!OVpkwq(AIu`Hv&d(e-ezf&#WWJ3Xw8be$2c|IlZ^X<T)#@seYtGiyCNq8|1z|s%|i)F>;dGeI}V{>25HlEDlw19Lld(4x_!zIgz0;)$5;G-Kt!ts*zlYUWivr@Li=yjz+jTDp|T{+|CYfs}U+iqt7unnM+Yt8?)<>CMlXwVsOAAU2l=WzT$v`vj=UtircnBMq*^@s551;aqHeGYm#gcsann~A+2&MHp{(JE|@43RW0QzY-W9wUTaw$*Rm$dBfFH;v-@KND%0}gjp@3ySk6d*gFqnp%!+J_7rBDlPE|T-bJjHq=(06uDkVpQN_Nuf)ol8e3ldEeQAEqpK%SzxMwywNt|Of&zCaLqDD4h`d*r?pB_vQ}kN)jXZQi<b<t5|BKtD&;%feFMz?9>|Fx`zQQjAD=Mms;V$t4}t1Qr=(-+>ZslS)!08ba#;4?b)&*jl1FaT%0QLv6%}w(dIs_1btdxs$X#_&deeD<oRl=#F-uib`dU`x+esK8o9yoKzbj+-9j1X_>yDBH#jyHJh%wx~L#Wn-)%j5iT1q^q@8E+}OtGEtRS@1=+eW3E#Ut2=UOJg#FB467}WD#JoZjT13+Dk}i>_kjx|FE>bS{uvcjV_O)wXo3<tV-c}Fl&34-<ts=>lsN&I0G%8B)zDeSbh)u+6)bqWoz^Y|cy1@v)1|DBjViRb}kdS|<xV&_9TS4Wqq#dGMm2QhtZ9oaB>VewrMbI_bw3#iN?EqHH;;A>BCZtApblZ-SRx*Z@yM??!Yd;30p<9EtSAl7Q6vPpGf?+4xnW7yXTW~nQQuSS?D|mf(d*iy#w-xjkmtr*bN4;P&31;}D--@b*vT9{3nAOeR8TYEi3(KdqzN%u7yrg1d$u$+%f^tFm7Kv2wH>bnfGmplAt=*n&T31gQHsXkU@A(*@*4+(8BaoC<D3CGH67`A_^f8|EBEw0Cyd(_9)_5}g2e=|lM+hLU<1k>u5*zAUM?`yDzZOL+xPAb_E6uu1XNMbNcf$cH%LtXJA)>WGk1&F%KjNyw%}UVi0s_&kYMc>bA|`Uw2_P=bT)c4ULq@&u07&#OhV#$yeZ^}thQgGAHa~@0sW8EaB<*!P9cAa@eYQ2>%)s}&_7$i-iq*JKYmP=Kk{foaMzWUrY7^$rUJMb2qj_%VV+-(XjfT^q#~(8v0O>jsW~92Ud!59Jl8P3bql8rm$0?~biAR*(n5SdZXP*bvlc3*nUIo))bdjL%#Vx4Q8l%z6Q$R8VW>wdLS;!DI@<YEy`$*^@B>AA}X7~f!u@|9m0mE_5j3<((G#QLwR{$X^Hv?b>!dL-u2q~N<s)|@DTLH|F(~%t|p*Xd2!^ya>Y?ICz&t(vy_UM{MtBeY3jB-bdn1PA#8kZ6PYM=*hXYJW{Ur`8tO`MSi-N@K8lVmuDyg^w2YkYev)I73n3f=>xG>y8`K``<981W1am-{}<RADFI+90)C&ud?^ZIt$T`>+Xu;32OK;(a(cNI!=W3xSJ@I~b%PDc+Cc)M!XhZR_}%-^^%iG{a|sexl89;^pALXycDBf6{BeT+qb)AQ@Y2opWw)uX~qs?=srbj|TueL}m6vasYqD6ZggF8*%20x3WS44($_8BHNQ^W<QifjV@)y5RsPcLOB<M>pMC#-8Re~BWZWTd9QKctn}d9E6pB<TIq@Cku#0hRd6u5)o6<yVY!f~!v#p>nR}<K<j|+i;B3(HhvU?fIa_AViEK8?78Y-x@0&nTb-_Zv(qu^7#y4`iEChBpQ|=O}OdT?D&*FkA6OxXbsXPSKLyCosBV5@6MleOd0Ko`hN+amN&*w7F232&;C+|v?Kaq+u=H$rGPTjJhFdsATOYv__qsq<fXH7eJn)+csC(>#L)1n#WB6HBZ+-4zjCNm$L);iCF@|JC`ysa!&<bZvnfb(8Kwx*{d8Z5BZW!00S(mExqpwU$mFG$cP#uzpV(F8QMDlLQ~s*_mTD3n)cP#*yQoMFvCwN!WER5qY8H6qS`zDgl8EYH*mQu&#ayDc4pGN5Ia#~6uI+Af@ik>SxAN5uw+2|Kpf{E$IVA5;cMW5T8;<FUfg08ueC|A@`)QyIOA?w=y*vt?XRa2R2~OP5vI<qADtxxR5b)9>VtG-e&J(8_ZR(VDk;LH^bjHdm8#YplBz;6?{F;m(htSYhvK$SP9WoLB&4`71~k$d>PZI7=$-v7Rj(K(*+`(72R(Kkp1TwT^Qag@Sv~nE`Qy6(`wmy3`ZD8SArWz{rtMX%WsPyC_P?i?A-ttRD?R_Z{Tsr*2<uU3uy1l|MoM8t?LGR`Xs!qp95OB!v$##+SgnQ5*tn2M`6w9S)1GFF{1ugH6&(fbDH**D2f*|AWh!#+_+@!7Ih=J~Bs2j&dn^xz`gds2f8#3=IIuU9@!D>Y8Hy#pNI-w5!QmCTyq+jrS5YEG$BvETSxh3*(oS7Hxu7OHinf-@0=7`PQ|K>sRl#ZeG4|^)A$U?cS-qxEpj@gchgP(K>|IkoV}lQx8;6cW-Z8y>hknv70Yky>orz=2f~?`aj?NjKoyyR#m~j;#d<a`<i{T>bM;BDccMAqix+4$aWhCx?L&sg<UnUQa7QeBTK({1%4P$D_W1OwqwL7v5}Mm*o#35;e?5W1V;KFbpZm5^v4pskQH)}b@J2UMAv57=C=<hbUO%y;*KIu$d-$5O|wN5iiDhw;HXgZxY3<(gx5BM87A6=^*Q6i4nN}oE(76%F;_(xV%zJ2VMWeqI7C!G%)C*Eq#A$0`OPT~(g>Z{0&T%{UNTOYP^g=PLjCEzQ_rp@&)qxKc3$3wqn@rL=ic<5rlb2y^aP~~yMk|{cz-0i@Gc8UodcUNLoF1b8elT%OAtvRR*t-l7g#Px$N-;(PAjO5I?rY365qVQJQ)yjBU9Xqh`UN8!-tUlPgv+Bo{4nKPK~0}9_oX{ws=e}l2zefp;V|CS4SnPu*<3mPc$L*5;SC?hHzq8G8W!&%-2Dm$bXb~_X*)EfuV^)rRr`QdriNplX)N8Vo&=7)2-X&o<n9M=&sq1Cssd4+q|pkBn(%F0bs*qm0f_U@&X0HQDd#X7qdP9S1a$xOyS3!N~tzm>$VbHb*S53_pY-2ZZhrQ-4zltL~L;w=Cv#BQ724N@)m>2cSSp7_)94nm_ANN(GI*omoJcEBSyUEZ(m(`0od)BtuAt^1*361>DFrX2W`@X%kkoF0&vsO>rk+pEuh=hI#&&o%ne5?0p~OBp?M8G>#adUfG^V_Q+OZ<+Go;c^$|~46Oii>C9B=p7}qR;!j(agrVf*5qGk!H6<H=*0YVNSg*PkE?uvBjT~W{o20JmFg1bZCx!A#Ayc^&u4da+lC0izal@xSqH2fYfkrq2@hGYPE9lZGi=m+eZkw<xtmbjNKvASNMO<H=%U{bV;=be<yy7s96IUY@i9rSMRD<^e6pXr8%9Cq}9^kU$&_R-1)**nMGf<;(KlbI(WyCH2qx&R~6PK{EVWKux)4Mb7}*`lPhZbPGt4@i^<mgD=R=X*}DWZNXEXclqWEPd;Cf!$Dv9Zxo>n(I(+$^)n7I*6O<oUOTzGiT+(bEIh-%~&*FvzkH4hmB@H5ONw0*?8Ezc)nJI9i+w9ze??aVQ^z>*m7*YP|w^urNkp!KbxKNdH<wLs;Q`AAX}geIpzU+1N|Y3Kh$KCCV8ah!UjoAYL>4`>j0X}+jA#jHq-*ori>RWor)^<xTd0YP&5@)q+lu;16w*3RqSz1MeCqwDym4Cr_-SC+uRbIun)+0!g?=nFWBYG{OD-bq^V7F7v&;EWKvI4qcSC<l^Q9Ol0kscwD;AcHEtU~+8|X#E(5()B%^d0qUCK&M3HP;**5frB(Ly`-GsF`P1iEpb#gS&eqmQIsiN^@7qv;Gg^4m1gt?5x8ctmL@z8g(=9M#NT+`K&g0!yW-)*4>(&I@7C^722`s)Qyh3@<Q1ar_=oiH4C8gy}EXusTLr57>Q@yRdzHkS9V0gHH-6oM7mYc5)qBpwq&2BW2$>f2rTRGCxN$A}`m>2NqhN9%rcAPrw#BlhjeH<16s@m8!a)nf2*OLu{VrShAxm}}J~1CwxfJq~&V6j`T7fO0}dnoMYF!y{|;(+j2&5S}+LWOo~kI>_y7sGPmb<FI}yA;(T6jHEoSr0tAj4-lKC@mO1p-sOi!ukiqo9$KW{1r8De%rLE~MAdaKB5pcDy$6hGC)76*&Y}9iZq%QSdqFA+26#}4_F5u>m170vz)unwCgEgG*rr0c5;>Nbk&6Jcw$3K%Ah>Mn@9P=|stPFP6GSwmAhXtPkXSvMfnRi{QZ-Y0iFYT2%@pow#<plvLNF6bIkv(gaTQg7qu5rk(E6Cx+ZfOUU3i5ZM9#)Ar=pP|U^1%R7FM7bqlhtI=}s1~KF$CxRY(X4w|%8FaM^evC&OzCWVAEaDoZXVfE9CYD*IsQgX0fNXpe(vlIym3xfhT&ODsukkyEMlXn5$?!W|0>*i@QATFYfvFfL{qrNIuZq6W&QC=GWg01c}|i9C21%cs7*?$xQM!0f)}t;yGg+T0wL1Akhwh1}K(lEqtKmpjm|sr@!Ky7r-7wy}w;FSlWlkzzgMqSiU4wYgN5#fgT^MV3Uu-E?bh8$)0M)1%F!iteXoT-$}p!(V5Cy4K)HQSBD8C&+V0eYT0kxTwmGl{6Rb5*3S7YlM0pIej6f?F8*2-|q!Gcq;_(o+cQAPi!fol3tYhF61=eQl-do6r@kS+FAi9HiZnMBntRqub;dJ_VGu>os5|Xwc0##d`R10GuwZFnpdwsMXMYS&h~^fc65`~33r3N2z6w^3`Wlu)xGI>fWFsZ&lxY^-YEi)AR?WRbdPj)I1@lS&{Dx;F?%Cb9I-41x|77ZF2!OLO{3Kk;WtGiQnw%)3NBLo#JPy&c>@t)5)I{(`<Wsdy@Rs5o$LlgTrMCj&xrF19&mQhvlQ9`hT#H8g&|*wVcJt&1;k<Z#nl-_>!8@R82GA>x2Bmd4}d==ZG*HL|NUeSmONsw>(D7Z8gWZD%eTkU$A@(e!1jZdF$6b<#95>?{MGxjS~#hF+nGj#9;?EW7P30lA*@@=%_;gJ*sssy5RF?4$;j)TbO9liaUE`;oMW#hcz+6rinM(cP`pMso-D;X-uN2K2VQ=wR(FHMwt6QX)4E_>kzVk!;=0%Vq=%aRwo%HS0vy-v%T&&LUKM{#Q~+t6uzcUO87>t>PaJF?HF~kdw@%RRcE$HscQ6b{FqxgE*DJ)FWX@4zRFF)_fWun(GAz}>E;yPZmQ3d*k4lTQHs){*dYcOmtX`eltx3vGlGkm4Y(k+`S}o&hE_<C<+Rz6zKMGOPYWZj(XsW`o*cMuaiD1j8w)4@yq@;vpPjp_1dtQk-FA#jqsp(U~I|N%?Dh_F#58(p@3!^uvn>YIk<5X^=7G96cxtBPOa@tf4qTbWiQd^qyNr<u7T4RhBGUYhw;<Tcm3wS$(2M+w}P(*TC3tZ3vY3I_3kBl3=sZi{8kN>iHs;~7@HT0B8Je?qIKSDZ-bDd|x3qj{%&^zD1bTK^F=$vaj)415b6xKV>^m~n`pT1OwFI~N2h%OL88aonN!1HjNw!+a)Gzvo=_3ZTZOXqcVJs8C!I1f>iOtmzS0qN!(C^|@6;Xwd6$WWlWXNM)dAj?MUJ?XAg!PchIDiD{fS6$BQ-t9`v#`4Up8y(w6Rq2!*wq!-ViPDdinD-G2kJOZO%!lXu;W+iZYnmY$*TT=3Gz^o4@xh9t0@h>Z#og|7GEvrLvXslCzk^m3SUE}AMUsLMVhwB-#ccSOx?o4#@fgz`B&d7@Bp=MG^E6APpmMHIv%HS~avV8@ciP&H9gHrXRbiGlVf#YLw}KG`Dy3ajO9XG{8Og&h8i<`JMHHWeelqP;DEQ31d&yZQ6dwLlP=m*hx8HWA6pa*_lufFR4SII(RAzP!y-_rLQ_Uvs(v??-Vz<%euv|Rm3y&lVGC`hz1KmS&0al2-C4{a~%^_!WeU?EY0`cd*3wTzY?kTpCi8M_0Ov8kcN>RI%>r^XI-$u0J`<TICjs{iDOIBXrF1T?^9^-YJx>h$G0ufj2PlZ=+19|PM&RD*H!n2P*sHUn_JJgW_BNz3pl7~ZP7%iibSf{`zLg;HH5Ps_%6QXTj(YA#HJrdz*B0`zyjvEyA7Rb1eUxLAYFiWUIBDZ>&8Vg6P;b=sBUAX4jki%S&b}J8F3TL=KW06V%w*+t^Z63}}1O&Ui1tj_<_<s%Qbtr$p*q+*M>q#4_;Q#sNXI`?(SR>s;NMO))o+}T|YvbtDl0G0RbBcY585#s%RiN#AFNG5_-V8>W88k{rZQYD1vl-{mMK=CHk@JRMaNBakvVkZX^xSaUI={Z%E~h*i9nZ5jjHwH7HBMUQT6Atv4znqYkPPO$?t52x5d<|o>j?vc@L=6DCcJRB<_;A^G(V1P5Dj$UCI-Q(${r$sNY5fx!T;ft`1PE`GeSB-wjye`Wwt6;e|<YgpPQb$D4A2)r1SU^m%O|eD(M(xRh^PKKI``-4qB!ij>7$bL<`gnW7q=i%9k(vOC2)Rva;O+_s=~)4j@E5Wx(J9g7#TP&7)8`Wk48l88kMg8&8IW0h+lGk;KFV$wqH8Cb6-?qbNW<^Jp^Glgu>|VLy~DYEr(cnMzTzIy4XDDa8$dP~?eI%bpGM??6tbotE0fTia&%c`0u+J*zfQhP_s#?o1svNOVapyka~zn{t}29f0|wAa20>+lCF3kW>Ne|0{9UJUNF}vT<EMwFJkcDITn;6-9^EpmVm#+@GVQS<1v5;ntjAh^o|BIU39Z-f>U8*!q^mzs}e8Q7{{6+kXDYY?L?}F!CEJYbP(F9GeOy0mxLDYnZlatuJD)(%W8y@$}=#Y@K6EqT+{qK=}mKt{G6O5sEw#b;zYRC+i4Qj1zwxq`Q8UKnE4Gr<}mg&ZmAz0Z4$2`grEvhK+cY*USbURP~TCo_cR4QOUq=Eu>+lD+lTloXuWqUc~|qAp5;jE3koJ()K4hZISfgmdN>2Y?)59+a;NT`)Y_mC&7PnYC7+KG;?Lld2+bSY)$Z8u9@|mD<->s=Z#mb#_b3o=K3$xS7s?|wvEE+zA}`JocldSqKbw#bix}p+iH&{%A8Z^L&;p$8MM70`TLV74J)pO-5LdZ&@zO8aK@lu7Jhn=Vz2;=?KVIkrm)$p`}H;Q^~X!FZrW@%{PX-A#pI9!Z_fL5`Dz#F*esld9XJ&5lYh~#i!W(9ZZ^;P^>h9?ev?EyuxGIg(IR?@qL{aW+$+>x^Pd(UW>MU1o>xOjW~1&(8Vy63RI_=(U*k^Fbb@_e5bbrMGy$CI%u;UbJddRVP(4SzLbZ$HMK_*=`n%@=_JfF~Pi=L9FNpf^qOk_;KSNXK?go=2Ojj~nU!320{x*oZ_nXaU{08-j>l9BX;~BPi&TrtWS$BXnphGSr@^mFQQJyza?MV-dT;yjE7il8&6h@<XOcQ&C>W2py^9<<V&Lp0Wxg#vs50W$(N7PfDLD26HaVTqIb#NUAF^-p(hh7bWWOoH1I>JSS8RI*cL>k8f8sUXY-1sDZH5{Q*2s?a+-z9iR5vb1l94KNZN_!owhO>p|7o(J&L#a#u`~@sC33|B97ZD(9>IDW50pw)FtJCW^-lNgi36Q(>Iz2FqcKa)`p*2wfULAxf;~7-|OqA=r=3f$T4zNOlLAlF;G5VMka(CLnFXt{&-KaMTXv}LeO#a3#9NM|3Y1#ic8oeAecvATM@-QS?$6aC5FUOrh)WPQqRN{WLA0_c%4^aUJh=3jhug+F5RT|?50%sUZ?)T#T(MpuA;L^d;(sXaY`|0>@&<}AGbx}4Q4-kDjSdLL?*qcrU8O(-5z7n7}u+8w*cqj(SXhWGNc*|deM(XmFa7v)v36*iYqcFv=dk5GZ<MIgMK)VMw5Z*S7rbEVlzzKRr4S2B9opu1z;Ga4`t&U5~xTZe$@DHH1UZmc5+A1>Io21?HB9rL|DMneL(MWu~gu5Q5r+@ADQ(3o;v!vBiUooqk{0ztfv#6gp8s|06S^z93)XNHtk(y+~Llmm(Ll_Qd<>AwXr!V*y>e-i3w>uCO8-9b4Kz`bLIRZ==-7lQ?XfHi*eoW&rqbVW`o_pQBm9f~gM4|kr>C5EhMC{`_t^GLY-Vbp55Gh`xH$dFzG+^Q~j?hYiAjLa<JP1<cBY}@0;&P5~0D}ytGm7F!&GQCFJ!%(Y&}182`jsH2o)d#eFyu(H`Lx(4<5>Wl8}!*YCpO}E*2DM=UD`Kw+5+-bi>U()mq|6eXCeUe;i@6h{*<P)Mq|A_+o2<rTDv4_jrN8Ef~p(?E-?WH_sKbWh45TKZvY-|!w0%b$!^!O&%E!TLmUH)LVS8voJI}m|5KN5Tt@(`(enhv@B{H3NYi<G#rYd)V%+^C>f(++2c1$0Kyq;);hd6l%;1Ytiq|dyQlrQvgAHHFuyC{{h<`~px(~F9vzjyL8rkbX7)(ZpT~EumOdRS=6q9gB3$Y^1G6|{-L_<F1F<iPN-cM#LN$)<*j}ar;4F~A7y&DV}m2s9yglbHQv<=3FBs!q}nOY{pVK8RKf`&JW`h7f6>H>i=<?sNIlW|bYVLt$3PD|9_e&EF*=<qR1@SF(}C_tHplVLX=#1lR*u;X+W%~dgfS~nfwYsPtCJ^FY^iv_*F$B)vV64)C|XVZ9yd`m+RVtf>%5{wtcA>~88E+2y_9hmrT?JUxQ^YRN{F;n{tsPvWI5Vz<>J`!{h5MQ5XwgBYDR37g13=G{wJEb{9(@Hc#P9YUN@S5?bmI}c_&DO=YS1GPuhW69ZtI?PbS8Nu}I5_priJ9Pwy$JR)wSJD@>?21g;Dt@@clV<no_n}#bp+6U*hzpHB2KI>c}t98kqQjKLI;E7fLQPeZ)QaKgM(dQnzB`!1!LMy=g*192{F43enjS*1LAUot~nvG@a)o_L?m$WNQWzm9~d=uRFsXD-Xk{xFv{}O(9IEjLSah2?}gzQpH;-IY>)fYc^1qm=(Y6dTew0#{xL&X0nKkk!=1)trz1<s=w*H|O=IEtWHcLuxuUP^hogop^nc?+E+Q5UK;fmC3TRWl&NbYDuLEwrH45PLll5)}(O!5351g_bM;uYi%5+3d_KU;Cb%C23XEf?(P`Wn79dZTEqRT8h#pKveKjFh2j8i+wAd|^;Dl8ec)<XEVIK^NK2w#n%4>RuGAIA}hm#}AAy(pn{jx|cQ+X{!{bk-7i+}+TWok*3%^Z4=#Wjd?zLyebEzN3e0@#IP{O@hJo8!|of$B78ZQu@0aLO+Sp?8mR)Xl=f<aq~}FcRzJ^^XiS(?K`(_+}=d(Dw<B<e;77aXk|D$K1Tewj{&x)sff)Tz_~zCK$y*yt~<jBlN*hCZ08}&KIjX%L1b2gK=v`L7mR1q-5y=W4dO(2A3^)i(kPzuo?R6`(f}hdVnB2CE#D9ti)U51UwWx#%X)wpuiiX1Sb#qp&4&sH(@M3JAqo!p295@XTstNdIG2J02r7!(2A;aWee8uBBZ}Arg8h1N!XQHEnzt7S-HMj^hu<nXV?DNZ(3cCyh+bk8?zb{On4E2&;Srqk+Yd6mk@{^H(!IiUm)~4VYEYuJRHKV#VSNTzSbr{U86jY|j_>SB22h}a@oX3F6Txduh_Im!JWl{C1vd)LTLzF-qIA}*UtFs#Q4JkNTFkjs>TBrEcLDx1E?lTC-Ab|>jKd~|jD2Mq5ZW%<M9xkU)n^%G@~1#W&bHB5YPP8MAxGxs`NsPXCA`v>tw44B@BM_b8bT)?^m5^D=KT*V;av&`k*F`-DO81>&|(gC?Wol)=q=HTUREQLL?U4;w@Cdg3`VQ#o>p4>O7S?&K4Agw@8O+4Y;fRxx7N06`4VU^RrY-xBx&n@I8zb8p=8FaRWgz)v#SmU!<P7oSz1N%&OtB(*~P%A2YL0mHXqHVi*q`_wnLSp($td)7J2x~DtWV7t-b2H9qY)L7l2PIyycv@em5fko3EBZ+%VukDCY}H;BT92Dgv9aqfkxuRa32KK$i~gY=WHFWErm_Tle>{dQ&#zDuOLHsri~`$B*Hbn_D>O?OIX1jHY=s=9)1MoM!$^sue^(%CD0)dl8+kMVhLXzy`tss~Vcj*?_HDd(PTW)?%W)t$Zpfu9g$cRrz;OQ3rNGrQF+*V~i)WS46V8EMcRuTZqpRF{6MP;pkA-4nm(zkUO9tdW1o#4}5nmvc1d;RTzBEpyRXv6@Ik79c^zSbH+ItDg{;wbGGOr+ceA4*4(0hw_UAd0(&#hOFG+JX5YHJu9x4xFs6#hjMuzHwsY|!M`_6-N2zx*s~n{t+Eq3Mqz}g`M~e|@_p1IkqtnO+KB3i5Xtk~RctWqA(Ca7k`h%s{Se}{bT=0fmq8rkX)CvG&yzlmRTC82@27?eydTQ3!Xt*Oj*8K}LliaJ?fb61mj$Z6J8kt)LmAL@8W@XJkU-Rm|sT#r}lMt=5TEl3BSLX#^$ywswshvUG2?l1&W$h>G4Dc2%NyBl{L|v9{y)HfgAftZT<SRbH@AZ$OXkUB;p=<OgeGJmPz(WwxB}HPS+&-Sb?ztE9H2Vkv5;&1Fw@wbcU3k4(P`S8?lC>+ZxX3!TWZiiSiUk)~R4g`UVX4T`7B}ZK1w_EHL_`{#D`XOlqbyLqg~iq<UvJ^%!AX7uS7%CwUSUrR(VT+33e2BB4l>xZPj7n)1>3S&^ug7D18JHC@9pX);WwTNhsx)l`B*fptgL!+7L})+{@^O8oq6VCNnttyt>>IZ)jZ?O$9$OU9P?Yetb|A7IElD4Q1wDWhFc4NJ1XI3o7c%J({M6tkuO=ecrQk+(rz*=q8&NE2dly&b5K>5o=2``QFpv67F-r6e1uiex>#UQ@_j6{CfLbBOQN;E;EHJDUvNS4bC|mxwxbQ%Hwa40K3#&sTGSYC%yiD);pP<ytFT<USwjKO&6#cK9PLXr?-MFBFMZZN?$*40`1q2yx~d-z;!Hzxc1{C?ZOkcR!a*H5{KKDw)ODp^qdsenrpR`Q&bZ~jc6Wo($c(G5M!@1%7>$zYuyXIzXm@}7N_XYCQFnK|{|Y*<bIIyrqc7|mCgz=)Gw}U6B`jn@_M#`;>PPZ^GE<fKS-Pr9wdzwzyEd7OEmob1mH;7*D_o|TwOsS4*QOe2wP-J7-3m{~4C;<5oYNYhwdP;&&f?#y-yHy+;T?Is3Y*6thm+wnr8Q~br8yLCT)5z9i_8isn`<zc1hWdajj>KDj2cros-J=O$J0vHCqvCj)$O8X?Z`Qjy>_K+i0WCAiggTp0Ykii#uvSq6Pkr8q&n+m({M=Xs-_e8(jUYD)<8*mz3?_)+J;v2OY9G`l)JQ`i}u~peY|3->}n3HrRvgbF)iNlxMF8kct>6mUiGH>mE-C%1ZZl%Yq`YiJvCv6%#9$|<Ey$(&2pBoYI?wGYm-a-Z69{)wY6<$b^akDMPYP{Pe;QR%;eJ;)PnZ8`H9wDd~^i@=k_d<!jFxjmp1NX=7bfOjiJ2u(DPVhvXEBXoe(z2Bv|nz+KDi$4e#b|JejbCd`sDi6ox%{^#A_+@atcA_=_Jt`su&lx^m~yH{W>loge@1?XNug-Y*}0^`{Si`_o51`2oLt^zENN`sTkLe(k%5|MX>k<5rWnqOchwT=(=we)skl-udy5AO7NX7{Kq|{$kEq*O<$?r;kWf{nb};Rp-b0<ar-{{cG?1?O(vO@)NQF1?N&grr1L~97PB(3dmQ%^zHBd_FIQ<fA8?kAHMUyzk2xX-yHt=n}=`w>fJZ~<-cD4Qf2E;|LgTHZ=bEgUx4i47v6g3tv3&U`SXWw{ovs*ennqDOJ6@rVN>koGp`fOVc4^hUopkby;CJ)dd+OdC;!h%z52j8MtA83mj0|6I@uAtKpT?<Qgdc7`A1*)%SZqA{YT&V;^CWr`Oeq=^}X-B{_a1123P#AehJHT_#IfLUt(CgXp}<?cg#`M_gNDKbx5*HyYbmQ#!h^IoDs%_!sX}4H}~#+8s8HYpr_|5_)qU_wO00H>rbDBCC}=I7(5tjn<4wusGv-!LT9?aw&Dq!ac&jdI#L2hLS9Te0}KcV5a^9k(I#dQlbl2Q0Wdc+`2PWLI5YS^`m~|>afTL2&ilL@!^rSdA~tH2Ru1Y_^aw~RGx_77A>TBr)gszKv)odx;Z+EBUs{}jA|qv}x&;?I90&%>^dgZbjvnhZe%)}uqeT#D5lrZJgM#I9y?p5gm2OaJf?=y6OE+w|q1)P|-okuR73>JG82hMtt6ouAg7J*Y&PvNRRJ#psw^3GBw%g!#on>XzfJ!#-m8>&hhk+l)#249Ev(GdBOn=#XxnOw`ObSLzTj2#pcjMf$h-eL230H&#G2UDR7V8mr2M;s10oi6S1!N<0Tll<YeZ@UTi;H_M8;%b+2ph#UDszWP+;H92(wO8b(i*x}MJp;6k(RKis5V>PYPiiRvagzIs_IPkl7ZqZj;aAGe2|L=ZKw9x+rb4@Ak<q~r4V4+LgUJ2k4(DHt+@zJ(xR}FO^QBQ$$>skqz2O=N92+7Hr7sSbK6<9g?^aHZanBg8|IvBly(4t7`vW3cF_JVhIHW)E{suiT}kKU<(D?UZD)RDY9R?zXYdeLJP}M_hIDaINBD#wrjnde5j|Cl!ZcwXPOO1lu9p~=qhJ<|rlBjPb%r?y6q$TFtQd&9jqy8a{dnyLL<kX==yyST3`hi(M4PhG*!;@`fQAzQThZD2`Zh*>MlWo5sFuwZpz`t}P_ZCWRHWn+L9MY6)EXMp8rAaA2c4uM7Gy>AQT3k9ZH{?!%htdrci!IIz9sr%R=mo~)rHzG4$@TuJZ&3Eh9VnnQNL<r(SrLb+o)Q9`8~kI2|tRp!eZ-%?fH0aR)F+12f?t@3%u^SXRUpGyMQL<>&An1Z)HCEXxLMoa|mD;1W2T^E+Vz?e2tbg9D?$5#vtP>>J3<yC`#zau>wv3=AG~T{Lxpxd-#*Te&@&k_})MK#iM`!@_T>vt#^O+Gn8SFKgr86GWzzzw|?~S7heXUr>u$jVvzPZ;eT_2>)f9FGk=Jp$*2N9s+B)nrw96T_S64Y)g`jlu!|RNZC<{9-I0VjKc@A9<7K|El5a~%vW3*v7GY?cEg;66kT~POB~zx6<1u+)-KYo(IOV|-9DMleufFr^pRz!WTAY9Q`$ymZ{L*Mx*vfoN#6d5Jg3ew}8Vj}Y93B0%MwZ9B4*=&B<~XHjL(&?AWVS8HvCS8jsA~?t@V)oG^+VLEErnoETCcwtQi)-@@X96cmGrko+P8c6lI=4(nl*Jaqkp#6RxbVS?Y~_4-P_;V{@vSOl(^?ScrA2vil72R=UBMr@z<QX3+F@$bpfN7SJ<kB>s*lMpT8@#ju+kxt@n8+7zj@M**x8lvQL=H1Tc;Pf&rQ_EyFGHnK$40r?=c_jb04!<(=>Te~*6h16E1D^VT;W{rz_i|M9cX(qel-jc<wF0Ccouu%9j(P<88fZ~yH&{GT^Va%LW7e$2ZAz;4gpg?Flq*b;Uf*QPk~CNMfH+y+RHaVy}zGx6_%;7OJ}BUoHOb%od&+_=~a+=KR@b_w@YtrpAKoiIt&wTG4P%8;$1ky4!E4Fui{<2_k4>f`m;a?%!p8C~INN;Uq8&;X#DDqbIeKP*tD>Ct9o15G~gICQM+0AtN60M3(9wK-baVr{^xwRsXL5_8$$vjz;;7jV0`U@m_yP}5jdHSpyY3AJ%qGYC;*JkNUdwY9QYxQOaj)}msEt&!zLG7#9DEU|U{k*#+QjrgV@CqVHAwC*5OGuv~|X`@Hibk_?NcD$9HASxVy10-9jwSpUy!ieKmJ6Wgsb}TsSdq8YjZTLkyHtOs8R-^Mp#;3;SmY@o%`Ro!@QM_@r3}+0!Vg19n-2ONoS5%8N&(2D+#EMf>8cDO8(p`is{f%}_V>N1kH@lE+s{2?MDpB45y^HdD$%0!Gv6<a?W9yr;p`E46^k=hs-cLXG+Clw6?X?;HIcVT7{DaLucvfELe)_qhyKbRp>g(6d^X<1HWg*gvq=`t*NlWKN$%YiQ*fMcD#h2TnX!vY_Mt`>rn;Qy>B{F~+7D_-Hk*H$^x)z(XI7_wS(O@R8B5Ia7SlKrYBIfvrhDtOlVJ9?d!kJ=tWn~6SxymAtN@lIJ=AWw-JEommz%k8fUcV)s$xLA89K32}zF{ZqhNjuqCKyerfg^;!hQHe+><<JR-i@cDGz&PCrF|>H8_O%3u|nRuZEY9Y0+NANE*?<TwJeJp5X!ZhUn4(cx=PHaCFKBpv>`FtVK+!pVQ(tIjzOZ)UN;<Jz+n;VNf@AV#SSsS^^2~vj<C8Q?TKr{6){rD3{Yq&g66<Hih>$Csuvvuu4W5@0y=6tmfIQINbkT5Z{#|gatpj@!&c2Xx)r+DB-)9wcJbiRDXDw4q6D+8kpz%crJS#YQppie{0Jb{N3%eyOTbuf*#TfDt`$Q{qbr3FQ#?33ziB~GwWf+@rmbkx5qJ>o=_%Knd{HTAzPx=NSyueWi^uL11*^v)GElGTq7o%?ynZ4W<MeUzTkGp9_3h)M>yl(<5=2SpeH`tTugXL@9~&jp@tF8M9wo-l4cEQb@E?6S9^_Nw<opzsr=-)j-31$`GOEb8z06D9hFW1o%X8h@P_?q|oL+-v{kU8Q3(GtG2VJGh9sQh99~O6!T-r<V2%d^Lv%`#?k+=N-uYyv@L7+K%ZxZxV5vBo3=Xb;L=n(W@&4&v)J}n`>*|B$}&xpQ*i255XxEuy77XgzCui1_W27h18<)m{gg&avW=c@6cWHiT<hyoxy_3#%zdGD`({^%z^`rX@qi+cUD3dYi{;1ui6PgS>8>JASDg=!*NXE?44e3K2$sX`_z#Z}Zug&O~<VwRbj9FALr=+vH_tk6-C!}-oHEoavdg;iu`2c3VcjQZo=z<~<7s{+>;5sQ7w@d6M1mR8i*0FLIuo*pB>Y_K~CnLZ4m=_rZisi-qpHGN!jh@OT6a(f!1sGa~BoYx?6?Q|g%j9@CZSZhZRQ!)-aZNa&NhNtk4kAIw3EGI%}DVmvkC@1Av;LrCe$W$w7cb8Z}ag7xW1{HOKRX}1*&4D}~a@*5+TP8X@iiJcG_o$76^mOQzwX~v6hJ4ShiIjcW8|dAi{paD&etG!oZybK{tM7gP-<842qaVEf=-<E0H>B_W_S^4%|10nO+ixEJ_NQc1FgpCte|zVxZyo;hZy)~l`;R{JGa~$>y6@?A55N5zcz^hve@4%!Z~x}uFTQm6=JyW2^gZ-?q3$02^Y;(`{>yBZ#TDQE#@{{q{^z-ihHMRc`Sn*1|L*ILzW2uA|ML}59vUI5F6iRX&)$6ai+?$M{af$7{zbm@^$d8y62J<;0s#C~ZQU|V0~_}Z|2_OQ4)oz$ZykQ=`#ilB@9^_qee~n+KYZ)2-g)cu_fEa@9h~C5Q@?xrD*z^RyrNlIAo#D>KeO00z5>gT2m0aX{`tMX`v(PjSgrTo`06{K{pI0Lzw+n{U+0VN!~guz!(V?#tikWz{^Gm;^x4Coeb*WX6g_<NOZlOYW7Ov#{r%S;{mst~zy6Q!eCNB5e)0n@2A|&j@{iv8yT8|AMdiOG44J<4C2pUM>evE>ZBN7+9%VFuFyz+-Y{lVE{stZ&{ms`N{mpNr9o6R<%!j}H4`HYCx&Lss!&X+XEbqPk*~1_H)8W6q$>YN=e)%8oe*Z^@pZgpy!n;5F(cusO!omrjz`3FczzQCI;eWmJSO0wL%AI%r{MUzH`u5>({`&9>{|Z=+rU&1A<M8vJJN)S{5jXzgKMueCwZk`mhI9Vh7etr8{T86?J70X`(NF)K{nsA;>T8GJ_?P$I_=>8+UV&m~YPO5cfu27Cq_NC*e`J_R5Dpx^@%eYY|5Jq1KYkVv(P>8kUl|A3o<c99wh;Wc;0{)fiN*BlY*VL;Wy0g~%BKYtPbaU`TTq+#p%GgOZq3<5=$*ITeD}9+Fj66QE#UXa?xA>!%ITEk!7k<c%W;ED1As%kUxZ>PAK*Fr&X0fmT6DJl0LeUUKV4!duJ)Q7{)4je<>G)i=?h12lm;EL(86^yaY^;><d~A_AoK3GT)DMXF?p}TX7~s2%Hu&mcY#lN|MmKpg~8QR{Bu#*n)re37FX2ea4uUl9Mz9NF64aY5)9n=C~t70nE7hGg;aZssP~Sg;yb#U?`W#Nqp16qSNR=H?RNy#-;vaRM^FK})PVCf8Jc!bqGokz+6mmzqrd(SINo2=q|lX2Jbh50j&U4$h1>UtnuctP7c_X{x`u_9U>TLeV^KRCMdx?|OxTfhkPe0f3uiL!y{AyXee<*LzW(RH3^zcg0OkY4%y+-|Z<rAYSYwbMP|^Sy_*Z{LaWmckwp4NN`2O3!sbf86Wxw*P!|#5VdDn-({ss$afbNGs{3<Xr`~#34xE)}jAO805xY&C?_}ZiIz5VbPUlVO%pKpB=WVpj0{hCWa(MMnT%)?*2{_Yz;$8Nv=6=>z*FTRRW#jpSO;WxgTuY!Z1VFFA2=zITp`1YI3PR-ZFX&JJ_{y;Ki?~&3YwPsvh`ZBDRZJ88w(Xm?FNqX1E*4Z5eOOJ@5N5;-M+Y`r{-=YTSj(9sl842bp?zDsi?qI0PT&x}uwoC9+3y5G#h+le(^Y8n$<B0u)4*&J@98rk0TarM0`!^ugJEU&Zh9No8i)ErbkrkoxF-Qgy1Qe}_vY=Q6l;y22y!)GP9KP`@tBfEH)N-B<9G7c7;vMu8X7vfPx-_#|e23N-2ScDFyWEwu5u=PMijPKmQ&@&Ut2I9(z`9Ciyrmlo3a1e>_B{pjQpJ;W;w*ON%>7`pljtUim^GP=5igHs+v~;FeCT*0=Dx04eu_nW@y0UvL;EBO(HJH4nbXu#xF~*+^SP3jrMCxRkc2A7Z;*QUBK5ElIt|vnFiLkr^g0oJ#1m@TcL?duRdJg3+PA3s4A4NxHR!B%Cb^*9mD!A(_TsJB2TbA02<F3K&YU$`lS^M+0Ab-jL{!Q^O+gbY^cS76c3EjRL~}r35q8GY=DkzTti2dLfA5rjDJVs=Q=5|jE%yTqh51_6j4(73rB8d;uix-EFQmr-Afu5CIiIDBgt=<Dswv=S>t*EKF!y;SPJC?He>sju#(?jg(y~rsdFZ>EYutb3erv=DcC}Cz+V?aHTl?sp6b?IK4=uzJD@Ke~m@---@TP;1?z?E<H{#6~D174NI)2yH&ZoVPN1H_;z?3fv1o?d2OF$xP$%cgX1%qr_Q`5fY@^*@p#D$+%yt*;Z%U21twCQE3f$*oAsYpOy8N@&_X$sAfsRf~*1pP1#M@c+Mw3#WYf<1HoW^6g^nd1Lx&WB#K25MAV3U!@???uTJMhUD$E=8^QG#%x$*Jdcw%zhdqI|FSq`&qN10LudEXU%E>H!ZFVM<T|pn*d>LK5g2;$D>i$P2))m1OQCc3lp8T#xt_$Xr;ku*UH+P3vb+P*scbm2z{(Ko>A8D_S$GrrQEhrFn}u75|LpFiCG~q&a>8j813w)mgmzT80}1v>659t?UeSF4ANDXrOtKHfWVdfXbixLPV`-QgJ@>9TFzc@yU*)DHcr5)m=mjI-PgR5ei*42>d99v!Y|S~@k2)NFpckCIH8H{H82`2oGzm<m}s-Dk>`;i*Qvn}WYCU#+Ql0nvB~l~aZ@Exzin+pJ-s?DtSZV>5L!&}$O(HId8pcztvvATFCX-Jtz;Z_BXlF;xWlS~F*;(y;FORl+3hjf+1;tUDlS|{F4P+}uYR%S>40)&Z?|eU*3LguRNG1vZ`Hi?4CekOx0UnN{80_76CA+W4dcno0}A22c>DjS?p?Us$c?n&ztTz0u1K4r`bswGVcsKK@{BgVeUdy$R?+FxSBWu2a@nLM#Yg}BeX0O-qc4;wk0<-RyBSOBMgs+)P<K>)CZ1ho_Yc*rl3`&t;ga=q2p$22j0NTMP94kY6he$R`8FNbMT>eAl87v$*fW{STm&O^y4{D$MW98%v44{N2Am?8%BuY1k9-&?aBUkyzHBE@v^qy3(>kVvRVn9JTaY}wNsHO@_?&*JBFF7jPjX*7WfJjfbe807=txrXNLS6*iV`n!(Wu!1sAYrjwR}3evAIBE{YAkJS~{Cn$C7WgMBgOxZrd)pGH#74_ZC<Ax=dwXE-v)@b*E)Xxwe7`LmZpcn03OLFMPkT`B}@Qp)cYVsv598@rQbQ@rAT19oii)g4VkzL6?o|AAZk@{I9Np{6cXo7Tx0OH2$)Lq_z&A`O!6YmJ}YP2<11A$$H|%RK>l@6~+SCN%?D~K})D=ueyHt=z6jEl2>H5Zf&Zy`U_8%T_{9d5}J#G3F}2i?@g;*P=wJMwFQ<m<7+tUY~iv(#$}y5PuDn&2DRR>Ih#M%L;(3iWqB4tg~lyw61rA?vZd~5tw3dk)0W!hR0t9>$g^@&X_^PiDNRzp-6Sg8>KB_zN+UiPH!f>(ISAUDWnWZ=XlmvR1t&1HJrAmi_F+>^t{0k6Uw5~hV)>cYb2<N<RCVDCtoS>6;*G%c!W|*1i->wRkZ05&@;g-s<8gzPwDostor+1lJN)qGzdxQ8k`mb%!`aMe1Y-!dsW_AuyN1*Ec8yH$?HX;bI8L{HB`VmZztgaWOfibaq<3^3u~RI4MWaBYn=i?9rM+$$wNGzFNKVo2;ozIvv*JF&fxL_G(4yNw@t!}Ui!xT_kNtlxuY_RJU*E0q{y_z<+|oEkm>XGz!Srue({<WW8{OS|c=*TsuOCnUa(o(`ef)g-<`~PkC!*timNhdH1FIT=g4`I<y|KPdX${i|RI+b;Sdh8ypAZ;;rgYaVXVNoYfB>v-7ppJ*BGQfykP!sghID`+knMGE8yd-n3##^UkAKKq(PNWyk4oakX%xm^dTd#mO(Qf~#9vaJp6f8D2V!A~4RUfTus5QNzq}28`TRQg`1b9)lMly2uGu?3J^XO?_T%aM<I}U+E3`j4JU=`;KCizyI(hr{^I7$a@>u5mbpCa^TFft0PB^@_grKCoEFn5NIXis)?l?F<es}!-`26(m<u#4g&(1&o<@m$Nf7AG%4o?r?y*qw)a`wLbC=<4uy^`J}>HI#BThU3saz!aN2#(7W!H*}T`HR);OA(Mlp=dKTdQ6{QSeL!}om@%s{$#u(qYV}2xZ6)aFR=k9pWcYP$iN+PyiR9OG6)<n%2$LaC!b1Osluf8uK^ZgxO{p=2`>^b!EB*#MtLdn6w)W~M_h|U6S;F^HL4NDahm6Z$Jf(nDsD?k1F(Qo9$6qH<v0sB2@Wl=zoRd1w_NjZy||giY?<pwf*$>|0F0h+`C-Q3{wMt8R^3<pjKm14$|OjS_(KXK29v8`@ui|=uWy#JyFL+<Am3!!^o}Ol$14K;6}1ruUwGUIvVO9;S%L*c|D^joIUUI`kEc_4i;W&aLxOfoc(GY$e%X^EKZ+#EX|GmEBn`*KWT7=Hv$R1e)Bu7o0WdN6+o$PWI{TRtavi)5^J#o`%DrGo18Uu0-W#36C6%1sxBF(dU-pWnUr!0uRw+9T664NY&NjKh)<<IwNi07Q{=WlRyZV*ECIU=}Eg}XsX^qV;kWNq@7Qo~q29h7BxAS@9RewHLZ?nxj7Oof~yl<5Wk!#hQf(s{`aCU^nPG8S}yPf&G|DI*@3epN^5u3XLu1wRoOO<b+DR;X4qE+|tYdltO(0Uyl@?p4MuR6*NMXKLd{CJ7e-I)Rf0N@+8_;d#D%g%J)>b`L7p5fRmef{K##PI_ck9t<mYFx@<7sU)av>|D1-hY>dU+z!Scb_|bo3f@c8Z|-zXaVGw3DE8u(91ea7k8IW&D`sbWozYNDS_%-=?l=9e^*%^!gWU$g2EKLTii_CEp_0ve)ghozr3p=#OcxNABY041Ozxe)}#g3wD<NC@1557Pi#<S8K{#^F<kMR)w%0#TZs?QZ-4k=&~}KFIFk~UogNTY;A<PYMvLU0TA!u=vmqJ-P3y%4?^TzvD=4wccDk3Xeek%dqv8aW<LcmTme!V5TYKIt?HxM4UhU8b>%Xoxa~>I;PtBx$kblEDzMItc2^jJUPw!q<*RfM<4WUqbC*nv~7Pj+NI7pWZGL&`hdK?N7WFeWiCU%dQ6|pxmWv4j_7Tw)@w|_+Mb>hJr(rF#>dd0`5s<W%3qp@L*cFjAoD%~35R&N3MeHJctnUAy})A$L>qj+chWKh#_Gm!+~KE+v>f@!yWpji{ps~D78$<AbL81u3)gfQXesu1E3nLY8*K7H}hVu1G*z*^VhOHzOSFwwrKQ+MrEZK+hnL$yxSBncopPV<$rGb~rBP{w)LjQY9>Rp!m7&m=N#ZZ<PkTZoe-DUD7dYj~Fmu@<rbeGkcGNm5R3Easf6$cR^qW!|q;MtfBDcv8Taipd!NL7bUV?a3XYzuPzT{P1WBS&P0pP_1t^`Pzt5{G6?)OEO?DGIXGA@OAHYB6VI8tS<KL@ntaqJf6IT@G9dVdf3>T@DBg-hTe92mA|jM-Qs+ZI0;d1t>e=2y0_dqSiZT*r4jF6>1vUK%{x8*0>p!2Z0m2W;;nQYZh0^JkAOspY(h0)gIjFy;<m#`0ai1GTpUX|SyV5jfTWI<?mN{kI{f2e+@sKs%NOD=j-eS}cEvCJMt%f+_b#8v^Si8_-JW`WTYaw9Bn@wZWSV7=iY^N=`3(oUraFSo?iH33X3DF}Q$o#jx&mIW!S@Rcic%2pRLll)vl5kw53UzW`s}`9SEd)UG-Ji>u3>S`S3TODi5y=b3RY*_LuX_B_mXEu9gOK8{<FzpWiL4urCdWd0OuDua%HJke!b;gOZu=4aJ_@hM1|!)Oc<LO5GEoVVZ?2?TdkF)wQ4Plwf*49GMQhe%sO&$d?_T5VsKkRnJ9atLS6au$#rXXF`<L=?C(J7meUl-tO4<7=wj*JlFa*IPA^&(R(;qH=lAVtg@FfHflWR^2M1d3)iY(amaQrAT~U6q&mGn2e!lAVshUprLP2ubze&S+r^$t>+SJPlFiRZKM!8o|lqZ*sI-r)VU1*b|c6t-Zi#yuNQvQJM7Q#(TOQFv-ctD-YXvm`q$<eMK$nLv5kMVVHe0$krjBbDRt?q`LaY^4X)uFl1yD#YHT~5xIJxMEMILHmV6vssB%Ps1G--ci-6SLoKcpDjdvPA~P493#;vL{1mxF_1zl&qR#<{2!JkQ4Hu7P>|TLM|#K92evSUgzNDOA*ko$v`SP!Z`Z)=JWgG59h(Jho>ipAI{GT$t78U&y$Oly5831b|x<CYRiV~#G~{^(lx{uMb4>G5u$jMiQtS7DD%}-I1lbzF>|ea$$Fw&T=cL~#k?430WDb%OD7xK+;6aH6k|oO(P$u$1d~JuPBYq&=`kwzN{J5EDI4iKd7>u99x!qb(~Nrc=i*tHIN1u{we>xr{)Km|EK6PssjMyLbvPr-c~7uoHtO~1S!G4!ZsJ|rG7A~7xx_*Q<Cpy+x>9ZPo5j_yz(-dEFZG1yb8$sD>OKlED@%k^wClp^dJPqcak;=d?oPtR0vQEH$gS+v@J_J$&r2*T&I$0DqFL6O8`fBD=DFhKKSO#2rxC$8<P#(dFIIe-Feegqo>%?m;#FqYsZPIBv1XPS7lpa>AQHxPdyTr=tzT8~wA<U#Ox&Z;;xa{Ri8QSTim1^#U$yP?;wHTcDU?ZV@JD(;)2p`>KG>6^_x%m%V$T1nkA$IoJ*C6Jfk`4bs+#y}Qgq!op|bVfWi3}$Uvn|%{d4l(d=gTWNY2ACnBwh(LbkDf9H2*`d*iZy_RlWjRa=d4JPT*b>o7{oFUb(LW%a*;#&V3vW%mI8GJFT;NAC}KQ4S)UmFt^u^@aUX!1RaON#C+kono?()(dJQXxvr|sSzk_O#qCz339aKSRk?sb@&lQLy9X1@)S{b{D9Yl+fW=^-|msqYmsYd-FBlUoB8y5x*{Jhh%gh^Hgw}vQdm>Yu~#o?t_Mx@*1fO2ulVPxH(mGU>q&<{lml)h6VIeUY}BpcM(ZY24|lH~AYL1?jQNs`PlS)xjUwcn1`0pa%L#`|avOBWuHP%73>s3UuqS?u@^tkzO^O(}_P`c0D@qHnv@G_hZcRahm36obYN(a9sdgo=cW^&K1%46sUcC5nyEPT72m_dum+-2eZA&OjD|A5%amhzrMElcI7SPDDBK2mvA;}V&a96YHQnZn24}?BLC<1Acrz>9c1Wxb=U18@}S3TON!b{eAX<EK(>`JH3{Oh<c<gE$tUWKGgfcz>ZW%9LGi9(qu91&4(gk@jX4VKdgT&aB~ucPCPb$gYFmP)%RJ0Yj&z7pm#nXl`#C|Jm8)3rZxGcHzvz;XV!-umnMMhl{xC=|S*?WnyZ2rBQdv)bEl58X$}RztxMbXW7}ZRRy`8XrsK{*Cf9y58Dli#*+<`(w*{vU<&zx`pi1$vf5XL`K@wzwOWG{dU9IGsN>($uzv0FY@&?Ci3(O9C#IKglcEG{ZGajj;_I%8-(7!3|V|RAiLm_ypth$3&Yv*+3C-RAJWw;<Mij(hKzVK*4L}W=IR<}x?jnWKWofGnrC~qH|V~?AocfBapNYgjwq#_@0xW1u|0jR!u(2BEtoI+9E{)TT)%L8xA@P2=*8i6c(nm%_AzK3<!kltWP5T+6ZY2v=;&q5wsc=QU(DWhyXl46-RbRax0`70%-+2@5q8PDeXLPe8Y%Djk5`<>Z9;~K+uj}P`7cEyJh|@OmrvT#I?zIZrj`+CecItphX?I(*IjMR*nZ*I-2>b6+A;(4{!7J<1tvZbov_cBov~nGyf>S5+Od0P&l2Xv%TkMy5s`8{{<l5DP-V-wrd#LE+9#l`{d=>!J(mtCZyf=qG>di^5c9HXgDU{FdH}WC7xQRcLbgT*w>|mGHE#u5UvBsB#5V9>*ZcR|Gmc!`zi`K$`<JSdE|B$(9^I)&w|I0pk(}>h3ybUA)AYG3{8H(ybBFceSM*Dz*B98$ZXZV$B(w{3;MCj233)o!P-Y|V33@|znVpOxJ$e2ENnEG}er=KY%MC9D3AVwALNT2hLa;zlK2xFvdA{f)_u)ATcGyJc`k85-4~g#wys&LJUm2k4qYHa{BdMcPH|(EA-(I+-V<MWy8~jJh3H|triwVs^othc}F-v9DW;<9ZMbe?*N18#SBJc`xc2DOo71vbQQ(h@eW0AD|Qh!#`^NL2Kb*zdsmHWpy!fSHzMDOa^C4m!cmLk=g{I;@yz3S@LvoB9N`HE;lNg=ingq@F$I2#uwqKTMMtSSLKsUIn6$e&)_ET-uB!t4Q>oyOelycUO5Tlw9M5q~m1QhgtBuv4!76|VN(v1ME@R?~ki=4;3ebo(lQWlZzM8Z2cR8?%(s%|?qmY7aaxR-AB07ZKuc{uL6702pE`J9d924XoFpdqtM*^WnawAT1Vo=%7hKK%6;#(qCV1ZlZZOops9ZM6ahQV+DDNUZ1@AjQ=WopJfmLCj8Yt_rwU(Jvgv?9rPle3ACYy7>+@PTmS{VLvMSckLo@!@2T$t$rDFd;PbOnnU%03dgD2Yc~=c+rz((4TGyvJUAMZU+ly?W59da>il*4v)xB}&7VesU6Bwym33e|)rmHLRH>>!~ynI=%L;3opz7Q=qCp9#UB~*ztVbq1(noAkiR9wd_6hIzC>4@98f`F>{N^U?;vzm-sEh)`DZ^PAGvLZ5*NPDHqt}(?}n>@{fe_eC9!@sVBc>b!eW@4(zR7mgvsbjgc{Qn5^iyk%J+)N_i!oA&cccI5gJ!H-&PTW+aWf7@KWr7t>$RhK=aw3&lq85=yslI5c5}2&WrHX&aKO04-w0)q=O8k)bVoN!)?RCM-UcDMvEg^~3@72qC_1d!HXqCNsRv(bmYkBpmc|mqTrAAe+UZaxNKlWrPuj(P^n-SG*EM&fI*Ct9^N|VT`+Y{}~XjvF&c^Z>%U&fL!kOKClTi({{)NfAo>Ng~M^_vhCd+OFP6~hNQfMc-Y{RFD86*eLB%Y@?->T6%TW)ILNpXf(9S>XJFRt8jeW_C{|v1!CN2!is%+MdWWxJq}K4;Kds`L0fVb?ag~kwwa}O;{oBJ_m%^OdGkpb+^<MIW_vl*ySRAdqsFeP$b0?E=_9WHqensTl%$_+gKUH$&w_lPf$1z)?~4v%VokIqzKbPi}G5picId{-<Zt)|7^)--_sDXj$u8KjK15wtgy+yr0LQSdvL&YTG)X^cZR~RkO`<TgebCjgH^=rHThFS3FSsk_one`cHghxT`ccxs2VW6wwiC+jRgHjrT0n_ku|UsCedvS$(8x{?ZcKS5qpgsm93_RY40n2n}Uw<Zr1?nU0im%PuDy9T7_GJ%~yiZ$wTmMIyUX(fieB-mLizCKp=5%%9qV_cc1QEXQf7Z0SIm9iyNkaGC3TvnpZ}F#lh9=etDe-8zP_6Y}&YOEYHF=gC`C0s73T)SF<FKe^1#e)cjUS-jUb7*+akuoLAFm!`x9kdeOr^Lw}=N{*>54o=0eg)A*iUAncZInN;=vT05a6C0y6F8Y*~5S49i~sJb8Z%HRT?)A@C};>#xIe8J$;{rY-A<yomi&;U{am@+1q?7B5vkQ^`+d>`jud-56Ii7I~Wt75QZQoMmKF<t1k(`6Wc39pEeVMQtNgX;KK@n?J6b$C@sPWgQv(BE_+i;_Cx_fjU?zS=BLtGhkLsw)|>{T5F(4(5@*RUD(}n8LSIr{!H*Aj+g%CGnfQC#~ZVS$*^M<R<Cs_|$12m+Dy<eoCu7_r#|AK(25Q&`k9Tz*s05bVqblu6##4f@ZvlFsVOG)N>kI?Us7`#EmB+B8LJ41waP{R@_W;W;P;6lbZE|@l6c>V0_JuZ~BlA+A`i=r?dmu{fb?eLjb3ou*(qOLNrLky(4`6*I$m0-?2{#?Ied07aUoqtp<uwL@}QBMd%~lZsHiI@M>%@l9V76mJNl59s2OX8(->y0A+i5NpY<`5{9B&NYovglEQ9^7~6f3b?o2}t?8WejjpG}8hJe9@b$@9tZW639xBmWMAjzF6+AtD^YQek8ZtyP5g&}V0QZKkSxVBWC{hJf(eIOBEpBD_HmD8*maaO}MgMF~nNUtXb_5t*ec34hkOIc=ZTAlohfxH-vxtN4c3Io^;f%CCy4e2Rpi2~|Tv?M!P=k2U5BOCObV>AI%)XWdj3mRMkMhZd*3V@{162n|*gQoT*xp$GSj8%~4*#YNg6{Sa*kn~xZr7{(@tpboR{EyC?2|bY!*bjJ8%=`EjfnfG>N9b=B<C9S!5gyk06X>n<noBo9$OnFk4!EuUjfOdajFGD6SDD@OJJKm*0_?721JmqI$o><aw^oMWv>9k$B~2P1Ojx(l-bo_nlNhBk_i&>t1)l_ETVmM8r3uOabaE7rNqa{42>@Cvfs|d<=TR*7so-Q+^ep4IJ;Uv?zp~zxVYr{?UO1P>bm(q_6W(UH-YGub-J3!%hqf~Ec)x#OUdz-RKD6yT4to~qYGWJO_Yx4rWxP81tIRt!kZ`w$D%n?8cw0~LKNSQB<~mtkQDvrE-FSxDQr%Ju+-nIW|Ti9hz2fSa7{b1gJcTTWGymx@N;Y`5tF4_iFz2K?D!88wT|OP#l_FqKe{e28N9mN&WRXQNJ~<`Q!nKvHaa$XyxKru#y9ztHLDdT>Y1kb*f>g~X*g%PxG#h?Xj(Bp$+C1Mq&RRi87tPvLn)=vibyr-Y@=$-KUwOXXymXpxxH%l_&HL2AE--weWq>QCNZw}zD*w2({An#*tXtU$c~;!XRA2+he;{bs$<{2*xRoVY$E}3o8@^IIdt;Pjok5E5r9`ldo33}c4WMe_zSU5o#ltgH}&+PSD<>efqMF6|5C&HwtbNvlxik*A&Gt8wr$2k%M}Sp_!SHJ#txQJ8X!K|R0=tcQR+J4y^?g<5LAnT?DkrV6YU90m{g)A%o1854FyWjmSzRDwgJ%|`i>R-*45vlj_P{$)=r8Y+TIBf9SQiE&^%e)l)@yLu9WOL1yi`EGsS2|B&e|ByiyoQ67<!I+Et7M4;6WWlI5tE#hq-m!nL~#-^2!~j*9Pkr8_~`6~yL06a{pILv|6RDrUbCt|U$Wh1R@PX<p94HOZcp4|ACsxe@7OTxe91Razyc@gJW~-icZUb+7mn-_eoYU^jF1V3}rx(p<!-Y47utTs{!prm2W!Dn~41*=x_2(`Cw9SOKEuqHF@le~65c>QNCyU4SXGh|b8DCA~lVLU?)9U+HwdSQfC0SZC?It3ITtYyK_g8nuFjuc~d%%Qh-=yR;N<Z6{Nt74vR)A~lyFo8jyQQ1<7K?FHo*nXwVA_--*@pDz}(q6zz>%4J`~7&@TCfg>jE$~$3}NaOI;ZeQ5^OX1$o$4Sy?wpp#DZ|?UTC%+Dp#qIAtP&A!~tLeWU*|;Au5*>~w*RJJ|Y#aMNyqVRGz#^toP{1I=+53VTpvyCM(G@@me7)fm)Xe0*cV~;a*ZT!sUBsLG$9GJG6ur(DWEB5VonD2TJcm^LgF42ZgQ~865A@>9n)j*?Ib2F?<@*P__R7bQO2WL>^Y`xIQ~lnC<oyX(U$Ch+i?8g3{jJ?UT=0p|m3k83M4~2U$YW!hJwmt8R$_{qA;m@d(rQH7f-I9AyJxqAYZl>3B9#6Q6K*$dPd)WvbMxumh*r}CVtW=+m{-H(mj}8%Lkf^e*eUHARrf_Lw0FCxEhW&x>aLq@bq`8IR;H!C=;)iM{m#zDa!0k_#T&W3BoItzvjyj-t4pLWIv1UJZ!X-j^TG|N3qG3M(`2>X%k4pj;sV3PWw&Bv;Y41Or@cu{z2Ol@QsuAu9Jl(8cG*8^JF5l+RWYUyd55dG(abKY!t-Ufj!cc#nn-<FA|;&|zG<>#7Zqzn_qnbph0#<!of1PLpCS{ieoVQ$$cWw(w~nMFy*F0B_th_*`pw|NfGSR;=0efp^5skUin9*u*f+*Ot$tX!dyLjW;k7P%<J`s5ynMy5?MXsI?c}clp;<Fo@ZK#cps+TIL@Y^@>N`f?)}QsZd==;_Pwi1{&_@ZtxQ?Tb;oJBbs96I-;7gF+rSWF1mY9&Fd?Mp5vgw_AC;sZG`(@Ra+9NIaYnS^QH6N0CT+LxfYh8xS6tX{EXA6WjDHw;)DV0ju-P#9TgVRt7@TnU*L`6Dgw-<-YP^i|zsnXuMY=v*S&6pfcs+e%fyfm+m7z+K1Y@5CHVhu6t>ngn20@sp?fUMS2@H>;PU{iB(Yp?t4o6`m9u3woxFA2AO^j=&btX;a+NW1#_=h=<k_B(?9N^Pykr&Bx$=!TFUmMwzc#3})rWKJ1B20fedXvrHDB_8$K5=&Pi&#D7;WTEWJkM1jNPxAYDe7#!Gt+&GR`6uHI*<M8a0GW(cQkVGNh-VAR;V4bb6kotI<N)Hu+Strj>5NmkUN7=>X=_%N;(oiYU*0GndS2|qY57bg>k-*uCKuldRzKEReXpkRj^*ELr2$v;t1)Xj(=XHYzIxOvL}z+QE`|8(zMwAq*J+qQbyxjJC5p$tmw!AkWvI#)%8VlliBtI~6_hYn5|%V{RlM7`*L-73Ot_Mcfh^?md4IWBc53bdB|^zg#}tw~oo~{54s^1KB}qkQRertJScG~*>8?YwPhx97EY@!+oL#u?T85=1p%`1v1uwkKyumq{t1p^fsq}v)8VYpi(-w<3n)OKs;96aYbWp2DeVLLxWiu^@Ey+D&YhfFd($7bHoYp}8+Mzz{OQ1o(bv?jRt3VV_DY4o*oX+!(5@x_#WI|DqbG7>ntNf968W-~nymc*I(uGO!;vawf0BHa5$IAz!Smtl|?V)E}0lIH3aTdkREQwr2$?)pT1?S-a$pVf*QiVXG7Ke|?dp+_|&EP6}DQ&2f!MBXbpxgzVJC^Q)=T$c?C&yh~eZ8<QyIR0PTNYW3mC$W}$+nAx0$m{okoOQ<18&qYDZ;mwbfAEnqV0qbEgR9`_gb=WIy=?FsEryb!=ShZKg|~+shi;UH7=rhQ~8KhHCaweOz22GTK2GS;{Q30chu#OXS(*%Z{$>zXH`ce(neE0sV#WN&P@=cGKVpdEvojgUUm4ZZd+jr^_gDSOQ%f0JK1E5*7{dYf=u@}II!nx6Rr}F3U#13`FCiP%g}<ndP-z%%Q#k%Q^^!p*u>s#m8|z96nY$^%)lJ*EGm+<oYw=XK7`*5No^v;I*B=DG6VTQig@)aRga=jfjHkYXLbHazNg>XAb(<XF0CYs%(mdKinrd(m-jtAS5blBIK}C@>J^pCA#y*cKx~<RMO>E=3a0q>krdXg8LZ1It^`}Sx}V5?>iMjf9QNEmE=77K7mWqbvcvQX+r<7p=@I*Cw|lL#t7W%ncc1O=7qh3VMx9~?T`^9niZl_b$i*Q5-KA-(X49=_Y!g!CO*j{~M2XgK?U7znf3DrFh@Ze$N&}OIU!?fY;r@BU3C~&;XbaJc{>=ge0U~SMXvWyqr*r~K<`wm{GEMS+z*699x?JZ6Pi$$oM;)^O#-WzPvpCLjD}EIj%mto3DR}=DDpo(yy>iQQip)+}BCfCBzFgQXc*h$<YFrTAuSICJHia_P3Z+%!APY%66VOI-8P|wtE0uX7`@(O+wE2>Cs>MzJ8DHtz*%q{WMQKDTS>U!&_VTblLF;up02_=T%dMtyE>!hOWYrJC4v}B&`M6OK2$}z5tm2R<!$y1lkZ%v&UiqE0D?K{j!j~_Fmt+2c#^di;iNXTTK@lWw@O)L7k3LxdLLB(JxYKq?aV6vwRZE+`{q}pCowgjv$L(zzyDqVe7JnyuefK$IN~7KPjVR4{4|RH=$8^(r<ZF;$eZ&8`h;+q(+ICf@kP4wk!;2y#k&qe0@6QqzcHlZSq5r9TF}t1i9<#+(*@Yt>#T_0Fd6X07oC-Wo4t-q3V{w%fzp0{{o!0xv!rt>KUG1;ZD~fR@N(goA3uo(H<FIU&Sb_sOUKAExxP&Xos23S+jX|M2D=mNTcKR*BmSp#`L8)u+|D$8Sc;VRl&C(CN9-{vu+=Rs`Xp>@*LBN{#<R^N{p$6?PQw2Rvv&)@K6H%W@AlZTN4%79lC3`)a8W{izaL{Wh$h3=MJ<cP}SH2kf{uE+G-r41j7kya0=f#gI5YnBomn`cF0K2bBn8Y=<YJ-+hS?SYBaJ_3T0Znv;hu=PeZ1=1q(zIHHUnWn=CbhbeRo{zn-kD7>L||`w=j!!$^{p?S#CY_#1g42ti#i`$d{I`+A1Tq%qn>)D=2L5<9&hcHbMcjl_nu<!z3&tv{?~`!t0dc?SkB_lp0Y9WixSx%TI{B+;>`D;L`AgO4K1FMfF9%($UhmckAFV-P-Iy%PLI#dKb{_*pM3ma93CBhIzK)=D@&<`RC+@FQI<)o2S9QfQ@sG2s@Nv^x8j<gSV&OKNTrgI-@<Wozf22H!5KS)$>5nUL#~A>U;JyTK6-b2c>3Y^C@3<<ogbe4<xB=wZjHC;lrK3_gm0=FX5Z^R7C6&UuZqitBGd7K-csG#UaX32<kud5VD?9q{%hegTYDGn>b>z5z$b%dwsu-+-rBdG7CQ|g>?J{wd{44XkxjELGd6oGq(7v6iypn8S&6u5CLb;8#xt~(p3e9K72+H`b?dW8(IRJ5q!ZNY$L7kgP95IiKSNv~Qne1aHix+3T)jiL)r)Ls#uokNs?3+Iy)J$qo4Z_MJxeKnhk@NP-hBFe@aFT;p-f5msyRausY;d<h=Qs#(d@pgCfT~voL^XM^VUS@&9vn^B8<`w!kg*rp6b8fL{v4FimJCGjmTPB+}a{z587@|EL=&`NQF*!*`jhs1z=Qmx<W4+eqk;6uDAg`J=gUOa*s`Fw36LOB6Ts}OjD`UHR2R3r*m3IA}|6@Iq8EU##2zAR9dpCC6YI@v^lD}ZbRb|1Puoq{mpt9`PrcHR_`@ec%F!!#u%8cNOU6ddEwObjPjUHd^P|T71b$^23@hgnG6Sb1$AS6J;4{`ip{C<&1@OaYbsh_ONhz^7bkLd{nyhu`}8{!l!^S=Q+db(Z4~)Hjiw&RQCr4oWUAjKE55NMExysri{CT0WU`mev>N~GNU$afyhft?o<!M+_|a|F6$K=JIX*o+3tk_-`OEQ#qq6j;3YV&$Y8gPOUC9s0oWc5)`q^Rft4CZtSwi#Zc>m|a52Xt;#6e2)YgsN+D1Q|=g_QnEUGtG6t1DXZHKpp*Lm{hx#zc^ypyurJEec<*QaT@yZ~A=gyVGeoT{6I59IqYyk_lzY;bc>E&EsNTMh(!AT!yTs+Rb|m##@<X_9)G|!sE;?T1kOQ$)eC`E0wc4+PJ(W)>91R6O$K1?ncqSHUYZgFXNtzN<r~rN$d7)J}y#Ri{<9hncxuh7|auC#9%VZmEw-nU+=@aPm2X5Ln@ygtrp8g`N8Mqnar0|Hi0-yn%`U&eR|MAyHKUfC2kAY^w1xoY8@H%)S-13RRcR*aZg*s9*d*F6pFx#uLNrX*S<<xv|lV+5t)tdSv?l;sC6reR{>_rj&`Ey6<s6^crhoQWf$_4<U?D<J)w*8_0~#2FylB;tHR|+g;c+$fnc~IhFqEU#Ix4a0P6ao_^Yt6Uxy&~yND|i&E)8k-D`KgZo>IA!_?_iOSF5%@`~`^hzj(=r=zElU_+5qBl_pcMRO$Yw%&Jy-IdO%(EVun)Lr7jqf@y9&rTgBSd=a~&;@bdUMvwG%}Sn3+BbZrZa0gSsaO@MhAz++jm-LOXQ|Wl|9iT8EAVnlu>yiyu3<}NBl(7QhbkHuz-B5@ClLm*gY2c0hJedm3sALhB6lpWjmjbyax+_uWISC3CmTB@kZj^)Q*uvkLUH~3wz&Q+j8SdiZoL97I~8}pUE>9x5#z-R!^RmcE`o2lfhx9j>;3S`#N_V2oG>|lw5_ySi)!kPmXvEmcGkXKbQUFK*Xf<QdtPl^oXjnm0e0Eg;!d!319hvUBqbc>6|F`FJtfSPQjC;qYQjuU5ONiaJrQJG56)Infqu{YG2Mnz$P0x;e%Lc)8OJRuPRo6dEp->ScnQ8wFT%{$*CZZMnX#Qzqu){C;Y1u%jk%GVyy1<Yf?x*yZ@h0^$_%htN?dgT24uCGv7Ds2Iq9ks!%-ZYOw!MU;+^2ZjSDvtDo*wM*w|LVU?{n&A=0U`KBW>ms_@lYC_7Y=y|d44jsGxM0os?J@SjCt$}maNM1U@?1<6(~sb-}ZjDAO-3YNA>RC>D@%RKwWn7=2fhZjZ3{~{qRkI6~*gW2>8<vOB|i|jUOIF~ER>v>`JUEd?Kl}nSs4|8z2WjTfYOi!MjuajD5wmqU=Y}*w-yxlwF?2;PKe9SlWSi*m!vedLGPv-uwY)IP|vJ6#B`!9FneteG)7zOUecjDK}cJIy(2d<gJA0ss`nW+@v6!*Sd<do3d0Q6SZ*yo+4Q-(chWcA<1o&rI3D5Wc%1*J$yVlA`u_YAEJs=z6kzJFV{MZHYXv#K5gw$-tE%$D)*$ll6~w=%Cqd^|rtHCD>NM;`YBUo-2eL!h!Qx9E)Z{}Yb7iChH=X(DI|P*KFbXZZX-r%XVOCC8DW7QmLC;dYC!oSwi4jTE$3LBN2*t5*~t6-5u^EkQ^xbhcjdu0GX~^i`HvVKG={cbnJm*?7ljCy=W9G**>2=^w)>Y%pC(YG%oA#iX*&8aAorrc>~2d{XuaiB6?z{CqSjc0+@-eQ(P$*DWEoppw&^Ra2EBR2BN%Hg3zj-nCkVDbq!=iBk&5o$re=n&m*-3S+cUIzjj2*>tH&K?zFK(v2YOFH2rqa4I=Kov~@xYzv7<OVcy1RtgYZN%yt!=kl92F+_1xQER*W?1j;m(x0q8l+wQ{Kb6u$&G=IEO5qUcG7Z-q@rWr!csI=_1qAhjSbQsfX%VVw^;+wx?8SFo((%daN_I@De=KB9>D8F<R#c3((ukXgt2x+0VuAGI`m%T)FsYhYeLTNP7vzlPRH(vb5)>{I)trqLlof{39j~GEvn)y(sN{uDaT!Y<Dsr_1T!=trKPYx^Qv6oPNde1#^P(4(RQmPLrvWd?hjd+Jb5>nG!i0r6MepeoUq}jO8%lGl(x1o*CPMJcmCYB}ADC9KUIdILJQL_o%h`e~@j)`pm$UFbpd2KEl;pzRC3GC6$SCt#p`UPg8UL<30jr8p+eeia`lsc+a0(oh>Io;)r%LGe0ezPSi&cOP46qDfh1gCixJxPD<HZcGC^_{~1riju^A~ejPlQTR1ri~EQwk{pRwo0sxMik(s!Jmxa-{Os=7(;$ogZ;E@8Vc)R2+M*3z^HkXl`H*`Lj*S7xVQii8%sayrM>ZVa<A$<GI#CcLPgLZUkz@fs4=aI9J~9+Uc&2=ayzW4AGj%rc=BrGUVtDv@iS1FSB}QZL3h}tYsM<>8zTPrmZ)<#S`6aLo=<v*1d1-ur0q#r+l@2x+DMqEw78tp2`c;(c6@mUQL-)`*6zuTN1>yjv(6>>21)MYPGT-vf0m;0b3Hsv<^n?^xw;GwsgEDc}#2PMO#5pquF$xY<XAoCR(Y%wQ!BXx+ER996HS|ghEz)o<q8j6t1hOaG&X1-p#Qrr5N$BGATMB^5%p5Quym?Inya=CY9a=>^FvDFNXNA;+i8f!eXO%2^CJ<%r_+Mzkhwy7bA<*meRz~U)~qjev=fOb9sQ4MhbLWk)1A_QGZw0bhR2VWQ)zpAV~|Y7g{n-3}N7W86tXvhAr-6(?TZ|7{jlNX#&25A<HY+<poXI<<%-&UiZP%4OOSoh!)9|9ILAoE2?bC<WiIg;!C%SmXa@09g@$4Q?9A1-|K5M6MM(i!26XYuUbR-ruE}?Q&B!=(m+-AugF~$Zdg$wTT5go@Su>%RN}L`car5?Kr+F9K>p=qzU+x%C;V^Wl_McQGW(K?l-J)XG`#VcBeSXxD^h@X#W&9%wtAhLlh#r0f79<wx7lfZG`;Jad{uO<4f;T;H-E6J<9QLEV)c*C&ptJeXVdfiTm80-I;G=QY)<}!+jfil;>8Q{e+y6U{IXmQp8i9(g&K3%h|DIrHI_;yrO>mD=<z^Rp^5i?+>9G(kK)=g^dp`O>3<*^CZe($QgSt`*1ta@OUOl4&j6cZ>x88xvZLhG^vxqACS0|-z1YT%?@m=Wo#}iTC<_1u7%fOGRDU}n;K`MtQ0(yoYT9f7!2jG!@j|P#B-^+uFzP~B@9?*e##u&Jl$PO~y<b^K&hClbA7J2U8p;}zJ>wmP5q!`qTm2PC^?Di#yie%n1{(+hHVU-dlv$^R;qQk<J{8w80u}RI%Xluvm6d@yoiDZYJ2L!>eZoZ*EdHihEJsb@+n0;s_;PG~6<_p>uf<2|4#q>RoOV6_i+}wMQCn>hTapS*9usiH8Jy5Rj~oUm)Q_8&Fhw4dYS3aOL!@e0B#f-KEfPnnUkXK9{=M~sTBgY&7mt)_kJM}31(#i6w+Q4uXIf{Zs~lj;j?us8)GWKQM!96_O3<5M4nKT2ei!^rAa3pH-z!hgPu?G&ogaTXt2{k>^ULwk=Xb~AAzet8FFu_f2Om#QgaLzX(c*wx6!S<%mMInFRJ4Y0s4e<ni<5)*s<29zx{c=OI1mZipR!Ol1C~|2JO)+7kD2B!a1~M23P6#LXho84PMsLs^(z4hL3=KN5`t}BAW;jAm3?+Zap8iTw(X<wa?3_46Wx}NWvh9k!TXO#crBZ$?Z4)N6pc>ae|~qa`jLcNw)pV*eQ<hw_WIq&H-9;+-!vnl!1?N>b6~OhUkf@VP=s=s?g)fx{X;=$kjO?-Es`)A1f-*bFHf7IPMzwpBlPY*I?mISyr*E9$e%4FdIwP?3iD~4uVj?z0e$g^U{b!=CYw;0+)b|3pNc@@>?V*vx)|{P{C}H3Lc+*fC@9h#PfMsGM|Gpi78^;ou+u*(*i@W}t>W8@7h?4aP4f2<eM%78#S6|s!hjU4rG8YtPtd5~XI(j2Vs^>_%LR8Rd7aNi@74=UEzQ<y166f%Sw2?hUb+&|pU?Zs8bQU$a^tORMPV+qt~{J@+FtD2V<iDnY{v4I!)Yj#MnTgv_HVU6^#~YZDQI5!r^v%CZfy$41%`BV7gn|6nuvg>q#n~)h@(mqlhRc=SC&aT%WLD$6XTkq@Bk`)y_IN4-_+c3Z~OY<N_gcm$}(-zEvKAqoNQ74(XF)X$>J|`-I@!hV?N3q2Zc{Qli?~HwnA@(MRdIgn2GYSHHA~QfQQOU7r)z=h3}<;BwSX$oQbR|Z_r}^CcHg0W88^$nLNBpp?ALd^cky;9ycOUmUf@&K>7M8=b!=Hp+3nh#rMx!OBm|<>JxY+{pgzYLjaC5%*EvBFXJb}FMj_I@q4HCm7J*fuv@e+jh`lJ5zr0wz%SJ!^!8+cwDQW+#|P>|RgX|%eonXY{a0-RmI05_9AqpPxv+^)0ky03b)!5UKc$}N`(~~_M&W8)f_x~`z)^C;ix)+>l<aR`tl&$+F#c`7z>iv+vc6uvVa}z66m>)(4ehIn6_WudRiC{ivdxdJ9X_rHmlaWC<<ea@!piQd4Q?Cqy9x;t$S8gG_v4%Cyh9-q0`soB6gNT7?uuu(jdV=)_z@9v(=B62ctR?cq!|$|!enG9P&Guxkq-ajuw2$2SeIQD^)KV?b`C*#6v0K<$=B&Z7_?=m1rK^42H0obU1%|FS-XHQ#lq5}>QfYR5kV={=ho#VMVyF-^~iLANZ;S9)kv*d3viJ{sH0H!&tO%}HEdhE&jmRwf_EArI*R4+Q94l5=?O5}C_05eOS|BcG0?w{X>tb$p6y+DW>ogG?VPKtXvYZX=U6VID^hIG_x1=Mb)q#1x#_*FIfxsF6H*b;;EZFVM2g&g5vdH3Pd-wi%zh&yhl*gvPB3`y!!PM;X3-tYp(h3~#TfhxHGKW?^yv5$itwj*CvOhVj~l%T2S6DXDD7hXijzgfEElVw?nG&eAOx{`H?3}&S_pJYbh&oXtRq`1lU=tSxTP<U5J@Xi-9@n;Vu3DoQM!~o>mnWtgJsNlw<hc{tda^@=u-dNyu55w&f0a^8@@2w+xog!!Y<m^z2-XOUUpn=My5yBdV5r=ScGj?4a#s`6(oJ3N1L}CvX=0YN>oSlj8&z5-3p*91BGhE>8e`ML6heSnXV!S)Q474%k}VD86dvs?BsH3D^jR!d_7L3h5su!_$QMz`?8-evh_`P*Ri^kaFc(d@U?VFrU+JX3u#7%NeU^&Y6&2f9EJPe5<0mZLqKMK3x4dhzt7&f@*csnwyL>PJ9q|435A@{O?;_YB?ZJ=LdWG!$gW-%T}jheM@xdQZl2Ked-98lA-sK)Tamf@yz3xB$Q1fWCH7#8T3cIusbBRB&!TICwYO5c;Ra5bYf_N>K$84G-Vq@ko2@+jtLFS_gg>(&fN#h64-2a^Yr$W~%*fT7N|1Ctk{RrvAJ4EQ9meSb$<>_z?S{eb5zkMHHNRK*vXIskb1lp&yt@<rcY9zlZ<8Lc{CIqF6|1!PW-jR}%K9)8ahPvmo6Y}cLs@5wTroW(L$_S3gMm?rrnS&l5nvnOqy#d?mcOHrSUmVTO>YH%;Is*eI{kk#jz1hdna)Ye{6x-*6XWMkpY``dpY-%vG7;KrdNW<C{DlpzAP$#JFMgRuP0=dL|9SelS#n;ps>Ebb!Tbw-6A$`0@`cM+cJP#%RE}L3PC^5yzE88|xj=CQm4&`g2a1*z^Jm!>nBEbb2yk4C_nphGc5m3xvd2@@bLE5DVsIfIv7)kda778U_UL+0Yp0_2fCS~IiDA-`sF8ju+dmm(!}bY$BlJKc+9Y(ZoeY?No>&GZZ<N3l1L(-5s2HOFxzHapBMIe+)b7e@De_d+8PYmCRW&eMrF6NQ>cSLkS)|@eZvv_#!<W{%Yn>H{jhh%`D=F5d$GsW*3iO(mB;W&OYB(`PZ~J1?YTHG+?0sXW?ADD^HZ<<;Cqutk?zB5j-!o>GB_9YWzcHW>W1<3-syrr|C8qZ4>EU}L%+^$YqOnsoZYehDf;*!-IQ(~&w2CIB89-Jp(+_=_8n1IBLieThfL#F+H-f@LP$_XClK@kK3$5Y_p-PeRQuf*B2eqV^ZOJa##@fhoAU3ul`EAp0W?_^PrjdhQWUFj1V8HEKYCMg%r3Pr<PtG!8+%7h=#K@_B4N%<p#+^U#wTVrf$`HlHN;<)f^p3=Rpb+O2XUmQ_s%^Kg)Z__uCD8bqQI}z*@)@<tG0HeCKK)7T=aY_oQIUkhw6JS{opU>^$u3{yzce)bt-&ghfs4^pj<ck0IvO!a5(#Yhqb`87pG<ys9a4d5ZP8RkiK4zu%0acl))>aCMV=Q%bb`FaawhS!JR*-(@iMlQh8BJbKs%+B5k`t*c5hG#SGi)ZjPUAem0r<Hs>~J3C|nK$4z&qE6lFhS&GV6I_Slw`q)%sG;zV(Z&I8xh`krcwQXc4w%il`)Wi!6=&e^!Q0#>GunTA9pe|G!%<_5AS#bs5~GPft%Z!nAk{Y0@F=?+Ju3I8j?;UBB5CVG^tO5{s@y5yPos8aS!FsWt-E&PssaL+F2CcZ5d_FSHehE!JI6)&=o*kxla?N?TGu-l<*R5AL-rvmIOZyeAf-PfNdwtgV;s$?h*tTer}R%UCIKNnHUa>Q|iD~hYoyGgcs3O(q+H=7&ELt&SIiIr=GOPIiHzZS~~R2E|ue2*4W_5N(3p;+OHz);eZiny1*D%BTGug0+gL~hiLJ#cHzu9VL76oSxGc9}3OyF?J6uc*;m92gGD`pFo2+veC#p3e9l4-4#{=hI(KPK|k*CWJNU2~z7TzCux;)kGm0m!MVj{heFKtj3Hg1zvZVi0yv&M5KkuI6pj^o|O|qSC32xvfewl?QA@?BjNnM)8QOIC86NDH0C9HxM`&l5xg(X_6|%#ZpG`j*5KeRRU<s4D<x7OmFN2K29z+}U##}$=`DoTb|%ZjSdKiAKYQBrAE8^?JueZe&pDtaDG5}cfhv0%uxW-tP#F+aufzj`Ur)|XUcWmIj*fpld2@Uwx)jGJP@GxRKC1@ssEk4szaw{n5syTK&NK3)sz-D;wfab0kb#n_tO>dwD~#iGM)?Secv?Q<e4AL4;ClkVGE|A}3Uu{YByyU6H-Xdw40vSa!F%!I%Wb&2$}8w39;pOcrG7pU-B+FSs`e#CfCMbr%CNa2c_?A^|96|t?e~lGBay<BL&}<USA==^Q5~ms-5RsukOfJ?yS4eN<$QDr9tT2va8XZPe%Tnd?n`}ScN^fS?G{(KCIaV*i$CXN;Zvw4;?jh(Hwb8jDY@3S7cbfyT+J3yILoPxy1G<fQNH-dZ$t?6qiBhy>W>HVXI&*!PN98lyb38AHwdcr{7C=hnQ9?>wSruR#Kb@*nV;-##Hx-(cGX&1dA@Bh7D{Kga^0Z(<T}$ns9!)Rswv7K1)bnXwzV_Ck(c71@QpWVirR!gqSBU~ExQnal{WZ#+1@uCn69YO;)=|fZY3hK@}^V$u3GSG+qQOl#+G)qEQ(v(HHw7E=0#9vxBFX2Cr?(AHhij;C+KTMi+Q-9MgsFPaRZlD<z%X2iC$A=uI5OPdMw)s+9+IOxE8X!jR33Xk49DPXj*C^a5d1_i>@PL6ZW#>NzpOaaFa@=*`p(f7T1f#m%I^$E2=t`5Egbx@n@%!l3n}mYnYOcuB{8b*+1zHI<u)=TI<-y`C{tNcIjkiYNX6@?bHWdwHrcO3L6RQAQd+ff!5<n2HJ`V;jwZm^*FolGBrHMPEc5+0D7C5Z-<hGt$oUQZ?`{iKSc$b7W+V3of~D8s8)1&?AxM@jWQF|84QD2_>fv1LzHzTGXl1Y1i_?MBylWC{G!p`Kjxb2DR}eo=r}k#`T4`)`RCJPV8EX5yU}Rq#$FN*EH|<|#~oO<@A!^oCWBFw;jw28y>O5vzVDdM&`T}HwLQ~~O)pOO{@D8?bDAK?HfyR&94H<W&gU|AIM+&$^r+fc&!D0!<Qb$a1~NG>*x3SauSw3v&rO6w2GxPC4$D-+J!7+(CXXbR5pSp}v!sj#EIDmA)@}ag-Qn5U$(zG><b~Z+aXovzJ$!MU;%79CH~25Ov-Lk*{2{NhB%t^&j{kOkdKeU)b^kH{aQyjH_DWYI(tJfhcl4oAs<re7dH&b=ihgaXi`<|k!3{req@pRGmf=?AvqpXw*?Bd6i}tYANh6(#_Ot;-Bew}XeI!Xq^>s5fNcDX+yGQk1GpPr7LbIX{cIx>2Y)e|9_B9`4FXamw^3h~M)SxYq5UH<iJI8&~<9g59{#6q2XdP**ROzM_3AT08#Ewc|?Z~sW%LZ1-wZl{TunpZPH;(>seOATCZD`s2xWv->*Dd&4|GaJ6<@YU^UhANa`0{&t+yq}G^5KskJ0D{vB=Wt$E(Jns;+|TEEhnncqijRJ?vSXbHbCu2Hae&G>Sr}EPpzYt^V{gG?V?%B^Oi(Bl!j-1`1$-}aQwG7?>-+LAL$3bQ&QuuMjFmOem;G39Go7%`FMIH2?QUEv2pQ@iXC0!w>_~~8u305iCm~Y6dE6#9v>dPKknZ!q%0BOE5~zfT5nDSPQ64)nnr^#c1`?}#hGn~RvP)S;|()wV2wse?8m-KHcmfFGq7Oa_XfV3*aOcrV{;E0`R*{iJ9%^b;q15yk>!pmAUUI<ox~6e+-%^EMwyw7B0Gt);b1TtWOkSi!^98ip_3%(V2}*L%pb&~B#WHE$R9ipBx$N7b0S3PyR{CRZJ8CwAcBNe7+V9|9C)s2;kU&0)6vNEhqmpyk>?J=!7wq0R_bO+lz7Q7wuWw&Im09|t>;0O&aXp}@kNKp@?Eb2l<9`p8`rZVKT0#p@l)6HtkiSeWH59+H%ZMXu`@_f_9#jtD~<=T<tM2>^8C~Z!{-52FM^_j=8lF{(0pu2<VBwCWr-De@h}R910NeVOw*BNrk<6Uerj5SXfTQgBir#pci=>3n3y(}EqWekDlt%XwVYuEBzurpapbrTw!{jfFdf*~#>7m03ov+z9VSs^C7$P|HZac`S^mI|-7t(Y(@Mg4@8Pl}yE`99(Yy`)Epu4KF~_zNcjRV{6Gl;%`hzI3e1MpxVK&66^rF~uJzlVIFtDAWp9}{XKuygw#!BzJW)dbzCXPSUD1~J@HBw<`gUB8E-e`m;S?r9w)U-W!Xb*5qaR59su|@-D5Ca|U)XIjw8K*HMcXN=&mgj#bWOcAiuex9Zn-SR=x@n5V0kyEb2#Xe`)(CqzGMy|Nn%G)5G(j@3O12wDae^5PGSIc>f~9gzR0svASOq9A0wRqf$4?^Hk8RV<T-U*>0jS|HiZe4DTE34p_FTO63Chqk)5OGa5oq>2pb|()2W66|>GFOaRRA0fER0owF&W|LIoT+)2VsiM9wzY+gcP*iwE(K^dx<lQy$~xFf`UOc%Rp2--waKx+%SADj7&2QWcr5sIdwriD$osy2n@2w8pK`(aur%502~j8junpFII$zswOtbj)s3-aIJ6OA<#1pQho)sm&xKCqoH(c@D5=vpi(C;JoT=%gR-C%n<0QrjNzIWTn!Y&-lh6lbI52hsZb=V`us8#p3s7kg=FCn;-uJ_~yb1Ef>?>p#S!%8Z>Hykx6;{szxfuD`&;<!`-2pZ;w4=-)#iVS<jtkTrM#JGCBxwNTF~tD_DY8KV6X@ID4QrA8rB0n~uZEHqaM4UM&m0V+DE6`8CLjSaI8hp!18bOou6UMhVNrn~qZB-5WM<A_;G~1+Z~7|6Gm4q2qvO!25}ARSxe16J#E`^FhH0E7=?GFligTO}lacR(IGZU}EcBzu_e^hOdSU2?L&#F2v|t;o&F#)MsChdf^j+Jpl4l5JS?JLE4I{FdfaHP4JHSl<3xO&QN5jnXvk;`&hLB(m;{nMEjuT~xGkC_@F;s2LY~<814oud{z!YNB18qxv69*vm!08>+4+lUp2glZQlF$ZA_5foR4S|AyD@i;M@aF>cm^5b%yc*+x^btYK1(EXsU>ae2v8=>NLhLG}OB>>X1Ih`R*p1u}9Agxk0Dd%d;_SJw$$JF#Q|b<?`#1zmhUfudH*qXGgNWi_un{O60h?ecN7=x!JmPd=Is#XTA=0^{ct}qo_qkxzibhtrhBvA$nK#OOHy%KK^pnv5!-AG&U@Su`8rsnaf*rU>W@8(j%qNK;vC@G9nlo@y&$o(Wu;TuAJq38&E;|iIH3=p30NKz7e;=jcFanx`+NMKifXxMZK~|>Y0H!kZOc%%09YMy=oW!w{%u1iJbOchnhBc_|r!#_p4u*h}2@y9MfZ>zOG_*iKKpR3Y!75sijBSvQ5E28lilOfgUDI`3hy>3Bip-Ccg_QoQ^DfV=QZ?eJmS;j38iG~;aA<?B55+6BXi2;w$iAJVc52384-o_!4;(J>W2iP@grna8TF@%3LlvyboI?n_&<xxx9Y#=Sh9m}qK}BXv;tIqk-yx!Bp@9dLYZSXS@C|AN6fU4){JjveC{;hHW=(*KNzMozP&J^BWyKCj!!eL9gMc3U0MgGuZXtAn$R@4_xdng6Db&{}9S(=z3!D(7f`Bh%1@$v)`ZX~x4hP_>*wrM7EZ-v;8RDk{mYPJ4Jpzvce}H^9NFY$fHg*@!MyX?hiCCfg+n|i2@TLYKWS>z@)U&`ELTH2u6zox&gieyV0~4%l<Ru`u(0(G%vwe5u4H6H+d~Bys;!<?vxnBI+AOsUy2D2~<9$nU|&XGb%ja}D<EEPjHb4OtoWp3<`Mvz!XHe~5Ab-^=0mLu@QI0WRQ5on}OYP#qBP>92PxvVeoz^b8Y2)aF>!yAV~Y&#(I1|D{LlsTgi@?Hc5GKL5Uo&p7K1X?*XLx>ESoko6^{7``D?q0N*r`y)rvj=rE!Vq$+7rB;i4nP`70Q6%YvcxC?Lxfxn{K%4_302s(JnVp<fv)+c3GI3`{C$8=R`Z6`IB;s#HG5>n5%j>=F~bye48$sf(qe<oK}}7;O@S&QJ{t}3b?T6L%Qh3!f{qOp@%MrJW)V$iDLay@>+YIW@Lf9|0bvrX_#n02XaIhi#Ici_5SpOjK(GbYfC553GQ$k&h3Da1VnPt9V(WJTA5ELE4?L$Ts#xAAaR+fsR>25d45u^BQqp71FgD}VB6WLc4o4;=^u&R5PY4Ead0=@q_{{GE`(2!J<VS<?d4np>n~?gfForM{LEi?*kONIWwm{1v-wz>ojRxTeY6mnB7Y##m1bP@l0*XfX!2O}%t8j%Hr@^VLp3`KMK?w)4LmTs>Q38S;`k>>`=SCsuyOWug52lhrbR>xy8{&qfK4lrUA^pKHr%U`*Cyb_3rR3QV#Beae`r9~wv2WVY-(t@Lg^7JAFHQ`J5x-jyiDKLId^ZY{#K-I`NFd3NU1thR2qJRmQB|64<k5Aj%kG$VN;b^Q4Iv-_XyEn;GHz-THk%eyR^J{$9d$!8NO?{MsVdCEBppB~h|}M53dL`AgttdEQaN%Ucsj`t>Yw95jdmi}x6RN?q1XCQ5J?`j@gvSKK|gX+e>AX6kWUDHL)&$J&soHao14ZaxHUwg#do7o?0ZRSxe!S`Cvp;dka=W7@vLD477Y9c()%V&6$CYmEmA}<_2l=2zepcZKMl53)m0)h%{<5}p63s+MJ}`xA7Tq6T9YiigCS`#R%C%I4DB#7F&&(pBzD0JMrLCFo|D+jsboQ3C;YxslXO77GZPwM7Ny=GB1@b<G`#_yxRFhIP3SpJ;>95y;t|9i*S6z<<EFz=3Z3z{PGO!CXN}`|)239XmqQF{4WL0-p+AiL6bfP%5_5vK1#08iwx322b?p%P1x~{NU_%%Vha;R1NW#Aj)T?QW2;f*FySkn(;BWv6#3?9?DexFXF-fw~U^uX#&6;+W(*GdVrnVP*8Dto12qZTlhGm1_h!PusIFIW<4*i;0!gP~??IpH71ZQ!fNMn2uML#o1!N(B<=7v$~;@o9qwX;mi9FVM-K!mY>3k9T@#(X_jD2v}H6KX^8u9=>o+hZlMnxiDNM&OSr{vAR2BL40`NdupXFbNO`(}vQRm`MVPIY@nfm_k4LVF0g}c|ft0({L7OA(S=v!8@w(r=w&DA)5@LDbWcZQXlyoLnuWf%eCAjOZ;R2ogM0rKNz_2!1uxHA*M&5hlB5h|Notpe(n+vU9(Dx{m>o`pzqot8MC0Dg;5033^_Hiq6o*&hrHrDP={j6hnATRp*}z;bkZaqB;PN;61Z2ZX%bwebt%;WSF4&91_Nkbkh4w8wZS+t94FvHm|3Pfgzn=GV>`}5Cw8Iwg1mu{q|hjB3o_~ml;V3~Y_lC<In0_p9A_U|Ih2^patZg`)FeNKGfX^4%)nmLB3l*^6&!oyf-l(7$WN>R7<u?z)O86?E5Zg3;?-gqV4tsKSW6vI2c}muCPfgD?SbjL9>g-sg2t7)fW{$jWi&`3yp2eF4YSxxLC(zB3x^?ib)1HS=($HqFDO&tQjTrb#7jGg(<s9KMt<gb#Qq^*+Mw;0hqE-WU4I0D1}uRz*x?{?vv3$e0`<HI3;W!4+*XUDDm~g})spFgb$Xyu*lXL0O^|%>Pmq7e#?UR|5!T2YhK_Gp7LGpD))9nXaEo+6ai-RD4Kk0q8T=X^TJF${-Nb@u1!f)+C(58ofK{1}KZ@Nrw6e@f$tMzGuK||tn`RdIgT!{84O**R)wOEtHGsM`7<jhpCO~^gtd0lDVP!UygmmDs<_S^Babn0#!)yo*Fbv%ht!$b>#1)48GDvDiTYmAu!I2E|Ix2#^4nB%I5$ctA6C7>A*(c804N(c&P?4Uy?t^!%TJ(txjUfpcZKEXdK)js6D2Ya18jZpv3SEa76_%N7wL{O1M-XA$p`8G^ZF6@=%6Cjk+yf2#M<)cGt!4~&B8qjeV$ccC^85%m4~>XieNZP{YXrF`@<X5zn1cm*F{YpbGT&qa*K_S3GNJaU3};X!8lx<Yd<SBn3n?-2hd7V6Yno)&jcm3C0ATX@Q{;mmTlOG{AeZ6rLgO>N@P~j3YRsjhQ+YjXi^s;Q1ys2MvIAwVnMN)&35wS6{LG4ds0u?&bm*o7KLhWL9H`>p892moN(y%dN*H?I1G@|yEYL+&Gk8ORg~jW~aHVGTPf3vmUL;X8AR0J;gbecRg;;t_Z7^~`m4-M3Lq8qaAy8|`s*>fT(5S5N-!X}^>D5hWc`bNfrJymS$$=Yr!;$BCkl{khcF1h!QS4wiFb7bq93aog9zn(!0{aQ=Mwos|!EXM)Z7%lHbMfL4n2s|H6N@iCfW|{00>KSOBMZwvOdu%ZSh!I2hGyzRZn4}r1%F7O7Y?$2+gv>J$+?XD5hxH8DARSw8Esl#3XPBA<syh#(Ex-gCTWz!Yuk2wAJVwzgb)&4E3@n$IhUVT;r&kMq~_6vIFQjP7(&4w;g4vTkbnUE^8FA(tp#`!h&zsNN46P;L&y@?7B3y8E*OsUZ<xcDW8#=KYyEH(fgFu+;IS2?CnONR6OTesAl2F)h2>d8-*!VY9YGQ%kBL8wQtVy~?cVx#O~rnCDoz@Obl(R@8)7?z4?ia11F{n6REQ~(`@ylZ5c(xZ8OXsPCHmk(*B~qHziTSZL(-T^2)zjZHv%mV9g65ovt*Q{;CWG!;?SocQx;fE3h^FdJOwCbq^3AYl#HOA{)nlZrr~U-!{Ju-JJ8T5BNI>T0f9KClU(gq<`1(dP3#bYBPc*Nvg|M&C2@=yL?LAJ$bq8o{H6om%BnrPCK7q^Fe7g^WH{R!nQ@l5Az%Tw7@DpNe&!6wTa^y|FtsW6$jLxUhmHyL%QSx*o7+{qaH^s=)VhJ2CRvtwkVQR6NxtI@qc{Y>Ni-lgn(KiGg@YlKWe)@g90DA`hjQaV-+w;HCm+tv58u73!)5zbH$2qH5ZDRD&CkL?44uGn66ndm$Dxy$*k|a)u@4ae?`%*GP!Q7&qXdiZLIZq0U^1qs5#n3pGtkPx;h~ja_uVKqA;HI90{%W6CBp%<`!I5`msnPd!+tXeIwPPdmMgSVZ}fb?-is?Pt#Z72^w>EXR)sOhBd+PBF=XM<&;ej3q{xf{^?ZPwL9Yk1Cq3TI9DfMzV<%W+3RRDwZw;LvF@bj69M;@nb`sev27>6i5ZWzgII^uM9XSc4F9!t7!_K>4FU}w$2_}q9b7*_%a1=tqjD7^9JD;msoC|Qku?CO_N0}KV7SvlGYCfisIFL}n2<kp`yflVb?+!hvv`IK}$ucmY@W}62<W{go1G^TV9$Il~CXkSZbk@Uo0L2p<Z$=ROpmIiL5>l9x<&q7}BlBK3j4U@Dx)gnwJ|C{#YKYYWuWHexPL$fNL$*6Vwd2T+N3KWy5uh$bUMFOyN}@P+Y#*mG!KsU#)UzWj<`DDy4#G+y>~*6k&O%jMbFG0JyTj2C%9xc6Ms{ja<hMzYtr<jyft3y^p3ikc*9Q_ncuUjRc2X!1t_`RzAGC{Be?30=`IqyjE?UWDY|cSkJ88X@NVjWObf<Cedc9ue<AZ}_HT|0QH+j0E3MT1%-M?Bau4Ysu;^rV(+|DW0#s8Tm6YIZ%_5S(4PVLp*a^#-<EBmxaH~%%Cp1pmS{oVQH{r-QZcXX?GqCWahuJ5?InN;sCL7a7bvu0%+Ss94B1!Z=am{<_g1StpmP9fi7N%3ohU37<GG#rFpmSJT*&r0wBg4NLEI|X0p_8X~XMg9zN(T<WX@%_a;F|&Q1@;+Ui-hXmZd<&3qA`n@SV|5#H?@cj|F`Ldg1^<GRoP{$Z31`!h4VmlU=y?C<!w-fylSa5UhPH2k{kxtATm^>(2AdfLwhx9-*D<x&i%py`8kwh(Z*Br{Bb)uX(^b9>&flE{YMp~6*QgGr^8mnwvjyL%q}?Q9wF=0tAFk=!e@&N<+!NZ4PckaKxTM*{qE&d?znZSEH<6A>ghHo_`*21X5Y!EegUq$<&>>^|z#HP|Mrj<5%*;;xfekG)9@-$>zUv)`*@s`#%E)EdH2#0+lVpgIspMZ20HG|VxoP61rbCx}2IO{&(rDngBXBnhKcQGb)+Kk#ZfLVBz<X7e6<W~v$)V>v=?KaX<WDyuSJTi=AzHh`LFx`;x<>_lEipsWA7pfQ$r_^bs1`l=<mMCwG5=RKs6KkRNtm0<hrPK;&;!A?BOC!U^l+X>(a;}-;4mS0wwJgrtG(v$xk=$@2niO_1ZmNbXq=F)1!j<U)7S|e$4_I^^-{d|-9b!S<5PsJ10HTAkm4V?Kk?X(l>zScd-1=8K)+djJsXAB|Gk)<{>#q)%l-7rB3fi0|Mu&Dy?+1K+aq&F0zLF>%kgUW6X{Ni8&X^*gly9#Z}!NC-VRch4HK->kchPn(aVN@>}9DLV;xhH(ubZq^1quvKQ*b}PM{a@@%;Q$qtK@5S`aEM6O23qema>4hLuA1C(}xTQ|iRf^uymmp|@T*cnob;g*$~W+Bb3KdUHi(i?c8$^f=%Qn)$*1Y@)B}Z2vktP?aR|eZ6Nc4uC^U-g8h%a{z%K`0ouyp$`cPvQ?BN(62dI*bJkCas%t|X!l#~fjP9Qdb^bjLLY;8Zbl&+eui%SfrrCtrkRtO4jHaTe&URPN0FV3!r?GZL9b&UEHO)VL!w=$@d)Oeaq3Y*8meM8U8V7Qc5g)YCH??wpl3;>dDQE)bwf)LbBFu-&Ak@XoT)&5`!B~lDD=JqS$8`x$iu-`t8eZ&OsVd=1IrqD6*@pUy>{9J3Bo=NEZ*_o0jDnuru6l|GED~(^Y(4{s8;*u>H3#Vr0R*kGL{>FoL{FaBUxY<D9d!dSuU|hXTKZ*QYv79L%6Tzk3GD?>KK%KEKNjuyEvVK-tNX^%dKipGVvpJXKT;eh73WP?9uPKAwv;T^Z(3-v?S8_5xdde?Z5SGxVE_L`TxvT?2;|zN9@gx<NSNijW2lJ|K4U*EC{E^lri|@hpO>l?>V|>sUtp1wk~Diz30eLg($R_IH?m_=rks0q?zKRd2wh{HW>8eJC^MyKH%T0vE03ZpTv|gW=O`uK?Y3^XVf!e^2|_lFxeKuk(2E`P{uP^<ZM|M_!OBnRX$pQ?*W)UCE$#|0N41h?zpWMC&{Xmb?&i}oU$yzOr*8tWS-?9-GinvYVSY{QWpK;G6)mnLJfCmyfiu&dll{LvRdt`*hi7_{o=A)FGb9&AaY0Fg!5?z06GH9qK--;Kl+n#LWQ|GmmDpYaTDHSr8cTiZq7-))pBY1+WW@gY$i%`P;gfc%vC*7<qIJn8{+g}Nh#faxS|q2Gp=7n)jYVJ>Up(Ttotf~=R8>yD_YsAgY5>s`r4}Gk6V%*U5M6~Wo?O-Jbd)sSSun%X+Xv5Pzto`P7^Au>V+VyYA>{68B5ADN!#1$j%9xHxA^PBV*Qq?oQWjo%j$PaDhETA2%&aR1urU^j3EU5yQB=P3L@$_pu~dcLbj3=@pAlBm6g3uVv|&UW3^jK12S%=`3*(IkBx83hdl}6bdLR5Q<19)XE)@m$azO1*ukDSbbWfYN2L@gZE$+G?f^>`TrX{Jv(9`<Wm{eOEw)m%Pvgo0@g9_m-sh9bWxa}&@X44fI8W!Bw7QuR(6%m~xe1*WXKYq0E-FFnuruAKx8_vvlf)?bSWy^m%f4jg)(Ef)rpMzqIZ4aXHXxB@g%lYvRG$V>AganEa#Yq3sT^5!EpUQ4m&TVxURZ@ViUJbOu=d*NQvNhS3}jpJht}Ysr5d2d=VWT-f}HMM%6d!%wpIwOK6Goxg6lSpcx~Ou?Lj5_ZYXW;D&H&4y}qu}iY(fc>x-==Vd#~y+RT}bQyD*1xmr(zE05XoeorGEYQA*{E;$>s1hVCIB{8eY>8%i%aroPwWG`6NqPF**BE4fpH>;x7c9i2>q4^iIp&t+3Mj=6!#q#X)r%xYG&ySBNiQS7L4{7g(NJH9FDMyLKfZ3r-^!4fV<FCgb4nMr1)UV%Oyx?RQgs5NA`@B<jf(@nQkJI@yUr*yQW?t~>Zci*rnLKr@q`^f*scmJd#Crb<vU$2$LFLfdf6-+F>Tz51UR&bS@!nNBr)=q>3M6P85PHL@^o1N0#Nm7~hpY;9M{eQmYPwD*mA`1k%apsPpFf<RygxR6wPbGD^)*p$%Hu6F%bvcYG~ZOe8nVT75s<t(;fEcpc);+1F<))MRWc!2T$X#+$LtW3?<o<WNADCV<U*>I&b4DE()=6MK{TQc3O=c@=*l=%)>`d^DfTDziFW=clWN0@J!;Tks#mMUEe5A*!e);v7SWvcIbnRAeGp5b$LSr=E$?)N_-1yyjg-*!p*NI__wuC}?9cVLoW#5V*_LW5Tw6eOz^u$j@8UG2nlBL)FNkR2dT}$24c@W6ZauejHra+>ZM$3jdds{fVq&e;cQIh2_{8%7AaZd0=}=~detUTG?(^w!aCUxpek^pUETsCWoNrVRUrAe89T966Od-77(AAnqxLnJ!tmW6#aIul<rX&<<ZA%SBr{VN1rIIE@NjoWR5N9)Ag5;)?a*#cBN^a6T$NEVo#vgPcf8D05jwS_^l?-K<opS*iqd>3ADf|m?IT^d`S3OUR+M97>6N>L28CR<w#ba`9pj;o2q(6Sp*+XB(A~Qtta9TvMb~)DvV9TyR9$8$9p3E19C@Dpih(*fj{ELxoW;3cH-U4%TUmk(JA*HwVSVNf#BW=eXrUSIWEK-2tWiw)@Qk!dH79o!Q|Aa(2q{Em?WV1R(xb!t#O~bj=c6+h~)Vy(}eiM(W-szHzR)uf!Z9e||;qX_SltZdeq$gH?e|-MSM+oKfOWwLUZJp|lr6GFo#K#X6daMoHCMJDYkR&u`{ZWaRnzurz4@Je=n{<7>fc9qWy`%%8^{rFcq9iHQ;$oee%fzYGiOR5eQ+}c+L$7}m&SpW1{p6kA_8C<Ci+Q<zGNC%Jo>XDl^Y|(;t+7b5LGb?L(Q#3FxB$OI20nv2gx!kWgf2gaGpbd<Zp15vJ8^b3(KJ2&HQC%ObJec<N@3pwZ7=rJUrMn2D(g{F-fX%aF3Sg^t)>K$?9b&&m>%f;1hR(7-leJ*Q4_&Q`NKE0T=nB}2?9DMHze%kopPLV|D>G76i)#^R=-@RU;13gpraahFDGsCSy{{wx`6t$yH9@=!Yorm5*I3Mx__9^v*Pb@bEiMgzswi6^Mhm~@7WjNYHDC9uqu3N%n4J01iOaNQa08)DD(L1U$d~(6kWFuA}dg|%ILMBwC=WzNYK=WYSCBGM_CoF{IpxCV9Myl7bm$Gl~CttGa4_t`(!b{qKXL+Q#MPgps<*2ZXj&8jD9j_TsO2HG!%z=Ab1@MVS*Eh67%Slo`DB0H|y%Nim--flJ5pNiT(*a5v7@Gk85Xy=7r%ef0~%%7IVWlyn84}$5K}B-YblRe@WA&Aq<@qw?Ip^rUFGq$=HnB>vS#($&z7jN-0gr3M&dn$LVaALvvga87EqhNMoN`h6V8Xp2&yLp}(ZicxM4;8x1OL_Il;S_HLmVYg=1ssF0<zM>Tn42ne!y%f9{P`1Hf^y8x`}%`eJO`03-jlQ(}a7DaSIc%l!=`0@y_my^ohB`#D5od_H$$y`yRnwx#!07WTN!Cy3Vpgt$9<6>K7Su<rc{|0;`R_0kYNX6H5H4B$}W5@2vf^kqssqPEbHcv^SaP5a~b)@G*FE^7u7RbOL^sFwBm)4)FO8+z|-z&I!X7{J&SFQM7)-e?a7L8>=4KBSCCanu}q0yqtvZPQ5U4}=y{13+_2>FSEiXOJ0oGiyl-yO)Nif$F9M01je9b3b2hKf^_uu{2q`G<+s()mX^hGnyfMa5@~u;9drl(X4Hx(abPs`SzNY0Lao83FtXiED{xKTT}2g{JIL;MqAQ$jypVWcB-KUY}5s=3mlVL51Z595O6jTff})K(yd$rkGdhm(Zqol`g1K)IFq|1z8tgQ8_OX1{3Y{>NvkgWwxh8Q!mWREltYeSF|>a_mDp4lysp!t6I}bjUbiZ^!5ny*$m4p94-~g-JJgM-;UpW7UHxn?Ug;!8aBEVJ<{~34sW^&WJyvY(1PcwsjGSC5Cvxw^Z$SJq<zDTi|Q;c7=GhFSjwtR@Uq2AL%d&5tFj#xt}kSQ^6}VsjG&M;?c^vY1v;N5Tw^*oIXcq~DO<LoK70knV)bhRc2O$3{`CZOL2Tm7iB(Zc(L+V-ZvjQ2Af8RQe%K;cMQq!|nNNR*V)x<G;rTDvH}S_WAKxDre|$bWK7D<7b_~=;Fa70W*}=CxE+<$Fq^h)vPd<F4kIqick0*Q9UQhOO{?_L|e|a1H^7(b}@$K7pCm%!$i2fhW-hMoNkFP2(%4Iw{dHa@ot-MEHM~CN!WNxlMr!kIB&RCH>KYn)%1@`pswN5{sd<s5%3{E~A9sf;$iU;Z=`u$g$2Gu0{<@n8C1gIbXa{S@szX=esOTBw{{O;uJ{bVnr@^yPX$RdYl!Rx~}XmvE%W45W#M{XgOtFndBVzGvN5-ux#$)2GM5So|!AAh9(KOtKO&WAK#7@v;bPB%B$&GiD*bh(;-tyJTra5->`S0J}IvZ9&b>V45)LHO8~x_1B=u4WYF4RMwVAS;t+X>QdZE3K~NDpmG7$P1nJ-byFUqm)wGw6Phd1(_r}q3}h|lj4Zh`q#>eiUpubos{fCtw#nuFvyo_Tq$e%>EvB+4z2jU7k{;t+<Z(Erv?77x%qT|@DcjZn_mSqi=`-ex?Vt9xlUJD+=X%Y`oy3dYN~8Kj#tPA5b8s%xZq1)?SA}q%{Qw%JZ>MyDerXLp4)S}Jy9T2yx3sF{4T-OfAjI($J5~S=hybnr-y&nyj*HdoxSD#GRc4q5Id+=JDpN(onx#V^Vb}r+lnd!HvQrK6<O@Hp7vk#Z?9*2x)Da!wxh2o!e<>`6-Dgw`#hk(w>B0Ps*4)4t2_;$e1P*PA7c3>^)n$Nt~v$Qywc`yv1C%VWYStZ7E&D``p?$xJ4^N!cQ2gF%LXSYdl_%@_Gf$V<+;<;CwT;UB%tbSacg|jSM}kQ5w6!^e9h|BBFiAKYVi|C!V<3YnRU1VYE2sF?LZW+1zPXD5^ef7U+|(s&PFogP?^#mpV?sXMHVeBdMyhU_R(qQ0t+iEg3`h2?d{)?Ofw}#d4GS3lS;A}{+J4q!;j~SeYT~jKN6B8dsPPj-U#6KdSuHJzjyifa)~eYE%5Um8w4fTU|Cz)nm!GH&F+}O5~7XLtyVEsXk4Me-!RD;c4=>nliv?0LX6y=LD1XdQ@1x3hp+MmhkhP^-5b+U)L&>Str%->OyCqF_kkTJpWlPcy#3h4^4_nn$rVVGkeyURpoF7b-*p-Pq(~;=@8p}N&Wxo(LiHb|j6YT~{5D7A=lS%PlheHi*0)6W6)l1W8O!AoF2OQjstqlrum7Mt!os{O9Xc{R1t;}aa7K_Zp{So-aNtynZE#uj3d#_u!Y}x0*%ui99P}@udJ3LK<l~Vzmp@1qcf9PgmrAq}&wBO76$~Q`5HMHb60~h@(X2aFM*k*UQnga18Gt*5r0_*DlD&e23KJmvV95(3o)URFr;gcHxe(6aH=mCVD=dF^Hc7utG&a*F*U*d^FQ`dhfTi<$w(;N<vu~}9@SO-0)0StpxB|_e&N}#o>r78?Ha8PW)~yx^+M$Lo?1bj=@j<swhqbe}iYTxrH-zt@=4F+I;i0R10R^$Uuk;7PD=eQtH=T95`x1z5x7FJz+Y&NC7!t{lI|4O&?3KzVdr~}5E0=PjXXE1;vleyQ_Bw+6VdJiJ7fW}qD*A`hb*Csli*;T?Z%akJT3`azOUJ^+<Q1C*DaDVlYhSG%2s%js6KdYSSpdJtE!*k7Qa#B-F8@+-iq~L>e2CPG;w;r{mfE-^iO?rBJnz2#lEg;h@r7!3GGAl4HcJ4fgu#+<$xk&ifC~7MY;C!qt<pyvw^ePffgWMkzCh$VH1cbp|NaalCZRsb7Zc7^U@!xZk4PL{-=^tYhTHYqLc%Jp`)u#8Vo&+&Zv_+P9VbkIgaEuVzG2g-oc7+!#v$ldCtTmR|J&S`F1L+kX}$_<M+FE8;-bwqXyk;`R#iF5+jz;UX(fXJagms!NESiLQYrZ`9sL3`e|vt+kB*r4s5hCj-3_?l;>@1t%&HU#UflhjdzSB<b07=)gRFmH*s8_r5zbFreWgtVYfe43EzNe11!C|ZHLr53uMW?GUCPIU!K_wq$#W#)wgF1d{YG%%hyf_5s{(Ez?4+r(_S+mSI^o?dQp>J|FA0WDE+{D!#t*9qsCJHK`z-H`Mv`>AD2TW#_w(Zi?gt&>u!GAZB~<fC@F1V?$G_^+o5*@Tb}1!Rhew%AT){_JXnsY;g(qAE^+p@#(2)h>fEftk31SWT63vZ805Sp=r8o~^TuBIs7(gp^r-c2*(cxel>BHWhYR<1ACO3uZYi?C&x|*%{WqOf^4NR$B$NwAyX}ucUwXL^23_w(<9u^_B-e`nOVDWj#RWb2SRcX0CzJ(Q+yqPmZTSL#AZRUScf%eh<6PAd%j0(=q=DcLrbE$-D!l{-O)^JMxYt7-4fVnvX;v7!N0#^WSkKoyILg9T!2H@TM)~}cKRjJoN%0@k`9eYpqKJHzC++Yb_PVOwMBt8nurFP-p6jy1ONoqnNCv~?v5i*o>tT*yNtF*DnLU)%-SyC8QEKv0V)VNSB-Z>(b>8kyRZxL1=rRATvp{g!Xw+6_iW%K(~8bju%n=gmM(Vf&>=WqLAeQ`4jfw!TQ^{`)O_tY)nVRcPWTYszzm!kSu+VY4S0QrNqNtiGh66koCsLiSr$%!TuOpc%g{Vht2?&gj$C3F9tC8?TocIgKLQ7@p0HMi<H?_KUXzuP7j)Z*%`E5sUSw1xt3tsbgX6Hf<eHr%(#6>cC{76kvcg$`N0`0IyHe>qdq(VDhh=2tI{-0P_&=zk<ycFrOx-W(u<9xPcfm-a2tHa?}9G0!@2c_-!z=$6yUn~!Jkavv_9)S{^P2<}b0<mWH1KYaZX)3r^fy<B@?*rXSzR=nfxVuI_KfZ}RRLJTA?6;Bny4(QeKbfyR6z&3_sR7XKO*#hTC4=4^_eL8#j<qS2+e|-b8?O)NB8F?0mc5Z5zcYDQRYtNm_=I_MnDp&VXT)L>Oc0`)^5y;7z${>^Q3*%`%P;8!IG6=3F90uu6>E1@j(V(x{h%%eNSKfZwYRk24AGIO${Sx&e4|d_C`Lw?Z+s=jUfVxEFsF_19Y+~`8o0oty03yVC-$V<aW*m+$LdfV0E?APwmBESzuG{9qS^z5YENVZ~QQ#wN(-7!hv>zfXAGJN@hvOxX`ZBOhwqeC4(Nf;l8w1ywmmf<MgfzHrcr)V`2&IuQDoae|0eEgFBuSo;%I5Q&a##AD!Oh_oDpg1mB12DG%|e}0pG9hvKyx=EpCYF&5roS17w<cf>-8`!5u9zH_tl0Q&7`}aJXXTI)Ru2#`EG;RV6KXfr@d^<>g1eYIkD({$h8rfxW#md*>W9y4t8gbMv*koghcwnlpoEy9(A93uk0K{M@s!1RkBM}x^n$yX`AcwYAVkz;5aT9j{go@P7$1m9^iv~jzP3g;aRTG(PS4YAVa6pjK+bFu_ZQ5mo9mY+Ec?CGFNYAnLCW{X{daEbfAx}tOt`d-N8D^?m2RCwVLwTCEPTJ4Fj225XRa))?bmg@A9e|wEsn%V;{dS&GOv2FI#Icg}vf)`3xGKsBYGaz-LYhu833@yr#UW@({k+{$(4~1`3;~&8ixBh>ay1H8tA2BMw}Bb)Ewb+Q!~pi|KTnH<XVT+aSAQ*8$cpIM6=g2MlJQ6FqhuzS!*wU4Efs=T?U;yVTWSJR=9?vWtwo?{{A1_k2!}^3qy&RDL_M%Wv<Ps&pfIJYJtvZo*L?k#aaK9IQG%?V_fd`-k(0n+T-q#f{@sC(8t<M!IimhAqAv56X4(bFQ5f+xVYCk7!@QR3dTd@(VToQ>JSCysox|^l0PtAQfpp!mUqILm!fc7=t_qR7klh>LucvjhzqbeZMJF{LKr7%<}Qpz1Vn;*=j44MZYFCAO2%mF}5*Af1e?m>SHXCWrnu;h`*lOuIPNMh2WfrT*x^&Y$q-pYBS*VM;jND%I<s}(_3z3<+IO)zVGP&mB!^e?o{h4ERSTJrJhHrto6wjO7=Gs7t;*AdHLfT)jn0_bb{ALVckh0rh-N^kirO;N{e2#xV~XR%M4ewvg-zxVK-(cq0yZ@<}c(QrC1gD#XML=zStDDegSELCD76hse$@ou#dexPg<vH<8aTk_dLFSF`6`_!y+f3ZN@GkCV!ifxH<<ZKp`-!6OI*aKHb8r*M#|xt$h?#wDGWV=$Ym7rAx&D0^YW$KtEQ1^?UZS$7L3fGS7~y#;f5EH8V%Edt}_}z2RkOF~^R?ydH275aNQAU$^-8;6UuS2L!^v2k^#XaC4A@czK|11o*|su~8=uEDS}kEr<f9fwzBAk5ufkr8rXk)x9$qk-&gN=!eV6bUrxH6U|Qzb|kRDBf%c`WS5cP6tK3*^!N4IPw&6JeQOQU^)Xd(RA`T~FOe#MJDA-x;9%o^NN_W(qOy%!HijdvSTbQDbPJ<u`Qlg6TO29yCk#$>Phg_)07OCDL*#Tus&=F_An%EHFXB^`HJ|n(d2ASuy2ud+@hVA8U>hPGZPl?9e{}<XY+@I7V-kc3YM78C5=eqVD1{Z~!EN}1Hf^CnKYbXe8!};UR5A&$v5w?qG}bX|#T`uDds+-5cG!_h{K*=x^kSeGY0SP=W(Zs>1vDX3v~)sb6d9V&?cpU(pA)iTgmF^bC?e8!<yz^bQdS!@HItUA{tN7&n!N7>cu^1@!`MhKH=}lTn6>YWuJb6}N}`!Xrr`Gc3{Uy`kUEx4*J2HL`z7vPbP2_p09U}M&K6yz8gZ<(g+pB(t;$t_q-nJVMYOm~5KO7CtxiRH9%62gv(KOY^zuEtJR2`_4sPYfamZ6QM2sD*Mb(9x`Lfk!Ft$2s;o2%X$tA)zLLpP7GJZ9^{Ao1!c5Le<%mAd)-t_Vo3w3dvy+IcfD>pe@YQY|*s*}LLwB?yj#^$%%Y|yZcN`;?LZL%bkAY2+`02g&nHUv^X$*1$Yx?L*^XjL0_kuBP)+k>cV7M(eX=B{biF7utuOs@wMIKOlAsoYGtjAO#vqH>hrDWxht$>Ca_m0HPBj|yV@+<bCc=@fBB_Sw`RYqiz1->j)WyB+kq@}Bf#GDX*<nS=OMe$=e3{4GlxZofW7(Q%gInQD}c+(ddUYORI~ExE&G?|q_Btg&NRy#aqHMhlyk;mY~>XXt4o^cV()>;**^qTtj~{I=Kx8?zZj=Mvutm)CN@jBIZAJLyx*aBDGj;|mlT+$vDmgR;&ToT-moOK#bw3VWvrjx261QoT|iX-&fQ8)*Yk)wtjalB#~UiM}9+>o8&HfJ<9)+MEAQq~BCnQM~29De^(p7{s^V9)gG7hS{L-U_8EY=}LGzTCAls74#@9(X!iVFS}4*D9<lvg7r*aaie*xlp5gTOc*0;DzFfxgGyMIaGxlROLG2_js7;8e4b7&U)C3kA#~V0`H`O#rs2z<55DAA_mhK<O?ksSX_cRxmL46_6K(L37<uWM7?_0+m|nJ_DwH2i^truFJHQo&PkV#jvwrhAW;r@(9zE?{WW(M?3;rEGYc)>~vwpvI+B&*8Ic*J3`cDUkeasX798=e4!;?WXT<gdrp2Ar_n=LS|tQ~nzhcA}^w3^ej82SQHuvHsswhFB0g^q^{v_};}7flS~ujz*hz3Z8#0+m|1j4sV$tm<{mp{xv+CM3W}T<OR5ntG%$Yaxn;cMQ9Q*Oh8HR>H3md{OOeF}h+wb8zhznot=+4Yv%d?u$e!Jf)uaT9L2ZaFIKgW5H|UNgz@_df;JfgUNv`2g&_VZnE$V6YsY`?s0+W(dt?R&boI!N7}=?Zen+Aca}9w+D6_i+=2<So2q6zzl+-ptJ&w@hieYYy#KZ=e`IMPL?cXX(3;hp@#3wqmVb*KR#6;)b{J9QO7A6Ms=*y>&?~=LfOmGUx{`)w6F+ckEBFBX-zs!j!SCV$x4&Ap5fb;D5p;}oph$(w1X$C#<m<=JUp}3^d}oE$R1RF7SaFa#xI?1f>=J$K{(YHY2qvaI-P=~EC^Tc7d?;#{_~&5*dc1+5)S;L$o_WeuJ2=x&Hjsgla2ca9$7@(B@Y<?+8B!qEJAg~uD_d|YA=HZ#xi|G-Xt^(%6eGJNkJeq*uZ1Yf4MdFrg%)oV(rIGsP#~@<cp2^!c4@hqPNB98ZuUWbr7;8A{W^4PF{uL|Qg@VX@P(aV8;ULNCKfD_m>>m%s+|ZNbh)?+*l(8^&C*Th@RE*uqp$`RmQEJk;yq*M%B_gM5<4y&RNNa0sS_(F24z&|KImXC^Ltoh%A=&QtXW@0(!tPJ<NHK@ae!#9VL(Q~qH3&mAz28j4FnhlA*|-3WWU7%J`SayK9hppnznk-3AZL%Li7N{0Qi>!G$&k=g;Ch7fMUfIJiHcEF3ZXKXGa{|v34s;6IP0*G6m2!<FHyYWN{ciV705{0qfmw!3VDT{O8A`{?Cs)D@|$eETwiQ1xd=k)TnpVIjJBV^<jS$JE(C`{2G&r!Ch~>>;tjW!8hV_3I-w`cW{^KDZ#))Do95#xWk7d%r~U!pp(D28s*XgB7~?zWtFQWeatXWaW=n(k=#sgF|->}40fV3eJmBj9?gM9^h00p%2Jm<skmvX&r*95Tn;8n1Zm7EN5wUVFu>+7fW<^~Glt>82t&FHo}rZW2f_o`V_44SbO~Qg=c9jM%ENJ2(t{u?&mcPqQDnc?yUOsYBNlRrNk5v=i}dg~@%mY=8M$5%=|{kg2c|m#p;gFN6r&j(2fFE;VmJUrUSQC{U_M=_gepUH6kj{+41LLAq&Lth^sM2Cc+iI_aAfRbw(VHHDi5A7SOt)Ev}mK4v&><D|G0wfehYsNdXqM+9S~IfF&iHs{|5vQ77K&SV+Vn=$vB-!-|GFv^e$EBN1V0bQP^qInM>i6-JKtH4)9MC1SR@@#NS&UAPO!dzW5@2mQ;>OMU&Ez{aP5E=i8{ixN2Mx*M{Gpg|<V}XNuT1z>}6uw2YVplR-8=n6S-E%)WRLV;_R-b~@_k!POwUMQbBS9Se;2Hgmn3vJ2oduzchApcVa4Z=O{5{QcAlNyRlD2or!Z+DYs1u66iK2jViM0b>XTH5C#JI~^P+tteFTp!jQza*BV25k4uhi7nT{F$Q*oB8lKPk{p4{$OjW5(G6^(t`mVQOv)P7YKBJKCEkFX#Rgrz`WXto!8p6n!AMMeU8Lu6$q`#exm5zaFER~=8TH2@ZOiriuxd-l<XoF!;e<@!_`KuDuiA%Q2aTPpL<%bQZ=Qs)g|Xidtk&o28U2@xb|c&(2s)!7>?0F$ZEcvFCxJRy*?3Cyy+{WQ;X5G0N3XZl?#p@z!pSkLtlC+)sgyhO`oLlOsujz1A+}ZGHFs})F=Djo`R04kWFVpD)wqRou%j`S2{7(TNd}ZYh?8+rCIQUQz%Llh*uC%r=}2xa&OrLu)|W1&y2K@+{J&&A*HK$n?e()%zH=?oYfaI{YxldS>9<c6ZMlH#=Cm{AMDyO!LauKn$}Vm`!+2?KD)hlRI;mt}fK$r_RHdwLthldzt$THzLs#MH)*`-<S{YGWlk&>M<Wuq>J#l5kXdC2z8EsImyF4~m30xS6SBf337h!eNQRR~94X-?U`3}KIqsE4|&W}23@UR*}7?n!$KD!wgEse$wkXl~EiXk-#=ycO|0ZWFC!%(;Ds`*OUArutKR(aG+Tgp_rYZ|8Vh`Nk*NhkOgqI8G?&<BF==xj0qO(9Gq{HpL&X=?R~%(!k#TWKA5!DY{TBa$*n;omlj|I>?7Mhm16^n)Y(BY{JTWgFQxsRK6AI$@4VUe4B(t5a?3{1hfw>0x*j+-HxgsbfNA5D`v(m=spSs2xg6Zs1L^O1bDMBUq`GuMd2<tl>DCO*eWI9fsdO5)~dP4Ax;AGBreE3IZCLH)s|`0P7JS#$bLKWfwy|lLek+eIGqX?i1sCsQ90~fByTwO|F8A|M=IbiZQ@Zl*IY&`szRawHI)5vm|)nuBbD!Y7<%$`2T<Y@Bh^c7XSSB{}YV=@BduFE}bGa(xaButTi8X6DY)jNNZPK6w%??c6`j5OeJ@{SaOt@{1uV8yeT=!U9U9t!n-ScixoBX0g7ulT$yFwGQLA?^ZY~=?)p3j9&Vq^DnQ>4tGxIrMHmlEvz|naSfWSE1IGDPp^RNev5%^E0n9bbicZ4w@%0~-Y>HUSBF0j+SWpD=C|oSD%7IWLdu43$EFb~3(t}iHQBW~Q?|{uR)hRWYR7<t0N-bZRk8j>8|Cl#e&}j~?YpCyPz}!`h<9a*Z_`UIH*5w38ajSk3y-1Ih{^yV)fSaNiSGao#oJJTwgK|fQ>*b|a=mSkTylw{pjP}oGM)onS2DN5<%t3Z_{lWQRe012cBW7!3sb<S#85vN}QEv(V@7^7T+LoZm;l?>M6Q!#;4plAm7RrYxDsvrm(unLG$NY^oBYVeDOp{;#G{`elFikG&KfZbO75)!2(o)WI{uH36(wao4=FAy*o#e3cKpKu$97~_UN$24R9rR}i<EMnz(*YWRn1}*r*HuY!fG$k0El9N)ouuJ&;xfs$>D?Z3Tva4qlbH>Afu14i<tj`L>uBSJp&?WL4UR0vWqgW0`IzfYKRIm0I_l7>q=;(1;406d#?FiML`BlZC&kJ;e>i-Uw(JsaQ|5Q9`RGMjS&{jvP{WJ#^k@^l*xknO-@Y@OxcTA=$3TH@`~84x&29DvWPhIaV7$D6T|0rh3cUdmku-rBgd+@9_b7B=-T>+kv@#~_Hd1iZE72*Z^JA&V#@R*c2B@9Yvpk#6fybwvSqw(|_T`U}^ZI!bCvmewwG6X-@qYYnkWKQ}Dt~v)lp%FfJOt)%QL`4X2&=1HfVg7*%f6X>h;YO_6laLgA!|@DphFl~1n0|~s>%4K3&px+q_BO;xI69vyUFfZbphQTWY_8>wULf@N0aXDC|^?YK$MuZZCM|*(!LRX+`1<i&GPr2is~b#SiUdxHL>PMt|9lRhQ$0sT_PXpp^vN`il6wgOHeJc7Z>2?j!Dv!V}|Q%Ii%*15fD`fXf)cf7k~_1XsFw12|lVQs{7;Gn+CWX@)<O_Cb-d{kBV7DS>(Zk)sIzJqWw;nQTMD(>8SpeFWj>OdkhVy)2OFDU3(`gRiI9s08_faP@tqr=4@<pL}d`TXgVIuvq{g&67yD*5ptoNkI-nC4N(OTtXqc0BTiI_+N}O4yPQlh@n+1rsNiyZKf5~628bLu@PSq^!<&P?!VsVZ1B3R4aKu<eidUeF7C(FLyAOX;$!IG{lSIv`4wPL@)ue-K-lZri3aORXF5xYE-*n>(m^+Z`u>#D#nO{xEeJU0=<HM5^s`<;GKb?L4bN9zL?_YlUAKf?af7-X6e)#hFYnf=FfI_{VPA0TxUztvDlhJvzbD)APGXELVs-;$cGZ>onVpNEQtQC5=NnCOiT8GbMrsALww}@#&YZs+FG6JKG7;w*8n7H44CtF8@wX4y09FPa#@D~zDFiiO7z_o;vmBET)<n$UnjSeo{?;wi6HB^8!n0e{)?1hQUqWu%iC`kSI$NP47Jk4_@aZ14#T+v?rc^uO5=&Iv^n4XE6K4-qBy?Hn5^_KIjcP}bd=rqbTRvJp8`Af{f$eB+y%4K;|D<!_x5z_5UsT|^cFN_Mfe)n2I`XPxqnaSnsja#LCeS&c<4;aOUp$7*O=tvLo?2-r)Ns$tmrWe1Etq!HTHV4%N?j-B?=PW|O`O`_$>|m?ZtCZPgm#j!E-*hys|A@MdHy>(In+qc9sV33MIAL%aetiF@kd6p8*c}gU2jld#;6|ECARw3l(w1ZJpsOrJnI8j6u-b)l$YO}f{^%01WoZ9mD^WTOd7OFU0Sccdhn*FDTGeN;XTINu&JJen_k4p!$~FXEOr6M8C&2wd{h)Hw?v{zRsH5AM<+WYtNUD2MB8LyGYDshjS`P+);R_~JN+@BYEHym9i0ee3RMEnMammG$p*@kacJ7H2SWuT30TY7-pJ8|=eIpcoT79cYZ;5JGwOj~$ix{KMqf}hdolv+)jITp)3OnUT+)&Pnh%rVSTUY=-k-n7%&wb(s=z(UR$)^@o2g<Kjp=sbzPT_P-wlg>Sjx*)>HocQnHBP_Ux-azdP=zU?cBxn)mabW)<z$_qdQCd&bvA76bxGPGN%be8MzxWbifya!+Rg$^HSTI^_M{YJuGI$qz?%4Tr`XAXaCT{!v28AjT{BNq#ssO1g-V9jQ-=@9cFwU`t6j=p(adA?d*-_-$sxvBmP(4TL}-Q5LOXg+upgCE=_$Wy4lT<(+c(875N(L9t`(YT5nAE38x}INKX4xNgw6t!%cRH)#6q?cHn@~>P%5eNN#JN7pqc~JdG-W*0<8TF0{vJ3pvxKv2`V-jtpbGzFNIOAn3WsC>ld&sG$iGfO7yLWB*!C$xw*th9hQ#{X?IbqkTC0*>=pwXaN{;6=D)vE8FX|OaS2CDGJ}L4;=QADv8LySb?wBZ^qr}hDJQFEC*Ip*%T#ucxw3UM6;4Qp%anR`)>AgBwXK5N5(Of-6O`UYoO!rA>ko4#I{GX+#tdo1B;Q1sWS6@vBpM%6rIhyX1_t(E=Vhx|{cN}e_Z7ywO7kwSR;kzVy2?|9++{6=ta))vF>PvD%;M_CI4D^t`se_N5e+3j(nt7_&Hz6WE{4|Ksz!fh<F{T>>vJsI(TNfdq8{2#)ZIj}b!nJ97z}>ZQ7jD{*xz<YOI+GkyqP(8(xbr6#_qOc1?o!iKTbq7otB)|2FH!2+StJkr)99R)QLG1x>WJQ-X6tBQd!`;*_iC9T^y8jByk67%F;-Sh18;2l9L`673t1L`L#%NUE+VbMus4>V}a}BSY04;2%u8WlOly`X^Km*^5MyuLXR^%z)-O`2zj97dXl2ZfyCJ<FDP!IgL~6&tw*@=m=t!-*h4$(ZCynz(>uyG>dD2SuP<kV5%~;gcJj6jpyr!{^BNLeC-6~V{3z5r_Pt^$)byy&cU|m`LojKAKZ_iyZ#p#uu2M;+R9nNT4b8lB5M5#YxW|b3%0@5}k#YM$wh+7~?S^{rNyg{w2h~Id&*#t+xJK$0;k+Dp1B)xCrEa)iOo|}XdQ8$U3N^bWNFwNUjyO%xI8bb{^t=NSEFO}yX#+Tc*Zr2O6r;#|^92D<m23?k?TZR%5=XVg7Yk{7s+0&E@Z_5bV~dn_2^-ilbmPw=7Ga*U;Q)pSn_X*HoDwMsPCnFw$zVELhRjfY(|AS%$V>ulRu%o1M5Pk2i62UVo#1WL&3kZJKm&vi7Bw=mIc++LA6^k2lN3LAB&L{FFyLP0;i@zzP9{~$0)f*miIpiwA#Q)o1vmw5z^0Y<70bC}_hNT^<q)w^3VhB=6DEru0*JdrXI_$^*tEBF=2PP*g!}W{Ud~R+4^3y(PH0B^Ev?yxO%;$aGgryy0|5adz@Kh;b+k3-0d!zQVcszJGUm+yf`ma_6@N?n)`+vcYq{C^<VpGv$v&mdthKZ8#K5N3Mr1M{JK?(v$G{vPI*bK47IubPCJ#M|p2**kk=qZu>>X?<#O59rpk8IDr5~bcLv=JKi>!lY&er6$5Y(DScVg)Pm_g;Dz-vX-)v=2nA$7?gh_O!^^tz@ao+MVkN1iI*KT0&wUKzwDlfRsiTH1H5_`Vh29me;EUiXdlZS<x$AcbO#i7D!hWD4mEI-H`)M?Iu3o&6R+X4mn;kMa7(m>c81DV>{=Dx9at<cY-99~R5tqg`5J#Z{s~aBLqiO|bBRy1#NE3p2NUgE;MpZz<vsLG)cTJeUIJ(N=AU=i8`nY9MDf!XY9W1NzS{h5V!Yc&jFij3ra2cAIv73lqi~62_gZFJbcW(}$nV-oJeR>WmXNZqpK*&@Dirx^yVyGMT93Q1Ui7np)MPb61_%S*>rBJrNntmbz{&8_v6DwH)^b{T!)>+f05vLI@b#Y9g~P)#Yp60;<nEu;Y>C`-ejJXpvs;hCEbG)n-pP9kfdB&_VsyE*;DVZUY{tRoXr-v2h8BIGL6);eG-JqKaL=%?5H;G>DP^#v!mz{Q+&<PzfUL=#lyAX3GUbWk77WoM(OJ*BC9Q*tc-Qrc;qNhSSub7IF1x5*&p~#pa_Sq-w*mh$c`9cD|uUrX6vDr}1MP8}Y~8d?1}=^Vb)vRC`feEFe=Hhwrra54XJK2zTizuym+A;MI@UuJx*I1Q5l|kO;E56@%5VaLk_CpeOL%B9CR<%4)mj*ohi#vY*Z{i#$24{qg0Sw_iV<bw7W3`Q_|$QUu&?H<q#&jhE+;G=2sUY6JseMXQAawV|Umz*ClxD&>Z)o7k|Wlpl?oi_F@rK6ri<J23m|SHftU+YMo7uvL1PsTWa}g)p1pr%$hcd=C{L#Yge!aeN4W4>5Q#p{ZOTRH7p=9ALc7S7*%F>!f~&9$p0b;1VTsW-~d>>^$6}r!3|cLHWib>&@$TjW6*3chh&le((i@%AimxUd-#D`@xSd|8(}Y@$T*0ci}3~E4bD4skCp!Whi<fY*m>`0a2<=yhmayfyx5UYr^Khye9H-$%D$s&NQM8m%x+Y<D1tdd=#pJfOd<eNR6nCuIH0W&NM<;abunnuo)5wQb5%=l2%o#R+VfNZ|nnPy=4numd5enI=N-jV0=q1MCCVCG+^w`HBk8(a3>p&uoBZ)(NR@1t&P%!aO4L5R-a90wQx8eDD^Dyc^lgFMC(y-BExZ7UNqH=K0D_ymt4wsxD98_RLl4a;b9IF+m-p%hu3f=y!q4nmtWv=FiGgU%Gsq0B<uDNBzh|){ibV3MfJgCZARVCx=u2*N{U4Ibva(7Tk+1qf9A_A_g=N?pxj&PMHL1^K`;=3v0a58oujB#N=hd$?_0=SV2s4^Uk3M*!=Rka%qwr;$aD<ZEhcT$yDnLkf6uw*{he0fG}QUna-!vu`z}<lK!Dfcja%Dw!{mEqbg&BtrcUnd9sZz=aRTp9cHk|iZY_XVDST~1iPKCw$U#0?aNr1n-DtZBSIXw5EhnB4I;hJhi3I*?ZE$KZ?IPY=@b$8L#TJaSqZ>~IdT!b)crpn@-QzE_aZqmP*DZR-y2~BT*Y-ZHhQl~lUY;Eul>LjqLON=sYh7I{^KVDncU(f*(W;A|D`)vEgU~GP3TGLL(J)4-W1e;5)@j6@E%kGU-tZ*wgwUXMC62=`-|tR=U4~rkyRh|Tz=(6wiCY;+fi@E~lT_rC2COgJJLPW$(CIGAB*41Z&O*V{CFr`2C%eKn{}j-pI&5wUKf2`^nn@;UpuaR3J9fu6#WR}F(am>MrP$pYA?5Ia$#^yOE+e_l2+~-f$ZWP|>u(r+U(a~y&goAlEn}z^fO@yuccgm`W_jj7Oxsr*;mW(W;||(`gBXTJ9LgIHS1St+v1NOI@1<TxC%J4q5ejxu_Ha`Bk}^kWR(XKGg$qzoph>_RAY|ZT3P%E-(16%Q6=VIf^V8NN@$~t7`ipO-YoWRMdAX^bE`lL7BEDLd!&Fi|S5TfN20mNfRvbWk>p3C^f#N}wU9;0+yvb%W${<v`J3P2+MGf1HD{jKSwfn<^d-DU6rI-wNot*?7C%mM%<{)Z8z}d;e-X6PSNEGYERJ+4SFITa;MH2oSZ@*L)ZXnSgV--o-6XW9Vy)gNE58ZY7>Q2bZtX7m#4p4T9PorJB(wk1_ec(|R==Vb->slK9$rPP}#vr#&ZnM10SPB*0r{5JflCI@xW1ar$T(@MdfsMy5N?qko7w*z=74j2rbk`YlHU&QT0_x5BeTu+@exdy?r3UC5*U-^~g3WcaOGFkey2cc2bKPPWpnmJMQc<(4;1^r+^IlgC7Os8B4ce{Imgnb5z)UpAgmZLDK&D*5if0-P*4BB@`c^PBgbfR1kQ1Wy8`%ODzgOQ`PAP>k+lEsk_bExo05daZ?D}_bQYHxUtiGI#N0aN?&b)nnenahVt*6)!KxB7fv^KVgCz8n4q~Skxo3@Jv6=PW1+v7s3t@kn_JuoVH7O<VHfkWyaVDLhbIXMy}hJL(T$mZUDw;H(>SrC$4-JnJ7wWyqG6Ph0B_gTxd%BldUOKilTp5N<43SW0vb<Fx7wHb6^<Y@KlgNr%X!dhCk*lnI&;&Vi<D>*W89FmeF`gXO-EN?UqZe<<BqRF;g3AL^Lwy+cU%)=Vk#m-AVcUC(#Xn{gHX*d}y=j1bNZqv3dYH-IYArI<aF|I9Md{C-8;*cYvTsP5x^<}>W14WRDirvXQsV`?}QmL__-NK5Bty@DSbO!hg12F``knY5wdybd%!k6z#0+acx-G&}mFk3hmd3rwKSXrLHoOE~|r5=9X4YZGMai&_N^SEp@pQtSyk6{VfAFO!no&=vM-M!=kpD!n9lI|_%2tG#sjDz#$2%%O{AoBnL*FY>sb3$2u%WjJnFCJCR9u_^K!8rQI&VG3oD_#@v&KRvx+Cb-1Z>+iroSMtolCd>WnzsHwwe&-)C}^SO5|+3&0oJZO0oA7WLdAui7LFK#Q?V>I*2NXcYt(LRp;lQzXlg9@Zx#ZGp_&e3BXD{fQP!Q^>OAfm!eFaG>GvYDW9PV`z9$>7wEiS}{AvNNj2koE2}<jO;$kfVv7HOVRt2!muQ2g^y*HiRBQH3;_(cKc6{UzgUGlJtI#sShO;7XsFu$MlOcGrRAUM^o#p1=zfu}63&^B)Dl1Ep0AIt|szpBS(4pr$~+}wG>Y`srD$!7W0bYVOpE9}qucRMpJN>`K^rc_I%TQcDQl&&=q^*uf8uq_#_*0vE@MXwGyQ~=!Sw~Lztx;_u=>n;w(`Why!*aBCozJ&>cpmkpf&uNG6lF7pB%1koy&J4&isSA<<MUQP<Vp0Msx=vCVhlC{9JlTH8=_k3EPRAxNU2KI0Ov<xsLqfY4li?C`5o4x9ldRDIMXg*sZs-gfgoxc{^S4HSuCs#ors7ExChS>;>;nceyJXp)EnLu}4H3du_9E}RsT~jj*AcC%eZ8cFj=1xa!g#Q+5HgP(K@x8V6)1(65alW)4v9=ZZ#aj%-u(@XKdtkGahzOfp-Y*TaCcg+e9|2`h|8NB3}EwMwk`wh^0>5V%>KlfWXM?Nn(IaPYW8{Lzs}~{AO(uobbJ&U??3VTD?3oy5{D~)*iDLWPGT0Ak&U>^!vt?&YtR4An`-OWl2zI2EhoZkfsxM^)q6|UFDj)sa$G});e#{gmGXzlMUJ)gFP!~I#Vv2U&e{z7q+G83dhmHL2+IA8tSmff5kN#~R11h9n>Mbjm(T43NT}bPh_P=IR1iM5Rfkr)kfEW4f~ar>TEDJCT{{pGZtv&ER)d%sr+T@$`Dp#lu@`J}EwY1(qxMZjTbI=?&lI3rGRfM_MjO0i(dlv9O>Lqa;ZrB*M-`!?Z59iCS1Y1OYx=PAc->d|Dt(u4QriRzl@89dVTDT-c+>edo=P?oEAf9lY=RHBqK1B6TLa7AZ^d)*HzC7XC!Na{84M;V`9QV#PH<*W5=;c;X;FRDEKVn?6WRq|ahWt!q*zmo1)??r_Sn5oy2}~}@gX4pQ0*@x542OF8zlq-n`o<^T(BnI369TWkor1NWg9vQm79T7#>Jb+mI$!Mde-kFtO6rJVJ}m?2pDw4DoayZ@NeXY>--ya{F8!G=22?->bP^#@kS2fI4er!5?FOb<(6z5<Rh;~8i!o@R`N|Ur3*i7443M-*C9+whdLxAt?O#1JL+hH3PG^QI#^}+v=wI2jt+QMtW0<6E2CzX-Hm_0_?g8y(ajViYMQ87kJ6eLIvF3QtebnKn3>zHm-4%{EV;KMj5$2uxiXnRq78jc?LdSaBFR>4x}IXN+3?q*uUo0yR^#~~&02%8k~tgCVcMGlKdBf^wI247u`pXfp&bLW9UfZj%WDp_#IYCqY3Na<07FDK$@JTr@LoQuK6Erbg%hXf>_a(EOr<N@D)@DsOUxz$p@yv1M|`79?n0o(kLUtZB-j_{>9gFVhBwYpje<zb>CI@dFmy(F^5jU<<QisK$+sZy)Gfi@KP*kj9^x$|jH;;}Q~X#rKn`S|RxDrdI}};@1-V<*!9D!Z;{wG_1S0fccqe#v&zZx<lDHa+hQ(;2EES@hf%6@qXeb%f!i0iPtPO_ghTJ0+n$DrB$c~m_-7q|+oZ|C@7feam5brf3C0L<)F<qW2&FEYCa;_x_IGHWqs<N%1v7BIzHPU|+vbaf2p{r0@&qtc(RVtHWvVejZ@8yQq%$&TI`es>$1DOv}&9=I+n~G)i+FQzl9>q{~+=aH*Nm%k(X$P-7k+0!6qhyCtdwI;98U8);%lWd0jyA*PSZhuA&dY;~d)ev@<S`pM!(Pnd9oA3U76eu1PPTc`WW#8KC5(OAA8<@UJ>Te6+nbMO&?zN~NH*5;%$s;B6vrzcFE1(aseR*;Au6>StDA_0;rl<n3S-JDOYcRc_n3)xIi*1z=44FqB5Psozm1~Ws}Jvg{rKtquP?v;?br8T-~RgQ!@Kj=!Sl|qZ$Eya4-vG2H8Lk(3+o|<A}(?Us?BZ~?9-Q0Oi%NW!f`?2huTbY`oXLeH6SvW3^TD>ViSSnaz0(os4`1JQJ5v-^;uO(ZQ7mY7wzaOW+Aj|5{$FngZ+w#j9L^2=kQ&u-l~kuI~#$#-53psj7x(ag>|vZ=DniPf&jWW%?AfK&yJ>o!=b8PMF{SuU`nv@OodVKL(mfZQ|K|hovR8u(29RTpk3~V5MN;DMl^Jl>=ADR_)7AlSQF%KG1DWQL1DtGCCgwp(Guz{3YV1<1LZg_%M8*(S4_iU@$<5z7Bq2cO0jH;Zn``dP?5{+x_JS)j4)Q8zkdAq;nSD1*PHmFAxN;-6SS-{+LWoK)RqJf{J{yRiHAw`h2?DX8;NV`$AS!JN?ng8dNfuzg;JzYOb7u}1!8866HkvOi-=QmKo?#Fl!l{LC`qkxQ7X$1pma0Z%p}@#U^3DkT#yqiQ4wLECX>M>WAnu+r|$w!QJ*lH==G@aOvOP+M;br$rsL(!B#gFSA?X)2h%*MugKapAV~%^M`;evqH_wN77#{F@4?Hul0h8uYf8`_so?b&8e7tStiX-4FqT8|Qws<V00VaXs&C}p4-v={F^PMzLplR-BF}gKfH(Y~7L$&&4<4nD_R4`ee?quH9I1-pHRr$)Rhw(IR7h_o|4a(j!MKP0IW}IQE6b3YGpfTSDa~K$s5t;%O3s;I6Ea1i=D?G}__kV%`^83#4gxG52l{lvAiH-y9;TotEAv$z+VPrbUN>n6H`8oeZawktXWgC5V37ul9xnpakj`vfCFH?et@MRb}{>~PQgl&*jk!xJqp)lO(IJ>#%XRM4%0#R+!jCA0xevq7p30;BmB6@~v^G<-!q?**M1}dhC2uljB$I0YRabOtMvdo2VSnBh{VOt!<`pS*JFKjK(gkckjp<Ey=8klF7%l$MMV)6+Np9^;)hT4^xY{T3PX2x2`<{g@g0AaWcS5y&0DOj@2<!b#lPP4KWZe+m~JFC0PF%9khTj8^<$J+)iRQ|ve{@HrJ5h`INbPA@!kb7D*-y*|ocS{*^`E5=#Ik+87mw9z)cdIXj5bx`<cM43m6a?Ee_^@&gm3?JOkr=mZv_=awa&JBAn>tZ&?+p5>s*2@gJ|iY>O}S}$Yqxf>Qi^J9#~QaC;Hb{T$r1*O&tWi?16!RQI)oPwGChqRrH7K%Ymoy}T4I&($3k^3xqjsLa`O;nHA3f<k$6~j2|1g^+G|DUJVDE)4-V<l@zI1nyH+bXnX3wHfn00^EMTw!+ioelynqXCU~>-d@t)BatGKdwtYl@vE_n;w<P%OzHz2={(!wY<&hyWYnPH=_EI@YxmDd3CPP(<b=`$A?RCSS7PP%(9IOE_~bs+vi9~lvqmIuqPl{LQnf@Tv7GZ53Jw}W&`m#UTQNGi^#bB29X+YlG2KhhVd_M70rz@%|6S7Ef(s_r(6ZJD{<0?+zq-3}06p5(N&kv#ds2dJ6BdrGP{fWELE?#n{~^LfMRE^cP_0WOD8z$eqZPSCu<Zj5#F5uC?28m=QbVA?@;JH?1=nw?ddalTy*FigJECKiyea$893iKKugT+@l7UPhkRg%Xr^uOF3+3#H%HC&WB*af(lZm%(K4%_I#7*x@N>4rMtJ%boBcbQDk(hpdRYkmUBLm#>UiPL{ekJRZ}Qv_V*CCau2BvSFk96Y$6$(!BhpP0JlFZX&M4?+}DL%so|J|DVFPFwsW}$O{jP;zM3Mgx1*ElgDO`V#Pzte69-G>4&{NVG%J6v@nC^YFt&W<P`C+yA)#UcHpwQpE)KL?IYT%C?26k4I~Cpl(54T95vbF!N0uuXlIrXvBRg${l{pAL;UT$uk>9~%*#M9TiJBHTxjGz+Isjm4?hBZ@#gwr`p&PhYw=Zfi}A$qjk4iEHLx3THbhjt?^03qbW1Yol2B5u%T(9@Wi-Q!t)^Ry!#k6d3uOmg^3BICqVCRK$AqPWH&2hb(}=WYuEm3yS#!_X5i3%wM{nS}doRaiyg*D{T5COj){I;5FPdkYK37NzL552)M^Dvkv6!dLX{*^hJwAT%qIKFPcgA@UeeJ1;rm8^QiwJYuC!T*3|I;w0!1UdJxD&;6WlaF*3q-S#>KO)5W4SsJ{RJsuTg}7cJ^W7;UxeX0pGY34a>1dBHM_vHiJy6<aHw9k;OLte$^Q(7vUFvt<7|Njo%?dPyV65Dr2(d#s<k*2P^_jHPQt$eQRUPM4mrKa?nXDuoAd}335#qrMg&V*tGD8M>m;rp!~ae+_vwwU!WlYW&$Dkkm=)W-t7$%%XleFI@T2(uPL?+pK#r#>j2(wLC+r(4d3)n!Ukfs>UAn;u0cf+Dw59d`a-g(I6dKQo@du)>(F<83gKx1T5|gf3bj;jfl<<p$C`->frPg#ycENIFQ6?R^t^tJ?y+b(9J0<7S&)^*DobJsRsrdnf?-$m$!_EtnP}?ooata<{!S%`CKT5U`t7lq`KST%-<!=crz!nbw90+x%kmC%ODK~TDVIMJsd-q#Rf3DJ_W-~rIX|`FdpQksYN$ohs8NrI8sLhz1v{uv4_NDAWD{3_4bF|0xS$gNI0q~w2AqUuH4kr}Xr2L-rLNWoz;gbRV6GrVm9Klo{#yb9|MwJy{r6U|F-+ap(ijTnED^ShxV7Q1E(^)$DL!s@F?tQ8A#ARaZq~xk$9sAF)mi?2c0LI)8gGLbUJN7ZSX~$`J0R;8B3N5t`!?!DrPtrYJ0m+%b_Nmo)fCsHpS>=Nh??q0t3cOPeQ^(Oa?2#Uo-l+)MBQB;IYo((p%VO4VmSYg;fWmZ3(|H8*8UFn9B&=f~*c!ij{!8-9{?Rj52mL}gE|jQh%@}v*;c=7qCzXw&_~;q@r(()c=1CdDt*(u*F$^&vBpl~ecSfID@QFSgcG$Gj#`j#VvP#MaMlhw)#6vQtTND1U%`;5dS>b_`niV*Oo0-0Lv4*HEv`P^%ydmiam(&^}QF2s$IUd81C+L5gU%}}Z+yU(vv+S8OTI9lHk1+5)(WRN#BzZwZ=QgX4^D&T-jU?juuxX$4@^S0h&6v~APdzBVElEHH8NfbUGyr;6e$DFDwzPof@*o?JE>ZI5m&0{5VHdivBW>_kfV|N;dn{S^)GwVq1C~Ai<qwAfOR-Mianz}-5zZ6c9{e%3ke;v$+I%hQ0xeZQvc$KldkTk=UbCU2z)MIw`>mMEMBK)aU0Cp>s&q@0KB`QyGDEIeWnTvV1mCMDig8?~b87AA*>Qv3l<plpYsz=I-_|w81tn#C4g1=Pd2OsJ_qa76bmamP^HY^r7mcWth5r57OZ4{I07)kMd+FV`FW<lVmk*yS1}BGxZx`z5hTBEoFl)~oycchK*AnqJw3+04`YK+}s*4CAZm*-`K3e#@QFW1#q}0!o$DR1G3XlFcd3en6$R8(<ho?`6+0pRmw0U}TF?{;`;`zzb!D;L1@OkUx;^f(JKRY=-dVX^BcojdEJYFnG!n=c~$H&i3n<veqi`K>Q;j^Pb);fCj{CReG)H-SQ53^^-@Nd6)^enqLgqnx&zhU$F>9F<mX!!j3>EqS^1A$^+tp'}


## 1. Thu thập cấu hình và kiểm tra các API/tensor nhỏ

Không cần sửa ô mã hỗ trợ phía trên. `failed`, `timeout`, `not_installed` được giữ
riêng. Nếu có nhiều Python, kết quả từng interpreter nằm ở `runtimes`; không lấy
package của Python 3.13 làm bằng chứng cho Python 3.10. Wheel tags chỉ kiểm tra
tương thích ABI/platform, chưa xác nhận native CUDA hoặc inference.
DocEnTR/DE-GAN có inventory riêng và đối chiếu kernel metadata với
`learned_documents_runtime.json`. Probe Keras dùng backend Torch trong subprocess
riêng và HDF5 bốn số; không dựng generator hoặc nạp weights. Wheel METADATA và
dependency được đọc mà không cài đặt. Metadata khớp chưa xác minh inference.


In [ ]:
report = collect_environment(
    run_probes=RUN_PROBES, input_root=INPUT_ROOT,
    configured_paths={'models':MODELS_DIR, 'packages':PACKAGES_DIR,
                      'pdfs':DATA_DIR, 'prepared_images':PREPARED_IMAGES},
    runtime_pythons=RUNTIME_PYTHONS, options=CHECK_OPTIONS,
    checkpoint_path=OUTPUT_DIR/'kaggle_environment_report.json')
print(json.dumps(report["hardware"], ensure_ascii=False, indent=2))
print()
print("GPU:", report["tools"].get("gpu", {}).get("stdout", "unavailable"))
print("Torch/CUDA:", report["probes"].get("torch_cuda", {}).get("stdout", "not available"))
print()
print("Các package liên quan:")
for name, version in report['packages'].items():
    print(f"{name:24s} {'not installed'}")
print()
print("Pip dependency check:", report.get("pip_check", {}).get("status"))
print("Runtime VTM từ phiên trước:", report["prior_vtm_runtime_exists"])
print('Trạng thái báo cáo:', report['status'])
print('Input:', json.dumps(report['asset_roots'], ensure_ascii=False, indent=2))
print('Asset:')
for name, details in report['assets'].items():
    print(name, details.get('state', details.get('status')), details.get('required_missing', []))
print('Runtime riêng:', {name:value['status'] for name,value in report['runtimes'].items()})


## 2. Build FlashAttention và kiểm tra UniHIR thật

Phần này mặc định tắt (`RESTORATION_GPU_CHECK='off'`); không cần bật để gửi log
môi trường. Giữ Internet Off và gắn models/packages mới nhất. Phép thử dùng đúng worker,
source upstream, kiểm checkpoint và pipeline Notebook 1 được nhúng có checksum.
Mặc định 4 L4 cùng phiên: kiểm từng GPU và peer access, UniHIR chia model qua
Accelerate với ngân sách weights 18 GiB/card, chừa activation/KV/VAE. VRAM không
tự gộp. Kiểm placement và VRAM trước inference; device map/đỉnh VRAM từng card
được lưu trong log. Một GPU lớn vẫn dùng được khi đổi gpu_ids/placement tương ứng.
Build thành công, CUDA kernel chạy đúng và inference là ba trạng thái riêng.
Kết quả phục hồi chưa chứng minh đúng chữ; original được giữ để đối chiếu.
Phép thử không dùng cache ảnh phục hồi. File log/build receipt được gom vào ZIP,
không chứa weights, venv hoặc source build nặng.


In [ ]:
try:
    report['restoration_gpu_check'] = run_restoration_gpu_check(
        report, mode=RESTORATION_GPU_CHECK, options=RESTORATION_CHECK_OPTIONS,
        payload=GPU_CHECK_PAYLOAD, output_dir=OUTPUT_DIR)
except Exception as error:
    report['restoration_gpu_check'] = {'status':'supervisor_failed', 'error':str(error),
        'traceback':traceback.format_exc(), 'model_inference_verified':False}
report['model_inference_performed'] = bool(report['restoration_gpu_check'].get('model_inference_attempted'))
save_report(report, OUTPUT_DIR)
print(json.dumps(report['restoration_gpu_check'], ensure_ascii=False, indent=2))


## 3. Gửi lại ZIP log

Mặc định gửi **`kaggle_environment_logs.zip`** trong Output; có JSON/TXT, inventory
DocEnTR/DE-GAN, metadata dependency và GPU. Nếu bật GPU check, gửi
**`restoration_check_*_logs.zip`** sau khi chạy xong (kể cả
khi build hoặc inference lỗi). Có JSON/TXT inventory, phase report, build log,
worker log và ảnh original/restored của một trang khi thành công. Nếu chỉ kiểm
inventory, ZIP tên `kaggle_environment_logs.zip`. Nếu bị ngắt, JSON đã lưu một phần vẫn hữu ích; trường
`status`/`complete` cho biết các phép kiểm tra đã kết thúc hay chưa.

Báo cáo giữ toàn bộ distributions/dependencies, ABI, vị trí module thật, log lỗi,
GPU architecture, compiler/header và file asset còn thiếu. Từ dữ liệu này mới
chọn package/runtime cần bổ sung cho UniHIR mà không hạ kernel đang dùng.
Không cần upload lại models/packages chỉ để lấy báo cáo. Nếu Input vừa cập nhật
UniHIR, gắn đúng phiên bản Dataset mới để kiểm tra checkpoint và inventory tương ứng.


In [ ]:
json_path, text_path = save_report(report, OUTPUT_DIR)
print("Báo cáo JSON:", json_path)
print("Báo cáo TXT:", text_path)
log_archive = package_gpu_check_logs(report, OUTPUT_DIR)
print('ZIP log gửi lại:', log_archive)
try:
    from IPython.display import display, FileLink
    display(FileLink(str(json_path)))
    display(FileLink(str(text_path)))
    display(FileLink(str(log_archive)))
except ImportError:
    pass
